# FreightQuote AI: Milestone 2 (multi-user platform)
**Infosys Springboard Internship 7.0 · Batch 4**  ·  FastAPI · SQLite · JWT · RBAC · Google Colab

This single notebook contains the complete FreightQuote AI project: the provided freight platform (six agents + SQL Copilot) merged with Milestone 1 authentication, Profile page, five-role RBAC, a **Run agents** page, Admin dashboard and improved Copilot. **The login page has one username/email + password form only. Users do not select a role; after successful authentication, the backend reads the account's role from SQLite and applies the matching permissions and dashboard.**

**How to run:** add the Colab Secrets below, then *Runtime → Run all*. The last cell prints the **Open the app** link.

| Secret name | Used for |
|---|---|
| `JWT_SECRET` | signing login tokens (required) |
| `NGROK_AUTHTOKEN` | fallback public link |
| `EMAIL_ADDRESS`, `EMAIL_PASSWORD` | Gmail sender + app password for OTP mails (Milestone 1) |
| `ADMIN_PASSWORD` *(optional)* | password of the first Admin. If missing, a random one is generated and printed once |
| `DEMO_PASSWORD` *(optional)* | password of the four demo accounts (one per other role) |

> Never type a password, token or key into a cell. Before submitting: **Edit → Clear all outputs**.

### What's new in this version
* **Profile picture** – pick one of 12 avatars or upload your own photo on *My profile* (camera button). It also shows in the sidebar and the Admin user list.
* **Voice Copilot** – a 🎤 button in the AI Copilot: say your question, it is sent automatically, and the answer can be read aloud.
* **Active / Inactive users** – the Admin dashboard shows who is active right now (signed in and in use in the last 3 minutes) and who is inactive, with filters and *last seen*.
* **No locked pages** – the sidebar shows only the pages your role can open (a Customer sees just AI Copilot and Quotes).
* **Temporary or permanent passwords** – when the Admin adds a user or customer, they choose the password and whether it is *temporary* (the user must change it at first sign-in) or *permanent* (the user keeps it and can change it any time from My profile). The Admin can also reset a password later from the Users table.


In [ ]:
!pip install -q fastapi 'uvicorn[standard]' python-multipart pandas numpy scikit-learn scipy matplotlib langgraph vaderSentiment pdfplumber requests pyngrok bcrypt PyJWT

In [ ]:
import os
os.makedirs('/content/app/static', exist_ok=True)
os.chdir('/content/app')

## Step 1 · Secrets
Reads the Colab Secrets into environment variables for the server. Values are never printed.

In [ ]:
import os
try:
    from google.colab import userdata
except Exception:
    userdata = None
NAMES = ["JWT_SECRET", "NGROK_AUTHTOKEN", "EMAIL_ADDRESS", "EMAIL_PASSWORD", "ADMIN_PASSWORD", "ADMIN_EMAIL", "DEMO_PASSWORD"]
status = {}
for n in NAMES:
    v = None
    if userdata is not None:
        try: v = userdata.get(n)
        except Exception: v = None
    v = v or os.environ.get(n)
    if v: os.environ[n] = str(v).strip()
    status[n] = bool(v)
for n, ok in status.items(): print(("  set      " if ok else "  missing  ") + n)
if not status["JWT_SECRET"]: print("\nJWT_SECRET is missing: a random key is used for this run only (everyone is signed out after a restart). Add it in Colab Secrets (key icon, left sidebar).")
if not (status["EMAIL_ADDRESS"] and status["EMAIL_PASSWORD"]): print("E-mail secrets missing: 'Forgot password' by e-mail code is disabled; the security question still works.")
os.environ["SEED_DEMO_USERS"] = "1"   # one demo account per role (set to "0" for a clean product)

## Step 2 · Project files
Each cell below writes one file of the application.

In [ ]:
%%writefile /content/app/alts.py
"""Alternatives shown in the detail panels: other sea routes for a shipment, nearby lower-congestion ports for a port, nearby outlets that can transfer stock to an outlet."""
import json, numpy as np, pandas as pd
import blocks as B
from blocks import Q, TEAL, BLUE, AMBER, RED, VIOLET
COLORS = [TEAL, BLUE, AMBER, VIOLET, "#d6409f", "#8b8d98"]
def hav(la1, lo1, la2, lo2, R=6371.0):
    p = np.pi / 180; a = np.sin((np.asarray(la2) - la1) * p / 2) ** 2 + np.cos(la1 * p) * np.cos(np.asarray(la2) * p) * np.sin((np.asarray(lo2) - lo1) * p / 2) ** 2; return 2 * R * np.arcsin(np.sqrt(a))
def usd(x): return "${:,.0f}".format(x)
VARIANTS = [((), "Default"), (("suez",), "Avoid Suez"), (("panama",), "Avoid Panama"), (("malacca",), "Avoid Malacca"), (("hormuz",), "Avoid Hormuz"), (("storm",), "Avoid storms")]
def route_alternatives(dom, o_xy, d_xy, tonnes=20.0, interp=None, rate=0.012, day_cost=120.0, dwell=0.0):
    """-> list of options sorted by total cost; each {label, name, nm, days, freight, delay_cost, total, wind, chain, avoid, flags}"""
    seen, out = {}, []
    for av, lab in VARIANTS:
        try: opts = dom.route_options(o_xy, d_xy, avoid=av)
        except Exception: continue
        if not opts: continue
        o = opts[0]; key = o["name"]
        if key in seen: continue
        seen[key] = 1; ch = o["chain"]; wind = 0.0
        if interp:
            try: wind = max(dom.weather_at(*interp(ch, f), allow_live=False)["wind"] for f in (.15, .35, .55, .75, .95))
            except Exception: pass
        days = o["nm"] / (14 * 24) + dwell; fr = o["nm"] * tonnes * rate; dl = days * day_cost; risk_cost = max(0, wind - 15) * 0.02 * days * day_cost * 3
        out.append({"label": lab, "name": o["name"], "nm": o["nm"], "days": round(days, 1), "freight": fr, "delay_cost": dl + risk_cost, "total": fr + dl + risk_cost, "wind": round(wind, 1), "chain": ch, "avoid": list(av)})
    if not out: return out
    out.sort(key=lambda r: r["total"]); fast = min(out, key=lambda r: r["days"]); short = min(out, key=lambda r: r["nm"])
    for r in out: r["tag"] = ", ".join(t for t, c in (("Recommended", r is out[0]), ("Fastest", r is fast), ("Shortest", r is short)) if c)
    return out
def route_blocks(opts, sid=None, title="Route alternatives", cur=None):
    if not opts: return [B.callout("No alternative sea route found for this lane.", "info")]
    lines = [{"pts": r["chain"], "color": COLORS[i % len(COLORS)], "w": 4 if i == 0 else 2.5, "dash": None if i == 0 else "6 6"} for i, r in enumerate(opts)]
    rows = [{"option": f"{i + 1}. {r['label']}", "route": r["name"], "nm": r["nm"], "days": r["days"], "freight": r["freight"], "delay": r["delay_cost"], "total": r["total"], "wind": r["wind"], "tag": r["tag"]} for i, r in enumerate(opts)]
    best, worst = opts[0], opts[-1]; pts = [p for r in opts for p in r["chain"]]; la = [p[0] for p in pts]; lo = [p[1] for p in pts]
    view = {"bounds": [[min(la) - 4, min(lo) - 6], [max(la) + 4, max(lo) + 6]]}
    ends = [{"id": "o", "lat": opts[0]["chain"][0][0], "lon": opts[0]["chain"][0][1], "color": "#1f2430", "r": 6, "tip": "start"}, {"id": "d", "lat": opts[0]["chain"][-1][0], "lon": opts[0]["chain"][-1][1], "color": RED, "r": 6, "tip": "destination"}]
    bl = [B.card(title, B.map_([{"type": "lines", "lines": lines}, {"type": "points", "id": "alt_ends", "pts": ends}], h=300, view=view, key="alt_map", legend=[(str(rows[i]["option"]) + (" (solid = recommended)" if i == 0 else " (dashed)"), COLORS[i % len(COLORS)]) for i in range(min(len(rows), len(COLORS)))] + [("Start", "#1f2430"), ("Destination", RED)]), B.table(rows, ["option", "route", B.num_col("nm", "nm", "int"), B.num_col("days", "days", "num"), B.num_col("freight", "freight cost", "money"), B.num_col("delay", "time + weather cost", "money"), B.num_col("total", "total", "money"), B.num_col("wind", "max wind m/s", "num"), "tag"], h=min(60 + 40 * len(rows), 330), key="alt_t"),
                       sub=f"Best: {best['label']} ({best['name']}), {best['days']} days, {usd(best['total'])}. Worst: {usd(worst['total'])}. Cost = $0.012 per tonne-nm + $120 per day + weather penalty above 15 m/s.")]
    return bl
def port_alternatives(port, k=6):
    p = Q("SELECT port_name, country, region, lat, lon, congestion_index ci, avg_dwell_days dw FROM ports"); me = p[p.port_name == port]
    if not len(me): return []
    me = me.iloc[0]; p["km"] = hav(me.lat, me.lon, p.lat.values, p.lon.values); c = p[(p.port_name != port) & (p.km < 1800)].copy()
    c["better"] = (me.ci - c.ci); c["score"] = c.ci * 12 + c.dw * 4 + c.km / 100; c = c[c.ci < me.ci].sort_values("score").head(k)
    rows = [{"port": r.port_name, "country": r.country, "km": r.km, "ci": r.ci, "dwell": r.dw, "days_saved": max(0, me.dw - r.dw), "haul_days": r.km / 600 / 1 if r.country == me.country else r.km / 450} for r in c.itertuples()]
    if not rows: return [B.callout(f"No less-congested port within 1,800 km of {port}.", "info")]
    return [B.card("Alternative ports nearby", B.table(rows, ["port", "country", B.num_col("km", "km away", "int"), B.num_col("ci", "congestion", "num"), B.num_col("dwell", "dwell days", "num"), B.num_col("days_saved", "dwell saved", "num"), B.num_col("haul_days", "extra haul days (approx.)", "num")], h=min(70 + 40 * len(rows), 340), open={"kind": "port", "col": "port"}, key="alt_p"),
                    sub="Cargo for the port could be diverted here; haul by road/rail is estimated at 600 km/day domestic, 450 km/day cross-border. Check customs and carrier agreements before diverting.")]
def outlet_alternatives(oid, max_km=400):
    g = Q("SELECT outlet_id, lat, lon FROM outlet_geo"); me = g[g.outlet_id == oid]
    if not len(me): return []
    me = me.iloc[0]; g["km"] = hav(me.lat, me.lon, g.lat.values, g.lon.values); near = g[(g.outlet_id != oid) & (g.km <= max_km)].sort_values("km").head(12)
    need = Q("SELECT sku_name, current_stock st, reorder_threshold thr, weekly_demand wk, lead_time_days lead FROM inventory WHERE outlet_id=? AND current_stock<=reorder_threshold*1.5 ORDER BY current_stock*1.0/MAX(weekly_demand,1) LIMIT 6", (oid,))
    if not len(need): return [B.callout("This outlet has no SKUs at or near the reorder level (within 150%), so no transfers are suggested.", "info")]
    if not len(near): return [B.callout(f"No outlet within {max_km} km to transfer stock from.", "info")]
    inv = Q(f"SELECT outlet_id, sku_name, current_stock st, reorder_threshold thr, weekly_demand wk FROM inventory WHERE outlet_id IN ({','.join('?' * len(near))}) AND sku_name IN ({','.join('?' * len(need))})", tuple(near.outlet_id) + tuple(need.sku_name))
    inv["spare"] = inv.st - (inv.thr + inv.wk); inv = inv[inv.spare > 0].merge(near[["outlet_id", "km"]], on="outlet_id"); rows = []
    for r in need.itertuples():
        want = max(1, int(r.thr + r.wk - r.st)); c = inv[inv.sku_name == r.sku_name].sort_values("km")
        if len(c):
            s = c.iloc[0]; q = int(min(want, s.spare)); days = max(.5, s.km * 1.3 / 320); rows.append({"sku": r.sku_name, "short_by": want, "from": s.outlet_id, "km": s.km, "units": q, "days": days, "vs_supplier": r.lead - days, "cost": s.km * 1.3 * 14})
        else: rows.append({"sku": r.sku_name, "short_by": want, "from": "none nearby has spare", "km": None, "units": 0, "days": None, "vs_supplier": None, "cost": None})
    d = pd.DataFrame(rows); mp = [{"type": "lines", "lines": [{"pts": [[float(me.lat), float(me.lon)], [float(g[g.outlet_id == r["from"]].lat.iloc[0]), float(g[g.outlet_id == r["from"]].lon.iloc[0])]], "color": COLORS[i % 6], "w": 3} for i, r in enumerate(rows) if r["units"]]},
                                {"type": "points", "id": "alt_o", "pts": [{"id": oid, "lat": float(me.lat), "lon": float(me.lon), "color": "#1f2430", "r": 8, "tip": oid}] + [{"id": r["from"], "lat": float(g[g.outlet_id == r["from"]].lat.iloc[0]), "lon": float(g[g.outlet_id == r["from"]].lon.iloc[0]), "color": TEAL, "r": 6, "tip": r["from"]} for r in rows if r["units"]]}]
    return [B.card("Suggested stock transfers from nearby outlets", B.map_(mp, h=240, key="alt_om", legend=[("This outlet (needs stock)", "#1f2430"), ("Outlet that can send stock", TEAL), ("Transfer line (colour per supplier outlet)", BLUE)]) if any(r["units"] for r in rows) else B.md("No nearby outlet has spare stock for these SKUs."), B.table(d, ["sku", B.num_col("short_by", "short by", "int"), "from", B.num_col("km", "km (straight line)", "int"), B.num_col("units", "transfer units", "int"), B.num_col("days", "transfer days (approx.)", "num"), B.num_col("vs_supplier", "days faster than supplier", "num"), B.num_col("cost", "est. cost ₹", "money")], h=min(80 + 40 * len(d), 330), open={"kind": "outlet", "col": "from"}, key="alt_ot"),
                    sub="Spare = stock above reorder level plus one week of demand. Distance is straight-line; road distance ≈ 1.3×. Cost ₹14 per road km. Compare with the supplier lead time before moving stock.")]


In [ ]:
%%writefile /content/app/analyst.py
"""Question understanding for the Copilot, ahead of the generic table query (qa.py). Each handler computes the answer from live data and returns
{"answer": lead text (markdown), "df": optional table, "sql": how it was computed, "extra": more markdown}. Nothing here asks the language model for a number."""
import re, json, datetime
import numpy as np, pandas as pd
from blocks import Q
import qa as QA, alts
CITY_STATE = {"Mumbai": "Maharashtra", "Pune": "Maharashtra", "Nashik": "Maharashtra", "Nagpur": "Maharashtra", "Delhi": "Delhi", "Bengaluru": "Karnataka", "Mysuru": "Karnataka", "Hyderabad": "Telangana", "Chennai": "Tamil Nadu", "Coimbatore": "Tamil Nadu", "Madurai": "Tamil Nadu",
              "Kolkata": "West Bengal", "Ahmedabad": "Gujarat", "Surat": "Gujarat", "Vadodara": "Gujarat", "Jaipur": "Rajasthan", "Lucknow": "Uttar Pradesh", "Kochi": "Kerala", "Thiruvananthapuram": "Kerala", "Chandigarh": "Chandigarh", "Bhopal": "Madhya Pradesh", "Indore": "Madhya Pradesh",
              "Patna": "Bihar", "Ranchi": "Jharkhand", "Raipur": "Chhattisgarh", "Bhubaneswar": "Odisha", "Guwahati": "Assam", "Goa": "Goa", "Visakhapatnam": "Andhra Pradesh", "Vijayawada": "Andhra Pradesh", "Kanpur": "Uttar Pradesh", "Dehradun": "Uttarakhand", "Amritsar": "Punjab", "Ludhiana": "Punjab"}
CHOKE = {"suez": ("Suez Canal", 30.5, 32.3), "panama": ("Panama Canal", 9.1, -79.7), "malacca": ("Strait of Malacca", 2.5, 101.5), "hormuz": ("Strait of Hormuz", 26.5, 56.5), "bosphorus": ("Bosphorus", 41.1, 29.0), "cape": ("Cape of Good Hope", -34.4, 18.5), "red sea": ("Red Sea", 18.0, 40.0)}
def _w(q, t): return re.search(r"(?<![a-z0-9])" + re.escape(t) + r"(?![a-z0-9])", q) is not None
def _md(df, n=10): return QA.md(df.head(n)) if df is not None and len(df) else ""
def _health(dec): return (((dec or {}).get("ev") or {}).get("health")) or {}
def _agent_scores(e):
    import entities as EN; out = {}
    for a in ("outlet", "inventory", "staff", "marketing", "audit", "feedback"):
        v = EN.latest_out(a).get("per_outlet", {}).get(e)
        if v is not None: out[a] = round(float(v), 1)
    return out
NEXT = {"staffing_shortage": "add a cover shift or move staff from a nearby outlet", "health_alert": "review the lowest agent score below and fix that first", "reorder": "place the reorder or transfer stock from a nearby outlet", "stockout": "reorder now or transfer stock",
        "corrective_action": "complete the corrective plan before the re-audit", "anomaly_review": "check the POS feed and the day's sales for this outlet", "campaign_review": "pause or re-target the campaign", "retention": "talk to the staff at risk and review overtime"}
# ------------------------------------------------------------------------------------------------ Franchise
def explain_outlet(e, dec):
    o = Q("SELECT * FROM outlets WHERE outlet_id=?", (e,))
    if not len(o): return None
    r = o.iloc[0]; h = _health(dec).get(e); sc = _agent_scores(e); tk = Q("SELECT action_id id, type, severity, status, detail FROM agent_actions WHERE target=? AND status IN ('open','proposed') ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT 8", (e,))
    avg = Q("SELECT AVG(revenue) rv, AVG(customer_satisfaction) cs FROM outlets").iloc[0]; low = Q("SELECT COUNT(*) n FROM inventory WHERE outlet_id=? AND current_stock<=reorder_threshold*1.5", (e,)).n[0]
    att = Q("SELECT AVG(predicted_attrition_prob) p, COUNT(*) n FROM staff WHERE outlet_id=?", (e,)).iloc[0]
    lines = []
    if h is not None: lines.append(f"Health score **{h:.1f}**/100 ({'Critical' if h < 60 else 'Needs attention' if h < 80 else 'Healthy'}).")
    if sc: worst = sorted(sc.items(), key=lambda kv: kv[1])[:2]; lines.append("Weakest areas: " + ", ".join(f"**{a}** {v}" for a, v in worst) + " (each agent scores 0-100, higher is better).")
    lines.append(f"Revenue {r.revenue:,.0f} ({(r.revenue / avg.rv - 1):+.0%} vs chain average), satisfaction {r.customer_satisfaction:.2f} (chain {avg.cs:.2f}), {int(att.n)} staff with average attrition risk {att.p:.0%}, {int(low)} SKUs at or near reorder level.")
    why = "\n".join(f"- {t.type} ({t.severity}): {t.detail}" + (f" → {NEXT[t.type]}" if t.type in NEXT else "") for t in tk.itertuples()) or "- no open tickets"
    return {"answer": (f"**{e}** ({r.outlet_name}, {r.location}) is flagged because:\n" if (h is None or h < 80 or len(tk)) else f"**{e}** ({r.outlet_name}, {r.location}) is not flagged right now; here is its status:\n") + "\n".join("- " + l for l in lines) + "\n\n**Open tickets:**\n" + why, "df": pd.DataFrame([{"agent": a, "score": v} for a, v in sc.items()]) if sc else None, "sql": f"decision health + agent per_outlet scores + agent_actions for {e}"}
def risk_rank(q, dec, scope, n=5):
    hl = _health(dec)
    if not hl: return None
    s = pd.Series(hl).astype(float); worst = bool(re.search(r"\b(highest risk|most at risk|worst|lowest|riskiest|critical|weakest|needs? attention|problem|bottom)\b", q)); best = bool(re.search(r"\b(best|top|highest health|strongest|healthiest)\b", q)) and not re.search(r"highest risk", q)
    both = re.search(r"best.*worst|worst.*best", q) is not None
    if scope: s = s[s.index == scope]
    o = Q("SELECT outlet_id, outlet_name, location, revenue, customer_satisfaction csat FROM outlets").set_index("outlet_id")
    def frame(ix):
        rows = []
        for e in ix:
            sc = _agent_scores(e); wk = min(sc, key=sc.get) if sc else "-"; ct = int(Q("SELECT COUNT(*) n FROM agent_actions WHERE target=? AND status IN ('open','proposed') AND severity='Critical'", (e,)).n[0])
            rows.append({"outlet": e, "name": o.name if False else (o.outlet_name.get(e, "") if e in o.index else ""), "location": o.location.get(e, ""), "health": round(float(s[e]), 1), "weakest_area": f"{wk} {sc.get(wk, '')}", "critical_tickets": ct, "revenue": o.revenue.get(e), "csat": o.csat.get(e)})
        return pd.DataFrame(rows)
    low = s.sort_values().head(n).index; hi = s.sort_values(ascending=False).head(n).index
    if both or (best and worst):
        a, b = frame(hi), frame(low); return {"answer": f"Best outlet: **{a.outlet[0]}** (health {a.health[0]}). Worst outlet: **{b.outlet[0]}** (health {b.health[0]}). Health combines outlet, inventory, staff, marketing, audit and feedback agent scores.", "df": None, "sql": "decision health per outlet", "extra": "**Top " + str(n) + " by health**\n" + QA.md(a) + "\n\n**Bottom " + str(n) + " by health**\n" + QA.md(b)}
    if best and not worst: d = frame(hi); return {"answer": f"Healthiest outlet: **{d.outlet[0]}** (health {d.health[0]}, {d.location[0]}). Top {n} below.", "df": d, "sql": "decision health per outlet, descending"}
    d = frame(low); crit = int((s < 60).sum()); return {"answer": f"Highest risk: **{d.outlet[0]}** (health {d.health[0]}, weakest area {d.weakest_area[0]}). {crit} outlets are below 60 (critical). Lowest {n} below; ask \"why is {d.outlet[0]} flagged\" for the reasons.", "df": d, "sql": "decision health per outlet, ascending"}
def inventory_status(scope):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q("SELECT outlet_id, sku_name, category, current_stock st, reorder_threshold thr, weekly_demand wk, lead_time_days lead, stockout_risk_prob p FROM inventory" + w, p)
    if not len(d): return None
    d["cover_d"] = d.st / (d.wk / 7).replace(0, np.nan); below = d[d.st <= d.thr]; wk1 = d[d.cover_d < 7]; lt = d[d.cover_d < d.lead]
    po = Q("SELECT status, COUNT(*) n FROM purchase_orders" + w + " GROUP BY status", p); cat = d.groupby("category").agg(skus=("sku_name", "count"), below_reorder=("st", lambda s: int((d.loc[s.index, "st"] <= d.loc[s.index, "thr"]).sum())), avg_cover_days=("cover_d", "mean"), avg_risk=("p", "mean")).reset_index().sort_values("avg_risk", ascending=False)
    worst = d.sort_values("cover_d").head(8)[["outlet_id", "sku_name", "st", "cover_d", "lead", "p"]].rename(columns={"st": "stock", "cover_d": "days_of_cover", "lead": "lead_days", "p": "stockout_risk"})
    return {"answer": f"Inventory: **{len(d):,}** SKU records; **{len(below):,}** at or below reorder level, **{len(wk1):,}** have under 7 days of cover, **{len(lt):,}** will run out before a new order can arrive (cover shorter than lead time). Average stock-out risk {d.p.mean():.0%}." + (" Open purchase orders: " + ", ".join(f"{r.status} {r.n}" for r in po.itertuples()) + "." if len(po) else ""), "df": worst, "sql": "inventory: stock vs reorder threshold, cover = stock ÷ daily demand", "extra": "**By category**\n" + QA.md(cat.round(2))}
def sales_trend(scope):
    w = " AND outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q(f"SELECT substr(ts,1,10) day, SUM(amount) amount, SUM(tx_count) tx FROM sales WHERE ts IS NOT NULL{w} GROUP BY day ORDER BY day DESC LIMIT 14", p)
    if len(d) < 2: return None
    d = d.iloc[::-1].reset_index(drop=True); last7, prev7 = d.tail(7).amount.sum(), d.iloc[:-7].tail(7).amount.sum() if len(d) > 7 else np.nan
    ch = f" ({(last7 / prev7 - 1):+.1%} vs the 7 days before)" if prev7 == prev7 and prev7 else ""
    return {"answer": f"Sales over the last 7 recorded days: **{last7:,.0f}**{ch}. Best day {d.loc[d.amount.idxmax(), 'day']} ({d.amount.max():,.0f}), weakest {d.loc[d.amount.idxmin(), 'day']} ({d.amount.min():,.0f}).", "df": d.tail(10), "sql": "sales grouped by day"}

def _place(ql):
    """Find a state or city named in the question. Returns ("state"|"city", name) or None."""
    flat = re.sub(r"[^a-z]", "", ql); cities = Q("SELECT DISTINCT location FROM outlets").location.dropna().tolist()
    for st in sorted(set(CITY_STATE.values()), key=len, reverse=True):
        if re.sub(r"[^a-z]", "", st.lower()) in flat: return "state", st
    for ct in sorted(cities, key=len, reverse=True):
        if re.search(r"\b" + re.escape(ct.lower()) + r"\b", ql): return "city", ct
    return None
def outlets_in_place(ql, scope=None):
    pl = _place(ql)
    if not pl: return None
    kind, name = pl; o = Q("SELECT outlet_id, outlet_name name, location, tier, revenue, customer_satisfaction csat, staff_headcount staff FROM outlets")
    o["state"] = o.location.map(CITY_STATE).fillna("Other"); d = o[o.state == name] if kind == "state" else o[o.location == name]
    if not len(d): return {"answer": f"No outlets found in {name}.", "df": None, "sql": "outlets filtered by place"}
    by = d.groupby("location").size().sort_values(ascending=False); txt = ", ".join(f"{k} {v}" for k, v in by.items())
    return {"answer": f"**{len(d)}** outlets in {name}" + (f": {txt}." if kind == "state" else "."), "df": d.drop(columns=["state"]).sort_values(["location", "outlet_id"]), "sql": f"outlets where {'state (from city)' if kind == 'state' else 'location'} = {name}"}
def staff_by_place(ql, scope=None):
    s = Q("SELECT s.staff_id, s.outlet_id, o.location FROM staff s LEFT JOIN outlets o USING(outlet_id)")
    if not len(s): return None
    s["state"] = s.location.map(CITY_STATE).fillna("Other"); pl = _place(ql)
    if pl:
        kind, name = pl; d = s[s.state == name] if kind == "state" else s[s.location == name]; g = d.groupby("location").size().sort_values(ascending=False).reset_index(name="employees")
        return {"answer": f"**{len(d):,}** employees in {name}" + (f"; most in **{g.location.iloc[0]}** ({int(g.employees.iloc[0]):,})." if len(g) else "."), "df": g, "sql": f"staff joined to outlets, place = {name}"}
    by_state = bool(re.search(r"\bstates?\b", ql)); by_out = bool(re.search(r"\boutlets?\b", ql)); col = "outlet_id" if by_out else ("state" if by_state else "location")
    g = s.groupby(col).size().sort_values(ascending=False).head(15).reset_index(name="employees"); low = bool(re.search(r"\b(fewest|least|lowest|less|low number)\b", ql)); r = (s.groupby(col).size().sort_values().reset_index(name="employees").iloc[0] if low else g.iloc[0])
    return {"answer": f"**{r[col]}** has the {'fewest' if low else 'most'} employees: **{int(r.employees):,}** (of {len(s):,} in total, by {'outlet' if by_out else 'state' if by_state else 'city'}).", "df": g, "sql": f"staff joined to outlets, grouped by {col}"}
def by_state(q):
    o = Q("SELECT outlet_id, location, revenue FROM outlets"); o["state"] = o.location.map(CITY_STATE).fillna("Other"); g = o.groupby("state").agg(outlets=("outlet_id", "count"), revenue=("revenue", "sum")).reset_index().sort_values("outlets", ascending=False)
    low = bool(re.search(r"\b(fewest|least|lowest)\b", q)); r = g.iloc[-1] if low else g.iloc[0]
    return {"answer": f"**{r.state}** has the {'fewest' if low else 'most'} outlets: **{int(r.outlets)}**. Your outlets table stores the city, so states are derived from the city name.", "df": g, "sql": "outlets grouped by state (city → state mapping)"}
def staff_status(scope):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    s = Q("SELECT outlet_id, role, overtime_hrs ot, predicted_attrition_prob pr FROM staff" + w, p); at = Q("SELECT status, COUNT(*) n FROM attendance" + (" WHERE outlet_id=?" if scope else "") + " GROUP BY status", p)
    if not len(s): return None
    hi = s[s.pr > .6]; g = s.groupby("outlet_id").pr.mean().sort_values(ascending=False).head(8).reset_index().rename(columns={"pr": "avg_attrition_risk"})
    return {"answer": f"Workforce: **{len(s):,}** staff, **{len(hi):,}** at high attrition risk (>60%), average overtime {s.ot.mean():.1f} h" + (". Attendance: " + ", ".join(f"{r.status} {r.n:,}" for r in at.itertuples()) if len(at) else "") + ".", "df": g, "sql": "staff + attendance", }
# ------------------------------------------------------------------------------------------------ Freight
STOP = {"long", "port", "new", "fort", "saint", "south", "north", "east", "west", "city", "bay", "cape", "grand", "great", "royal", "santa", "puerto", "king", "bandar"}
def _ports_in(q):
    p = Q("SELECT port_name FROM ports").port_name.tolist(); spans = []; first = {}
    for name in p:
        core = [w for w in re.sub(r"\(.*?\)", "", name).lower().split() if w not in QA.GENERIC]
        if core and len(core[0]) >= 4 and core[0] not in STOP: first.setdefault(core[0], []).append(name)
    for name in p:
        als = set(QA._alias(name)); core = [w for w in re.sub(r"\(.*?\)", "", name).lower().split() if w not in QA.GENERIC]
        if core and core[0] not in STOP and len(first.get(core[0], [])) == 1 and len(core[0]) >= 4: als.add(core[0])
        for al in sorted(als, key=len, reverse=True):
            m = re.search(r"(?<![a-z0-9])" + re.escape(al) + r"(?![a-z0-9])", q)
            if m: spans.append((m.start(), m.end(), len(al), name)); break
    spans.sort(key=lambda x: (-x[2], x[0])); kept = []
    for s_ in spans:
        if not any(s_[0] < k[1] and k[0] < s_[1] for k in kept): kept.append(s_)
    kept.sort(); return [k[3] for k in kept], kept
def plan_lane(q, dom):
    ps, hits = _ports_in(re.sub(r"how long", "how_long", q.lower()))
    if len(ps) < 2: return None
    pre = q.lower(); o, d = ps[0], ps[1]; m = re.search(r"from\s+(.+?)\s+to\s+", pre)
    cargo = next((k for k in dom.CARGO if k.lower() in pre or k.lower().rstrip("s") in pre), "General"); wm = re.search(r"(\d+(?:\.\d+)?)\s*(t|tons?|tonnes?|mt|kg|kgs)\b", pre); wt = 20.0
    if wm: wt = float(wm.group(1)) / (1000 if wm.group(2).startswith("kg") else 1)
    po, pd_ = dom.port_xy(o), dom.port_xy(d)
    if not po or not pd_: return None
    cust = Q("SELECT customer_id FROM customers LIMIT 1").customer_id[0]; req = {"origin": o, "dest": d, "cargo": cargo, "weight_t": wt, "hs_code": "HS-8501", "customer_id": cust, "docs_provided": None}
    pq = dom.preview_quote(req); opts = alts.route_alternatives(dom, (po[0], po[1]), (pd_[0], pd_[1]), tonnes=wt, interp=dom.interp, dwell=(po[3] + pd_[3]) * .5)
    if not pq and not opts: return None
    rows = [{"option": f"{i + 1}. {x['label']}", "route": x["name"], "nm": x["nm"], "days": round(x["days"], 1), "freight_cost": round(x["freight"]), "time_cost": round(x["delay_cost"]), "total": round(x["total"]), "tag": x["tag"]} for i, x in enumerate(opts)]
    head = f"**{o} → {d}**, {wt:g} t of {cargo}: "
    if pq: head += f"recommended route **{pq['route']}**, about **{pq['days']} days** at sea and in port, quoted price **${pq['price']:,.0f}** (margin {pq['margin_pct']}%), weather risk {pq['weather']}, decision: {pq['decision']}."
    elif opts: head += f"best route **{opts[0]['name']}**, about **{opts[0]['days']} days**."
    ex = (pq["narrative"] if pq and pq.get("narrative") else "") + ("\n\nOnly one sea route exists for this lane in the routing model (it does not pass a closable canal or strait), so the alternatives are commercial: another carrier, or diverting to a nearby less-congested port (open the port in the map to see candidates)." if len(opts) == 1 else "")
    return {"answer": head, "df": pd.DataFrame(rows) if rows else None, "sql": "dom.preview_quote (route, weather, ML price, customs, margin) + sea-route alternatives; nothing was saved", "extra": ex, "map": opts}
def choke_ports(q):
    k = next((k for k in CHOKE if _w(q, k)), None)
    if not k: return None
    name, la, lo = CHOKE[k]; p = Q("SELECT port_name, country, congestion_index ci, avg_dwell_days dw, lat, lon FROM ports"); p["km"] = alts.hav(la, lo, p.lat.values, p.lon.values); near = p[p.km <= 1800].sort_values("ci", ascending=False)
    v = Q("SELECT COUNT(*) n FROM vessels WHERE route_name LIKE ?", (f"%{name.split()[0]}%",)).n[0]; sh = Q("SELECT COUNT(*) n FROM shipments s JOIN vessels v USING(shipment_id) WHERE v.route_name LIKE ? AND s.status LIKE '%Delay%'", (f"%{name.split()[0]}%",)).n[0]
    if not len(near): return None
    return {"answer": f"Ports within 1,800 km of the **{name}**: **{len(near)}**; **{int((near.ci >= 4).sum())}** are congested (index ≥ 4). Most congested: **{near.port_name.iloc[0]}** ({near.ci.iloc[0]:.1f}, dwell {near.dw.iloc[0]:.1f} days). {int(v)} tracked vessels are routed via this passage, {int(sh)} of them delayed.", "df": near[["port_name", "country", "ci", "dw", "km"]].rename(columns={"ci": "congestion", "dw": "dwell_days", "km": "km_from_passage"}), "sql": f"ports within 1,800 km of {name}"}
def freight_summary(q):
    if re.search(r"\bcustoms\b", q) and re.search(r"\b(hold|held|status|summary|how many|why|docs?|documents?)\b", q):
        d = Q("SELECT dest_port, COUNT(*) n FROM shipments WHERE status='Customs Hold' GROUP BY dest_port ORDER BY n DESC LIMIT 8"); t = int(Q("SELECT COUNT(*) n FROM shipments WHERE status='Customs Hold'").n[0]); tk = Q("SELECT detail FROM agent_actions WHERE type='customs_docs' AND status IN ('open','proposed') LIMIT 400")
        miss = pd.Series([x for dd in tk.detail for x in re.findall(r"missing ([^:]+?)(?:\.|$)", dd)[0].split(", ")] if len(tk) and all("missing" in x for x in tk.detail) else []).value_counts().head(6)
        return {"answer": f"**{t}** shipments are on customs hold. Most held at **{d.dest_port.iloc[0] if len(d) else '-'}**." + (" Most often missing: " + ", ".join(f"{k} ({v})" for k, v in miss.items()) + "." if len(miss) else ""), "df": d.rename(columns={"n": "held"}), "sql": "shipments with status Customs Hold; customs_docs tickets"}
    return None
def greeting(q, kind):
    ql = re.sub(r"[^a-z0-9 ?']", " ", q.lower()).strip()
    hi = re.match(r"^(hi+|hello+|hey+|hola|namaste|good (morning|afternoon|evening)|greetings|yo|sup|howdy)\b( there| team| copilot| buddy)?$", ql)
    thanks = re.match(r"^(thanks?|thank you|thx|ok(ay)?|cool|great|nice|got it|awesome)( a lot| so much)?$", ql)
    who = re.search(r"\b(who are you|what are you|what can you do|what do you do|help me|how can you help|your capabilities|what is this)\b", ql) or ql in ("help", "?")
    how = re.match(r"^how are you", ql)
    if not (hi or thanks or who or how): return None
    ex = (["Which stock needs immediate refill?", "Which outlets are at highest risk and why?", "Top 5 outlets by revenue", "Why is OUT-007 flagged?", "Which state has the most outlets?"] if kind == "Franchise" else
          ["Mumbai to Singapore 50 t electronics: days, cost and best route", "Which ports are most congested?", "How many shipments are on customs hold?", "Which quotes have the lowest margin?"])
    what = ("outlets, stock, staff, marketing, audits, customer feedback and anomalies" if kind == "Franchise" else "routes, ports, weather, customs, quotes and carriers")
    if thanks: return {"answer": "You're welcome. Ask me anything else about " + what + ".", "df": None, "sql": ""}
    head = "Hello! " if hi else ("I'm doing well, thank you. " if how else "")
    return {"answer": head + f"I'm your operations copilot. I answer from your live data on {what}, and I can explain *why* something is flagged, suggest actions and send them to the approval queue. I understand 37 languages.\n\nTry one of these:\n" + "\n".join(f"- {e}" for e in ex), "df": None, "sql": ""}

def refill(scope, n=10):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q("SELECT outlet_id, sku_name sku, category, current_stock stock, reorder_threshold thr, weekly_demand wk, lead_time_days lead FROM inventory" + w, p)
    if not len(d): return None
    d["daily"] = d.wk / 7.0; d["cover"] = d.stock / d.daily.replace(0, np.nan); d["cover"] = d.cover.fillna(999)
    d["order_qty"] = np.maximum(d.thr * 3, np.ceil(d.daily * (d.lead + 7)) - d.stock).astype(int)
    d["gap"] = d.cover - d.lead
    d["urgency"] = np.where(d.cover < d.lead, "Critical: runs out before delivery", np.where(d.stock <= d.thr, "High: below reorder level", np.where(d.cover < 7, "Medium: under 7 days", "OK")))
    d["rank"] = d.urgency.str[:4].map({"Crit": 0, "High": 1, "Medi": 2}).fillna(3); need = d[d.urgency != "OK"].sort_values(["rank", "gap", "cover"])
    if not len(need): return {"answer": "No SKU needs an urgent refill right now: all have stock above the reorder level and cover longer than the supplier lead time.", "df": None, "sql": "inventory: cover = stock / daily demand vs lead time"}
    crit = need[need.urgency.str.startswith("Critical")]; po = Q("SELECT outlet_id, sku_name FROM purchase_orders WHERE status IN ('open','ordered','pending','Pending','Ordered','Open')" + (" AND outlet_id=?" if scope else ""), p) if "purchase_orders" in [r[0] for r in Q("SELECT name FROM sqlite_master WHERE type='table'").values] else None
    top = need.head(n)[["outlet_id", "sku", "stock", "thr", "cover", "lead", "order_qty", "urgency"]].rename(columns={"thr": "reorder_level", "cover": "days_of_cover", "lead": "lead_days"}); top["days_of_cover"] = top.days_of_cover.round(1)
    pool = crit if len(crit) else need[need.urgency.str.startswith("High")] if (need.urgency.str.startswith("High")).any() else need; by = pool.groupby("outlet_id").size().sort_values(ascending=False).head(5)
    outs = ", ".join(f"{k} ({v} SKUs)" for k, v in by.items()) if len(by) else "none"
    return {"answer": f"**{len(crit):,}** SKU records will run out *before a new order can arrive* (days of cover < supplier lead time) and **{len(need) - len(crit):,}** more are low. Most urgent outlets: {outs}. The table lists the {len(top)} most urgent SKUs, with the quantity to order.",
            "df": top, "sql": "inventory: days_of_cover = stock / (weekly_demand/7); urgent when cover < lead_time_days; order_qty = max(3 x reorder level, daily demand x (lead + 7 days) - stock)",
            "extra": "**How to read it:** *days_of_cover* is how long current stock lasts at current demand; *lead_days* is how long the supplier takes. When cover is shorter than lead time, stock-out is certain unless you order now (or transfer stock from a nearby outlet). The matching *reorder* tickets are raised by the agent."}

# ------------------------------------------------------------------------------------------------ router
def answer(kind, q, scope=None, dec=None, dom=None):
    r = _answer(kind, q, scope, dec, dom)
    if r: r["complete"] = True
    return r
def _answer(kind, q, scope=None, dec=None, dom=None):
    g = greeting(q, kind)
    if g: return g
    ql = q.lower().strip(); ids = re.findall(r"\b(OUT-\d+|STF-\d+|SHP-[A-Z0-9-]+|QTE-[A-Z0-9-]+)\b", q, re.I)
    try:
        if kind == "Franchise":
            if ids and ids[0].upper().startswith("OUT") and re.search(r"\b(why|reason|flag\w*|wrong|problem|issue|explain|risk)\b", ql):
                if scope and ids[0].upper() != scope: return {"answer": f"{ids[0].upper()} is outside your outlet scope ({scope}).", "df": None, "sql": ""}
                return explain_outlet(ids[0].upper(), dec)
            _pl = (bool(re.search(r"\b(where|which|list|all|show|located|locations?|states?|cities|city|how many|number)\b", ql)) or bool(_place(ql))) and not re.search(r"\b(risk\w*|health\w*|revenue|sales|csat|satisfaction|inventory|stock|attrition|salary|overtime|best|worst|weakest|critical)\b", ql)
            if _pl and re.search(r"\b(employees?|staff|workers?|people|headcount|manpower|workforce)\b", ql) and (_place(ql) or re.search(r"\b(where|which|most|highest|more|largest|big|fewest|least|lowest|number)\b", ql)): return staff_by_place(ql, scope)
            if _pl and re.search(r"\b(outlets?|stores?|branch\w*|franchises?)\b", ql) and _place(ql): return outlets_in_place(ql, scope)
            if re.search(r"\b(state|states)\b", ql) and re.search(r"\b(outlets?|stores?|branch\w*)\b", ql): return by_state(ql)
            if re.search(r"\b(outlets?|stores?|branch\w*)\b", ql) and re.search(r"\b(risk\w*|health\w*|best|worst|weakest|strongest|critical|attention|problem\w*|performing|performance)\b", ql) and not re.search(r"\b(revenue|sales|csat|satisfaction|headcount)\b", ql): return risk_rank(ql, dec, scope)
            if re.search(r"\b(inventory|stock|stocks|stocked|sku|skus|items?|products?|ingredients?)\b", ql) and re.search(r"\b(refill\w*|restock\w*|re-?order\w*|replenish\w*|immediate\w*|urgent\w*|attention|run(ning)? (out|low)|shortage\w*|low|critical|need\w*|order now)\b", ql) and not re.search(r"\b(how many|count|average|total)\b", ql): return refill(scope)
            if re.search(r"\b(inventory|stock)\b", ql) and re.search(r"\b(how|status|holding|summary|overview|situation|look|doing|week|today|health)\b", ql) and not re.search(r"\b(how many|count|average|total|top|lowest|highest|which)\b", ql): return inventory_status(scope)
            if re.search(r"\b(sales|revenue|takings)\b", ql) and re.search(r"\b(trend|this week|last week|today|yesterday|last 7|past week|recent|how (is|are|did))\b", ql): return sales_trend(scope)
            if re.search(r"\b(staff|workforce|employees?|team)\b", ql) and re.search(r"\b(how|status|summary|overview|situation|doing|look)\b", ql) and not re.search(r"\b(how many|count|average|total|top|lowest|highest|which)\b", ql): return staff_status(scope)
        else:
            if dom is not None and re.search(r"\b(days?|time|transit|cost|price|quote|route|how long|how much|eta)\b", ql):
                r = plan_lane(q, dom)
                if r: return r
            if re.search(r"\b(congest\w*|delay\w*|busy|ports?)\b", ql) and any(_w(ql, k) for k in CHOKE): return choke_ports(ql)
            r = freight_summary(ql)
            if r: return r
    except Exception as e:
        import sys; print("analyst failed:", repr(e), file=sys.stderr)
    return None


In [ ]:
%%writefile /content/app/blocks.py
"""Server-driven UI blocks. Pages are Python functions returning JSON blocks; the single-page front end renders them and live-updates only what changed.
Charts are ECharts option dicts built here with one consistent palette."""
import math, json, datetime
import numpy as np, pandas as pd
import live_engine as LE

TEAL, BLUE, AMBER, RED, VIOLET, GREEN, PINK, GREY = "#12a594", "#3e63dd", "#f5a524", "#e5484d", "#8e4ec6", "#30a46c", "#d6409f", "#8b8d98"
PRI = "#5b5bd6"
PAL = [PRI, TEAL, AMBER, VIOLET, RED, GREEN, PINK, BLUE, GREY]
TONE = {"Critical": RED, "High": RED, "Warning": AMBER, "Medium": AMBER, "Info": BLUE, "Low": TEAL, "Pass": TEAL, "Fail": RED}

def Q(sql, p=()):
    with LE._DB_LOCK:
        return pd.read_sql(sql, LE.conn(), params=p)

def clean(v):
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.floating, float)): return None if (v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))) or (isinstance(v, np.floating) and (np.isnan(v) or np.isinf(v)))) else round(float(v), 4)
    if isinstance(v, (np.bool_,)): return bool(v)
    if isinstance(v, (pd.Timestamp, datetime.datetime, datetime.date)): return str(v)
    if isinstance(v, (list, tuple)): return [clean(x) for x in v]
    if isinstance(v, dict): return {str(k): clean(x) for k, x in v.items()}
    if v is pd.NA or v is pd.NaT: return None
    return v
def L(x): return [clean(v) for v in (x.tolist() if hasattr(x, "tolist") else list(x))]

# ------------------------------------------------------------------ charts (ECharts options)
def _base(extra=None, legend=False, grid=None):
    o = {"animation": True, "animationDuration": 350, "color": PAL, "tooltip": {"trigger": "axis", "confine": True}, "grid": grid or {"left": 8, "right": 14, "top": 28 if legend else 14, "bottom": 6, "containLabel": True}}
    if legend: o["legend"] = {"top": 0, "type": "scroll"}
    if extra: o.update(extra)
    return o
def bar(cats, series, horizontal=False, stack=False, colors=None, fmt=None, max_=None, pct=False, itemcolors=None, sort=False):
    """series: {name: values} or a list of values."""
    series = {"": series} if not isinstance(series, dict) else series; cats = L(cats)
    if sort and len(series) == 1:
        v = L(list(series.values())[0]); order = sorted(range(len(v)), key=lambda i: (v[i] is None, v[i] or 0)); cats = [cats[i] for i in order]; series = {k: [L(x)[i] for i in order] for k, x in series.items()}
    ser = []
    for i, (n, v) in enumerate(series.items()):
        d = L(v); s = {"type": "bar", "name": n, "data": d, "barMaxWidth": 22, "itemStyle": {"borderRadius": [0, 4, 4, 0] if horizontal else [4, 4, 0, 0], "color": (colors or PAL)[i % len(colors or PAL)]}, "emphasis": {"focus": "series"}}
        if itemcolors and i == 0: s["data"] = [{"value": x, "itemStyle": {"color": c}} for x, c in zip(d, itemcolors)]
        if stack: s["stack"] = "s"
        ser.append(s)
    cat = {"type": "category", "data": cats, "axisLabel": {"hideOverlap": True}}; val = {"type": "value", "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    if max_: val["max"] = max_
    if pct: val["axisLabel"] = {"formatter": "{value}%"}
    o = _base(legend=len(series) > 1 and any(series)); o.update({"xAxis": val if horizontal else cat, "yAxis": cat if horizontal else val, "series": ser})
    return o
def line(x, series, area=True, band=None, colors=None, dashed=(), ymin=None, ymax=None, step=False, markers=False):
    series = {"": series} if not isinstance(series, dict) else series; ser = []
    for i, (n, v) in enumerate(series.items()):
        c = (colors or PAL)[i % len(colors or PAL)]; s = {"type": "line", "name": n, "data": L(v), "smooth": not step, "showSymbol": markers, "symbolSize": 5, "lineStyle": {"width": 2, "color": c, "type": "dashed" if n in dashed else "solid"}, "itemStyle": {"color": c}}
        if step: s["step"] = "end"
        if area and i == 0 and len(series) <= 2: s["areaStyle"] = {"color": {"type": "linear", "x": 0, "y": 0, "x2": 0, "y2": 1, "colorStops": [{"offset": 0, "color": c + "55"}, {"offset": 1, "color": c + "00"}]}}
        ser.append(s)
    if band:
        lo, hi = L(band[0]), L(band[1]); ser += [{"type": "line", "data": lo, "stack": "b", "lineStyle": {"opacity": 0}, "showSymbol": False, "silent": True, "tooltip": {"show": False}}, {"type": "line", "data": [None if (a is None or b is None) else b - a for a, b in zip(lo, hi)], "stack": "b", "lineStyle": {"opacity": 0}, "areaStyle": {"color": "rgba(251,191,36,.16)"}, "showSymbol": False, "silent": True, "tooltip": {"show": False}}]
    y = {"type": "value", "scale": True, "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    if ymin is not None: y["min"] = ymin
    if ymax is not None: y["max"] = ymax
    o = _base(legend=len([s for s in series if s]) > 1); o.update({"xAxis": {"type": "category", "data": L(x), "boundaryGap": False, "axisLabel": {"hideOverlap": True}}, "yAxis": y, "series": ser}); return o
def pie(items, donut=True, colors=None):
    """items: {name: value} or list of (name, value, color?)"""
    its = list(items.items()) if isinstance(items, dict) else items; data = []
    for i, it in enumerate(its):
        d = {"name": str(it[0]), "value": clean(it[1])}
        if (colors and str(it[0]) in colors) or len(it) > 2: d["itemStyle"] = {"color": (colors or {}).get(str(it[0])) or it[2]}
        data.append(d)
    return {"animation": True, "color": PAL, "tooltip": {"trigger": "item", "confine": True}, "legend": {"bottom": 0, "type": "scroll"}, "series": [{"type": "pie", "radius": ["52%", "78%"] if donut else "75%", "center": ["50%", "44%"], "avoidLabelOverlap": True, "label": {"show": False}, "itemStyle": {"borderColor": "#ffffff", "borderWidth": 2, "borderRadius": 4}, "data": data}]}
def scatter(groups, xname="", yname="", colors=None, sized=False):
    """groups: {name: [ {x,y,label,size?} ]}. Points are [x, y, label, size]; the client resolves the {"__fn": ...} formatters."""
    ser = []
    for i, (n, pts) in enumerate(groups.items()):
        ser.append({"type": "scatter", "name": n, "symbolSize": {"__fn": "size3"} if sized else 10, "data": [[clean(p["x"]), clean(p["y"]), p.get("label", ""), clean(p.get("size", 10))] for p in pts],
                    "itemStyle": {"color": (colors or {}).get(n, PAL[i % len(PAL)]), "opacity": .85}, "emphasis": {"focus": "series"}})
    o = _base(legend=len(groups) > 1); o["tooltip"] = {"trigger": "item", "confine": True, "formatter": {"__fn": "scatterTip"}}
    ax = lambda n: {"type": "value", "name": n, "scale": True, "nameTextStyle": {"color": "#6b7080"}, "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    o.update({"xAxis": ax(xname), "yAxis": ax(yname), "series": ser}); return o
def hist(values, bins=24, color=TEAL, pct=False):
    v = np.asarray([x for x in values if x is not None and not (isinstance(x, float) and math.isnan(x))], dtype=float)
    if not len(v): return bar([], [])
    h, e = np.histogram(v, bins=bins); labels = [f"{(e[i] * (100 if pct else 1)):.{0 if pct else 1}f}" + ("%" if pct else "") for i in range(len(h))]
    return bar(labels, [int(x) for x in h], colors=[color])
def waterfall(labels, deltas, total_label="Total"):
    base, run, up = [], 0.0, []
    for d in deltas: base.append(run if d >= 0 else run + d); up.append(abs(d)); run += d
    cats = list(labels) + [total_label]; base.append(0); up.append(run)
    o = _base(); o.update({"xAxis": {"type": "category", "data": cats}, "yAxis": {"type": "value", "splitLine": {"lineStyle": {"color": "#eceef6"}}},
                           "series": [{"type": "bar", "stack": "w", "data": [clean(x) for x in base], "itemStyle": {"color": "transparent"}, "tooltip": {"show": False}}, {"type": "bar", "stack": "w", "data": [{"value": clean(u), "itemStyle": {"color": TEAL if i == len(up) - 1 else (BLUE if deltas[i] >= 0 else RED)}} for i, u in enumerate(up)], "barMaxWidth": 34, "itemStyle": {"borderRadius": 4}}]}); return o
def radar(names, series, mx=100):
    o = {"animation": True, "color": PAL, "tooltip": {"confine": True}, "legend": {"bottom": 0}, "radar": {"indicator": [{"name": n, "max": mx} for n in names], "splitArea": {"show": False}, "axisName": {"color": "#6b7080"}}, "series": [{"type": "radar", "data": [{"name": k, "value": L(v), "areaStyle": {"opacity": .18}} for k, v in series.items()]}]}; return o
def gauge(value, name="", mx=100):
    c = RED if value < 60 else AMBER if value < 80 else TEAL
    return {"series": [{"type": "gauge", "startAngle": 210, "endAngle": -30, "min": 0, "max": mx, "progress": {"show": True, "width": 14, "itemStyle": {"color": c}}, "axisLine": {"lineStyle": {"width": 14, "color": [[1, "rgba(148,163,184,.15)"]]}}, "axisTick": {"show": False}, "splitLine": {"show": False}, "axisLabel": {"show": False}, "pointer": {"show": False}, "anchor": {"show": False},
                         "detail": {"valueAnimation": True, "fontSize": 30, "fontWeight": 700, "offsetCenter": [0, "8%"], "color": "#1c2024", "formatter": "{value}"}, "title": {"offsetCenter": [0, "62%"], "color": "#6b7080", "fontSize": 12}, "data": [{"value": clean(round(value, 1)), "name": name}]}]}
def graph(nodes, links, h=520):
    """nodes: [{id,name,kind,size,color,tip}] ; links: [(a,b)]"""
    cat = sorted({n.get("kind", "x") for n in nodes}); return {"animation": True, "tooltip": {"confine": True, "formatter": {"__fn": "nodeTip"}}, "legend": {"data": cat, "bottom": 0, "textStyle": {"color": "#6b7080"}}, "color": PAL,
        "series": [{"type": "graph", "layout": "force", "roam": True, "draggable": True, "force": {"repulsion": 260, "edgeLength": [40, 120], "gravity": .08}, "label": {"show": True, "position": "right", "color": "#454a58", "fontSize": 10, "formatter": {"__fn": "nodeLabel"}},
                    "categories": [{"name": c} for c in cat], "lineStyle": {"color": "#cfd3e3", "width": 1}, "emphasis": {"focus": "adjacency", "lineStyle": {"width": 2}},
                    "data": [{"id": n["id"], "name": n["id"], "category": cat.index(n.get("kind", "x")), "symbolSize": n.get("size", 12), "itemStyle": {"color": n.get("color")} if n.get("color") else {}, "tip": n.get("tip", ""), "showLabel": n.get("label", False), "open": n.get("open")} for n in nodes],
                    "links": [{"source": a, "target": b} for a, b in links]}]}

# ------------------------------------------------------------------ blocks
def kpis(*items):
    out = []
    for it in items:
        d = {"label": it[0], "value": clean(it[1])}
        if len(it) > 2 and it[2] is not None: d["sub"] = str(it[2])
        if len(it) > 3 and it[3]: d["tone"] = it[3]
        out.append(d)
    return {"t": "kpis", "items": out}
def chart(opt, title=None, h=290, open=None, sub=None, key=None): return {"t": "chart", "opt": opt, "title": title, "h": h, "open": open, "sub": sub, "key": key or title}
def table(df, cols=None, title=None, h=None, open=None, key=None, sub=None, search=False, page=None):
    """cols: list of names or dicts {k,label,type: text|num|int|pct|pill|bar|money|time|id, max}. open = {"kind":"outlet","col":"outlet_id"} makes rows clickable."""
    if isinstance(df, list): df = pd.DataFrame(df)
    cols = cols or list(df.columns); cs = [{"k": c, "label": c.replace("_", " ")} if isinstance(c, str) else c for c in cols]
    rows = [{c["k"]: clean(r.get(c["k"])) for c in cs} for r in df.to_dict("records")] if len(df) else []
    return {"t": "table", "cols": cs, "rows": rows, "title": title, "h": h, "open": open, "key": key or title, "sub": sub, "search": search}
def callout(text, tone="info", title=None): return {"t": "callout", "text": text, "tone": tone, "title": title}
def md(text): return {"t": "md", "text": text}
def row(*cols, ratio=None): return {"t": "row", "cols": list(cols), "ratio": ratio}
def col(*blocks): return {"t": "col", "blocks": list(blocks)}
def card(title, *blocks, sub=None): return {"t": "card", "title": title, "sub": sub, "blocks": list(blocks)}
def form(id, fields, submit="Submit", url=None, title=None, tone="primary", result=True, sub=None):
    """fields: {name,label,type: select|number|text|textarea|range|checks|multi|file|date, options, value, min,max,step}"""
    return {"t": "form", "id": id, "fields": fields, "submit": submit, "url": url, "title": title, "tone": tone, "sub": sub}
def button(id, label, url, tone="primary", confirm=None, payload=None): return {"t": "button", "id": id, "label": label, "url": url, "tone": tone, "confirm": confirm, "payload": payload or {}}
def map_(layers, h=470, view=None, open=None, key="map", legend=None, live=None): return {"t": "map", "layers": layers, "h": h, "view": view, "open": open, "key": key, "legend": legend, "live": live}
def tickets(items, key="tickets", title=None, sub=None): return {"t": "tickets", "items": items, "key": key, "title": title, "sub": sub}
def hero(agent, title, blurb, r, can_run=True):
    return {"t": "hero", "agent": agent, "title": title, "blurb": blurb, "can_run": can_run, "score": None if not r else clean(r["score"]), "risk": None if not r else r["risk"], "summary": None if not r else r["summary"], "ts": None if not r else r["ts"], "ms": None if not r else r["ms"]}
def tabs(items): return {"t": "tabs", "tabs": items}
def page(title, sub, blocks, tabs=None, live=8, actions=None): return {"title": title, "sub": sub, "blocks": blocks, "tabs": tabs, "live": live, "actions": actions or []}
def pill_col(k, label=None): return {"k": k, "label": label or k.replace("_", " "), "type": "pill"}
def num_col(k, label=None, fmt="num", mx=None): return {"k": k, "label": label or k.replace("_", " "), "type": fmt, "max": mx}


In [ ]:
%%writefile /content/app/config.py
"""Shared configuration. APP_KIND (Franchise | Freight) is set by the launcher. No Google Drive prompts: data lives in DATA_DIR (set PERSIST_DIR to keep it between sessions)."""
import os
KIND = os.environ.get("APP_KIND", "Franchise")
APP_DIR = os.path.dirname(os.path.abspath(__file__))
_default = "/content/app_data" if os.path.isdir("/content") else os.path.join(APP_DIR, "runtime_data")
DATA_DIR = os.environ.get("PERSIST_DIR") or os.environ.get("APP_DATA_DIR") or os.environ.get("FRANCHISEOPS_DATA_DIR") or os.environ.get("FREIGHTQUOTE_DATA_DIR") or _default
os.makedirs(DATA_DIR, exist_ok=True)
DB_PATH = os.path.join(DATA_DIR, "franchise_database.db" if KIND == "Franchise" else "freight_database.db")
def _secret(key, default=""):
    try:
        from google.colab import userdata
        v = userdata.get(key)
        if v: return str(v).strip()
    except Exception: pass
    return str(os.environ.get(key, default) or "").strip()
HF_TOKEN = _secret("HF_TOKEN") or _secret("HUGGINGFACE_TOKEN")
def _persisted_secret():
    """No JWT_SECRET_KEY given: generate a random one once and keep it beside the database, so sessions survive restarts and are never signed with a public default."""
    import secrets as _s
    p = os.path.join(DATA_DIR, ".jwt_secret")
    try:
        if os.path.exists(p): return open(p).read().strip()
        v = _s.token_urlsafe(48); fd = os.open(p, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600); os.write(fd, v.encode()); os.close(fd); return v
    except Exception: return _s.token_urlsafe(48)
JWT_SECRET_KEY = _secret("JWT_SECRET_KEY") or _persisted_secret()
ADMIN_EMAIL_ID = _secret("ADMIN_EMAIL_ID", "admin@infosys.com"); DEMO_MODE = os.environ.get("DEMO_MODE", "1") != "0"     # 1 = seeded demo accounts shown on the login page (Colab demo). 0 = production: only the admin exists, with ADMIN_PASSWORD (or a random one printed once in the server log)
def _admin_pw():
    v = _secret("ADMIN_PASSWORD")
    if v: return v
    if DEMO_MODE: return "Admin@123"
    import secrets as _s; p = os.path.join(DATA_DIR, ".admin_initial");
    if os.path.exists(p): return open(p).read().strip()
    v = _s.token_urlsafe(9) + "1aA"; open(p, "w").write(v); print("\n*** DEMO_MODE=0: initial admin password:", v, "(stored in", p, ") ***\n"); return v
ADMIN_PASSWORD = _admin_pw()
EMAIL_ID = _secret("EMAIL_ID"); EMAIL_PASSWORD = _secret("EMAIL_PASSWORD").replace(" ", "")
MODEL_NAME = os.environ.get("LLM_MODEL", "")
NLLB_MODEL = os.environ.get("NLLB_MODEL", "facebook/nllb-200-distilled-600M")


In [ ]:
%%writefile /content/app/db.py
import sqlite3, os, threading
import pandas as pd
from config import DB_PATH

# A single, long-lived connection reused across the whole app process
# instead of opening (and never explicitly closing) a brand-new SQLite
# connection + re-running PRAGMA setup on every single call. That old
# pattern was leaking file handles under load and was the single
# biggest source of slowness/lock contention across the app.
_conn = None
_conn_lock = threading.Lock()

def get_conn():
    global _conn
    if _conn is None:
        with _conn_lock:
            if _conn is None:
                os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
                _conn = sqlite3.connect(DB_PATH, timeout=30, check_same_thread=False)
                _conn.execute("PRAGMA journal_mode=WAL;")
                _conn.execute("PRAGMA synchronous=NORMAL;")
                _conn.execute("PRAGMA cache_size=-64000;")   # ~64MB page cache
                _conn.execute("PRAGMA temp_store=MEMORY;")
    return _conn

def save_chat_message(username, role, message):
    try:
        with get_conn() as conn:
            conn.execute("INSERT INTO chat_history (username, role, message) VALUES (?, ?, ?);", (username, role, message))
            conn.commit()
    except Exception: pass

def load_chat_history(username=None, limit=100):
    try:
        with get_conn() as conn:
            if username:
                df = pd.read_sql("SELECT role, message FROM chat_history WHERE username=? ORDER BY id ASC LIMIT ?;", conn, params=(username, limit))
                if df.empty:
                    df = pd.read_sql("SELECT role, message FROM chat_history ORDER BY id ASC LIMIT ?;", conn, params=(limit,))
            else:
                df = pd.read_sql("SELECT role, message FROM chat_history ORDER BY id ASC LIMIT ?;", conn, params=(limit,))

            res = []
            for _, r in df.iterrows():
                content_val = str(r.get("message") or r.get("content") or "")
                res.append({
                    "role": str(r.get("role", "assistant")),
                    "content": content_val,
                    "message": content_val
                })
            return res
    except Exception: return []

def clear_chat_history(username=None):
    try:
        with get_conn() as conn:
            if username:
                conn.execute("DELETE FROM chat_history WHERE username=?;", (username,))
            else:
                conn.execute("DELETE FROM chat_history;")
            conn.commit()
    except Exception: pass

def init_db():
    with get_conn() as conn:
        conn.execute("""
        CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT,
            role TEXT,
            message TEXT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password_hash TEXT,
            security_question TEXT,
            security_answer_hash TEXT,
            role TEXT,
            failed_attempts INTEGER DEFAULT 0,
            lock_until TIMESTAMP DEFAULT NULL,
            account_status TEXT DEFAULT 'active',
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)
        # Milestone 2 authentication columns — safe migrations for existing databases.
        for _col, _ddl in [
            ("username", "ALTER TABLE users ADD COLUMN username TEXT"),
            ("security_question", "ALTER TABLE users ADD COLUMN security_question TEXT"),
            ("security_answer_hash", "ALTER TABLE users ADD COLUMN security_answer_hash TEXT"),
            ("failed_attempts", "ALTER TABLE users ADD COLUMN failed_attempts INTEGER DEFAULT 0"),
            ("lock_until", "ALTER TABLE users ADD COLUMN lock_until TIMESTAMP DEFAULT NULL"),
            ("account_status", "ALTER TABLE users ADD COLUMN account_status TEXT DEFAULT 'active'"),
            ("profile_picture", "ALTER TABLE users ADD COLUMN profile_picture TEXT DEFAULT NULL"),
        ]:
            try:
                conn.execute(_ddl)
            except Exception:
                pass

        conn.execute("""
        CREATE TABLE IF NOT EXISTS alerts (
            alert_id INTEGER PRIMARY KEY AUTOINCREMENT,
            shipment_id TEXT,
            outlet_id TEXT,
            severity TEXT,
            category TEXT,
            message TEXT,
            date TEXT,
            resolved INT DEFAULT 0
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS shipments (
            shipment_id TEXT PRIMARY KEY,
            origin_port TEXT,
            dest_port TEXT,
            carrier TEXT,
            status TEXT,
            eta TEXT,
            weight_kg REAL,
            hs_code TEXT,
            predicted_delay_risk REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS freight_quotes (
            quote_id TEXT PRIMARY KEY,
            shipment_id TEXT,
            customer_id TEXT,
            carrier TEXT,
            base_cost REAL,
            insurance REAL,
            customs_fee REAL,
            fuel_surcharge REAL,
            final_price REAL,
            margin_pct REAL,
            status TEXT,
            created_at TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS ports (
            port_id TEXT PRIMARY KEY,
            port_name TEXT,
            country TEXT,
            congestion_index REAL,
            avg_dwell_days REAL,
            lat REAL,
            lon REAL,
            region TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS carriers (
            carrier_id TEXT PRIMARY KEY,
            name TEXT,
            rating REAL,
            on_time_pct REAL,
            avg_cost_index REAL,
            risk_level TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS customers (
            customer_id TEXT PRIMARY KEY,
            name TEXT,
            industry TEXT,
            priority_tier TEXT,
            credit_risk REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS ml_metrics (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            module TEXT,
            model_name TEXT,
            metric_name TEXT,
            metric_value REAL,
            trained_at DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS outlets (
            outlet_id TEXT PRIMARY KEY,
            outlet_name TEXT,
            location TEXT,
            tier TEXT,
            revenue REAL,
            operating_costs REAL,
            customer_satisfaction REAL,
            staff_headcount INT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS staff (
            staff_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            name TEXT,
            role TEXT,
            salary REAL,
            overtime_hrs REAL,
            job_satisfaction INT,
            age INT,
            tenure_years INT,
            work_life_balance INT,
            predicted_attrition_prob REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS inventory (
            record_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            sku_name TEXT,
            category TEXT,
            current_stock INT,
            reorder_threshold INT,
            weekly_demand REAL,
            lead_time_days INT,
            stockout_risk_prob REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS marketing (
            campaign_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            campaign_name TEXT,
            channel TEXT,
            budget REAL,
            actual_roi REAL,
            reach INT,
            conversions INT,
            start_date TEXT,
            end_date TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS feedback (
            feedback_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            rating INT,
            comment TEXT,
            date TEXT,
            sentiment_score REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS audits (
            audit_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            audit_date TEXT,
            score REAL,
            violations INT,
            category TEXT,
            status TEXT,
            notes TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS weather_risks (
            port_name TEXT PRIMARY KEY,
            current_severity INT,
            forecast TEXT,
            wind_speed REAL,
            wave_height REAL,
            temperature REAL
        );
        """)
        conn.commit()

In [ ]:
%%writefile /content/app/domain_freight.py
"""FreightQuote AI live domain (PRS v3): 7 agents, ML pricing + congestion models, vessel tracking, reroute loop, PDF documents."""
import sys, os, re, json, math, random, time, datetime, zlib
import numpy as np, pandas as pd, requests
from live_engine import AgentResult, risk_label, conn, now, _DB_LOCK, set_state, get_state, emit, run_cycle, _log_run, act, init_live_tables
from config import DB_PATH
KIND = "Freight"; NAME = "FreightQuote AI"; ENTITY_COL = "shipment_id"; SMS = False; ESCALATE_TO = "Operations Manager"
OWNER = {"quote_anomaly": "Finance Lead", "reroute": "Operations Manager", "quote_approval": "Admin", "customs_docs": "Customs Officer", "congestion_notice": "Operations Manager", "margin_review": "Finance Lead", "quote_hold": "Freight Broker"}
ACTION_AGENT = {"quote_anomaly": "anomaly_watch", "reroute": "weather_watch", "customs_docs": "customs_watch", "congestion_notice": "route_watch", "quote_approval": "quote_watch", "margin_review": "quote_watch", "quote_hold": "quote_watch"}
DATA_DIR = os.path.dirname(DB_PATH); TICK_NM = 84.0       # 1 tick = 1 sailing hour
CARGO = {"Electronics": 1.15, "Pharmaceuticals": 1.35, "Automotive Parts": 1.1, "Textiles": .95, "Heavy Machinery": 1.3, "Perishables": 1.4, "General": 1.0, "Hazardous": 1.6, "Bulk": .8}
CODES = {k: i for i, k in enumerate(CARGO)}
def Q(sql, p=()):
    with _DB_LOCK, conn() as c: return pd.read_sql(sql, c, params=p)

# ---------------------------------------------------------------- geo + routing
def nm(a, b):
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1])); h = math.sin((la2 - la1) / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    return 3440.065 * 2 * math.asin(min(1, math.sqrt(h)))
def chain_nm(ch): return sum(nm(ch[i], ch[i + 1]) for i in range(len(ch) - 1))
def interp(ch, frac):
    legs = [nm(ch[i], ch[i + 1]) for i in range(len(ch) - 1)]; tot = sum(legs) or 1; d = min(max(frac, 0), 1) * tot
    for i, L in enumerate(legs):
        if d <= L or i == len(legs) - 1:
            t = d / L if L else 0; return (ch[i][0] + (ch[i + 1][0] - ch[i][0]) * t, ch[i][1] + (ch[i + 1][1] - ch[i][1]) * t)
        d -= L
SUEZ_S, SUEZ_N, CAPE = (12.6, 43.4), (30.0, 32.55), (-34.4, 18.5)
_ACC = {}
def _access(xy):
    """Sea entry point for river / bay ports (Hamburg, Wuhan, Houston ...), looked up by the port's coordinates."""
    if not _ACC:
        try:
            import seanav
            for n, la, lo in Q("SELECT port_name, lat, lon FROM ports").itertuples(index=False): _ACC[(round(float(la), 2), round(float(lo), 2))] = seanav.access_for(n)
        except Exception: _ACC[None] = None
    return _ACC.get((round(float(xy[0]), 2), round(float(xy[1]), 2)))
def best_route(o, d):
    import seanav
    r = seanav.sea().route((float(o[0]), float(o[1])), (float(d[0]), float(d[1])), (), (), _access(o), _access(d))
    return {"name": "via " + " + ".join(r["via"]) if r["via"] else "direct sea route", "chain": r["pts"], "nm": round(r["nm"]), "via": r["via"]}
def route_options(o, d, avoid=()):
    """Real sea routes (water only: see seanav.py). avoid may contain 'suez', 'panama', 'malacca', 'hormuz', 'bosphorus' and/or 'storm' (steer around the active storms)."""
    import seanav
    S = seanav.sea(); blocked = tuple(a for a in avoid if a in seanav.BLOCK_GROUPS); st = tuple((s["lat"], s["lon"], s.get("r", 700)) for s in storms()) if "storm" in avoid else ()
    opts = S.options((float(o[0]), float(o[1])), (float(d[0]), float(d[1])), blocked, st, _access(o), _access(d))
    if not opts and st: opts = S.options((float(o[0]), float(o[1])), (float(d[0]), float(d[1])), blocked, (), _access(o), _access(d))
    return [{"name": x["name"], "chain": x["pts"], "nm": round(x["nm"]), "via": x["via"]} for x in opts]
def port_xy(name):
    r = Q("SELECT lat, lon, congestion_index ci, avg_dwell_days dw, country FROM ports WHERE port_name=? LIMIT 1", (name,))
    return (float(r.lat[0]), float(r.lon[0]), float(r.ci[0]), float(r.dw[0]), r.country[0]) if len(r) else None

# ---------------------------------------------------------------- weather (Open-Meteo live + cached port rows + simulated storms)
_WX = {}; _OFF = [0.0]
def storms(): return get_state("storms", []) or []
def inject_storm(lat=None, lon=None, wind=22.0):
    s = storms(); s.append({"lat": lat if lat is not None else random.uniform(-30, 45), "lon": lon if lon is not None else random.uniform(-60, 120), "r": 700, "wind": wind, "ts": now()}); set_state("storms", s[-6:])
def weather_at(lat, lon, allow_live=True):
    k = (round(lat / 2) * 2, round(lon / 2) * 2); hit = _WX.get(k); wind, wave, src = 6.0, 1.2, "cached"
    if hit and time.time() - hit[0] < 900: wind, wave, src = hit[1:]
    elif allow_live and time.time() > _OFF[0]:
        try:
            r = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=wind_speed_10m&wind_speed_unit=ms", timeout=3).json()["current"]
            wind, src = float(r["wind_speed_10m"]), "Open-Meteo live"
            try: wave = float(requests.get(f"https://marine-api.open-meteo.com/v1/marine?latitude={lat}&longitude={lon}&current=wave_height", timeout=3).json()["current"]["wave_height"] or 1.2)
            except Exception: wave = 1.2
            _WX[k] = (time.time(), wind, wave, src)
        except Exception: _OFF[0] = time.time() + 120; wind, src = 5 + 4 * random.random(), "simulated (offline)"
    else: wind, src = 5 + 4 * random.random(), "simulated (offline)"
    for s in storms():
        d = nm((lat, lon), (s["lat"], s["lon"]))
        if d < s["r"]: wind = max(wind, s["wind"] * (1 - .5 * d / s["r"])); src += " + storm"
    return {"wind": round(wind, 1), "wave": round(wave, 1), "src": src, "level": "High" if wind > 15 else "Medium" if wind >= 8 else "Low"}
def wscore(level): return {"High": 1.0, "Medium": .5, "Low": .12}[level]

# ---------------------------------------------------------------- ML models
def _gen_rates(n=4000):
    R = np.random.default_rng(42); dist = R.uniform(500, 12000, n); w = R.uniform(1, 600, n); cg = R.integers(0, len(CARGO), n); cong = R.integers(0, 4, n); mult = np.array(list(CARGO.values()))
    cost = (250 + .21 * dist) * (.6 + .012 * w) * (1 + .08 * cong) * mult[cg] * R.normal(1, .06, n)
    return pd.DataFrame({"distance_nm": dist, "cargo_weight": w, "cargo_type": cg, "congestion_level": cong, "freight_cost_usd": cost})
_ML = {}
def pricing_model():
    from sklearn.ensemble import RandomForestRegressor; from sklearn.model_selection import train_test_split; from sklearn.metrics import r2_score, mean_squared_error; import joblib
    path = os.path.join(DATA_DIR, "pricing_rf.joblib")
    if "rf" in _ML: return _ML
    if os.path.exists(path): _ML.update(joblib.load(path)); return _ML
    src = "bootstrap synthetic (feed a Kaggle freight-rate CSV to replace)"
    try:
        d = Q("SELECT * FROM freight_rates")
        if len(d) >= 300: src = "fed dataset (freight_rates table)"
        else: d = _gen_rates()
    except Exception: d = _gen_rates()
    X, y = d[["distance_nm", "cargo_weight", "cargo_type", "congestion_level"]], d.freight_cost_usd
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=42); rf = RandomForestRegressor(100, random_state=42, n_jobs=-1).fit(Xtr, ytr)
    m = {"r2": round(r2_score(yte, rf.predict(Xte)), 3), "rmse": round(float(np.sqrt(mean_squared_error(yte, rf.predict(Xte)))), 1), "source": src, "importance": dict(zip(X.columns, rf.feature_importances_.round(3).tolist()))}
    _ML.update(rf=rf, metrics=m); joblib.dump({"rf": rf, "metrics": m}, path)
    with _DB_LOCK, conn() as c: c.execute("UPDATE ml_models SET is_active=0 WHERE model_name='rf_freight_price'"); c.execute("INSERT INTO ml_models(model_name,metrics,trained_at,is_active) VALUES(?,?,?,1)", ("rf_freight_price", json.dumps(m), now()))
    return _ML
def congestion_model():
    from sklearn.ensemble import RandomForestClassifier
    if "cg" in _ML: return _ML["cg"]
    R = np.random.default_rng(3); n = 3000; ci = R.uniform(1, 4.8, n); dw = R.uniform(1.5, 7, n); mo = R.integers(1, 13, n); dow = R.integers(0, 7, n); cg = R.integers(0, len(CARGO), n)
    lvl = np.clip(np.round((ci - 1) / 1.3 + .25 * (dw > 5) + .35 * np.isin(mo, [10, 11, 12, 1, 2]) + R.normal(0, .3, n)), 0, 3).astype(int)
    clf = RandomForestClassifier(100, random_state=42).fit(np.c_[ci, dw, mo, dow, cg], lvl); _ML["cg"] = clf; return clf
def predict_congestion(ci, dw, cargo="General"):
    t = datetime.date.today(); return int(congestion_model().predict([[ci, dw, t.month, t.weekday(), CODES.get(cargo, 6)]])[0])

# ---------------------------------------------------------------- bootstrap, simulator, vessel tracking
def bootstrap():
    from scale_data import scale_freight; scale_freight()
    with _DB_LOCK, conn() as c:
        c.executescript("""CREATE TABLE IF NOT EXISTS quote_requests(req_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, origin TEXT, dest TEXT, cargo TEXT, weight_t REAL, hs_code TEXT, customer_id TEXT, language TEXT, status TEXT DEFAULT 'new', quote_id TEXT, result TEXT, docs_provided TEXT);
        CREATE TABLE IF NOT EXISTS shipment_docs(ref TEXT, doc TEXT, ts TEXT, by TEXT, UNIQUE(ref, doc));
        CREATE TABLE IF NOT EXISTS vessels(shipment_id TEXT PRIMARY KEY, lat REAL, lon REAL, progress REAL, chain TEXT, route_name TEXT, speed_kn REAL, updated TEXT);
        CREATE TABLE IF NOT EXISTS ml_models(model_id INTEGER PRIMARY KEY AUTOINCREMENT, model_name TEXT, metrics TEXT, trained_at TEXT, is_active INT);
        CREATE TABLE IF NOT EXISTS customs_rules(chapter TEXT PRIMARY KEY, goods TEXT, duty_pct REAL, docs TEXT, restricted INT);""")
        try: c.execute("ALTER TABLE quote_requests ADD COLUMN docs_provided TEXT")
        except Exception: pass
        if not c.execute("SELECT 1 FROM customs_rules").fetchone():
            base = "Commercial Invoice;Packing List;Bill of Lading"
            for ch, g, d, ex, rs in [("30", "Pharmaceuticals", 5, "Import License;Drug Control Certificate", 1), ("84", "Machinery", 7.5, "Certificate of Origin", 0), ("85", "Electronics", 10, "BIS Certificate;Certificate of Origin", 0), ("87", "Vehicles and parts", 15, "Homologation Certificate", 0),
                                     ("10", "Cereals", 12, "Phytosanitary Certificate", 0), ("62", "Apparel", 12, "Certificate of Origin", 0), ("27", "Mineral fuels", 5, "Hazmat Declaration;IMDG Certificate", 1), ("39", "Plastics", 8, "Certificate of Origin", 0), ("72", "Iron and steel", 7.5, "Mill Test Certificate", 0), ("22", "Beverages", 20, "Excise Permit", 1)]:
                c.execute("INSERT INTO customs_rules VALUES(?,?,?,?,?)", (ch, g, d, base + ";" + ex, rs))
        n = c.execute("SELECT COUNT(*) FROM vessels").fetchone()[0]
        if n == 0:
            R = random.Random(5)
            for sid, o, d in c.execute("SELECT shipment_id, origin_port, dest_port FROM shipments WHERE status IN ('In Transit','Port Congestion Delay','Anchorage Pending')").fetchall():
                po, pd_ = port_xy(o), port_xy(d)
                if not po or not pd_: continue
                opt = best_route(po[:2], pd_[:2]); p = R.uniform(.05, .9); la, lo = interp(opt["chain"], p)
                c.execute("INSERT OR REPLACE INTO vessels VALUES(?,?,?,?,?,?,?,?)", (sid, la, lo, p, json.dumps(opt["chain"]), opt["name"], R.uniform(12, 17), now()))
    pricing_model(); congestion_model()

def tick():
    reasons = []; R = random
    with _DB_LOCK, conn() as c:
        real = set()
        for sid, p, chain, spd in c.execute("SELECT shipment_id, progress, chain, speed_kn FROM vessels").fetchall():
            if sid in real: continue
            ch = json.loads(chain); tot = chain_nm(ch) or 1; p2 = p + spd * 1.0 / tot
            if p2 >= 1:
                c.execute("DELETE FROM vessels WHERE shipment_id=?", (sid,)); c.execute("UPDATE shipments SET status='Delivered' WHERE shipment_id=?", (sid,)); emit("delivered", sid); continue
            la, lo = interp(ch, p2); c.execute("UPDATE vessels SET lat=?, lon=?, progress=?, updated=? WHERE shipment_id=?", (la, lo, p2, now(), sid))
        _tickn = get_state("tickn", 0) + 1; set_state("tickn", _tickn)
        ports = c.execute("SELECT port_name, lat, lon FROM ports ORDER BY port_id").fetchall()
        try: import weather as _WXL; _fresh = _WXL.fresh_ports()
        except Exception: _fresh = set()
        for i in range(12):                                         # round-robin refresh of port weather + congestion drift
            nme, la, lo = ports[(_tickn * 12 + i) % len(ports)]; w = weather_at(la, lo, allow_live=False)
            if nme in _fresh:                                       # real Open-Meteo reading is recent: keep it, only drift congestion
                c.execute("UPDATE ports SET congestion_index=MIN(5,MAX(0.5,congestion_index+?)) WHERE port_name=?", (R.uniform(-.12, .12), nme)); continue
            c.execute("INSERT OR REPLACE INTO weather_risks(port_name,current_severity,forecast,wind_speed,wave_height,temperature) VALUES(?,?,?,?,?,?)", (nme, w["level"], w["src"], w["wind"] * 3.6, w["wave"], 24.0))
            c.execute("UPDATE ports SET congestion_index=MIN(5,MAX(0.5,congestion_index+?)) WHERE port_name=?", (R.uniform(-.12, .12), nme))
        for _ in range(max(0, min(4, 230 - c.execute("SELECT COUNT(*) FROM vessels").fetchone()[0]))):   # keep the tracked fleet topped up as ships arrive
            o, d = R.sample(ports, 2); sid = f"SHP-L{R.randint(0, 10**8):08d}"; opt = best_route((o[1], o[2]), (d[1], d[2]))
            c.execute("INSERT OR REPLACE INTO shipments(shipment_id,origin_port,dest_port,carrier,status,eta,weight_kg,hs_code,predicted_delay_risk) VALUES(?,?,?,?,?,?,?,?,?)", (sid, o[0], d[0], R.choice([r[0] for r in c.execute("SELECT name FROM carriers").fetchall()]), "In Transit", (datetime.date.today() + datetime.timedelta(days=20)).isoformat(), R.uniform(2000, 25000), f"HS-{R.choice([30,84,85,87,10,62])}{R.randint(10,99)}", .1))
            c.execute("INSERT OR REPLACE INTO vessels VALUES(?,?,?,?,?,?,?,?)", (sid, o[1], o[2], R.uniform(0, .05), json.dumps(opt["chain"]), opt["name"], R.uniform(12, 17), now()))
        if _tickn % 6 == 0 and R.random() < .6:                      # incoming customer quote request
            o, d = R.sample(ports, 2); cu = c.execute("SELECT customer_id FROM customers ORDER BY RANDOM() LIMIT 1").fetchone()[0]
            hs = f"HS-{R.choice([30,84,85,87,10,62,27])}{R.randint(10,99)}"; req_docs = _required_docs(hs)[1]; given = [x for x in req_docs if R.random() < (.97 if x in ("Commercial Invoice", "Packing List", "Bill of Lading") else .82)]
            c.execute("INSERT INTO quote_requests(ts,origin,dest,cargo,weight_t,hs_code,customer_id,language,docs_provided) VALUES(?,?,?,?,?,?,?,?,?)", (now(), o[0], d[0], R.choice(list(CARGO)), round(R.uniform(5, 450), 1), hs, cu, "English", ";".join(given))); reasons.append("quote-request")
        if _tickn % 40 == 0 and R.random() < .5: inject_storm(); reasons.append("storm")
        if _tickn % 25 == 0: set_state("storms", [s for s in storms() if (time.time() % 1e9) > 0][-3:])
    return reasons

def feed():
    with _DB_LOCK, conn() as c: ports = c.execute("SELECT port_name, lat, lon FROM ports ORDER BY port_id").fetchall()
    k = get_state("feedk", 0) or 0
    for i in range(10): _, la, lo = ports[(k * 10 + i) % len(ports)]; weather_at(la, lo, allow_live=True)
    set_state("feedk", k + 1)
DEFAULT_POLICY = {"reroute": "approval"}
def process():
    for r in Q("SELECT req_id FROM quote_requests WHERE status='new' ORDER BY req_id LIMIT 3").req_id.tolist(): run_quote(int(r))

# ---------------------------------------------------------------- the 7 pipeline agents
def route_agent(s):
    req = s["req"]; po, pd_ = port_xy(req["origin"]), port_xy(req["dest"])
    if not po or not pd_: raise ValueError("unknown port")
    opts = route_options(po[:2], pd_[:2], s.get("avoid", ())); best = opts[0]
    cong = predict_congestion((po[2] + pd_[2]) / 2, (po[3] + pd_[3]) / 2, req["cargo"]); days = best["nm"] / (14 * 24) + po[3] * .5 + pd_[3] * .5
    cars = Q("SELECT name, rating, on_time_pct, avg_cost_index, risk_level FROM carriers"); cars["score"] = cars.rating / 5 * 40 + cars.on_time_pct * .4 - cars.avg_cost_index * 12 - (cars.risk_level == "High") * 8
    top = cars.sort_values("score", ascending=False).head(3)
    return AgentResult("route", 100 - cong * 20, risk_label(100 - cong * 20), f"{best['name']}, {best['nm']:,} nm, {days:.1f} days, congestion level {cong}/3 (ML)",
        {"route": best["name"], "chain": best["chain"], "distance_nm": best["nm"], "transit_days": round(days, 1), "congestion": cong, "alternatives": [o["name"] for o in opts[1:]], "carriers": top.name.tolist(), "ports": [po[:2], pd_[:2]]})
def weather_agent(s):
    r = s["results"]["route"].findings; ch = r["chain"]; pts = [interp(ch, f) for f in (0, .25, .5, .75, 1)]; ws = [weather_at(la, lo) for la, lo in pts]; worst = max(ws, key=lambda w: w["wind"])
    sc = max(wscore(w["level"]) for w in ws); delay = round(min(.95, sc * .6 + worst["wind"] / 60), 2)
    return AgentResult("weather", 100 - sc * 100, risk_label(100 - sc * 100), f"{worst['level']} risk: wind {worst['wind']} m/s, waves {worst['wave']} m ({worst['src']}), delay probability {delay}", {"level": worst["level"], "wind": worst["wind"], "wave": worst["wave"], "risk": sc, "delay_prob": delay, "src": worst["src"]})
def pricing_agent(s):
    m = pricing_model(); rt = s["results"]["route"].findings; req = s["req"]
    ml = float(m["rf"].predict(pd.DataFrame([[rt["distance_nm"], req["weight_t"], CODES.get(req["cargo"], 6), rt["congestion"]]], columns=["distance_nm", "cargo_weight", "cargo_type", "congestion_level"]))[0])
    fuel = ml * float(get_state("fuel_pct", .12) or .12); fees = 350 + 40 * rt["congestion"] * 2
    return AgentResult("pricing", 90, "Low", f"ML base ${ml:,.0f} (R2 {m['metrics']['r2']}), fuel 12% ${fuel:,.0f}, port fees ${fees:,.0f}", {"base": round(ml, 2), "fuel": round(fuel, 2), "port_fees": fees, "model": m["metrics"]})
def _required_docs(hs):
    ch = (re.findall(r"\d+", str(hs)) or ["00"])[0][:2]; r = Q("SELECT * FROM customs_rules WHERE chapter=?", (ch,))
    base = ["Commercial Invoice", "Packing List", "Bill of Lading"]
    return ch, (r.docs[0].split(";") if len(r) else base), (int(r.restricted[0]) if len(r) else 0), (float(r.duty_pct[0]) if len(r) else 10.0), (r.goods[0] if len(r) else "General cargo")
def _attached(ref):
    try: return set(Q("SELECT doc FROM shipment_docs WHERE ref=?", (ref,)).doc)
    except Exception: return set()
def customs_agent(s):
    """Customs check against the customs_rules table: duty, restricted goods, and which required documents the customer actually provided or attached later."""
    req = s["req"]; ch, docs, restricted, duty, goods = _required_docs(req["hs_code"]); prov = req.get("docs_provided")
    have = set(str(prov).split(";")) if isinstance(prov, str) and prov else set(docs)                # legacy requests without a document list count as complete
    have |= _attached(s.get("quote_id", "")); missing = [d for d in docs if d not in have]
    risk = round(min(1, .12 + .35 * restricted + .18 * len(missing)), 2)
    return AgentResult("customs", 100 - risk * 100, risk_label(100 - risk * 100), f"Chapter {ch} ({goods}): {'restricted, ' if restricted else ''}{len(missing)} document(s) missing, duty {duty}%", {"risk": risk, "docs": docs, "missing": missing, "duty_pct": duty, "restricted": bool(restricted), "status": "Hold" if len(missing) or risk > .6 else "Ready", "goods": goods})
def margin_agent(s):
    p = s["results"]["pricing"].findings; cust = Q("SELECT priority_tier t, credit_risk cr FROM customers WHERE customer_id=?", (s["req"]["customer_id"],)); tier = cust.t[0] if len(cust) else "Silver"; cr = float(cust.cr[0]) if len(cust) else .1
    target = .20 + {"Platinum": -.02, "Gold": 0, "Silver": .02, "Bronze": .04}.get(tier, .02) + (.02 if cr > .3 else 0)
    cost = p["base"] + p["fuel"] + p["port_fees"]; op = p["base"] * .15; duty = p["base"] * s["results"]["customs"].findings["duty_pct"] / 100 * .1; total_cost = cost + op + duty
    price = total_cost / (1 - target); margin = (price - total_cost) / price * 100
    return AgentResult("margin", min(100, margin * 4), "Low" if margin >= 10 else "High", f"Target {target*100:.0f}% ({tier}), price ${price:,.0f}, margin {margin:.1f}%", {"price": round(price, 2), "margin_pct": round(margin, 1), "op_cost": round(op, 2), "target": target, "tier": tier, "cost": round(total_cost, 2)})
def document_agent(s):
    q = s["quote_id"]; out = os.path.join(DATA_DIR, "docs"); os.makedirs(out, exist_ok=True); req = s["req"]; rt = s["results"]["route"].findings; paths = []
    try:
        from reportlab.lib.pagesizes import A4; from reportlab.pdfgen import canvas
        for kind, lines in (("BillOfLading", [f"Shipper/Customer: {req['customer_id']}", f"Port of loading: {req['origin']}", f"Port of discharge: {req['dest']}", f"Cargo: {req['cargo']} ({req['hs_code']}), {req['weight_t']} t", f"Carrier: {rt['carriers'][0]}", f"Route: {rt['route']}, {rt['distance_nm']:,} nm"]),
                            ("PackingList", [f"Quote: {q}", f"Cargo type: {req['cargo']}", f"Gross weight: {req['weight_t']} t", f"Documents: {', '.join(s['results']['customs'].findings['docs'])}"])):
            p = os.path.join(out, f"{q}_{kind}.pdf"); cv = canvas.Canvas(p, pagesize=A4); cv.setFont("Helvetica-Bold", 16); cv.drawString(50, 800, f"{kind} - {q}"); cv.setFont("Helvetica", 11)
            for i, ln in enumerate(lines): cv.drawString(50, 770 - 22 * i, ln)
            cv.save(); paths.append(p)
    except Exception as e: return AgentResult("document", 60, "Medium", f"PDF generation unavailable: {e}")
    return AgentResult("document", 100, "Low", f"{len(paths)} PDFs generated", {"paths": paths})
def alert_agent(s):
    val = s["results"]["margin"].findings["price"]; w = s["results"]["weather"].findings; acts = []
    if val > 50000: acts.append({"type": "quote_approval", "target": s["quote_id"], "detail": f"Quote ${val:,.0f} exceeds $50,000: admin approval required", "severity": "Critical", "owner": "Admin"})
    elif s["results"]["margin"].findings["margin_pct"] < 10: acts.append({"type": "margin_review", "target": s["quote_id"], "detail": "Margin below 10%", "severity": "Warning"})
    if w["level"] == "High": acts.append({"type": "reroute", "target": s["quote_id"], "detail": f"High weather risk on lane (wind {w['wind']} m/s)", "severity": "Warning"})
    return AgentResult("alert", 100, "Low", f"{len(acts)} alerts raised", {}, acts)

def _compute(req, qid, cid=None, avoid=()):
    """The quote pipeline as a LangGraph graph: route -> weather -> [High risk: re-plan avoiding chokepoints, retry] -> pricing -> customs -> margin.
    cid=None runs it without logging (used by the Digital Twin and the quote preview)."""
    st = {"req": req, "results": {}, "avoid": tuple(avoid), "quote_id": qid, "tries": 0}
    def step(name, fn):
        t = time.time()
        try: res = fn(st)
        except Exception as e: res = AgentResult(name, 40, "High", f"error: {e}")
        st["results"][name] = res
        if cid: _log_run(cid, "quote", res, int((time.time() - t) * 1000))
        return res
    def replan():
        st["avoid"] = ("suez",); st["tries"] += 1
        if cid: _log_run(cid, "quote", AgentResult("re-plan", 100, "Low", f"weather High: re-routing avoiding Suez/Red Sea (try {st['tries']})"), 0)
    def need_replan(): return st["results"]["weather"].findings.get("level") == "High" and st["tries"] < 1 and st["results"]["route"].findings.get("alternatives") and "suez" not in st["avoid"]
    try:
        from langgraph.graph import StateGraph, END
        from typing import TypedDict
        class G(TypedDict, total=False): n: int
        gr = StateGraph(G)
        gr.add_node("route", lambda g: (step("route", route_agent), {})[1]); gr.add_node("weather", lambda g: (step("weather", weather_agent), {})[1]); gr.add_node("replan", lambda g: (replan(), {})[1])
        gr.add_node("rest", lambda g: ([step(n, f) for n, f in (("pricing", pricing_agent), ("customs", customs_agent), ("margin", margin_agent))], {})[1])
        gr.set_entry_point("route"); gr.add_edge("route", "weather"); gr.add_edge("replan", "route"); gr.add_conditional_edges("weather", lambda g: "replan" if need_replan() else "rest"); gr.add_edge("rest", END); gr.compile().invoke({"n": 0})
    except ImportError:
        step("route", route_agent); step("weather", weather_agent)
        if need_replan(): replan(); step("route", route_agent); step("weather", weather_agent)
        for nme, fn in (("pricing", pricing_agent), ("customs", customs_agent), ("margin", margin_agent)): step(nme, fn)
    return st, step

def _decide(st):
    res = st["results"]; req = st["req"]; wr, cr, rt = res["weather"].findings["risk"], res["customs"].findings["risk"], res["route"].findings["congestion"] / 3
    risk = round(wr * .4 + cr * .3 + rt * .3, 3); m = res["margin"].findings; val = m["price"]          # PRS v3: weather x0.40 + customs x0.30 + congestion x0.30
    decision = "On Hold" if res["customs"].findings["status"] == "Hold" or risk > .7 else "Pending Approval" if val > 50000 or m["margin_pct"] < 10 else "Approved"
    narr = f"Quote {st['quote_id']}: {req['origin']} to {req['dest']}, {res['route'].findings['route']}, {res['route'].findings['transit_days']} days. Risk score {risk} (weather {wr}, customs {cr}, congestion {rt:.2f}). Price ${val:,.0f} at {m['margin_pct']}% margin. Decision: {decision}."
    return risk, decision, val, m, narr

def preview_quote(req, avoid=()):
    """Price a lane without saving anything. The Digital Twin calls this on its database copy."""
    st, _ = _compute(dict(req), "PREVIEW", None, avoid)
    if not all(k in st["results"] and st["results"][k].findings for k in ("route", "weather", "pricing", "customs", "margin")): return None
    risk, decision, val, m, narr = _decide(st); r = st["results"]
    return {"price": val, "margin_pct": m["margin_pct"], "risk": risk, "decision": decision, "route": r["route"].findings["route"], "days": r["route"].findings["transit_days"], "weather": r["weather"].findings["level"], "narrative": narr}

def run_quote(req_id):
    from live_engine import _log_run, init_live_tables, act
    import sys; dom = sys.modules[__name__]; init_live_tables()
    r = Q("SELECT * FROM quote_requests WHERE req_id=?", (req_id,)); req = r.iloc[0].to_dict(); cid = f"Q{req_id}-" + datetime.datetime.now().strftime("%H%M%S"); qid = f"QTE-L{req_id:05d}"
    st, step = _compute(req, qid, cid); res = st["results"]
    if not all(k in res and res[k].findings for k in ("route", "weather", "pricing", "customs", "margin")):
        with _DB_LOCK, conn() as c: c.execute("UPDATE quote_requests SET status='error', result=? WHERE req_id=?", (json.dumps({k: v.summary for k, v in res.items()}), req_id)); return None
    risk, decision, val, m, narr = _decide(st); step("document", document_agent); step("alert", alert_agent); wr, cr, rt = res["weather"].findings["risk"], res["customs"].findings["risk"], res["route"].findings["congestion"] / 3
    with _DB_LOCK, conn() as c:
        sid = f"SHP-{qid[4:]}"
        c.execute("INSERT OR REPLACE INTO shipments(shipment_id,origin_port,dest_port,carrier,status,eta,weight_kg,hs_code,predicted_delay_risk) VALUES(?,?,?,?,?,?,?,?,?)", (sid, req["origin"], req["dest"], res["route"].findings["carriers"][0], "Customs Hold" if decision == "On Hold" else "Quoted", (datetime.date.today() + datetime.timedelta(days=int(res["route"].findings["transit_days"]) + 3)).isoformat(), req["weight_t"] * 1000, req["hs_code"], risk))
        p = res["pricing"].findings; c.execute("INSERT OR REPLACE INTO freight_quotes VALUES(?,?,?,?,?,?,?,?,?,?,?,?)", (qid, sid, req["customer_id"], res["route"].findings["carriers"][0], p["base"], round(p["base"] * .02, 2), round(p["base"] * res["customs"].findings["duty_pct"] / 100 * .1, 2), p["fuel"], val, m["margin_pct"], decision, now()[:10]))
        c.execute("UPDATE quote_requests SET status='done', quote_id=?, result=? WHERE req_id=?", (qid, json.dumps({"decision": decision, "risk": risk, "price": val, "narrative": narr, "missing": res["customs"].findings["missing"], "route": res["route"].findings["route"], "days": res["route"].findings["transit_days"], "weather": res["weather"].findings["level"], "margin_pct": m["margin_pct"], "parts": {"weather": wr, "customs": cr, "congestion": round(rt, 2)}}), req_id))
    dec = {"score": round((1 - risk) * 100, 1), "risk": risk_label((1 - risk) * 100), "narrative": narr, "actions": res["alert"].actions + ([{"type": "quote_hold", "target": qid, "detail": "Customs hold, missing: " + ", ".join(res["customs"].findings["missing"]), "severity": "Warning"}] if decision == "On Hold" and res["customs"].findings["missing"] else [])}
    act(dom, dec, cid, {}); return dec

def attach_doc(ref, doc, by="user"):
    """Attach a missing customs document to a quote (QTE-...) or a shipment (SHP-...). Re-runs the quote so the hold lifts when nothing is missing."""
    with _DB_LOCK, conn() as c:
        c.execute("CREATE TABLE IF NOT EXISTS shipment_docs(ref TEXT, doc TEXT, ts TEXT, by TEXT, UNIQUE(ref, doc))"); c.execute("INSERT OR IGNORE INTO shipment_docs VALUES(?,?,?,?)", (ref, doc, now(), by))
    emit("doc_attached", ref, {"doc": doc})
    if ref.startswith("QTE-L"):
        try: run_quote(int(ref[5:]))
        except Exception: pass

# ---------------------------------------------------------------- fleet monitoring cycle (the same agents, run continuously)
def route_watch(s):
    p = Q("SELECT port_name, congestion_index ci, avg_dwell_days dw FROM ports WHERE congestion_index>=4").sort_values("ci", ascending=False)
    n = Q("SELECT COUNT(*) n FROM shipments WHERE status='Port Congestion Delay'").n[0]; sc = 100 - min(60, len(p) * 3 + n * .5)
    return AgentResult("route_watch", sc, risk_label(sc), f"{len(p)} ports with congestion index >= 4, {n} shipments delayed", {"ports": p.head(8).port_name.tolist(), "n_congested": len(p), "delayed": int(n)}, [{"type": "congestion_notice", "target": r.port_name, "detail": f"Congestion index {r.ci:.1f}, dwell {r.dw:.1f} days", "severity": "Warning"} for r in p.head(6).itertuples()])
def weather_watch(s):
    v = Q("SELECT shipment_id, lat, lon, chain FROM vessels"); bad = []; acts = []
    for r in v.itertuples():
        w = weather_at(r.lat, r.lon, allow_live=False)
        if w["level"] == "High": bad.append(r.shipment_id)
    for sid in bad[:15]:
        acts.append({"type": "reroute", "target": sid, "detail": "High weather at current vessel position: alternate route proposed", "severity": "Critical"})
    sc = 100 - min(90, len(bad) / max(len(v), 1) * 400); return AgentResult("weather_watch", sc, risk_label(sc), f"{len(bad)} of {len(v)} vessels in High-risk weather ({len(storms())} active storms)", {"vessels_high": bad}, acts)
def customs_watch(s):
    """Shipments on customs hold: required documents (customs_rules by HS chapter) minus documents attached. Clears the hold when nothing is missing."""
    h = Q("SELECT shipment_id, hs_code FROM shipments WHERE status='Customs Hold'"); acts = []; cleared = 0
    for r in h.itertuples():
        ch, docs, restricted, duty, goods = _required_docs(r.hs_code); miss = [d for d in docs if d not in _attached(r.shipment_id)]
        if not miss and not restricted:
            with _DB_LOCK, conn() as c: c.execute("UPDATE shipments SET status='In Transit' WHERE shipment_id=?", (r.shipment_id,)); emit("customs_cleared", r.shipment_id); cleared += 1; continue
        acts.append({"type": "customs_docs", "target": r.shipment_id, "detail": f"Customs hold ({r.hs_code}, {goods}): missing {', '.join(miss[:4]) if miss else 'restricted goods clearance'}", "severity": "Warning"})
    n = len(h) - cleared; sc = 100 - min(70, n * 2)
    return AgentResult("customs_watch", sc, risk_label(sc), f"{n} shipments on customs hold, {cleared} cleared this cycle", {"holds": n, "cleared": cleared}, acts[:25])
def quote_watch(s):
    p = Q("SELECT q.quote_id, q.final_price, q.margin_pct, q.status, r.result FROM freight_quotes q LEFT JOIN quote_requests r ON r.quote_id=q.quote_id WHERE q.status IN ('Pending Approval','On Hold')"); acts = []
    for r in p.itertuples():
        if r.status == "Pending Approval": acts.append({"type": "quote_approval", "target": r.quote_id, "detail": f"Quote ${r.final_price:,.0f} exceeds $50,000: admin approval required" if r.final_price > 50000 else "Margin below 10%: approval required", "severity": "Critical" if r.final_price > 50000 else "Warning", "owner": "Admin"})
        else:
            try: miss = json.loads(r.result or "{}").get("missing", [])
            except Exception: miss = []
            acts.append({"type": "quote_hold", "target": r.quote_id, "detail": "Customs hold, missing: " + ", ".join(miss) if miss else "On hold: shipment risk score above 0.7", "severity": "Warning"})
    sc = 100 - min(60, len(p) * 3); return AgentResult("quote_watch", sc, risk_label(sc), f"{len(p)} quotes awaiting approval or on hold", {"pending": int((p.status == 'Pending Approval').sum()), "on_hold": int((p.status == 'On Hold').sum())}, acts[:30])

def anomaly_watch(s):
    """IsolationForest over quote economics (price/base ratio, margin, fuel share) to catch mispriced or tampered quotes."""
    from sklearn.ensemble import IsolationForest
    q = Q("SELECT quote_id, base_cost, fuel_surcharge, final_price, margin_pct FROM freight_quotes WHERE base_cost>0")
    if len(q) < 30: return AgentResult("anomaly_watch", 100, "Low", "waiting for more quotes", {"flagged": []})
    X = pd.DataFrame({"ratio": q.final_price / q.base_cost, "margin": q.margin_pct, "fuel": q.fuel_surcharge / q.base_cost}); iso = IsolationForest(contamination=.06, random_state=42).fit(X); q["flag"] = iso.predict(X) == -1
    bad = q[q.flag]; sc = 100 - len(bad) / len(q) * 250
    return AgentResult("anomaly_watch", round(max(sc, 0), 1), risk_label(sc), f"{len(bad)} of {len(q)} quotes look anomalous (IsolationForest)", {"flagged": bad.quote_id.tolist()},
        [{"type": "quote_anomaly", "target": r.quote_id, "detail": f"Price/base ratio {r.final_price / r.base_cost:.2f}, margin {r.margin_pct:.1f}%: review before sending", "severity": "Warning"} for r in bad.head(10).itertuples()])

AGENTS = {"route_watch": route_watch, "weather_watch": weather_watch, "customs_watch": customs_watch, "quote_watch": quote_watch, "anomaly_watch": anomaly_watch}
REPLAN = {"weather_watch": lambda r: ["route_watch"] if r.risk != "Low" else []}
def plan(req): return list(AGENTS)
def engine(s):
    r = s["results"]; score = float(np.mean([v.score for v in r.values()])); acts = [a for v in r.values() for a in v.actions]; order = {"Critical": 0, "Warning": 1, "Info": 2}; acts.sort(key=lambda a: order.get(a.get("severity"), 3))
    ev = {"fleet_health": round(score, 1), "vessels": int(Q("SELECT COUNT(*) n FROM vessels").n[0]), "agents": {k: {"score": v.score, "risk": v.risk, "summary": v.summary} for k, v in r.items()}, "critical": sum(a.get("severity") == "Critical" for a in acts)}
    n = f"Fleet health {ev['fleet_health']}/100 with {ev['vessels']} vessels tracked. " + " ".join(f"{k}: {v['summary']}." for k, v in ev["agents"].items())
    return {"score": round(score, 1), "risk": risk_label(score), "narrative": n, "actions": acts, "agents": ev["agents"], "evidence": ev}
def apply(a, c):
    if a["type"] == "reroute":
        sid = a["target"]; v = c.execute("SELECT lat, lon, chain FROM vessels WHERE shipment_id=?", (sid,)).fetchone()
        if v:
            ch = json.loads(v[2]); dest = ch[-1]
            if a.get("opt_idx") is not None:
                import alts; row = c.execute("SELECT s.weight_kg FROM shipments s WHERE shipment_id=?", (sid,)).fetchone(); al = alts.route_alternatives(sys.modules[__name__], (v[0], v[1]), dest, tonnes=float((row or [20000])[0] or 20000) / 1000, interp=interp); i = int(a["opt_idx"]); opt = al[i] if i < len(al) else al[0]
            else: opt = route_options((v[0], v[1]), dest, avoid=("storm",))[0]
            c.execute("UPDATE vessels SET chain=?, progress=0, route_name=? WHERE shipment_id=?", (json.dumps([(v[0], v[1])] + opt["chain"][1:]), ("rerouted: " if a.get("opt_idx") is not None else "rerouted around storm: ") + opt["name"], sid)); return f"vessel re-planned {opt['name']} ({opt['nm']:,} nm)"
    return ""
def decide(action_id, approve, by="user"):
    from live_engine import resolve_action
    with _DB_LOCK, conn() as c:
        t = c.execute("SELECT target FROM agent_actions WHERE action_id=?", (action_id,)).fetchone()
        if t and t[0].startswith("QTE"): c.execute("UPDATE freight_quotes SET status=? WHERE quote_id=?", ("Approved" if approve else "Rejected", t[0]))
    resolve_action(action_id, "approved" if approve else "rejected", f"by {by}")
def submit_request(origin, dest, cargo, weight_t, hs_code, customer_id, language="English", docs_provided=None):
    with _DB_LOCK, conn() as c: return c.execute("INSERT INTO quote_requests(ts,origin,dest,cargo,weight_t,hs_code,customer_id,language,docs_provided) VALUES(?,?,?,?,?,?,?,?,?)", (now(), origin, dest, cargo, weight_t, hs_code, customer_id, language, ";".join(docs_provided) if docs_provided is not None else None)).lastrowid
def feed_status():
    o = time.time() < _OFF[0]; return [("Open-Meteo weather and marine", "Offline: simulated winds in use" if o else "Real · live (cached 15 min)"), ("Vessel positions", "Simulated along real route waypoints (connect AIS via Data Feeder)"), ("Port congestion", "ML predicted, drifts live"), ("Freight-rate model data", "Bootstrap synthetic unless a Kaggle CSV is fed")]
def kpi_extra():
    return [("Vessels tracked", int(Q("SELECT COUNT(*) n FROM vessels").n[0])), ("Quotes awaiting approval", int(Q("SELECT COUNT(*) n FROM freight_quotes WHERE status='Pending Approval'").n[0]))]

def _prewarm():
    try:
        import seanav; seanav.sea()
    except Exception: pass
import threading as _th; _th.Thread(target=_prewarm, daemon=True).start()


In [ ]:
%%writefile /content/app/entities.py
"""One description of any entity (outlet, staff, quote, shipment, vessel, port): master data, relations, and everything flagged on it. Feeds the drawer, the knowledge graph and the Copilot."""
import json
from blocks import Q, clean

def _tickets(e, n=30):
    return Q("SELECT action_id id, type, severity, status, detail, owner, assignee, due FROM agent_actions WHERE target=? ORDER BY action_id DESC LIMIT ?", (e, n))
def latest_out(agent):
    r = Q("SELECT output FROM agent_runs WHERE agent=? ORDER BY run_id DESC LIMIT 1", (agent,))
    try: return json.loads(r.output[0]) if len(r) else {}
    except Exception: return {}
def kind_of(e):
    e = str(e)
    if e.startswith("OUT-"): return "outlet"
    if e.startswith("QTE"): return "quote"
    if e.startswith("SHP"): return "shipment"
    if e.startswith("EMP") or e.startswith("STF"): return "staff"
    return "port"

def describe(kind, dom, e):
    """-> {"kind","info":{label:value},"rel":[(name,kind,tip)], "tickets": DataFrame}"""
    k = kind_of(e); info, rel = {}, []
    if kind == "Franchise" and k == "outlet":
        o = Q("SELECT * FROM outlets WHERE outlet_id=?", (e,))
        if len(o):
            r = o.iloc[0]; info = {"Outlet": r.outlet_name, "Location": r.location, "Revenue": f"{r.revenue:,.0f}", "Customer satisfaction": round(float(r.customer_satisfaction), 2), "Headcount": int(r.staff_headcount)}
            for a in ("outlet", "inventory", "staff", "marketing", "audit", "feedback"):
                v = latest_out(a).get("per_outlet", {}).get(e)
                if v is not None: rel.append((f"{a} agent: {v:.0f}", "agent", f"{a} score for {e}"))
            for s in Q("SELECT staff_id, predicted_attrition_prob p FROM staff WHERE outlet_id=? ORDER BY p DESC LIMIT 4", (e,)).itertuples(): rel.append((s.staff_id, "staff", f"attrition {s.p:.0%}"))
            for s in Q("SELECT sku_name, stockout_risk_prob p FROM inventory WHERE outlet_id=? ORDER BY p DESC LIMIT 4", (e,)).itertuples(): rel.append((s.sku_name, "sku", f"stock-out risk {s.p:.0%}"))
            for s in Q("SELECT campaign_name, channel FROM marketing WHERE outlet_id=? AND end_date>=date('now')", (e,)).itertuples(): rel.append((s.campaign_name, "campaign", f"live via {s.channel}"))
    elif kind == "Freight" and k == "quote":
        q = Q("SELECT * FROM freight_quotes WHERE quote_id=?", (e,))
        if len(q):
            r = q.iloc[0]; info = {"Quote": e, "Price": f"${r.final_price:,.0f}", "Margin": f"{r.margin_pct}%", "Status": r.status}
            for n, tip in ((r.shipment_id, "shipment"), (r.customer_id, "customer"), (r.carrier, "carrier")): rel.append((n, tip, tip))
    elif kind == "Freight" and k == "shipment":
        s = Q("SELECT * FROM shipments WHERE shipment_id=?", (e,))
        if len(s):
            r = s.iloc[0]; info = {"Shipment": e, "Route": f"{r.origin_port} to {r.dest_port}", "Carrier": r.carrier, "Status": r.status, "Delay risk": r.predicted_delay_risk}
            for n, tip in ((r.origin_port, "port"), (r.dest_port, "port"), (r.carrier, "carrier")): rel.append((n, tip, tip))
            v = Q("SELECT lat, lon, route_name, progress FROM vessels WHERE shipment_id=?", (e,))
            if len(v): info["Vessel"] = f"{v.lat[0]:.1f}, {v.lon[0]:.1f} · {v.route_name[0]} · {v.progress[0]:.0%}"
    elif kind == "Freight":
        p = Q("SELECT * FROM ports WHERE port_name=?", (e,))
        if len(p):
            r = p.iloc[0]; info = {"Port": e, "Country": r.country, "Congestion index": round(float(r.congestion_index), 2), "Dwell days": r.avg_dwell_days}
            for s in Q("SELECT shipment_id FROM shipments WHERE origin_port=? OR dest_port=? LIMIT 6", (e, e)).itertuples(): rel.append((s.shipment_id, "shipment", "via port"))
    return {"kind": k, "info": {a: clean(b) for a, b in info.items()}, "rel": rel, "tickets": _tickets(e)}

def facts(kind, dom, text):
    import re
    out = []
    for e in dict.fromkeys(m.upper() for m in re.findall(r"\b(OUT-\d+|QTE-L\d+|SHP-[A-Z0-9-]+)\b", text, re.I)):
        d = describe(kind, dom, e); t = d["tickets"]; t = t[t.status.isin(["open", "proposed"])].head(8)
        if d["info"] or len(t): out.append(f"{e}: " + "; ".join(f"{k}={v}" for k, v in d["info"].items()) + (f". Related: {', '.join(n for n, _, _ in d['rel'][:8])}." if d["rel"] else "") + (" Open tickets: " + " | ".join(f"{r.type} ({r.severity}): {r.detail}" for r in t.itertuples()) if len(t) else " No open tickets."))
    return out


In [ ]:
%%writefile /content/app/gpu.py
"""Minimal stub (no language model in this version). Kept so hardening.py can report health."""
def status(): return {"cuda": False, "gpu": None, "models": {}, "stats": {}}


In [ ]:
%%writefile /content/app/hardening.py
"""Production hardening for the FastAPI app: request ids + JSON access log, security headers (CSP, frame, permissions), per-IP rate limits on auth routes,
a global JSON error handler that never leaks tracebacks, and Prometheus-style metrics. Install with hardening.install(app)."""
import os, time, json, uuid, threading, logging, traceback, collections
from logging.handlers import RotatingFileHandler
from fastapi import Request
from fastapi.responses import JSONResponse, PlainTextResponse
import config

_LOG = logging.getLogger("access"); _LOG.setLevel(logging.INFO); _LOG.propagate = False
try:
    _h = RotatingFileHandler(os.path.join(config.DATA_DIR, "access.log"), maxBytes=2_000_000, backupCount=3); _h.setFormatter(logging.Formatter("%(message)s")); _LOG.addHandler(_h)
except Exception: pass
_ERR = logging.getLogger("errors"); _ERR.propagate = False
try:
    _e = RotatingFileHandler(os.path.join(config.DATA_DIR, "errors.log"), maxBytes=1_000_000, backupCount=2); _e.setFormatter(logging.Formatter("%(asctime)s %(message)s")); _ERR.addHandler(_e)
except Exception: pass

M = {"requests": collections.Counter(), "status": collections.Counter(), "lat_sum": collections.Counter(), "lat_n": collections.Counter(), "errors": 0, "limited": 0, "started": time.time()}
_L = threading.Lock(); _BUCKET = {}
CSP = "default-src 'self'; img-src 'self' data: blob:; style-src 'self' 'unsafe-inline' https://fonts.googleapis.com; script-src 'self' 'unsafe-inline'; connect-src 'self'; font-src 'self' data: https://fonts.gstatic.com; frame-ancestors 'none'; base-uri 'self'; form-action 'self'"
# path prefix -> (max requests, per seconds). Everything else: 900 per minute per IP.
LIMITS = {"/api/auth/login": (15, 60), "/api/auth/signup": (8, 600), "/api/auth/forgot": (6, 300), "/api/auth/reset": (10, 600), "/api/auth/question": (15, 300), "/api/profile/password": (8, 600), "/api/copilot": (40, 60), "/api/translate": (60, 60)}
def _ip(r):
    f = r.headers.get("x-forwarded-for"); return (f.split(",")[0].strip() if f else (r.client.host if r.client else "?"))
def _allow(ip, path):
    key, (n, per) = next(((p, v) for p, v in LIMITS.items() if path.startswith(p)), ("*", (900, 60))); k = (ip, key); now = time.time()
    with _L:
        q = _BUCKET.setdefault(k, collections.deque())
        while q and now - q[0] > per: q.popleft()
        if len(q) >= n: return False, int(per - (now - q[0])) + 1
        q.append(now)
        if len(_BUCKET) > 20000: _BUCKET.clear()
    return True, 0
def _route(path):
    parts = path.split("/")
    return "/".join(parts[:3]) if path.startswith("/api/") else "static"

def install(app, kind="app"):
    @app.middleware("http")
    async def guard(request: Request, call_next):
        rid = request.headers.get("x-request-id") or uuid.uuid4().hex[:12]; t0 = time.time(); path = request.url.path
        if os.environ.get("RATE_LIMIT_OFF") != "1" and (path.startswith("/api/") and request.method != "GET" or path.startswith("/api/login")):
            ok, wait = _allow(_ip(request), path)
            if not ok:
                M["limited"] += 1; return JSONResponse({"ok": False, "msg": f"Too many requests. Try again in {wait}s.", "request_id": rid}, status_code=429, headers={"Retry-After": str(wait), "X-Request-ID": rid})
        try: r = await call_next(request)
        except Exception as e:
            M["errors"] += 1; _ERR.error("%s %s %s\n%s", rid, request.method, path, traceback.format_exc())
            return JSONResponse({"ok": False, "msg": "Something went wrong on the server. Quote this id to the admin: " + rid, "request_id": rid}, status_code=500, headers={"X-Request-ID": rid})
        ms = (time.time() - t0) * 1000; rt = _route(path)
        with _L: M["requests"][rt] += 1; M["status"][r.status_code // 100] += 1; M["lat_sum"][rt] += ms; M["lat_n"][rt] += 1
        h = r.headers; h["X-Request-ID"] = rid; h["X-Frame-Options"] = "DENY"; h["Content-Security-Policy"] = CSP; h["Permissions-Policy"] = "microphone=(self), camera=(), geolocation=()"; h["Cross-Origin-Opener-Policy"] = "same-origin"
        if request.headers.get("x-forwarded-proto") == "https": h["Strict-Transport-Security"] = "max-age=31536000"
        if path.startswith("/api/") and path not in ("/api/stream", "/api/pulse"): _LOG.info(json.dumps({"t": round(t0), "id": rid, "ip": _ip(request), "m": request.method, "p": path, "s": r.status_code, "ms": round(ms)}))
        return r

    @app.get("/api/ready")
    def ready(): return {"ok": True, "kind": kind, "uptime_s": round(time.time() - M["started"])}

    @app.get("/api/metrics")
    def metrics(request: Request):
        """Prometheus text format. Set METRICS_TOKEN to require ?token=..., otherwise only logged-in admins (cookie) may read it."""
        tok = os.environ.get("METRICS_TOKEN")
        if tok and request.query_params.get("token") != tok: return PlainTextResponse("forbidden", status_code=403)
        if not tok:
            try:
                import auth as _A; u, _e = _A.user_from_request(request)
                if not u or u.get("role") != "Admin": return PlainTextResponse("forbidden", status_code=403)
            except Exception: return PlainTextResponse("forbidden", status_code=403)
        L = ["# TYPE app_requests_total counter"] + [f'app_requests_total{{route="{k}"}} {v}' for k, v in M["requests"].items()] + ["# TYPE app_responses_total counter"] + [f'app_responses_total{{class="{k}xx"}} {v}' for k, v in M["status"].items()]
        L += ["# TYPE app_latency_ms_avg gauge"] + [f'app_latency_ms_avg{{route="{k}"}} {M["lat_sum"][k] / max(1, M["lat_n"][k]):.1f}' for k in M["lat_n"]] + [f"app_errors_total {M['errors']}", f"app_rate_limited_total {M['limited']}", f"app_uptime_seconds {time.time() - M['started']:.0f}"]
        try:
            import live_engine as LE; c = LE.conn(); L += [f"app_open_tickets {c.execute('SELECT COUNT(*) FROM agent_actions WHERE status=?', ('open',)).fetchone()[0]}"]
        except Exception: pass
        try:
            import gpu; g = gpu.status(); L += [f'app_model_ready{{model="{k}"}} {1 if v["state"] == "ready" else 0}' for k, v in g["models"].items()]
        except Exception: pass
        return PlainTextResponse("\n".join(L) + "\n")


In [ ]:
%%writefile /content/app/intent_router.py
import re as _re_live
import pandas as pd
import re, math
from db import get_conn

def df_to_markdown_safe(df):
    if df is None or df.empty: return ""
    try: return df.to_markdown(index=False)
    except: pass
    headers = list(df.columns)
    lines = ["| " + " | ".join([str(h) for h in headers]) + " |"]
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for _, row in df.iterrows():
        vals = [str(v) if v is not None else "" for v in row.values]
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)

INTENT_MAP = {
    "ports": ["port", "ports", "harbor", "terminal", "congestion", "dwell"],
    "shipments": ["shipment", "shipments", "cargo", "carrier", "freight", "teu", "transit"],
    "quotes": ["quote", "spot", "price", "rate", "baf", "margin", "cost"],
    "weather": ["weather", "typhoon", "storm", "cyclone", "wind", "wave"],
    "customs": ["customs", "tariff", "duty", "hs code", "import", "export"],
    "alerts": ["alert", "incident", "delay", "disruption", "hold"]
}

def classify_intent(query):
    q = query.lower()
    for intent, keywords in INTENT_MAP.items():
        if any(kw in q for kw in keywords):
            return intent
    return "general"

def handle_freight_intent(query):
    q_low = query.lower()
    try:
        with get_conn() as conn:
            # 1. Ports / Terminals
            if any(k in q_low for k in ["port", "ports", "harbor", "terminal", "congestion"]):
                tot_ports = conn.execute("SELECT COUNT(*) FROM ports;").fetchone()[0]
                df_ports = pd.read_sql("SELECT port_name as 'Port Name', country as 'Country', region as 'Region', congestion_index as 'Congestion (1-5)', avg_dwell_days as 'Avg Dwell Days' FROM ports ORDER BY congestion_index DESC LIMIT 5;", conn)
                return f"CRITICAL FACT: Total Monitored Ports = {tot_ports}.\n\n[Sample Subset Preview - First 5 Records Only]:\n{df_to_markdown_safe(df_ports)}", "Ports Ledger DB"

            # 2. Shipments / Cargo
            if any(k in q_low for k in ["shipment", "shipments", "carrier", "cargo"]):
                tot_shipments = conn.execute("SELECT COUNT(*) FROM shipments;").fetchone()[0]
                df_ship = pd.read_sql("SELECT shipment_id, origin_port, dest_port, carrier, status, predicted_delay_risk FROM shipments ORDER BY predicted_delay_risk DESC LIMIT 5;", conn)
                return f"CRITICAL FACT: Total Active Shipments = {tot_shipments}.\n\n[Sample Subset Preview - First 5 Records Only]:\n{df_to_markdown_safe(df_ship)}", "Shipments Ledger DB"

            # 3. Quotes / Margins
            if any(k in q_low for k in ["quote", "quotes", "spot", "margin", "price", "rate"]):
                tot_quotes = conn.execute("SELECT COUNT(*) FROM freight_quotes;").fetchone()[0]
                df_q = pd.read_sql("SELECT quote_id, shipment_id, ROUND(base_cost, 0) AS base_cost_usd, ROUND(final_price, 0) AS final_price_usd, ROUND(margin_pct, 1) AS margin_pct FROM freight_quotes ORDER BY margin_pct DESC LIMIT 5;", conn)
                return f"CRITICAL FACT: Total Active Freight Quotes = {tot_quotes}.\n\n[Sample Subset Preview - First 5 Records Only]:\n{df_to_markdown_safe(df_q)}", "Freight Quotes DB"

            # 4. Weather
            if any(k in q_low for k in ["weather", "typhoon", "storm", "wind", "wave", "cyclone", "risk", "risks"]):
                tot_weather = conn.execute("SELECT COUNT(*) FROM weather_risks;").fetchone()[0]
                df_w = pd.read_sql("SELECT port_name AS 'Port Name', current_severity AS 'Severity (1-5)', forecast AS 'Forecast', wind_speed AS 'Wind (Knots)', wave_height AS 'Wave (m)' FROM weather_risks ORDER BY current_severity DESC LIMIT 5;", conn)
                return f"CRITICAL FACT: Total Monitored Weather Risk Hubs = {tot_weather}.\n\n[Sample Subset Preview - First 5 Records Only]:\n{df_to_markdown_safe(df_w)}", "Weather Risks DB"
    except Exception:
        pass
    return None


_AGENTIC_Q = _re_live.compile(r"(need[s]? attention|what should i do|action plan|prioriti[sz]e|overall (status|health)|run (the )?agents?|full analysis|full status|why is .{0,40}(low|down|failing|underperform|risky)|what is going wrong)")
def _agentic_query(query):
    """Open-ended 'what should I do / what needs attention' questions run the whole agent graph instead of a single table lookup."""
    if not _AGENTIC_Q.search(query.lower()): return None
    try:
        from live_engine import run_cycle
        import domain_freight as _dom
        dec = run_cycle(_dom, "copilot", query)
        top = "\n".join(f"- **{a['severity']}** · {a['type']} · {a['target']}: {a['detail']}" for a in dec["actions"][:8])
        return f"### Agent orchestration result\n\n{dec['narrative']}\n\n**Top actions**\n{top}", "Agent Orchestrator (live)"
    except Exception:
        return None

def _run_centralized_orig(query):
    freight = handle_freight_intent(query)
    if freight:
        return freight

    try:
        from rag_engine import answer_with_citation
        ctx, src = answer_with_citation(query)
        return ctx, src
    except Exception:
        return f"Retrieved enterprise freight intelligence for query: '{query}'.", "General Knowledge Index"

def run_grounded_query(query):
    return run_centralized_brain_query(query)

def text_to_sql(query):
    return run_centralized_brain_query(query)

def run_centralized_brain_query(query):
    _ag = _agentic_query(query)
    return _ag if _ag else _run_centralized_orig(query)

In [ ]:
%%writefile /content/app/live_engine.py
"""Shared live agent runtime (FranchiseOps AI + FreightQuote AI).
Domain modules (domain_franchise.py / domain_freight.py) plug in agents, a simulator and action handlers."""
import os, re, json, time, sqlite3, threading, datetime, smtplib, ssl
from dataclasses import dataclass, field
from email.message import EmailMessage
import requests
from config import DB_PATH

_DB_LOCK = threading.RLock()
_TL = threading.local()
def conn():
    """One SQLite connection per thread: nested helpers (emit/set_state) share the caller's transaction instead of deadlocking."""
    c = getattr(_TL, "c", None)
    if c is None:
        c = sqlite3.connect(DB_PATH, timeout=30); c.execute("PRAGMA journal_mode=WAL;"); c.execute("PRAGMA busy_timeout=30000;"); _TL.c = c
    return c
def now(): return datetime.datetime.now().isoformat(timespec="seconds")

@dataclass
class AgentResult:
    agent: str
    score: float = 100.0
    risk: str = "Low"
    summary: str = ""
    findings: dict = field(default_factory=dict)
    actions: list = field(default_factory=list)   # {"type","target","detail","severity","owner"?,"sla_h"?}
    confidence: float = 0.8

def risk_label(score): return "High" if score < 60 else "Medium" if score < 80 else "Low"

def init_live_tables():
    with _DB_LOCK, conn() as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS agent_runs(run_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, cycle_id TEXT, trigger_src TEXT, agent TEXT, risk TEXT, score REAL, latency_ms INT, summary TEXT, output TEXT);
        CREATE TABLE IF NOT EXISTS agent_actions(action_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, type TEXT, target TEXT, detail TEXT, severity TEXT, owner TEXT, status TEXT DEFAULT 'open', due TEXT, resolved_ts TEXT, source_cycle TEXT, note TEXT);
        CREATE TABLE IF NOT EXISTS live_events(event_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, etype TEXT, entity TEXT, payload TEXT);
        CREATE TABLE IF NOT EXISTS live_state(k TEXT PRIMARY KEY, v TEXT, ts TEXT);
        CREATE TABLE IF NOT EXISTS decisions(decision_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, cycle_id TEXT, trigger_src TEXT, score REAL, risk TEXT, narrative TEXT, evidence TEXT);
        CREATE TABLE IF NOT EXISTS audit_log(audit_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, actor TEXT, event TEXT, target TEXT, detail TEXT);
        CREATE TABLE IF NOT EXISTS ticket_comments(comment_id INTEGER PRIMARY KEY AUTOINCREMENT, action_id INT, ts TEXT, author TEXT, body TEXT);
        CREATE INDEX IF NOT EXISTS ix_act_status ON agent_actions(status, type, target);
        CREATE INDEX IF NOT EXISTS ix_runs_agent ON agent_runs(agent, run_id);
        CREATE INDEX IF NOT EXISTS ix_runs_cycle ON agent_runs(cycle_id);
        CREATE INDEX IF NOT EXISTS ix_cm_act ON ticket_comments(action_id);
        CREATE INDEX IF NOT EXISTS ix_audit_ts ON audit_log(audit_id);
        """)
        for col in ("assignee TEXT", "approval TEXT"):
            try: c.execute(f"ALTER TABLE agent_actions ADD COLUMN {col}")
            except sqlite3.OperationalError: pass
        for t, cols in (("sales", "outlet_id, ts"), ("attendance", "outlet_id, ts"), ("feedback", "outlet_id, date")):
            try: c.execute(f"CREATE INDEX IF NOT EXISTS ix_{t}_o ON {t}({cols})")
            except sqlite3.OperationalError: pass

def audit(actor, event, target="", detail=""):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO audit_log(ts,actor,event,target,detail) VALUES(?,?,?,?,?)", (now(), str(actor), event, str(target), str(detail)[:500]))

RETAIN = {"sales": ("ts", 6000), "attendance": ("ts", 3000), "feedback": ("date", 2000), "agent_runs": ("ts", 4000), "decisions": ("ts", 800), "live_events": ("ts", 2000), "audit_log": ("ts", 20000)}
def maintenance():
    """Keep the live DB bounded: prune to the newest N rows per table (never touches seed/master data)."""
    n = 0
    with _DB_LOCK, conn() as c:
        for t, (col, keep) in RETAIN.items():
            try:
                pk = c.execute(f"PRAGMA table_info({t})").fetchall()
                if not pk: continue
                if t in ("sales", "attendance"):
                    cnt = c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
                    if cnt > keep * 4: n += c.execute(f"DELETE FROM {t} WHERE rowid IN (SELECT rowid FROM {t} ORDER BY rowid ASC LIMIT ?)", (cnt - keep * 2,)).rowcount
                else:
                    cnt = c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
                    if cnt > keep * 2: n += c.execute(f"DELETE FROM {t} WHERE rowid IN (SELECT rowid FROM {t} ORDER BY rowid ASC LIMIT ?)", (cnt - keep,)).rowcount
            except sqlite3.OperationalError: pass
    return n

def set_state(k, v):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT OR REPLACE INTO live_state(k,v,ts) VALUES(?,?,?)", (k, json.dumps(v, default=str), now()))
def get_state(k, default=None):
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT v FROM live_state WHERE k=?", (k,)).fetchone()
    return json.loads(r[0]) if r else default
def emit(etype, entity, payload=None):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO live_events(ts,etype,entity,payload) VALUES(?,?,?,?)", (now(), etype, str(entity), json.dumps(payload or {}, default=str)))

# ---------------- notifications (real channels, all optional via env) ----------------
_SENT = {}
def _nk(s): return re.sub(r"\d+(?:\.\d+)?", "#", str(s))
def notify(dom, target, severity, category, message):
    with _DB_LOCK, conn() as c:
        c.execute(f"INSERT INTO alerts ({dom.ENTITY_COL}, severity, category, message, date) VALUES (?,?,?,?,?)",
                  (target, severity, category, message, datetime.datetime.now().strftime("%Y-%m-%d %H:%M")))
    if severity not in ("Critical", "High"): return
    key = (target, category, _nk(message))
    if time.time() - _SENT.get(key, 0) < 3600: return          # no alert spam
    _SENT[key] = time.time()
    text = f"[{dom.NAME}] {severity} · {target} · {category}: {message}"
    for var in ("SLACK_WEBHOOK_URL", "TEAMS_WEBHOOK_URL"):
        if os.environ.get(var):
            try: requests.post(os.environ[var], json={"text": text}, timeout=5)
            except Exception: pass
    em, pw, to = os.environ.get("EMAIL_ID"), (os.environ.get("EMAIL_APP_PASSWORD") or os.environ.get("EMAIL_PASSWORD") or "").replace(" ", ""), os.environ.get("ALERT_EMAIL_TO")
    if em and pw and to:
        try:
            m = EmailMessage(); m["Subject"] = text[:90]; m["From"] = em; m["To"] = to; m.set_content(text)
            with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=ssl.create_default_context(), timeout=15) as s:
                s.login(em, pw); s.send_message(m)
        except Exception: pass
    def _mail_subscribers():
        try:
            import mailer
            if not mailer.configured(): return
            with conn() as cc: rows = [r[0] for r in cc.execute("SELECT email FROM user_prefs WHERE email_alerts=1")]
            for e in rows: mailer.send(e, text[:90], text)
        except Exception: pass
    threading.Thread(target=_mail_subscribers, daemon=True).start()
    sid, tok, frm, tph = (os.environ.get(k) for k in ("TWILIO_SID", "TWILIO_TOKEN", "TWILIO_FROM", "ALERT_SMS_TO"))
    if sid and tok and frm and tph and getattr(dom, "SMS", False):
        try: requests.post(f"https://api.twilio.com/2010-04-01/Accounts/{sid}/Messages.json", auth=(sid, tok),
                           data={"From": frm, "To": tph, "Body": text[:300]}, timeout=8)
        except Exception: pass

# ---------------- action layer ----------------
SLA_H = {"Critical": 24, "Warning": 72, "Info": 168}
def policy(dom):
    """Autonomy policy per action type: 'auto' (agent executes) or 'approval' (human approves first). Admin-editable."""
    return {}      # no approval queue in this version: agents act on their own
def set_policy(t, mode, actor="admin"):
    p = get_state("policy", {}) or {}; p[t] = mode; set_state("policy", p); audit(actor, "policy", t, mode)
# ---------------------------------------------------------------- multi-level approval chains
# Each ticket that needs a human gets an ordered chain of steps (table approval_steps). Every step names a role; the next step opens only after the previous one is approved.
# Four-eyes rule: nobody can approve two steps of the same ticket, and nobody can approve a request they raised. Any step rejection rejects the whole ticket.
LADDER = ["Operations Manager", "Franchise Owner", "Admin"]
def _role_ok(role, need): return role == "Admin" or role == need or (need == "Operations Manager" and role == "Franchise Owner")
def chain_for(severity, typ, levels=None):
    cfg = get_state("approval_chain", {}) or {}
    if levels: return LADDER[:1] + (["Admin"] if int(levels) >= 2 else [])
    if typ in cfg and cfg[typ]: return list(cfg[typ])
    return ["Operations Manager", "Admin"] if severity == "Critical" else ["Operations Manager"]
def _steps_ddl(c): c.execute("CREATE TABLE IF NOT EXISTS approval_steps(action_id INT, step INT, role TEXT, status TEXT DEFAULT 'pending', by TEXT, ts TEXT, note TEXT, PRIMARY KEY(action_id, step))")
def _ensure_chain(c, aid, severity, typ):
    _steps_ddl(c)
    if not c.execute("SELECT 1 FROM approval_steps WHERE action_id=?", (aid,)).fetchone():
        for i, r in enumerate(chain_for(severity, typ)): c.execute("INSERT INTO approval_steps(action_id,step,role) VALUES(?,?,?)", (aid, i + 1, r))
def chains(ids):
    """{action_id: [{step, role, status, by}]} for the ticket list (creates the default chain for tickets that have none yet)."""
    out = {}
    if not ids: return out
    with _DB_LOCK, conn() as c:
        _steps_ddl(c)
        for aid, sev, typ in c.execute("SELECT action_id, severity, type FROM agent_actions WHERE action_id IN (%s) AND (status='proposed' OR type='quote_approval')" % ",".join("?" * len(ids)), list(ids)).fetchall(): _ensure_chain(c, aid, sev, typ)
        for r in c.execute("SELECT action_id, step, role, status, by FROM approval_steps WHERE action_id IN (%s) ORDER BY action_id, step" % ",".join("?" * len(ids)), list(ids)).fetchall(): out.setdefault(r[0], []).append({"step": r[1], "role": r[2], "status": r[3], "by": r[4] or ""})
    return out
def next_role(chain):
    for s in chain or []:
        if s["status"] == "pending": return s["role"]
    return None
def advance(aid, actor, role, approve=True, note=""):
    """Record one decision. Returns ('final'|'partial'|'rejected'|'denied'|'missing', message). Does not execute anything."""
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT severity, type, note FROM agent_actions WHERE action_id=? AND (status='proposed' OR type='quote_approval')", (aid,)).fetchone()
        if not r: return "missing", "nothing to approve on this ticket"
        _ensure_chain(c, aid, r[0], r[1]); steps = c.execute("SELECT step, role, status, by FROM approval_steps WHERE action_id=? ORDER BY step", (aid,)).fetchall()
        cur = next((s for s in steps if s[2] == "pending"), None)
        if not cur: return "missing", "this ticket has no pending approval step"
        if not _role_ok(role, cur[1]): return "denied", f"step {cur[0]} needs a {cur[1]} (you are {role})"
        if approve:
            if actor in [s[3] for s in steps if s[3]]: return "denied", "four-eyes rule: you already approved an earlier step of this ticket"
            if f"requested by {actor}" in (r[2] or ""): return "denied", "four-eyes rule: you cannot approve your own request"
        c.execute("UPDATE approval_steps SET status=?, by=?, ts=?, note=? WHERE action_id=? AND step=?", ("approved" if approve else "rejected", actor, now(), note, aid, cur[0]))
        left = [s for s in steps if s[2] == "pending" and s[0] != cur[0]]
        if not approve: return "rejected", f"rejected at step {cur[0]} ({cur[1]})"
        if left:
            c.execute("UPDATE agent_actions SET note=COALESCE(note,'')||? WHERE action_id=?", (f" step {cur[0]}/{len(steps)} approved by {actor};", aid)); audit(actor, "approve_step", aid, f"step {cur[0]}/{len(steps)} {note}".strip())
            return "partial", f"step {cur[0]} of {len(steps)} approved; waiting for {left[0][1]}"
    return "final", "all steps approved"
def request_approval(dom, typ, target, detail, severity, requester, levels=None):
    """Anyone with access can ask for a decision: creates a proposed ticket with its own approval chain."""
    a = {"type": "manual_request", "target": target, "detail": detail, "request": typ}
    with _DB_LOCK, conn() as c:
        cur = c.execute("INSERT INTO agent_actions(ts,type,target,detail,severity,owner,status,note,approval) VALUES(?,?,?,?,?,?,'proposed',?,?)", (now(), "manual_request", target, f"{typ}: {detail}", severity, requester, f"requested by {requester};", json.dumps(a)))
        aid = cur.lastrowid; _steps_ddl(c)
        for i, r in enumerate(chain_for(severity, "manual_request", levels)): c.execute("INSERT INTO approval_steps(action_id,step,role) VALUES(?,?,?)", (aid, i + 1, r))
    audit(requester, "request_approval", f"{typ}:{target}", detail[:120]); return aid
def approve_action(dom, aid, actor="user", approve=True, note="", role="Admin"):
    """Human-in-the-loop: runs the approval chain; the action executes only when the last step is approved. Returns (state, message)."""
    st, msg = advance(aid, actor, role, approve, note)
    if st in ("denied", "missing", "partial"): return st, msg
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT type,target,detail,severity,approval FROM agent_actions WHERE action_id=?", (aid,)).fetchone()
        if st == "final":
            a = json.loads(r[4] or "{}") or {"type": r[0], "target": r[1], "detail": r[2]}
            try: res = dom.apply(a, c) or ""
            except Exception as ex: res = f"apply failed: {ex}"
            if r[0] == "manual_request": c.execute("UPDATE agent_actions SET status='resolved', resolved_ts=?, note=COALESCE(note,'')||?, approval='approved' WHERE action_id=?", (now(), f" approved by {actor}; request granted", aid))
            else: c.execute("UPDATE agent_actions SET status='open', note=COALESCE(note,'')||?, approval=? WHERE action_id=?", (f" approved by {actor}; {res}", "approved", aid))
        else:
            c.execute("UPDATE agent_actions SET status='rejected', resolved_ts=?, note=COALESCE(note,'')||?, approval='rejected' WHERE action_id=?", (now(), f" rejected by {actor} {note}", aid))
    audit(actor, "approve" if approve else "reject", f"{r[0]}:{r[1]}", note); return st, msg
def assign_action(aid, assignee, actor="user"):
    with _DB_LOCK, conn() as c: c.execute("UPDATE agent_actions SET assignee=? WHERE action_id=?", (assignee, aid))
    audit(actor, "assign", aid, assignee)
def comment_action(aid, author, body):
    with _DB_LOCK, conn() as c: c.execute("INSERT INTO ticket_comments(action_id,ts,author,body) VALUES(?,?,?,?)", (now(), author, body) if False else (aid, now(), author, body))
    audit(author, "comment", aid, body[:80])
def act(dom, decision, cid, results):
    raised = set(); sent = 0; extra = 0; pol = policy(dom)
    for a in decision.get("actions", []):
        key = (a["type"], str(a["target"]), _nk(a["detail"])); raised.add(key)
        sev = a.get("severity", "Warning")
        due = (datetime.datetime.now() + datetime.timedelta(hours=a.get("sla_h", SLA_H.get(sev, 72)))).isoformat(timespec="seconds")
        with _DB_LOCK, conn() as c:
            hit = next((r[0] for r in c.execute("SELECT action_id, detail FROM agent_actions WHERE type=? AND target=? AND status IN ('open','proposed')", key[:2]).fetchall() if _nk(r[1].replace("OVERDUE: ", "")) == key[2]), None)
            if hit:
                c.execute("UPDATE agent_actions SET detail=CASE WHEN detail LIKE 'OVERDUE:%' THEN 'OVERDUE: '||? ELSE ? END WHERE action_id=?", (a["detail"], a["detail"], hit)); continue
            note = ""; status = "open"; appr = None
            if pol.get(a["type"]) == "approval": status = "proposed"; note = "awaiting human approval"; appr = json.dumps(a, default=str)
            else:
                try: note = dom.apply(a, c) or ""
                except Exception as ex: note = f"apply failed: {ex}"
            c.execute("INSERT INTO agent_actions(ts,type,target,detail,severity,owner,due,source_cycle,note,status,approval) VALUES(?,?,?,?,?,?,?,?,?,?,?)",
                      (now(), a["type"], str(a["target"]), a["detail"], sev, a.get("owner") or dom.OWNER.get(a["type"], "Operations"), due, cid, note, status, appr))
        if sev == "Critical":
            if sent < 8: notify(dom, a["target"], "Critical", a["type"], a["detail"]); sent += 1
            else: extra += 1
    if extra: notify(dom, "MULTIPLE", "Critical", "Summary", f"{extra} more critical actions raised this cycle; see Command Center")
    with _DB_LOCK, conn() as c:      # auto-close: the owning agent re-ran and no longer raises it
        for aid, t, tg, d in c.execute("SELECT action_id,type,target,detail FROM agent_actions WHERE status IN ('open','proposed')").fetchall():
            if dom.ACTION_AGENT.get(t) in results and (t, tg, _nk(d.replace("OVERDUE: ", ""))) not in raised:
                c.execute("UPDATE agent_actions SET status='auto-closed', resolved_ts=? WHERE action_id=?", (now(), aid))

def escalate_overdue(dom):
    n = 0
    with _DB_LOCK, conn() as c:
        rows = c.execute("SELECT action_id,target,detail,type FROM agent_actions WHERE status='open' AND due<? AND detail NOT LIKE 'OVERDUE:%'", (now(),)).fetchall()
        for aid, tg, d, t in rows:
            nd = (datetime.datetime.now() + datetime.timedelta(hours=24)).isoformat(timespec="seconds")
            c.execute("UPDATE agent_actions SET severity='Critical', owner=?, detail=?, due=? WHERE action_id=?", (dom.ESCALATE_TO, "OVERDUE: " + d, nd, aid)); n += 1
    for aid, tg, d, t in rows: notify(dom, tg, "Critical", "Escalation", "OVERDUE: " + d)
    return n

def resolve_action(aid, status="resolved", note="", actor="user"):
    audit(actor, status, aid, note)
    with _DB_LOCK, conn() as c:
        c.execute("UPDATE agent_actions SET status=?, resolved_ts=?, note=COALESCE(note,'')||? WHERE action_id=?", (status, now(), (" " + note) if note else "", aid))

# ---------------- orchestration (LangGraph when installed, plain loop otherwise) ----------------
def _log_run(cid, trig, res, ms):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO agent_runs(ts,cycle_id,trigger_src,agent,risk,score,latency_ms,summary,output) VALUES(?,?,?,?,?,?,?,?,?)",
                  (now(), cid, trig, res.agent, res.risk, res.score, ms, res.summary, json.dumps(res.findings, default=str)[:30000]))

def make_nodes(dom):
    def planner(s): return {"queue": dom.plan(s["request"]), "results": {}}
    def agent(s):
        name = s["queue"][0]; t = time.time()
        try: res = dom.AGENTS[name](s)
        except Exception as e: res = AgentResult(name, 50.0, "Medium", f"agent error: {e}")
        _log_run(s["cycle_id"], s["trigger"], res, int((time.time() - t) * 1000))
        new = [n for n in dom.REPLAN.get(name, lambda r: [])(res) if n not in s["results"] and n not in s["queue"]]
        if new: _log_run(s["cycle_id"], s["trigger"], AgentResult("re-plan", 100, "Low", f"{name} risk {res.risk}: adding {', '.join(new)}"), 0)
        return {"results": {**s["results"], name: res}, "queue": s["queue"][1:] + new}
    def engine(s):
        d = dom.engine(s)
        with _DB_LOCK, conn() as c:
            c.execute("INSERT INTO decisions(ts,cycle_id,trigger_src,score,risk,narrative,evidence) VALUES(?,?,?,?,?,?,?)",
                      (now(), s["cycle_id"], s["trigger"], d["score"], d["risk"], d["narrative"], json.dumps({k: v for k, v in d.items() if k not in ("narrative", "actions")}, default=str)))
        return {"decision": d}
    def act_node(s): act(dom, s["decision"], s["cycle_id"], s["results"]); return {}
    return planner, agent, engine, act_node

def _compile(dom):
    from typing import TypedDict
    from langgraph.graph import StateGraph, END
    class S(TypedDict, total=False):
        cycle_id: str; trigger: str; request: str; queue: list; results: dict; decision: dict
    planner, agent, engine, act_node = make_nodes(dom)
    g = StateGraph(S)
    for n, f in (("planner", planner), ("agent", agent), ("engine", engine), ("act", act_node)): g.add_node(n, f)
    g.set_entry_point("planner"); g.add_edge("planner", "agent")
    g.add_conditional_edges("agent", lambda s: "agent" if s["queue"] else "engine")
    g.add_edge("engine", "act"); g.add_edge("act", END)
    return g.compile()

_GRAPHS = {}
def run_cycle(dom, trigger="manual", request=""):
    init_live_tables()
    cid = datetime.datetime.now().strftime("%m%d-%H%M%S") + "-" + trigger[:4]
    s = {"cycle_id": cid, "trigger": trigger, "request": request, "queue": [], "results": {}, "decision": {}}
    try:
        if dom.NAME not in _GRAPHS: _GRAPHS[dom.NAME] = _compile(dom)
        out = _GRAPHS[dom.NAME].invoke(s)
    except ImportError:
        planner, agent, engine, act_node = make_nodes(dom)
        s.update(planner(s))
        while s["queue"]: s.update(agent(s))
        s.update(engine(s)); act_node(s); out = s
    return out["decision"]

# ---------------- background runtime ----------------
class Runtime:
    def __init__(self, dom, tick=5, cycle=60, min_gap=12):
        self.dom, self.tick, self.cycle, self.min_gap = dom, tick, cycle, min_gap
        self.stop = threading.Event(); self.ticks = 0; self.cycles = 0; self.last_cycle = 0; self.last_err = ""
        self.th = threading.Thread(target=self._loop, daemon=True); self.th.start()
        if hasattr(dom, "feed"): self.fth = threading.Thread(target=self._feed_loop, daemon=True); self.fth.start()
    def _feed_loop(self):          # network polling lives here so slow/offline APIs never stall the tick
        while not self.stop.is_set():
            try: self.dom.feed()
            except Exception as e: self.last_err = f"feed: {e}"
            self.stop.wait(int(os.environ.get("FEED_SEC", 60)))
    def _loop(self):
        init_live_tables(); set_state("boot", {"stage": "bootstrap", "ts": now()})
        try: self.dom.bootstrap()
        except Exception as e: self.last_err = f"bootstrap: {e}"
        set_state("boot", {"stage": "first_cycle", "ts": now()})
        while not self.stop.is_set():
            try:
                reasons = self.dom.tick() or []; self.ticks += 1
                if self.ticks % 120 == 0: maintenance()
                reasons = reasons + (self.dom.process() or [])
                due = time.time() - self.last_cycle
                if (reasons and due >= self.min_gap) or due >= self.cycle:
                    run_cycle(self.dom, "event" if reasons else "sched", ""); escalate_overdue(self.dom)
                    self.cycles += 1; self.last_cycle = time.time()
                    set_state("runtime", {"ticks": self.ticks, "cycles": self.cycles, "last_cycle": now(), "reasons": reasons[:3], "err": self.last_err})
                    if self.cycles == 1: set_state("boot", {"stage": "ready", "ts": now()})
                elif self.ticks % 6 == 0: set_state("runtime", {**(get_state("runtime", {}) or {}), "ticks": self.ticks})
            except Exception as e: self.last_err = str(e)
            self.stop.wait(self.tick)

_RT = None; _RT_LOCK = threading.Lock()
def start_runtime(dom):
    global _RT
    with _RT_LOCK:
        if _RT is None:
            _RT = Runtime(dom, int(os.environ.get("LIVE_TICK_SEC", 5)), int(os.environ.get("LIVE_CYCLE_SEC", 60)))
    return _RT

In [ ]:
%%writefile /content/app/mailer.py
"""One place that sends e-mail (OTP codes, reports, alerts). Gmail SMTP with an app password: set EMAIL_ID + EMAIL_PASSWORD (Colab secrets or env)."""
import os, ssl, smtplib
from email.message import EmailMessage
def creds(): return os.environ.get("EMAIL_ADDRESS") or os.environ.get("EMAIL_ID") or "", (os.environ.get("EMAIL_PASSWORD") or os.environ.get("EMAIL_APP_PASSWORD") or "").replace(" ", "")
def configured(): a, b = creds(); return bool(a and b)
def send(to, subject, body, attachments=(), html=None):
    """attachments: [(filename, bytes, mimetype)]. Returns (ok, message)."""
    em, pw = creds()
    if not (em and pw): return False, "E-mail is not configured (set EMAIL_ADDRESS and EMAIL_PASSWORD)."
    try:
        m = EmailMessage(); m["Subject"] = subject; m["From"] = em; m["To"] = to; m.set_content(body)
        if html: m.add_alternative(html, subtype="html")
        for fn, data, mt in attachments: m.add_attachment(data, maintype=mt.split("/")[0], subtype=mt.split("/")[1], filename=fn)
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=ssl.create_default_context(), timeout=20) as s: s.login(em, pw); s.send_message(m)
        return True, f"Sent to {to}"
    except Exception as e: return False, f"Could not send: {e}"


In [ ]:
%%writefile /content/app/qa.py
"""Exact-answer engine for the Copilot. Turns a plain-English question into a read-only, parameterised SQL query over the live tables
(counts, top/bottom N, averages, totals, group-by, filters, ID lookups) and returns the real rows. The LLM only explains afterwards;
it never has to guess a number. Table and column names come from a fixed whitelist, values are always bound parameters."""
import re
from blocks import Q

TABLES = {
    "Franchise": {
        "outlets": (["outlet", "store", "branch", "franchise"], "outlet_id", ["outlet_id", "outlet_name", "location", "tier", "revenue", "customer_satisfaction", "staff_headcount"], "revenue"),
        "staff": (["staff", "employee", "worker", "workforce", "people"], "staff_id", ["staff_id", "name", "role", "outlet_id", "salary", "overtime_hrs", "predicted_attrition_prob"], "predicted_attrition_prob"),
        "inventory": (["inventory", "sku", "item", "product", "stock level"], "record_id", ["outlet_id", "sku_name", "category", "current_stock", "reorder_threshold", "stockout_risk_prob"], "stockout_risk_prob"),
        "marketing": (["campaign", "marketing", "promotion", "advert"], "campaign_id", ["campaign_id", "campaign_name", "outlet_id", "channel", "budget", "actual_roi", "conversions"], "actual_roi"),
        "feedback": (["feedback", "review", "complaint", "comment"], "feedback_id", ["feedback_id", "outlet_id", "rating", "sentiment_score", "date", "comment"], "rating"),
        "audits": (["audit", "inspection", "compliance"], "audit_id", ["audit_id", "outlet_id", "audit_date", "score", "violations", "category", "status"], "score"),
        "sales": (["sale", "transaction", "takings"], "sale_id", ["outlet_id", "ts", "amount", "tx_count", "source"], "amount"),
        "purchase_orders": (["purchase order", "reorder order", "po"], "po_id", ["po_id", "outlet_id", "sku_name", "qty", "eta_ts", "status"], "qty"),
        "attendance": (["attendance", "absent", "absence", "present"], "att_id", ["staff_id", "outlet_id", "ts", "status"], None),
        "agent_actions": (["ticket", "action", "task", "approval", "alert"], "action_id", ["action_id", "type", "target", "severity", "status", "assignee", "detail"], None),
    },
    "Freight": {
        "shipments": (["shipment", "cargo", "container", "consignment"], "shipment_id", ["shipment_id", "origin_port", "dest_port", "carrier", "status", "eta", "weight_kg", "predicted_delay_risk"], "predicted_delay_risk"),
        "freight_quotes": (["quote", "quotation", "price", "pricing"], "quote_id", ["quote_id", "shipment_id", "customer_id", "carrier", "final_price", "margin_pct", "status"], "final_price"),
        "ports": (["port", "terminal", "harbour", "harbor"], "port_id", ["port_name", "country", "region", "congestion_index", "avg_dwell_days"], "congestion_index"),
        "carriers": (["carrier", "shipping line", "liner", "line"], "carrier_id", ["name", "rating", "on_time_pct", "avg_cost_index", "risk_level"], "on_time_pct"),
        "customers": (["customer", "client", "shipper"], "customer_id", ["customer_id", "name", "industry", "priority_tier", "credit_risk"], "credit_risk"),
        "weather_risks": (["weather", "storm", "wave", "wind", "forecast"], "port_name", ["port_name", "current_severity", "forecast", "wind_speed", "wave_height"], "wave_height"),
        "vessels": (["vessel", "ship ", "ais", "voyage"], "shipment_id", ["shipment_id", "route_name", "lat", "lon", "progress", "speed_kn"], "progress"),
        "quote_requests": (["request", "enquiry", "enquiries", "inquiry"], "req_id", ["req_id", "ts", "origin", "dest", "cargo", "weight_t", "status", "quote_id"], None),
        "customs_rules": (["customs", "duty", "hs code", "tariff"], "chapter", ["chapter", "goods", "duty_pct", "docs", "restricted"], "duty_pct"),
        "agent_actions": (["ticket", "action", "task", "approval", "alert"], "action_id", ["action_id", "type", "target", "severity", "status", "assignee", "detail"], None),
    },
}
CSYN = {"revenue": ["revenue", "turnover"], "customer_satisfaction": ["satisfaction", "csat"], "staff_headcount": ["headcount"], "salary": ["salary", "pay", "wage"], "overtime_hrs": ["overtime"],
        "job_satisfaction": ["job satisfaction"], "tenure_years": ["tenure"], "predicted_attrition_prob": ["attrition", "churn", "likely to leave", "quit"], "current_stock": ["current stock", "units in stock"],
        "reorder_threshold": ["reorder level", "threshold"], "weekly_demand": ["demand"], "lead_time_days": ["lead time"], "stockout_risk_prob": ["stockout", "stock-out", "stock out", "shortage", "run out"],
        "actual_roi": ["roi", "return on"], "sentiment_score": ["sentiment"], "score": ["audit score", "score"], "violations": ["violation"], "amount": ["amount", "sales value"], "qty": ["quantity", "qty"],
        "congestion_index": ["congestion", "congested", "crowded"], "avg_dwell_days": ["dwell"], "on_time_pct": ["on-time", "on time", "punctual", "punctuality", "reliab"], "avg_cost_index": ["cost index"],
        "predicted_delay_risk": ["delay risk", "delay", "delayed"], "weight_kg": ["weight"], "final_price": ["final price", "price", "quote value", "worth"], "margin_pct": ["margin"], "base_cost": ["base cost"],
        "insurance": ["insurance"], "customs_fee": ["customs fee"], "fuel_surcharge": ["fuel"], "credit_risk": ["credit risk", "credit"], "duty_pct": ["duty"], "wave_height": ["wave"], "wind_speed": ["wind"],
        "speed_kn": ["speed"], "progress": ["progress", "completed"], "operating_costs": ["operating cost", "costs"], "rating": ["rating", "rated", "stars"], "budget": ["budget", "spend"], "reach": ["reach"], "conversions": ["conversion"]}
BAD = {"predicted_attrition_prob", "stockout_risk_prob", "violations", "congestion_index", "avg_dwell_days", "predicted_delay_risk", "credit_risk", "operating_costs", "overtime_hrs", "lead_time_days", "avg_cost_index", "wave_height", "wind_speed"}
PROB = {"predicted_attrition_prob", "stockout_risk_prob", "predicted_delay_risk", "credit_risk", "progress", "sentiment_score"}
LABEL = {"outlets": "outlet_name", "staff": "name", "carriers": "name", "customers": "name", "ports": "port_name", "inventory": "sku_name", "marketing": "campaign_name"}
GENERIC = {"port", "line", "terminal", "cargo", "ocean", "shipping", "the", "of"}

def _w(q, term): return re.search(r"(?<![a-z0-9])" + re.escape(term.strip()) + r"(?:s|es)?(?![a-z0-9])", q) is not None
def _num(s, unit):
    v = float(s.replace(",", "")); u = (unit or "").lower()
    return v * {"k": 1e3, "m": 1e6, "lakh": 1e5, "lakhs": 1e5, "cr": 1e7, "crore": 1e7, "crores": 1e7}.get(u, 1)
_NUM = {}
def _cols(t):
    df = Q(f"SELECT * FROM {t} LIMIT 20"); _NUM[t] = {c for c in df.columns if str(df[c].dtype)[:3] in ("flo", "int") }
    return list(df.columns)
PL = {"inventory": "inventory records", "staff": "staff", "marketing": "campaigns", "feedback": "feedback entries", "audits": "audits", "sales": "sales records", "attendance": "attendance records", "freight_quotes": "quotes", "ports": "ports", "carriers": "carriers", "customs_rules": "customs rules", "weather_risks": "port weather entries", "agent_actions": "tickets", "purchase_orders": "purchase orders", "quote_requests": "quote requests", "outlets": "outlets", "shipments": "shipments", "customers": "customers", "vessels": "vessels"}
def pct(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns and c in PROB: df[c] = df[c].map(lambda x: f"{x:.0%}" if isinstance(x, (int, float)) and abs(x) <= 1 else x)
    return df
def _alias(v):
    v = str(v); out = {v.lower()}; base = re.sub(r"\(.*?\)", "", v).strip().lower(); out.add(base)
    out.update(m.lower() for m in re.findall(r"\((.*?)\)", v))
    core = " ".join(w for w in base.split() if w not in GENERIC)
    if len(core) >= 3: out.add(core)
    return {a for a in out if len(a) >= 3}
def _fmt(v):
    if isinstance(v, float): return f"{v:,.2f}" if abs(v) < 1000 else f"{v:,.0f}"
    if isinstance(v, int): return f"{v:,}" if abs(v) >= 10000 else str(v)
    return str(v)
def md(df):
    if df is None or not len(df): return ""
    h = "| " + " | ".join(map(str, df.columns)) + " |\n|" + "---|" * len(df.columns) + "\n"
    return h + "\n".join("| " + " | ".join(_fmt(x).replace("|", "/").replace("\n", " ")[:90] for x in r) + " |" for r in df.itertuples(index=False))

def lookup(kind, q, scope=None):
    """ID in the question -> the exact row(s) plus its open tickets."""
    ids = list(dict.fromkeys(re.findall(r"\b(OUT-\d+|STF-\d+|EMP-\d+|SHP-[A-Z0-9-]+|QTE-[A-Z0-9-]+|PORT-\d+|CAR-\d+|CUST-\d+|CAMP-\d+|AUD-\d+|FB-\d+)\b", q, re.I)))
    ids = [i.upper() for i in ids]
    if not ids: return None
    parts, sqls = [], []
    for e in ids[:4]:
        if scope and e != scope and not e.startswith("PORT") and not e.startswith("CAR"): parts.append(f"{e} is outside your outlet scope ({scope})."); continue
        for t, (_, idc, cols, _m) in TABLES[kind].items():
            if t == "agent_actions": continue
            try: df = Q(f"SELECT * FROM {t} WHERE {idc}=?", (e,))
            except Exception: continue
            if len(df):
                r = df.iloc[0]; parts.append(f"**{e}** ({t}): " + "; ".join(f"{c}={_fmt(r[c])}" for c in df.columns if str(r[c]) not in ("None", "nan"))); sqls.append(f"SELECT * FROM {t} WHERE {idc}='{e}'"); break
        tk = Q("SELECT action_id, type, severity, status, detail FROM agent_actions WHERE target=? AND status IN ('open','proposed') ORDER BY action_id DESC LIMIT 5", (e,))
        if len(tk): parts.append(f"Open tickets on {e}:\n" + md(tk)); sqls.append(f"SELECT … FROM agent_actions WHERE target='{e}' AND status IN ('open','proposed')")
    return {"answer": "\n\n".join(parts), "df": None, "sql": "; ".join(sqls)} if parts else None

def answer(kind, q, scope=None, mask=False):
    """-> dict(answer, sql, df, table) when the question maps to the data, else None."""
    ql = q.lower().strip()
    if not ql: return None
    d = lookup(kind, q, scope)
    if d: return d
    T = TABLES[kind]; sc = {}
    for t, (syn, _, _, _) in T.items(): sc[t] = sum(3 for s in syn if _w(ql, s))
    cols_of = {}
    for t in T:
        try: cols_of[t] = _cols(t)
        except Exception: cols_of[t] = []
    hits = {c: any(_w(ql, s) for s in (CSYN.get(c, []) + [c.replace("_", " ")])) for c in {c for v in cols_of.values() for c in v}}
    for t, cs in cols_of.items(): sc[t] += sum(2 for c in cs if hits.get(c) and c not in ("status", "type", "date", "name"))
    if not sc or max(sc.values()) <= 0: return None
    order = list(T); t = max(order, key=lambda x: (sc[x], -order.index(x)))
    syn, idc, show, dmetric = T[t]; cs = cols_of[t]
    # aggregation
    agg = "list"
    if re.search(r"\b(how many|number of|count)\b", ql): agg = "count"
    elif re.search(r"\b(average|avg|mean)\b", ql): agg = "avg"
    elif re.search(r"\b(total|sum|overall|combined)\b", ql): agg = "sum"
    elif re.search(r"\b(highest|most|top|best|largest|biggest|maximum|max|greatest|riskiest|worst|lowest|least|smallest|minimum|min|bottom|weakest|cheapest|fewest|poorest)\b", ql): agg = "rank"
    # metric
    mc = [c for c in cs if hits.get(c) and c in _NUM.get(t, ()) and c not in ("lat", "lon")]
    metric = mc[0] if mc else dmetric
    if metric and metric not in cs: metric = None
    # filters
    where, params, desc = [], [], []
    def val_filter(tab, col, sub=False):
        try: vals = Q(f"SELECT DISTINCT {col} v FROM {tab} WHERE {col} IS NOT NULL LIMIT 600").v.tolist()
        except Exception: return
        found = []
        for v in vals:
            if isinstance(v, (int, float)): continue
            for a in _alias(v):
                if _w(ql, a): found.append((len(a), str(v), a)); break
        found.sort(reverse=True); used, taken = [], ql
        for _, v, a in found:
            if a in taken and not any(a in u or u in a for u in used):
                used.append(a)
                if sub: where.append(f"outlet_id IN (SELECT outlet_id FROM {tab} WHERE {col}=?)")
                else:
                    where.append(f"{col}=?")
                params.append(v); desc.append(f"{col} = {v}")
    for c in cs:
        if c in ("status", "category", "channel", "role", "tier", "severity", "type", "location", "origin_port", "dest_port", "carrier", "region", "country", "risk_level", "industry", "priority_tier", "source", "sku_name", "name", "port_name", "customer_id", "forecast", "route_name", "origin", "dest", "cargo"):
            if c in ("origin_port", "dest_port", "origin", "dest"): continue
            val_filter(t, c)
    if t in ("shipments", "quote_requests"):
        oc, dc = ("origin_port", "dest_port") if t == "shipments" else ("origin", "dest")
        try: pv = Q("SELECT DISTINCT port_name v FROM ports").v.tolist()
        except Exception: pv = []
        for pn in pv:
            for a_ in sorted(_alias(pn), key=len, reverse=True):
                mm = re.search(r"(?<![a-z0-9])" + re.escape(a_) + r"(?![a-z0-9])", ql)
                if not mm: continue
                pre = ql[max(0, mm.start() - 22):mm.start()].strip()
                frm = re.search(r"(from|origin|out of|leaving|departing|ex)$", pre); to = re.search(r"(to|towards|into|destination|dest|bound for|arriving at|arriving in|reaching)$", pre)
                if t == "quote_requests": pn_ = a_.split(" (")[0]; pat = pn_
                else: pat = pn
                if frm: where.append(f"{oc}=?"); params.append(pat); desc.append(f"{oc} = {pn}")
                elif to: where.append(f"{dc}=?"); params.append(pat); desc.append(f"{dc} = {pn}")
                else: where.append(f"({oc}=? OR {dc}=?)"); params += [pat, pat]; desc.append(f"{pn} is origin or destination")
                break
    if "outlet_id" in cs and t != "outlets":
        try: val_filter("outlets", "location", sub=True); val_filter("outlets", "tier", sub=True)
        except Exception: pass
    m = re.search(r"\b(outlet|store|branch)\s*(?:no\.?|number|#)?\s*(\d{1,3})\b", ql)
    if m and "outlet_id" in cs: where.append("outlet_id=?"); params.append(f"OUT-{int(m.group(2)):03d}"); desc.append(f"outlet_id = OUT-{int(m.group(2)):03d}")
    for pat, op in ((r"(?:below|under|less than|lower than|smaller than|<)", "<"), (r"(?:above|over|more than|greater than|higher than|exceed\w*|>)", ">"), (r"(?:at least|minimum of|>=)", ">="), (r"(?:at most|no more than|<=)", "<=")):
        mm = re.search(pat + r"\s*(?:rs\.?|₹|\$|inr|usd)?\s*([\d,]*\.?\d+)\s*(%|k|m|lakhs?|cr(?:ores?)?)?", ql)
        if mm and metric:
            v = _num(mm.group(1), mm.group(2) if mm.group(2) != "%" else "");
            if metric in PROB and v > 1: v /= 100
            where.append(f"{metric}{op}?"); params.append(v); desc.append(f"{metric} {op} {v:g}"); break
    if re.search(r"\b(open|pending|unresolved)\b", ql) and "status" in cs and not any("status" in x for x in desc) and t in ("agent_actions", "purchase_orders"):
        where.append("status IN ('open','proposed')" if t == "agent_actions" else "status NOT IN ('delivered','received','closed')"); desc.append("status = open")
    if re.search(r"\b(critical|high severity|urgent)\b", ql) and "severity" in cs and not any("severity" in x for x in desc): where.append("severity IN ('critical','high')"); desc.append("severity = critical/high")
    if re.search(r"\b(delayed|late)\b", ql) and t == "shipments" and not any("status" in x for x in desc): where.append("(status LIKE '%delay%' OR predicted_delay_risk>=0.5)"); desc.append("delayed or delay risk ≥ 0.5")
    if re.search(r"\b(low stock|running low|below reorder|need reorder|reorder)\b", ql) and t == "inventory": where.append("current_stock<=reorder_threshold"); desc.append("stock ≤ reorder threshold")
    if re.search(r"\b(high risk|at risk)\b", ql) and metric in PROB and not any(metric in x for x in desc): where.append(f"{metric}>=0.5"); desc.append(f"{metric} ≥ 0.5")
    if scope and t != "agent_actions":
        if "outlet_id" in cs: where.append("outlet_id=?"); params.append(scope); desc.append(f"outlet_id = {scope}")
        else: return {"answer": "That data is not available for your outlet role.", "sql": "", "df": None}
    elif scope: return {"answer": "Ticket lists are limited to your outlet; open the Actions page.", "sql": "", "df": None}
    # an unfiltered, unaggregated, unranked question is too vague to answer exactly
    if agg == "list" and not where and not re.search(r"\b(show|list|which|what|who|give)\b", ql): return None
    W = (" WHERE " + " AND ".join(where)) if where else ""
    nm = re.search(r"\b(?:top|bottom|first|best|worst|lowest|highest)\s+(\d{1,2})\b", ql); n = int(nm.group(1)) if nm else (5 if agg == "rank" else 10)
    tdesc = PL.get(t, syn[0] + "s"); fdesc = (" where " + ", ".join(desc)) if desc else ""
    grp = re.search(r"\b(?:by|per|for each|across|grouped by|each)\s+(location|city|tier|role|category|channel|carrier|region|country|status|type|severity|industry|outlet|port|origin|destination|sku|customer)\b", ql)
    gcol = {"city": "location", "outlet": "outlet_id", "port": "port_name" if "port_name" in cs else "origin_port", "origin": "origin_port", "destination": "dest_port", "sku": "sku_name", "customer": "customer_id"}.get(grp.group(1), grp.group(1)) if grp else None
    if gcol and gcol not in cs and "outlet_id" in cs and gcol in ("location", "tier"):
        j = f" JOIN outlets o ON o.outlet_id={t}.outlet_id"; gq = f"o.{gcol}"
    elif gcol in cs: j, gq = "", gcol
    else: j, gq = "", None
    if gq:
        fn = {"count": "COUNT(*)", "avg": f"AVG({metric})", "sum": f"SUM({metric})"}.get(agg, f"AVG({metric})" if metric else "COUNT(*)")
        if agg == "rank" and re.search(r"\b(lowest|least|smallest|minimum|min|bottom|worst|fewest|weakest|poorest|cheapest)\b", ql) and not re.search(r"\b(worst|riskiest)\b", ql) or (re.search(r"\b(worst)\b", ql) and metric not in BAD): order_dir = "ASC"
        else: order_dir = "DESC"
        if re.search(r"\bhow many|number of|count\b", ql): fn = "COUNT(*)"
        sql = f"SELECT {gq} AS grp, {fn} AS value, COUNT(*) AS rows_n FROM {t}{j}{W} GROUP BY {gq} ORDER BY value {order_dir} LIMIT 25"
        df = Q(sql, tuple(params))
        if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
        lab = {"COUNT(*)": "count"}.get(fn, fn.lower().replace("(", " of ").replace(")", ""))
        top = df.iloc[0]; df = df.rename(columns={"grp": gcol, "value": lab});
        return {"answer": f"{lab.capitalize()} of {tdesc} by {gcol}{fdesc}. Highest: **{top.iloc[0]}** ({_fmt(top.iloc[1])}); lowest: **{df.iloc[-1, 0]}** ({_fmt(df.iloc[-1, 1])}).", "sql": sql, "df": df}
    if agg == "count":
        sql = f"SELECT COUNT(*) AS n FROM {t}{W}"; n_ = int(Q(sql, tuple(params)).n[0])
        return {"answer": f"**{n_:,}** {tdesc}{fdesc}.", "sql": sql, "df": None}
    if agg in ("avg", "sum") and metric:
        fn = "AVG" if agg == "avg" else "SUM"; sql = f"SELECT {fn}({metric}) AS v, COUNT(*) AS n, MIN({metric}) AS lo, MAX({metric}) AS hi FROM {t}{W}"; r = Q(sql, tuple(params)).iloc[0]
        if not r.n: return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
        pv = f"{r.v:.1%}" if metric in PROB and abs(r.v) <= 1 else _fmt(float(r.v))
        return {"answer": f"{'Average' if agg == 'avg' else 'Total'} {metric.replace('_', ' ')} of {tdesc}{fdesc}: **{pv}** across {int(r.n):,} rows (min {_fmt(float(r.lo))}, max {_fmt(float(r.hi))}).", "sql": sql, "df": None}
    if mask and t == "staff" and (metric in ("salary", "age") or re.search(r"\b(salary|salaries|pay|wage|age)\b", ql)): return {"answer": "Salary and personal details are visible to Admin, Owner and Operations roles only.", "sql": "", "df": None}
    sel = list(dict.fromkeys([c for c in show if c in cs] + ([metric] if metric else [])))
    if mask: sel = [c for c in sel if not (t == "staff" and c in ("name", "salary", "age"))]
    if t == "inventory" and "outlet_id" in cs and "sku_name" not in sel: sel.append("sku_name")
    if agg == "rank" and metric:
        low = re.search(r"\b(lowest|least|smallest|minimum|min|bottom|fewest|weakest|poorest|cheapest)\b", ql) is not None; worst = re.search(r"\b(worst|riskiest)\b", ql) is not None
        asc = (low and not worst) or (worst and metric not in BAD) or (re.search(r"\b(best)\b", ql) is not None and metric in BAD)
        sql = f"SELECT {', '.join(sel)} FROM {t}{W} ORDER BY {metric} {'ASC' if asc else 'DESC'} LIMIT {n}"; df = Q(sql, tuple(params))
        if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
        r0 = df.iloc[0]; lab = r0[LABEL[t]] if t in LABEL and LABEL[t] in df.columns else r0[idc] if idc in df.columns else r0.iloc[0]; pv = f"{r0[metric]:.1%}" if metric in PROB and abs(r0[metric]) <= 1 else _fmt(r0[metric])
        return {"answer": f"{'Lowest' if asc else 'Highest'} {metric.replace('_', ' ')} among {tdesc}{fdesc}: **{lab}** at **{pv}**. Top {len(df)} below.", "sql": sql, "df": pct(df, sel)}
    sql = f"SELECT {', '.join(sel)} FROM {t}{W}" + (f" ORDER BY {metric} {'ASC' if metric in BAD and False else 'DESC'}" if metric else "") + f" LIMIT {max(n, 10) if not nm else n}"
    tot = int(Q(f"SELECT COUNT(*) n FROM {t}{W}", tuple(params)).n[0]); df = Q(sql, tuple(params))
    if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
    return {"answer": f"**{tot:,}** {tdesc}{fdesc}; showing {len(df)}" + (f", sorted by {metric.replace('_', ' ')}" if metric else "") + ".", "sql": sql, "df": pct(df, sel)}


In [ ]:
%%writefile /content/app/scale_data.py
"""Scale both apps to production-like volume. Idempotent: only tops tables up to the target counts.
Coordinates are approximate (1-2 decimals): replace/extend through the Data Feeder with UN/LOCODE or World Port Index data."""
import random, datetime, math
from live_engine import conn, _DB_LOCK, now

PORTS = {
"China": "Dalian 38.9 121.7;Xiamen 24.5 118.1;Fuzhou 26.0 119.4;Lianyungang 34.7 119.4;Yantai 37.6 121.4;Rizhao 35.4 119.5;Nanjing 32.1 118.8;Zhanjiang 21.2 110.4;Haikou 20.0 110.3;Zhuhai 22.2 113.6;Taicang 31.5 121.2;Wuhan 30.6 114.3;Shantou 23.4 116.7;Dongguan 22.9 113.6",
"Japan": "Yokohama 35.4 139.7;Nagoya 35.1 136.9;Osaka 34.6 135.4;Kobe 34.7 135.2;Hakata 33.6 130.4;Chiba 35.6 140.1",
"South Korea": "Incheon 37.4 126.6;Gwangyang 34.9 127.7;Ulsan 35.5 129.4;Pyeongtaek 36.9 126.8",
"Taiwan": "Keelung 25.1 121.7;Taichung 24.3 120.5", "Vietnam": "Hai Phong 20.8 106.7;Da Nang 16.1 108.2;Vung Tau 10.4 107.1;Quy Nhon 13.8 109.3",
"Thailand": "Bangkok 13.7 100.6;Songkhla 7.2 100.6", "Malaysia": "Penang 5.4 100.3;Johor 1.4 103.9;Kuantan 3.97 103.4;Kuching 1.6 110.3",
"Indonesia": "Surabaya -7.2 112.7;Semarang -6.9 110.4;Belawan 3.8 98.7;Makassar -5.1 119.4;Balikpapan -1.3 116.8;Palembang -3.0 104.8;Bitung 1.4 125.2",
"Philippines": "Cebu 10.3 123.9;Davao 7.1 125.6;Subic Bay 14.8 120.3", "Myanmar": "Yangon 16.8 96.2", "Cambodia": "Sihanoukville 10.6 103.5",
"Bangladesh": "Chittagong 22.3 91.8;Mongla 22.5 89.6", "Sri Lanka": "Hambantota 6.1 81.1;Galle 6.0 80.2", "Pakistan": "Port Qasim 24.8 67.3;Gwadar 25.1 62.3", "Iran": "Chabahar 25.3 60.6",
"India": "Mormugao 15.4 73.8;Pipavav 20.9 71.5;Hazira 21.1 72.6;Ennore 13.2 80.3;Krishnapatnam 14.3 80.1;Kakinada 16.9 82.3;Gangavaram 17.6 83.2;Dahej 21.7 72.5;Mumbai Port 18.9 72.8;Navlakhi 22.9 70.5;Karaikal 10.9 79.9;Kamarajar 13.3 80.3;Vizhinjam 8.4 77.0;Okha 22.5 69.1;Porbandar 21.6 69.6",
"UAE": "Khalifa Port 24.8 54.6;Port Rashid 25.3 55.3;Fujairah 25.1 56.4;Khor Fakkan 25.3 56.4", "Oman": "Sohar 24.5 56.6;Duqm 19.7 57.7",
"Saudi Arabia": "Dammam 26.5 50.2;King Abdullah Port 22.5 39.1;Yanbu 24.1 38.1", "Qatar": "Hamad Port 25.0 51.6", "Kuwait": "Shuwaikh 29.4 47.9", "Bahrain": "Khalifa Bin Salman 26.2 50.6",
"Iraq": "Umm Qasr 30.0 47.9", "Jordan": "Aqaba 29.5 35.0", "Israel": "Haifa 32.8 35.0;Ashdod 31.8 34.6", "Turkey": "Mersin 36.8 34.6;Ambarli 41.0 28.7;Izmir 38.4 27.1",
"Yemen": "Aden 12.8 45.0", "Djibouti": "Djibouti 11.6 43.1", "Sudan": "Port Sudan 19.6 37.2",
"Germany": "Bremerhaven 53.6 8.5", "Belgium": "Zeebrugge 51.3 3.2", "Netherlands": "Amsterdam 52.4 4.9", "United Kingdom": "Southampton 50.9 -1.4;London Gateway 51.5 0.5;Liverpool 53.4 -3.0",
"Ireland": "Dublin 53.3 -6.2", "France": "Marseille Fos 43.4 4.9", "Spain": "Barcelona 41.4 2.2;Bilbao 43.3 -3.0", "Portugal": "Lisbon 38.7 -9.1;Sines 37.9 -8.9",
"Italy": "Naples 40.8 14.3;La Spezia 44.1 9.8;Livorno 43.6 10.3;Trieste 45.6 13.8;Gioia Tauro 38.4 15.9;Taranto 40.5 17.2", "Slovenia": "Koper 45.5 13.7", "Croatia": "Rijeka 45.3 14.4",
"Greece": "Thessaloniki 40.6 22.9", "Romania": "Constanta 44.2 28.7", "Bulgaria": "Varna 43.2 27.9", "Ukraine": "Odesa 46.5 30.7", "Sweden": "Gothenburg 57.7 11.9", "Denmark": "Aarhus 56.2 10.2",
"Norway": "Oslo 59.9 10.7", "Finland": "Helsinki 60.2 25.0", "Poland": "Gdynia 54.5 18.5", "Lithuania": "Klaipeda 55.7 21.1", "Latvia": "Riga 57.0 24.1", "Estonia": "Tallinn 59.4 24.8",
"Russia": "St Petersburg 59.9 30.2;Novorossiysk 44.7 37.8", "Malta": "Marsaxlokk 35.8 14.5", "Cyprus": "Limassol 34.7 33.0",
"Egypt": "Alexandria 31.2 29.9;Port Said 31.3 32.3;Damietta 31.5 31.8;Ain Sokhna 29.6 32.4", "Libya": "Tripoli 32.9 13.2", "Tunisia": "Rades 36.8 10.3", "Algeria": "Algiers 36.8 3.1",
"Morocco": "Casablanca 33.6 -7.6", "Senegal": "Dakar 14.7 -17.4", "Ivory Coast": "Abidjan 5.3 -4.0", "Ghana": "Tema 5.6 0.0", "Togo": "Lome 6.1 1.3", "Benin": "Cotonou 6.4 2.4",
"Nigeria": "Onne 4.7 7.2", "Cameroon": "Douala 4.0 9.7", "Angola": "Luanda -8.8 13.2", "Namibia": "Walvis Bay -22.9 14.5", "South Africa": "Cape Town -33.9 18.4;Ngqura -33.8 25.7",
"Mozambique": "Maputo -25.97 32.6;Beira -19.8 34.9", "Tanzania": "Dar es Salaam -6.8 39.3", "Kenya": "Mombasa -4.1 39.7", "Somaliland": "Berbera 10.4 45.0", "Mauritius": "Port Louis -20.2 57.5", "Madagascar": "Toamasina -18.1 49.4",
"USA": "Oakland 37.8 -122.3;Seattle 47.6 -122.3;Tacoma 47.3 -122.4;Houston 29.7 -95.3;New Orleans 29.9 -90.1;Miami 25.8 -80.2;Charleston 32.8 -79.9;Norfolk 36.9 -76.3;Baltimore 39.3 -76.6;Boston 42.4 -71.0",
"Canada": "Vancouver 49.3 -123.1;Prince Rupert 54.3 -130.3;Halifax 44.6 -63.6;Montreal 45.5 -73.5", "Mexico": "Manzanillo 19.1 -104.3;Veracruz 19.2 -96.1;Lazaro Cardenas 17.9 -102.2",
"Panama": "Balboa 8.95 -79.6;Colon 9.35 -79.9", "Colombia": "Cartagena 10.4 -75.5;Buenaventura 3.9 -77.1", "Peru": "Callao -12.05 -77.15", "Ecuador": "Guayaquil -2.3 -79.9",
"Chile": "San Antonio -33.6 -71.6;Valparaiso -33.0 -71.6", "Argentina": "Buenos Aires -34.6 -58.4", "Uruguay": "Montevideo -34.9 -56.2",
"Brazil": "Santos -23.98 -46.3;Paranagua -25.5 -48.5;Rio Grande -32.0 -52.1;Itajai -26.9 -48.7;Suape -8.4 -35.0;Pecem -3.5 -38.8", "Jamaica": "Kingston 17.97 -76.8", "Dominican Republic": "Caucedo 18.4 -69.6", "Bahamas": "Freeport 26.5 -78.8",
"Australia": "Sydney -33.9 151.2;Melbourne -37.8 144.9;Brisbane -27.4 153.2;Fremantle -32.05 115.7;Adelaide -34.8 138.5;Port Hedland -20.3 118.6", "New Zealand": "Auckland -36.8 174.8;Tauranga -37.65 176.2;Lyttelton -43.6 172.7",
"Fiji": "Suva -18.1 178.4", "New Caledonia": "Noumea -22.3 166.4", "Papua New Guinea": "Port Moresby -9.5 147.1"}
CARRIERS = ["Evergreen Marine","HMM","Yang Ming","ZIM Integrated","Pacific International Lines","Wan Hai Lines","OOCL","APL","Hamburg Sud","Matson","SM Line","Sinokor","KMTC","Namsung Shipping","SITC",
"X-Press Feeders","Gold Star Line","Arkas Line","Turkon Line","Grimaldi Lines","Messina Line","CCNI","Seaboard Marine","Crowley Maritime","Tropical Shipping","Swire Shipping","Hoegh Autoliners",
"Wallenius Wilhelmsen","NYK Line","Mitsui OSK Lines","K Line","Safmarine","Emirates Shipping Line","Shipping Corporation of India","Great Eastern Shipping","Samudera Shipping","Salam Pacific",
"Interasia Lines","Heung-A Line","Unifeeder","Containerships","Finnlines","DFDS","Stena Line","Atlantic Container Line","King Ocean","Bengal Tiger Line","Transworld Shipping","Sealand Maersk Asia","Pan Ocean","Cosco Hong Kong"]
def region_of(lat, lon):
    if -170 < lon < -30: return "Americas"
    if -30 <= lon < 50 and lat > 12: return "Europe/Mediterranean" if lat > 30 else "Africa"
    if -30 <= lon < 52 and lat <= 12: return "Africa"
    if 112 < lon and lat < -10: return "Oceania"
    if 25 < lon < 60 and 12 < lat < 33: return "Middle East"
    return "Asia"

def scale_freight(target_ports=220, target_carriers=110, target_shipments=400):
    R = random.Random(42)
    with _DB_LOCK, conn() as c:
        have = {r[0].lower() for r in c.execute("SELECT port_name FROM ports")}; n = c.execute("SELECT COUNT(*) FROM ports").fetchone()[0]
        for ctry, spec in PORTS.items():
            for e in spec.split(";"):
                if n >= target_ports + 40: break
                nm, la, lo = e.rsplit(" ", 2); name = f"{nm} Port"
                if name.lower() in have: continue
                n += 1; have.add(name.lower())
                c.execute("INSERT OR REPLACE INTO ports(port_id,port_name,country,congestion_index,avg_dwell_days,lat,lon,region) VALUES(?,?,?,?,?,?,?,?)",
                          (f"PORT-{n:03d}", name, ctry, round(R.uniform(1.0, 4.6), 1), round(R.uniform(1.5, 7.0), 1), float(la), float(lo), region_of(float(la), float(lo))))
        k = c.execute("SELECT COUNT(*) FROM carriers").fetchone()[0]; names = {r[0] for r in c.execute("SELECT name FROM carriers")}
        pool = [x for x in CARRIERS if x not in names] + [f"Regional Feeder Line {i:02d}" for i in range(1, 80)]
        for nm in pool:
            if k >= target_carriers: break
            k += 1
            c.execute("INSERT OR REPLACE INTO carriers(carrier_id,name,rating,on_time_pct,avg_cost_index,risk_level) VALUES(?,?,?,?,?,?)",
                      (f"CAR-{k:03d}", nm, round(R.uniform(3.2, 4.9), 2), round(R.uniform(68, 97), 1), round(R.uniform(.8, 1.25), 2), R.choice(["Low", "Low", "Medium", "High"])))
        ports = c.execute("SELECT port_name FROM ports").fetchall(); cars = [r[0] for r in c.execute("SELECT name FROM carriers")]
        s = c.execute("SELECT COUNT(*) FROM shipments").fetchone()[0]
        while s < target_shipments:
            s += 1; a, b = R.sample(ports, 2)
            c.execute("INSERT OR REPLACE INTO shipments(shipment_id,origin_port,dest_port,carrier,status,eta,weight_kg,hs_code,predicted_delay_risk) VALUES(?,?,?,?,?,?,?,?,?)",
                      (f"SHP-{s:04d}", a[0], b[0], R.choice(cars), R.choices(["In Transit", "Delivered", "Customs Hold", "Port Congestion Delay", "Anchorage Pending"], [55, 20, 8, 10, 7])[0],
                       (datetime.date.today() + datetime.timedelta(days=R.randint(2, 40))).isoformat(), round(R.uniform(800, 28000), 1), f"HS-{R.choice([30,84,85,87,10,62,27,39,72,22])}{R.randint(10,99)}", round(R.uniform(.03, .6), 2)))

CITIES = {"Mumbai": (19.08, 72.88), "Delhi": (28.61, 77.21), "Bengaluru": (12.97, 77.59), "Chennai": (13.08, 80.27), "Hyderabad": (17.39, 78.49), "Pune": (18.52, 73.86), "Ahmedabad": (23.03, 72.57),
"Jaipur": (26.91, 75.79), "Kolkata": (22.57, 88.36), "Surat": (21.17, 72.83), "Kochi": (9.93, 76.27), "Coimbatore": (11.02, 76.96), "Lucknow": (26.85, 80.95), "Nagpur": (21.15, 79.09), "Indore": (22.72, 75.86),
"Bhopal": (23.26, 77.41), "Patna": (25.59, 85.14), "Chandigarh": (30.73, 76.78), "Visakhapatnam": (17.69, 83.22), "Vadodara": (22.31, 73.18), "Thiruvananthapuram": (8.52, 76.94), "Mysuru": (12.30, 76.64),
"Guwahati": (26.14, 91.74), "Bhubaneswar": (20.30, 85.82), "Ranchi": (23.34, 85.31), "Raipur": (21.25, 81.63), "Dehradun": (30.32, 78.03), "Goa": (15.50, 73.83), "Madurai": (9.93, 78.12), "Nashik": (19.99, 73.79)}
SKUS = [("Buns", "Food"), ("Cooking Oil", "Food"), ("Sauce", "Food"), ("Frozen Fries", "Food"), ("Chicken", "Food"), ("Cheese", "Food"), ("Lettuce", "Food"), ("Packaging", "Supplies"), ("Beverage Syrup", "Beverage"), ("Potatoes", "Food"), ("Flour", "Food"), ("Spice Mix", "Food")]
def scale_franchise(outlets=120, staff=1250, per_outlet_sku=12):
    R = random.Random(42)
    with _DB_LOCK, conn() as c:
        c.executescript("""CREATE TABLE IF NOT EXISTS outlet_geo(outlet_id TEXT PRIMARY KEY, lat REAL, lon REAL);
        CREATE TABLE IF NOT EXISTS customers_fr(customer_id TEXT PRIMARY KEY, outlet_id TEXT, total_spend_90d REAL, visit_freq REAL, avg_order_value REAL, days_since_last_visit INT);""")
        n = c.execute("SELECT COUNT(*) FROM outlets").fetchone()[0]; tiers = ["Metro Flagship", "Urban", "Suburban", "Highway"]; cities = list(CITIES)
        while n < outlets:
            n += 1; oid = f"OUT-{n:03d}"; city = R.choice(cities); rev = R.uniform(900000, 5200000)
            c.execute("INSERT OR REPLACE INTO outlets VALUES(?,?,?,?,?,?,?,?)", (oid, f"Franchise Outlet {n:03d}", city, R.choice(tiers), rev, rev * R.uniform(.55, .82), round(R.uniform(3.0, 4.9), 2), R.randint(12, 30)))
        for oid, city in c.execute("SELECT outlet_id, location FROM outlets").fetchall():
            if not c.execute("SELECT 1 FROM outlet_geo WHERE outlet_id=?", (oid,)).fetchone():
                la, lo = CITIES.get(city, CITIES["Mumbai"]); c.execute("INSERT INTO outlet_geo VALUES(?,?,?)", (oid, la + R.uniform(-.06, .06), lo + R.uniform(-.06, .06)))
        s = c.execute("SELECT COUNT(*) FROM staff").fetchone()[0]; oids = [r[0] for r in c.execute("SELECT outlet_id FROM outlets")]
        while s < staff:
            s += 1; sal = R.uniform(18000, 80000); sat = R.randint(1, 5); ot = R.uniform(0, 35)
            p = max(.05, min(.95, .9 - sal / 100000 - sat / 10 + ot / 100))
            c.execute("INSERT OR REPLACE INTO staff VALUES(?,?,?,?,?,?,?,?,?,?,?)", (f"STF-{s:04d}", R.choice(oids), f"Employee {s:04d}", R.choice(["Crew", "Crew", "Cashier", "Cook", "Shift Lead", "Store Manager"]), sal, round(ot, 1), sat, R.randint(19, 58), R.randint(0, 12), R.randint(1, 5), round(p, 2)))
        for oid in oids:
            have = {r[0] for r in c.execute("SELECT sku_name FROM inventory WHERE outlet_id=?", (oid,))}
            for sku, cat in SKUS[:per_outlet_sku]:
                if sku in have or sku in {"Oil"} and "Cooking Oil" == sku: continue
                wk = R.uniform(80, 700); lead = R.randint(2, 9)
                c.execute("INSERT OR REPLACE INTO inventory VALUES(?,?,?,?,?,?,?,?,?)", (f"INV-{oid[4:]}-{sku[:3].upper()}", oid, sku, cat, int(wk / 7 * (lead + R.uniform(2, 12))), int(wk / 7 * lead), round(wk, 1), lead, .1))
            if c.execute("SELECT COUNT(*) FROM marketing WHERE outlet_id=?", (oid,)).fetchone()[0] < 2:
                for j in range(2):
                    c.execute("INSERT OR REPLACE INTO marketing VALUES(?,?,?,?,?,?,?,?,?,?)", (f"CMP-{oid[4:]}-{j}", oid, f"Campaign {oid[4:]}-{j}", R.choice(["Social", "Email", "Influencer", "Offline"]), R.uniform(20000, 120000), round(R.uniform(1.2, 5.2), 2), R.randint(8000, 60000), R.randint(200, 2500), "2024-01-01", "2024-12-31"))
            if c.execute("SELECT COUNT(*) FROM audits WHERE outlet_id=?", (oid,)).fetchone()[0] < 2:
                for j in range(2):
                    sc = max(40, min(100, R.gauss(84, 10)))
                    c.execute("INSERT OR REPLACE INTO audits VALUES(?,?,?,?,?,?,?,?)", (f"AUD-{oid[4:]}-{j}", oid, (datetime.date.today() - datetime.timedelta(days=30 * (j + 1))).isoformat(), sc, int((100 - sc) / 6), R.choice(["Food Safety", "Hygiene", "SOP Compliance"]), "Pass" if sc >= 80 else "Warning" if sc >= 60 else "Fail", "Monthly audit"))
            if c.execute("SELECT COUNT(*) FROM feedback WHERE outlet_id=?", (oid,)).fetchone()[0] < 8:
                for j in range(8):
                    r = R.choices([1, 2, 3, 4, 5], [5, 8, 15, 35, 37])[0]
                    c.execute("INSERT OR REPLACE INTO feedback VALUES(?,?,?,?,?,?)", (f"FB-{oid[4:]}-{j}", oid, r, R.choice(["Great taste", "Slow service", "Clean ambience", "Staff was rude", "Cold food", "Good value", "Long wait time", "Dirty tables"]), datetime.date.today().isoformat(), (r - 3) / 2))
            if c.execute("SELECT COUNT(*) FROM customers_fr WHERE outlet_id=?", (oid,)).fetchone()[0] < 30:
                for j in range(30):
                    seg = R.choices(["hv", "freq", "churn"], [.2, .4, .4])[0]
                    f = {"hv": (6, 12), "freq": (3, 8), "churn": (.2, 1.5)}[seg]; aov = R.uniform(250, 900) if seg == "hv" else R.uniform(150, 450)
                    fr = R.uniform(*f); c.execute("INSERT OR REPLACE INTO customers_fr VALUES(?,?,?,?,?,?)", (f"C-{oid[4:]}-{j}", oid, round(fr * aov * 3, 0), round(fr, 1), round(aov, 0), R.randint(60, 120) if seg == "churn" else R.randint(1, 20)))

In [ ]:
%%writefile /content/app/seanav.py
"""Sea navigation: an ocean mask built from the world coastline (Natural Earth 110m land, bundled with the app), real canal/strait corridors
(Suez, Panama, Malacca, Gibraltar, Dover, Danish straits, Bosphorus, Hormuz, Sunda, Lombok...) and Dijkstra routing over a 1 degree water grid.
Routes are then straightened with a line-of-sight check on a finer 0.25 degree mask, so no leg crosses land. Everything is pure numpy/scipy."""
import os, json, math, heapq, threading
import numpy as np
from matplotlib.path import Path as MPath
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra, connected_components

HERE = os.path.dirname(os.path.abspath(__file__))
TOPO = [os.path.join(HERE, "static", "vendor", "world.topo.json")]
R_NM = 3440.065
ACCESS = {   # river / bay / canal-side ports: where a ship actually enters open water (lat, lon). Matched on the port name.
    "wuhan": (31.2, 122.6), "nanjing": (31.2, 122.6), "taicang": (31.3, 122.4), "shanghai": (31.1, 122.4), "guangzhou": (22.0, 114.0), "dongguan": (22.0, 114.0), "zhuhai": (21.9, 113.8),
    "tokyo": (34.9, 139.9), "yokohama": (34.9, 139.9), "chiba": (34.9, 139.9), "nagoya": (34.2, 137.0), "osaka": (34.2, 135.0), "kobe": (34.2, 135.0), "hamburg": (54.1, 8.2), "antwerp": (51.45, 3.4),
    "houston": (29.1, -94.6), "new orleans": (28.9, -89.3), "baltimore": (37.0, -75.9), "norfolk": (36.9, -75.9), "montreal": (49.3, -66.0), "palembang": (-2.0, 105.4), "yangon": (15.7, 96.3), "mongla": (21.6, 89.5), "kolkata": (21.6, 88.2),
    "ho chi minh": (10.3, 107.1), "kandla": (22.8, 69.6), "navlakhi": (22.9, 70.0), "dahej": (21.7, 72.0), "hazira": (21.0, 72.4), "ambarli": (40.9, 28.3), "izmir": (38.4, 26.6), "amsterdam": (52.5, 4.3), "london gateway": (51.5, 1.2),
    "oslo": (59.2, 10.6), "douala": (3.9, 9.4), "fuzhou": (26.0, 119.9), "lianyungang": (34.7, 119.6), "port klang": (2.9, 101.0), "belawan": (4.1, 98.6), "rotterdam": (51.95, 3.9), "buenos aires": (-35.0, -57.0), "montevideo": (-35.0, -56.2), "guayaquil": (-2.8, -80.4),
    "seattle": (47.9, -122.7), "tacoma": (47.8, -122.8), "vancouver": (49.2, -123.4), "oakland": (37.8, -122.6), "kuching": (2.0, 110.3), "bangkok": (13.3, 100.6), "haikou": (20.3, 110.3), "port said": (31.4, 32.3), "ain sokhna": (29.6, 32.6), "aqaba": (28.9, 34.9),
}
CORRIDORS = {   # name -> centre line (lat, lon); cells along it (and 0.3 degree either side) are forced to water
    "Suez Canal": [(31.9, 32.3), (31.3, 32.33), (30.6, 32.34), (30.0, 32.55), (29.4, 32.55), (28.9, 32.8), (28.3, 33.2)],
    "Panama Canal": [(9.9, -79.95), (9.35, -79.88), (9.05, -79.6), (8.6, -79.5), (8.0, -79.4)],
    "Strait of Malacca": [(1.1, 104.3), (1.25, 103.7), (1.9, 102.6), (2.7, 101.5), (3.6, 100.5), (4.6, 99.9), (5.6, 99.0), (6.3, 97.5)],
    "Singapore Strait": [(1.2, 104.0), (1.4, 104.6), (2.2, 105.5)],
    "Strait of Gibraltar": [(36.5, -7.0), (36.0, -5.8), (36.0, -5.3), (36.2, -4.5), (36.4, -3.5)],
    "Dover Strait": [(49.6, -6.0), (49.8, -3.0), (50.4, -0.5), (50.9, 1.1), (51.4, 2.1), (52.0, 3.0)],
    "Danish Straits": [(57.7, 7.5), (57.8, 10.3), (57.0, 11.6), (56.2, 12.5), (55.5, 12.8), (55.0, 13.0), (54.9, 14.0), (55.2, 16.0), (56.0, 18.0), (57.5, 19.7), (58.8, 21.0), (59.6, 23.5), (59.9, 26.0), (60.0, 29.0)],
    "Bosphorus": [(39.9, 25.9), (40.2, 26.5), (40.7, 27.7), (41.0, 28.9), (41.4, 29.15), (42.0, 29.4), (42.8, 29.8)],
    "Strait of Hormuz": [(24.5, 58.6), (25.7, 56.8), (26.5, 56.3), (26.4, 54.8), (26.6, 52.8), (27.4, 51.0), (28.8, 49.8), (29.6, 48.9)],
    "Bab-el-Mandeb": [(12.0, 43.9), (12.7, 43.3), (13.6, 42.6)],
    "Sunda Strait": [(-5.7, 105.9), (-6.1, 105.8), (-6.8, 105.3)],
    "Lombok Strait": [(-7.6, 115.7), (-8.4, 115.7), (-9.2, 115.6)],
    "Messina": [(38.4, 15.6), (38.0, 15.6), (37.5, 15.3)],
    "Seto Inland Sea": [(33.9, 135.0), (34.4, 134.9), (34.5, 135.1)],
    "Korea Strait": [(35.0, 129.2), (34.3, 129.8), (33.5, 130.3)],
    "Aegean": [(36.2, 22.8), (36.6, 24.0), (37.5, 24.2), (38.5, 24.6), (39.5, 24.6), (40.0, 25.8)],
    "Adriatic": [(39.8, 18.9), (41.0, 18.7), (42.5, 17.5), (43.8, 15.3), (45.0, 13.5), (45.6, 13.4)],
    "Red Sea": [(27.5, 34.2), (24.0, 36.2), (20.0, 38.4), (16.0, 41.2), (13.5, 42.8)],
    "Gulf of Aden": [(12.5, 43.5), (12.0, 47.0), (12.4, 51.5), (12.0, 54.5)],
    "Persian Gulf": [(24.5, 58.6), (25.5, 56.7), (26.4, 54.5), (27.0, 52.0), (28.0, 50.5), (29.3, 48.8)],
    "Gulf of Mexico": [(24.0, -84.0), (23.0, -87.0), (26.5, -90.0), (28.0, -91.0), (28.0, -95.0), (26.0, -97.0), (21.0, -97.0), (19.0, -95.0)],
    "Baltic South": [(54.2, 18.9), (54.6, 19.5)],
}
BLOCK_GROUPS = {"suez": ["Suez Canal"], "panama": ["Panama Canal"], "malacca": ["Strait of Malacca"], "hormuz": ["Strait of Hormuz"], "bosphorus": ["Bosphorus"]}
CHOKE_LABEL = {"Suez Canal": "Suez Canal", "Panama Canal": "Panama Canal", "Strait of Malacca": "Strait of Malacca", "Strait of Gibraltar": "Gibraltar", "Dover Strait": "English Channel", "Strait of Hormuz": "Hormuz", "Bab-el-Mandeb": "Bab-el-Mandeb", "Bosphorus": "Bosphorus", "Danish Straits": "Danish Straits", "Sunda Strait": "Sunda Strait", "Lombok Strait": "Lombok Strait"}

def _hav(a, b):
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1])); h = math.sin((la2 - la1) / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    return R_NM * 2 * math.asin(min(1.0, math.sqrt(h)))

def _land_polys():
    p = next((x for x in TOPO if os.path.exists(x)), None)
    if not p: raise FileNotFoundError("world.topo.json (bundled with the app) not found")
    t = json.load(open(p)); sx, sy = t["transform"]["scale"]; tx, ty = t["transform"]["translate"]; arcs = []
    for a in t["arcs"]:
        x = y = 0; pts = []
        for dx, dy in a: x += dx; y += dy; pts.append((x * sx + tx, y * sy + ty))
        arcs.append(pts)
    def ring(idx):
        out = []
        for i in idx:
            seg = arcs[i] if i >= 0 else arcs[~i][::-1]; out += seg if not out else seg[1:]
        return out
    polys = []
    def geoms(o):
        if o["type"] == "GeometryCollection":
            for g in o["geometries"]: yield from geoms(g)
        else: yield o
    for g in geoms(t["objects"]["land"]):
        pl = g["arcs"] if g["type"] == "MultiPolygon" else [g["arcs"]]
        for poly in pl: polys.append([_unwrap_ring(ring(r)) for r in poly])        # [outer, holes...]
    return polys
def _unwrap_ring(pts):
    """Rings that cross the antimeridian jump by ~360 degrees between two points; make the longitude continuous (it may then exceed +-180)."""
    out = [pts[0]]
    for x, y in pts[1:]:
        px = out[-1][0]
        while x - px > 180: x -= 360
        while x - px < -180: x += 360
        out.append((x, y))
    return out

def gc_arr(a, b, step_nm=60.0, include_ends=True):
    """Vectorised great-circle sampling: (lat array, lon array); longitudes continuous with a's."""
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1]))
    v1 = np.array([math.cos(la1) * math.cos(lo1), math.cos(la1) * math.sin(lo1), math.sin(la1)]); v2 = np.array([math.cos(la2) * math.cos(lo2), math.cos(la2) * math.sin(lo2), math.sin(la2)])
    om = math.acos(max(-1.0, min(1.0, float(v1 @ v2)))); n = max(1, int(om * R_NM / step_nm)); k = np.arange(0 if include_ends else 1, n + 1 if include_ends else n); t = k / n
    if om > 1e-9: w1 = np.sin((1 - t) * om) / math.sin(om); w2 = np.sin(t * om) / math.sin(om)
    else: w1, w2 = 1 - t, t
    v = w1[:, None] * v1[None, :] + w2[:, None] * v2[None, :]; la = np.degrees(np.arcsin(np.clip(v[:, 2], -1, 1))); lo = np.degrees(np.arctan2(v[:, 1], v[:, 0]))
    lo = a[1] + (((lo - a[1]) + 180) % 360 - 180)          # continuous with a's longitude
    return la, lo
def gc_points(a, b, step_nm=60.0, include_ends=True):
    """Points along the great circle from a to b (lat, lon), spaced about step_nm apart; longitudes continuous with a's."""
    la, lo = gc_arr(a, b, step_nm, include_ends); out = list(zip(la.tolist(), lo.tolist()))
    for q in range(1, len(out)):          # keep successive longitudes continuous (no jumps > 180)
        while out[q][1] - out[q - 1][1] > 180: out[q] = (out[q][0], out[q][1] - 360)
        while out[q][1] - out[q - 1][1] < -180: out[q] = (out[q][0], out[q][1] + 360)
    return out

class Sea:
    def __init__(self, step=1.0, fine=0.25, latmin=-78.0, latmax=82.0):
        self.step, self.fine, self.latmin, self.latmax = step, fine, latmin, latmax
        self.lats = np.arange(latmin + step / 2, latmax, step); self.lons = np.arange(-180 + step / 2, 180, step); self.nlat, self.nlon = len(self.lats), len(self.lons)
        self.flats = np.arange(latmin + fine / 2, latmax, fine); self.flons = np.arange(-180 + fine / 2, 180, fine)
        polys = _land_polys(); self.land = self._raster(polys, self.lats, self.lons); self.fland = self._raster(polys, self.flats, self.flons)
        self.corr_cells = {}                                                   # corridor name -> set of coarse node ids
        for name, line in CORRIDORS.items(): self._carve(name, line)
        pol = (self.lats < -52) | (self.lats > 76); drake = (self.lons > -82) & (self.lons < -48)
        self.land[np.ix_(pol & (self.lats < 0), ~drake)] = True; self.land[np.ix_(self.lats > 76, np.ones(self.nlon, bool))] = True
        fpol = (self.flats < -52); fdrake = (self.flons > -82) & (self.flons < -48); self.fland[np.ix_(fpol, ~fdrake)] = True; self.fland[self.flats > 76, :] = True
        self.ocean = ~self.land; self.nid = -np.ones(self.land.shape, int); self.nid[self.ocean] = np.arange(int(self.ocean.sum()))
        self.nodes = np.argwhere(self.ocean); self.n = len(self.nodes); self._build_graph()
        self._cache = {}; self._variants = {}; self._lock = threading.RLock()
    def _raster(self, polys, lats, lons):
        LON, LAT = np.meshgrid(lons, lats); pts = np.c_[LON.ravel(), LAT.ravel()]; inside = np.zeros(len(pts), bool)
        for rings in polys:
            outer = np.array(rings[0]); x0, y0 = outer.min(0); x1, y1 = outer.max(0)
            for off in (0, 360, -360):
                mo = (pts[:, 0] + off >= x0) & (pts[:, 0] + off <= x1) & (pts[:, 1] >= y0) & (pts[:, 1] <= y1)
                if not mo.any(): continue
                idx = np.where(mo)[0]; q = pts[idx] + np.array([off, 0]); ins = MPath(outer).contains_points(q)
                for h in rings[1:]: ins &= ~MPath(np.array(h)).contains_points(q)
                inside[idx[ins]] = True
        return inside.reshape(len(lats), len(lons))
    def _carve(self, name, line):
        cells = set()
        for (a, b) in zip(line[:-1], line[1:]):
            n = max(2, int(_hav(a, b) / 8))
            for t in np.linspace(0, 1, n):
                la, lo = a[0] + (b[0] - a[0]) * t, a[1] + (b[1] - a[1]) * t
                i = int((la - self.latmin) / self.step); j = int((lo + 180) / self.step) % self.nlon
                if 0 <= i < self.nlat: self.land[i, j] = False; cells.add((i, j))
                fi = int((la - self.latmin) / self.fine); fj = int((lo + 180) / self.fine) % len(self.flons)
                for di in (-1, 0, 1):
                    for dj in (-1, 0, 1):
                        a_, b_ = fi + di, (fj + dj) % len(self.flons)
                        if 0 <= a_ < len(self.flats): self.fland[a_, b_] = False
        self.corr_cells[name] = cells
    def _build_graph(self):
        rows, cols, w = [], [], []
        for k, (i, j) in enumerate(self.nodes):
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    if di == 0 and dj == 0: continue
                    a, b = i + di, (j + dj) % self.nlon
                    if not (0 <= a < self.nlat) or not self.ocean[a, b]: continue
                    if di != 0 and dj != 0 and not (self.ocean[i, b] or self.ocean[a, j]): continue         # no diagonal squeeze between two land cells
                    rows.append(k); cols.append(self.nid[a, b]); w.append(_hav((self.lats[i], self.lons[j]), (self.lats[a], self.lons[b])))
        self.A = csr_matrix((w, (rows, cols)), shape=(self.n, self.n))
        nc, lab = connected_components(self.A, directed=False); self.main = lab == np.bincount(lab).argmax(); self.comp = lab
    # ------------------------------------------------------------ helpers
    def is_land(self, la, lo):
        i = int((la - self.latmin) / self.fine); j = int(((lo + 180) % 360) / self.fine)
        return bool(self.fland[i, j]) if 0 <= i < len(self.flats) else True
    def dist_to_water_nm(self, la, lo, maxr=2.0):
        """Distance in nm from a point to the nearest water cell of the fine mask (0 if the point is in water)."""
        i0 = int((la - self.latmin) / self.fine); j0 = int(((lo + 180) % 360) / self.fine); r = int(maxr / self.fine); best = None
        for di in range(-r, r + 1):
            for dj in range(-r, r + 1):
                i, j = i0 + di, (j0 + dj) % len(self.flons)
                if 0 <= i < len(self.flats) and not self.fland[i, j]:
                    d = _hav((la, lo), (self.flats[i], self.flons[j])); best = d if best is None or d < best else best
        return best
    def _clear(self, a, b, ends=0.6):
        la, lo = gc_arr(a, b, 7.0)
        keep = ~(((np.abs(la - a[0]) < ends) & (np.abs(lo - a[1]) < ends)) | ((np.abs(la - b[0]) < ends) & (np.abs(lo - b[1]) < ends)))
        la, lo = la[keep], lo[keep]
        if not len(la): return True
        i = ((la - self.latmin) / self.fine).astype(int); j = (((lo + 180) % 360) / self.fine).astype(int); ok = (i >= 0) & (i < len(self.flats))
        if not ok.all(): return False
        return not bool(self.fland[i, j].any())
    def _near_nodes(self, la, lo, k=12, main_only=True):
        d = (self.lats[self.nodes[:, 0]] - la) ** 2 + ((((self.lons[self.nodes[:, 1]] - lo + 180) % 360) - 180) * math.cos(math.radians(la))) ** 2
        if main_only: d = np.where(self.main, d, 1e9)
        return np.argsort(d)[:k]
    def _snap(self, la, lo):
        near = self._near_nodes(la, lo)
        for k in near:
            p = (float(self.lats[self.nodes[k, 0]]), float(self.lons[self.nodes[k, 1]]))
            if self._clear((la, lo), p, 0.45): return int(k)
        return int(near[0])
    def _graph_for(self, blocked):
        key = tuple(sorted(blocked))
        if not key: return self.A
        with self._lock:
            if key in self._variants: return self._variants[key]
            ok = np.ones(self.n, bool)
            for g in key:
                for name in BLOCK_GROUPS.get(g, [g]):
                    for (i, j) in self.corr_cells.get(name, ()):
                        if self.nid[i, j] >= 0: ok[self.nid[i, j]] = False
            D = csr_matrix(ok.astype(float)).T if False else None
            A = self.A.multiply(ok[:, None]).multiply(ok[None, :]).tocsr(); A.eliminate_zeros(); self._variants[key] = A; return A
    # ------------------------------------------------------------ public API
    def route(self, o, d, blocked=(), storms=(), oa=None, da=None):
        """o, d = (lat, lon). Returns {"pts": [(lat, lon), ...] (longitudes unwrapped), "nm": float, "via": [names]} or None."""
        key = (round(o[0], 2), round(o[1], 2), round(d[0], 2), round(d[1], 2), oa, da, tuple(sorted(blocked)), tuple((round(s[0]), round(s[1]), round(s[2])) for s in storms))
        with self._lock:
            if key in self._cache: return self._cache[key]
        A = self._graph_for(blocked)
        if storms:
            ok = np.ones(self.n, bool)
            for (sla, slo, rn) in storms:
                dd = np.array([_hav((sla, slo), (self.lats[i], self.lons[j])) for i, j in self.nodes]) if False else None
                la_ = self.lats[self.nodes[:, 0]]; lo_ = self.lons[self.nodes[:, 1]]
                dist = R_NM * np.arccos(np.clip(np.sin(np.radians(la_)) * math.sin(math.radians(sla)) + np.cos(np.radians(la_)) * math.cos(math.radians(sla)) * np.cos(np.radians(lo_ - slo)), -1, 1)); ok &= dist > rn
            A = A.multiply(ok[:, None]).multiply(ok[None, :]).tocsr(); A.eliminate_zeros()
        s, t = self._snap(*(oa or o)), self._snap(*(da or d))
        dist, pred = dijkstra(A, directed=True, indices=s, return_predecessors=True)
        if not np.isfinite(dist[t]): res = None
        else:
            path = [t]
            while path[-1] != s: path.append(pred[path[-1]])
            path.reverse(); pts = [o] + ([oa] if oa else []) + [(float(self.lats[self.nodes[k, 0]]), float(self.lons[self.nodes[k, 1]])) for k in path] + ([da] if da else []) + [d]; pts = self._unwrap(self._straighten(pts)); pts = self._densify(pts)
            res = {"pts": [(round(a, 3), round(b, 3)) for a, b in pts], "nm": round(sum(_hav(pts[i], pts[i + 1]) for i in range(len(pts) - 1)), 1), "via": self._via(pts)}
        with self._lock: self._cache[key] = res
        return res
    def _straighten(self, pts):
        out = [pts[0]]; i = 0
        while i < len(pts) - 1:
            j = len(pts) - 1
            while j > i + 1 and not self._clear(pts[i], pts[j], 0.5): j -= 1
            out.append(pts[j]); i = j
        return out
    @staticmethod
    def _densify(pts, step=90.0):
        out = [pts[0]]
        for a, b in zip(pts[:-1], pts[1:]): out += gc_points(a, b, step)[1:]
        return out
    @staticmethod
    def _unwrap(pts):
        out = [pts[0]]
        for la, lo in pts[1:]:
            prev = out[-1][1]
            while lo - prev > 180: lo -= 360
            while lo - prev < -180: lo += 360
            out.append((la, lo))
        return out
    def _via(self, pts):
        names = []; dense = []
        for a, b in zip(pts[:-1], pts[1:]):
            for t in np.linspace(0, 1, max(2, int(_hav(a, b) / 15))): dense.append((a[0] + (b[0] - a[0]) * t, a[1] + (b[1] - a[1]) * t))
        order = []
        for la, lo in dense:
            i = int((la - self.latmin) / self.step); j = int(((lo + 180) % 360) / self.step) % self.nlon
            for name, cells in self.corr_cells.items():
                if name in CHOKE_LABEL and (i, j) in cells and CHOKE_LABEL[name] not in order: order.append(CHOKE_LABEL[name])
        if any(-47 < la < -30 and 10 < (lo % 360 if lo < 0 else lo) < 35 for la, lo in dense): order.append("Cape of Good Hope")
        if any(la < -50 and (-80 < lo < -55) for la, lo in dense): order.append("Cape Horn / Drake Passage")
        return order
    def options(self, o, d, blocked=(), storms=(), oa=None, da=None):
        """Up to three sea routes: shortest, and the alternatives that avoid Suez / Panama / Malacca when the shortest uses them."""
        base = self.route(o, d, tuple(blocked), storms, oa, da); out = []
        if not base: return out
        def label(r):
            v = r["via"]; return ("via " + " + ".join(v[:3])) if v else "direct sea route"
        out.append({"name": label(base), "pts": base["pts"], "nm": base["nm"], "via": base["via"]})
        for g, nm_ in (("suez", "Suez Canal"), ("panama", "Panama Canal"), ("malacca", "Strait of Malacca")):
            if nm_ in base["via"] and g not in blocked:
                alt = self.route(o, d, tuple(blocked) + (g,), storms, oa, da)
                if alt and abs(alt["nm"] - base["nm"]) > 50 and all(abs(alt["nm"] - x["nm"]) > 50 for x in out): out.append({"name": label(alt) + f" (without {nm_})", "pts": alt["pts"], "nm": alt["nm"], "via": alt["via"]})
        return out

def access_for(name):
    n = (name or "").lower()
    for k, v in ACCESS.items():
        if k in n: return v
    return None
_SEA = None; _SL = threading.Lock(); _VER = "v5"
def _cache_path():
    d = os.environ.get("APP_DATA_DIR") or os.path.dirname(os.path.abspath(__file__)); return os.path.join(d, f"seanav_{_VER}.pkl")
def sea():
    """Singleton. The raster + graph (about 6 s to build) is pickled next to the database, so later starts load in well under a second."""
    global _SEA
    with _SL:
        if _SEA is None:
            import pickle
            p = _cache_path()
            try:
                with open(p, "rb") as f: _SEA = pickle.load(f)
                _SEA._cache = {}; _SEA._variants = {}; _SEA._lock = threading.RLock()
            except Exception:
                _SEA = Sea()
                try:
                    st = {k: v for k, v in _SEA.__dict__.items() if k not in ("_lock", "_cache", "_variants")}
                    class _P: pass
                    o = Sea.__new__(Sea); o.__dict__.update(st)
                    with open(p + ".tmp", "wb") as f: pickle.dump(o, f, protocol=4)
                    os.replace(p + ".tmp", p)
                except Exception: pass
    return _SEA


In [ ]:
%%writefile /content/app/seed_data.py
import random, sqlite3, datetime
import pandas as pd
from db import get_conn

BASE_PORTS = [
    ("JNPT Nhava Sheva (Mumbai)", "India", 3.4, 4, 18.95, 72.95, "Asia"),
    ("Mundra Port", "India", 2.8, 3, 22.84, 69.70, "Asia"),
    ("Chennai Port", "India", 3.1, 4, 13.10, 80.30, "Asia"),
    ("Tuticorin VOC Port", "India", 2.5, 3, 8.75, 78.18, "Asia"),
    ("Cochin Port", "India", 2.2, 3, 9.96, 76.26, "Asia"),
    ("Visakhapatnam Port", "India", 2.9, 3, 17.68, 83.28, "Asia"),
    ("Kolkata Haldia Port", "India", 3.5, 5, 22.03, 88.11, "Asia"),
    ("Kandla Deendayal Port", "India", 3.0, 4, 23.01, 70.22, "Asia"),
    ("New Mangalore Port", "India", 2.3, 3, 12.92, 74.81, "Asia"),
    ("Paradip Port", "India", 3.2, 4, 20.26, 86.67, "Asia"),
    ("Shanghai Port", "China", 4.2, 5, 31.23, 121.47, "Asia"),
    ("Singapore Port", "Singapore", 1.2, 2, 1.29, 103.85, "Asia"),
    ("Busan Port", "South Korea", 1.8, 3, 35.10, 129.04, "Asia"),
    ("Tokyo Port", "Japan", 2.2, 3, 35.62, 139.77, "Asia"),
    ("Colombo Port", "Sri Lanka", 2.7, 3, 6.94, 79.84, "Asia"),
    ("Dubai Jebel Ali Port", "UAE", 1.9, 2, 25.20, 55.27, "Middle East"),
    ("Rotterdam Port", "Netherlands", 1.5, 2, 51.92, 4.47, "Europe"),
    ("Antwerp Port", "Belgium", 2.6, 3, 51.22, 4.40, "Europe"),
    ("Hamburg Port", "Germany", 2.5, 3, 53.55, 9.99, "Europe"),
    ("Los Angeles Port", "USA", 3.6, 4, 33.74, -118.27, "Americas"),
    ("Ningbo-Zhoushan Port", "China", 3.9, 5, 29.87, 121.55, "Asia"),
    ("Shenzhen Port", "China", 3.7, 4, 22.54, 114.05, "Asia"),
    ("Guangzhou Port", "China", 3.3, 4, 23.10, 113.30, "Asia"),
    ("Qingdao Port", "China", 3.4, 4, 36.07, 120.38, "Asia"),
    ("Tianjin Port", "China", 3.6, 4, 39.00, 117.72, "Asia"),
    ("Hong Kong Port", "Hong Kong", 2.1, 3, 22.29, 114.16, "Asia"),
    ("Kaohsiung Port", "Taiwan", 2.4, 3, 22.62, 120.28, "Asia"),
    ("Laem Chabang Port", "Thailand", 2.6, 3, 13.08, 100.88, "Asia"),
    ("Port Klang", "Malaysia", 2.0, 3, 3.00, 101.39, "Asia"),
    ("Tanjung Pelepas Port", "Malaysia", 1.7, 2, 1.36, 103.55, "Asia"),
    ("Jakarta Tanjung Priok", "Indonesia", 3.3, 4, -6.10, 106.88, "Asia"),
    ("Manila Port", "Philippines", 3.5, 4, 14.58, 120.95, "Asia"),
    ("Ho Chi Minh Cat Lai Port", "Vietnam", 3.0, 4, 10.76, 106.77, "Asia"),
    ("Karachi Port", "Pakistan", 3.8, 5, 24.85, 66.98, "Asia"),
    ("Bandar Abbas Port", "Iran", 3.6, 4, 27.19, 56.28, "Middle East"),
    ("Jeddah Islamic Port", "Saudi Arabia", 2.2, 3, 21.49, 39.17, "Middle East"),
    ("Salalah Port", "Oman", 1.8, 2, 17.02, 54.09, "Middle East"),
    ("Piraeus Port", "Greece", 2.3, 3, 37.94, 23.65, "Europe"),
    ("Valencia Port", "Spain", 2.1, 3, 39.44, -0.32, "Europe"),
    ("Algeciras Port", "Spain", 2.0, 2, 36.13, -5.45, "Europe"),
    ("Felixstowe Port", "UK", 2.8, 3, 51.96, 1.35, "Europe"),
    ("Le Havre Port", "France", 2.4, 3, 49.49, 0.11, "Europe"),
    ("Gdansk Port", "Poland", 2.2, 3, 54.35, 18.65, "Europe"),
    ("Genoa Port", "Italy", 2.6, 3, 44.41, 8.93, "Europe"),
    ("Durban Port", "South Africa", 3.4, 4, -29.87, 31.02, "Africa"),
    ("Tangier Med Port", "Morocco", 1.9, 2, 35.88, -5.50, "Africa"),
    ("Lagos Apapa Port", "Nigeria", 4.0, 5, 6.45, 3.37, "Africa"),
    ("New York/New Jersey Port", "USA", 2.9, 3, 40.68, -74.02, "Americas"),
    ("Savannah Port", "USA", 2.7, 3, 32.08, -81.09, "Americas"),
    ("Long Beach Port", "USA", 3.4, 4, 33.75, -118.19, "Americas"),
]


def safe_exec(conn, sql, params=()):
    try:
        conn.execute(sql, params)
    except Exception as e:
        pass

def seed_all():
    with get_conn() as conn:
        # 1. Ports
        safe_exec(conn, "DELETE FROM ports;")
        for i, p in enumerate(BASE_PORTS, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO ports (port_id, port_name, country, congestion_index, avg_dwell_days, lat, lon, region) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"PORT-{i:03d}", p[0], p[1], p[2], p[3], p[4], p[5], p[6])
            )

        # 2. Users
        safe_exec(conn, "INSERT OR REPLACE INTO users (id, email, password_hash, role) VALUES (1, 'admin@infosys.com', 'admin123', 'Admin');")
        safe_exec(conn, "INSERT OR REPLACE INTO users (id, email, password_hash, role) VALUES (2, 'broker@infosys.com', 'admin123', 'Freight Broker');")
        safe_exec(conn, "INSERT OR REPLACE INTO users (id, email, password_hash, role) VALUES (3, 'customer@infosys.com', 'admin123', 'Customer');")

        # 3. Shipments
        safe_exec(conn, "DELETE FROM shipments;")
        carriers_list = ["Maersk Line", "MSC Cargo", "CMA CGM", "COSCO Shipping", "Hapag-Lloyd", "ONE Ocean Express"]
        statuses = ["In Transit", "Customs Hold", "Delivered", "Port Congestion Delay", "Anchorage Pending"]

        for i in range(1, 101):
            p1 = BASE_PORTS[i % len(BASE_PORTS)][0]
            p2 = BASE_PORTS[(i+3) % len(BASE_PORTS)][0]
            shp_id = f"SHP-{i:04d}"

            # Generate the exact 9 fields expected by the database
            weight = round(random.uniform(500.0, 45000.0), 1)
            eta_val = (datetime.date.today() + datetime.timedelta(days=random.randint(2, 30))).isoformat()
            sev = random.randint(1, 5)
            ch_prob = round(random.uniform(0.02, 0.45), 2)
            cong = round(random.uniform(1.0, 4.8), 1)
            d_risk = round((cong / 5.0) * 0.5 + (ch_prob) * 0.3 + (sev / 5.0) * 0.2, 2)
            hs = f"HS-{random.randint(8400, 8900)}"

            # Insert matching the 9 columns exactly
            safe_exec(conn,
                "INSERT OR REPLACE INTO shipments (shipment_id, origin_port, dest_port, carrier, status, eta, weight_kg, hs_code, predicted_delay_risk) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (shp_id, p1, p2, random.choice(carriers_list), random.choice(statuses), eta_val, weight, hs, d_risk)
            )

        # 4. Weather Risks
        safe_exec(conn, "DELETE FROM weather_risks;")
        for i in range(1, 21):
            pname = BASE_PORTS[(i - 1) % len(BASE_PORTS)][0]
            sev = random.randint(1, 4)
            fore = "Category 3 Typhoon Warning" if sev >= 3 else "Clear Maritime Conditions"
            w_spd = round(random.uniform(12.0, 58.0), 1)
            wv_ht = round(random.uniform(0.8, 5.5), 1)
            temp = round(random.uniform(14.0, 38.0), 1)
            safe_exec(conn,
                "INSERT OR REPLACE INTO weather_risks (port_name, current_severity, forecast, wind_speed, wave_height, temperature) VALUES (?, ?, ?, ?, ?, ?);",
                (pname, sev, fore, w_spd, wv_ht, temp)
            )

        # 5. Alerts
        safe_exec(conn, "DELETE FROM alerts;")
        categories = ["Customs Hold", "Typhoon Storm", "Port Congestion", "Vessel Mechanical", "Bunker Fuel Surcharge"]
        severities = ["Critical", "High", "Medium", "Low"]
        for i in range(1, 51):
            shp_id = f"SHP-{i:04d}"
            sev = random.choice(severities)
            cat = random.choice(categories)
            msg = f"Alert #{i:03d}: Severe {cat} operational delay reported on {shp_id}."
            safe_exec(conn,
                "INSERT INTO alerts (shipment_id, severity, category, message, date, resolved) VALUES (?, ?, ?, ?, ?, ?);",
                (shp_id, sev, cat, msg, "2024-08-11", 0)
            )

        # 6. Carriers
        safe_exec(conn, "DELETE FROM carriers;")
        for idx, name in enumerate(carriers_list, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO carriers (carrier_id, name, rating, on_time_pct, avg_cost_index, risk_level) VALUES (?, ?, ?, ?, ?, ?);",
                (f"CAR-{idx:03d}", name, round(random.uniform(3.8, 4.9), 2), round(random.uniform(78, 96), 1), round(random.uniform(0.86, 1.18), 2), "Low")
            )

        # 7. Customers
        safe_exec(conn, "DELETE FROM customers;")
        for i in range(1, 25):
            safe_exec(conn,
                "INSERT OR REPLACE INTO customers (customer_id, name, industry, priority_tier, credit_risk) VALUES (?, ?, ?, ?, ?);",
                (f"CUST-{i:03d}", f"Corporate Client {i:03d}", random.choice(["Food Service", "Retail", "QSR", "Hospitality"]), random.choice(["Platinum", "Gold", "Silver"]), round(random.uniform(0.02, 0.18), 2))
            )

        # 8. Freight Quotes
        safe_exec(conn, "DELETE FROM freight_quotes;")
        for i in range(1, 51):
            base = round(random.uniform(1200, 9000), 2)
            margin_pct = round(random.uniform(9, 24), 1)
            final = round(base * (1 + margin_pct / 100), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO freight_quotes (quote_id, shipment_id, customer_id, base_cost, insurance, customs_fee, fuel_surcharge, final_price, margin_pct, status, created_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"QTE-{i:04d}", f"SHP-{random.randint(1, 50):04d}", f"CUST-{random.randint(1, 20):03d}", base, round(base * 0.02, 2), round(random.uniform(100, 600), 2), round(base * 0.08, 2), final, margin_pct, random.choice(["Draft", "Accepted", "Submitted"]), datetime.date.today().isoformat())
            )

        # 9. Customs Tariffs
        safe_exec(conn, "DELETE FROM customs_tariffs;")

        # ADD THE MISSING LIST HERE:
        cargos = ["Electronics", "Pharmaceuticals", "Automotive Parts", "Textiles", "Heavy Machinery", "Perishables"]

        for i, cargo in enumerate(cargos, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO customs_tariffs (tariff_id, hs_code, cargo_type, origin_country, destination_country, duty_rate, clearance_risk, required_docs, advisory) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"TAR-{i:03d}", f"HS-{8400 + i * 31}", cargo, "India", "UAE", round(random.uniform(4, 16), 2), round(random.uniform(0.08, 0.42), 2), "Commercial invoice, packing list, bill of lading", "Validate HS code and pre-clear high-risk lanes.")
            )

        # 10. Outlets (FranchiseOps)
        outlet_cities = ["Chennai", "Bengaluru", "Hyderabad", "Mumbai", "Pune", "Delhi", "Kochi", "Coimbatore", "Ahmedabad", "Kolkata"]
        tiers = ["Metro Flagship", "Urban", "Express", "Mall"]
        safe_exec(conn, "DELETE FROM outlets;")
        for i in range(1, 51):
            revenue = round(random.uniform(850000, 6400000), 2)
            cost = round(revenue * random.uniform(0.58, 0.82), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO outlets (outlet_id, outlet_name, location, tier, revenue, operating_costs, customer_satisfaction, staff_headcount) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"OUT-{i:03d}", f"Franchise Outlet {i:03d}", random.choice(outlet_cities), random.choice(tiers), revenue, cost, round(random.uniform(3.2, 4.9), 2), random.randint(12, 55))
            )

        # 11. Staff
        roles = ["Store Manager", "Shift Lead", "Crew", "Chef", "Cashier", "Inventory Associate"]
        safe_exec(conn, "DELETE FROM staff;")
        for i in range(1, 151):
            satisfaction = random.randint(1, 5)
            overtime = round(random.uniform(0, 42), 1)
            attrition = min(0.95, max(0.03, 0.55 - satisfaction * 0.08 + overtime * 0.009 + random.uniform(-0.08, 0.08)))
            safe_exec(conn,
                "INSERT OR REPLACE INTO staff (staff_id, outlet_id, name, role, salary, overtime_hrs, job_satisfaction, age, tenure_years, work_life_balance, predicted_attrition_prob) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"STF-{i:04d}", f"OUT-{random.randint(1, 50):03d}", f"Employee {i:04d}", random.choice(roles), round(random.uniform(18000, 95000), 2), overtime, satisfaction, random.randint(19, 56), random.randint(0, 14), random.randint(1, 5), round(attrition, 2))
            )

        # 12. Inventory
        skus = ["Buns", "Cheese", "Sauce", "Chicken", "Paneer", "Coffee Beans", "Packaging", "Oil", "Frozen Fries", "Dessert Mix"]
        safe_exec(conn, "DELETE FROM inventory;")
        for i in range(1, 151):
            demand = round(random.uniform(20, 420), 1)
            threshold = random.randint(30, 180)
            stock = random.randint(5, 420)
            risk = min(0.95, max(0.02, (threshold - stock) / max(threshold, 1) + random.uniform(0.05, 0.28)))
            safe_exec(conn,
                "INSERT OR REPLACE INTO inventory (record_id, outlet_id, sku_name, category, current_stock, reorder_threshold, weekly_demand, lead_time_days, stockout_risk_prob) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"INV-{i:04d}", f"OUT-{random.randint(1, 50):03d}", random.choice(skus), random.choice(["Food", "Beverage", "Packaging", "Consumable"]), stock, threshold, demand, random.randint(1, 9), round(risk, 2))
            )

        # 13. Marketing
        channels = ["Digital Ads", "Social Media", "Local Print", "Influencer Campaign", "Radio Spots"]
        safe_exec(conn, "DELETE FROM marketing;")
        for i in range(1, 51):
            budget = round(random.uniform(15000, 120000), 2)
            roi = round(random.uniform(1.8, 5.4), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO marketing (campaign_id, outlet_id, campaign_name, channel, budget, actual_roi, reach, conversions, start_date, end_date) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"CMP-{i:03d}", f"OUT-{random.randint(1, 50):03d}", f"Campaign {i:03d}", random.choice(channels), budget, roi, random.randint(5000, 80000), random.randint(200, 4500), "2024-01-01", "2024-12-31")
            )

        # 14. Feedback
        safe_exec(conn, "DELETE FROM feedback;")
        comments = ["Great food!", "Slow service", "Clean ambience", "Polite staff", "Average experience"]
        for i in range(1, 101):
            rating = random.randint(1, 5)
            sentiment = round((rating - 3) / 2.0, 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO feedback (feedback_id, outlet_id, rating, comment, date, sentiment_score) VALUES (?, ?, ?, ?, ?, ?);",
                (f"FB-{i:04d}", f"OUT-{random.randint(1, 50):03d}", rating, random.choice(comments), "2024-08-01", sentiment)
            )

        # 15. Audits
        safe_exec(conn, "DELETE FROM audits;")
        categories_audit = ["Food Safety", "Hygiene & Sanitation", "Fire & Safety", "Financial Compliance"]
        for i in range(1, 51):
            score = round(random.uniform(65, 99), 1)
            status = "Pass" if score >= 85 else ("Conditional Pass" if score >= 75 else "Action Required")
            safe_exec(conn,
                "INSERT OR REPLACE INTO audits (audit_id, outlet_id, audit_date, score, violations, category, status, notes) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"AUD-{i:03d}", f"OUT-{random.randint(1, 50):03d}", "2024-08-01", score, int((100-score)/5), random.choice(categories_audit), status, "Audit completed cleanly.")
            )

        conn.commit()

In [ ]:
%%writefile /content/app/server.py
"""Web server: FastAPI JSON API + server-sent events + the single-page front end. Models live on the GPU in this process; the simulator and agent cycles run in worker.py."""
import sqlite3, os, sys, io, re, csv, json, time, asyncio, subprocess, threading, importlib, datetime
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config
from config import KIND, DB_PATH
from fastapi import FastAPI, Request, Response, HTTPException, UploadFile, File, Form, Header
from fastapi.responses import JSONResponse, StreamingResponse, FileResponse, HTMLResponse, PlainTextResponse
from fastapi.staticfiles import StaticFiles
import pandas as pd
import numpy as np
import gpu, qa as QA, analyst as AN, live_engine as LE, entities as EN, blocks as B
import views_core as VC
import auth as AUTH, rbac as RB, copilot_plus as CP, views_platform as VP
from fastapi.responses import FileResponse as _FR
dom = importlib.import_module("domain_franchise" if KIND == "Franchise" else "domain_freight")
V = importlib.import_module("views_franchise" if KIND == "Franchise" else "views_freight")
HERE = os.path.dirname(os.path.abspath(__file__)); STATIC = os.path.join(HERE, "static")
app = FastAPI(title=f"{V.NAME} Platform", docs_url=None, redoc_url=None, openapi_url=None) if os.environ.get("PUBLIC_DOCS") != "1" else FastAPI(title=f"{V.NAME} Platform"); _worker = None; _ready = {"db": False}
import hardening; hardening.install(app, KIND)

@app.middleware("http")
async def sec(request, call_next):
    r = await call_next(request); r.headers["X-Content-Type-Options"] = "nosniff"; r.headers["Referrer-Policy"] = "same-origin"; r.headers["Cache-Control"] = "no-store" if request.url.path.startswith("/api") else r.headers.get("Cache-Control", "public, max-age=300"); return r

def _boot():
    from db import init_db; from seed_data import seed_all
    init_db(); seed_all(); LE.init_live_tables(); AUTH.init_db(); _creds = AUTH.seed_defaults(); _ready["db"] = True
@app.on_event("startup")
def startup():
    global _worker
    _boot()
    if os.environ.get("NO_WORKER") != "1":
        env = {**os.environ, "APP_KIND": KIND}; _worker = subprocess.Popen([sys.executable, os.path.join(HERE, "worker.py")], env=env, cwd=HERE)
@app.on_event("shutdown")
def shutdown():
    if _worker:
        try: _worker.terminate()
        except Exception: pass

# ------------------------------------------------------------------ auth + context
TEMP_PW_OK = ("/api/me", "/api/ping", "/api/profile", "/api/profile/password", "/api/auth/logout")     # all a user with a temporary password may call
def user_of(request: Request):
    """The only place identity enters: reads the Bearer token, re-checks the user row (exists, not locked, token still valid).
    A user whose administrator-set password is still temporary can only open the change-password screen until they choose their own."""
    u, err = AUTH.user_from_request(request)
    if not u: raise HTTPException(401, err or "Please sign in.")
    if u.get("must_change") and request.url.path not in TEMP_PW_OK: raise HTTPException(403, "Your password is temporary. Set a new password first.")
    return u
def ctx(u, params=None): return {"dom": dom, "kind": KIND, "role": u["role"], "email": u["email"], "user": u, "customer_id": u.get("customer_id"), "scope": None, "params": params or {}}
def need(c, cap):
    if not RB.has_cap(c["role"], cap):
        AUTH.log_activity(c["user"]["username"], "access_denied", cap, c["role"])
        raise HTTPException(403, f"Access denied (403): the {c['role']} role does not have the '{cap}' permission.")
def menu_for(u):
    """Pages the role may open (the server also refuses everything else with 403)."""
    return [m for m in RB.nav_for(u["role"]) if m["enabled"]]
def audit(u, event, target="", detail=""):
    LE.audit(u["email"], event, target, detail); AUTH.log_activity(u["username"], event, target, detail)
def ok(msg="Done", **kw): return {"ok": True, "msg": msg, **kw}
def bad(msg): return JSONResponse({"ok": False, "msg": msg}, status_code=400)

@app.get("/api/public")
def public():
    st = LE.get_state("boot", {}) if _ready["db"] else {}
    return {"name": V.NAME, "stage": (st or {}).get("stage", "starting"), "questions": AUTH.SECURITY_QUESTIONS, "email_enabled": AUTH.email_ready(), "jwt_secret": AUTH.JWT_FROM_SECRET, "roles": [{"role": r, "who": RB.ROLE_BLURB[r]} for r in RB.ROLES], "demo_users": ({"Admin": "admin", "Operations Manager": "ops_manager", "Freight Broker": "freight_broker", "Auditor": "auditor", "Customer": "customer_demo"} if os.environ.get("SEED_DEMO_USERS", "1") == "1" else {})}
@app.get("/api/ping")
def ping(request: Request): user_of(request); return {"ok": True}          # heartbeat: keeps the user "Active" while the app is open
@app.get("/api/me")
def me(request: Request):
    u = user_of(request)
    return {"user": AUTH.public_user(u), "kind": KIND, "name": V.NAME, "menu": menu_for(u), "nav": menu_for(u), "caps": sorted(RB.caps(u["role"])), "blurb": RB.ROLE_BLURB.get(u["role"], "")}
# ------------------------------------------------------------------ live pulse
def pulse(full=True):
    if not full: return {"stage": (LE.get_state("boot", {}) or {}).get("stage", "starting")}
    rt = LE.get_state("runtime", {}) or {}; boot = LE.get_state("boot", {}) or {}; d = VC.decision()
    oa = B.Q("SELECT status, severity, COUNT(*) n FROM agent_actions WHERE status IN ('open','proposed') GROUP BY status, severity")
    n = lambda st, sv=None: int(oa[(oa.status == st) & ((oa.severity == sv) if sv else True)].n.sum()) if len(oa) else 0
    return {"stage": boot.get("stage", "starting"), "ticks": rt.get("ticks", 0), "cycles": rt.get("cycles", 0), "last_cycle": rt.get("last_cycle"), "health": d["score"] if d else None, "risk": d["risk"] if d else None, "dec": d["id"] if d else 0,
            "open": n("open"), "critical": n("open", "Critical"), "proposed": 0, "gpu": {"cuda": True, "name": "", "used": 0, "total": 0, "llm": "off", "llm_name": "", "llm_done": 0, "llm_total": 0}, "worker_alive": bool(rt.get("last_cycle"))}
@app.get("/api/pulse")
def pulse_http(request: Request): u = user_of(request); return pulse(RB.has_cap(u["role"], "view"))
@app.get("/api/stream")
async def stream(request: Request):
    u = user_of(request); tok = AUTH.bearer(request); full = RB.has_cap(u["role"], "view")
    async def gen():
        last = None; t0 = time.time()
        while True:
            if await request.is_disconnected(): break
            if (await asyncio.to_thread(AUTH.user_from_token, tok))[0] is None: yield "data: " + json.dumps({"error": "signed_out"}) + "\n\n"; break     # locked / deleted / signed out: stop at once
            try: p = await asyncio.to_thread(pulse, full)
            except Exception as e: p = {"error": str(e)}
            k = json.dumps(p, sort_keys=True, default=str)
            if k != last or time.time() - t0 > 10: last = k; t0 = time.time(); yield f"data: {k}\n\n"
            await asyncio.sleep(1.5)
    return StreamingResponse(gen(), media_type="text/event-stream", headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

# ------------------------------------------------------------------ pages
_TABS = {}
def _tabs_of(c, f):
    if f not in _TABS: _TABS[f] = f(c, None).get("tabs")
    return _TABS[f]
def _page(c, pid, tab):
    if not RB.can_open_page(c["role"], pid):
        AUTH.log_activity(c["user"]["username"], "access_denied", "page:" + pid, c["role"])
        raise HTTPException(403, f"Access denied (403): the {c['role']} role cannot open the '{pid}' page.")
    if pid == "copilot": return B.page("AI Copilot", "Ask about your data in plain English. Every answer is an exact database query, shown under the answer.", [{"t": "chat", "examples": CP.suggestions(c["role"], "", 5)}], [], live=0)
    if pid == "profile": return VP.profile_page(c)
    if pid == "runagents": need(c, "run"); return VP.run_agents_page(c)
    if pid == "admin": need(c, "users"); return VP.admin_page(c)
    f = V.PAGES.get(pid)
    if not f: raise HTTPException(404, "unknown page")
    o = f(c, tab); _TABS[f] = o.get("tabs"); return o
@app.get("/api/page/{pid}")
def page(pid: str, request: Request, tab: str = "", p: str = ""):
    u = user_of(request); c = ctx(u, json.loads(p) if p else {})
    t = time.time(); out = _page(c, pid, tab or None); out["id"] = pid; out["ms"] = round((time.time() - t) * 1000); return JSONResponse(B.clean(out))
@app.get("/api/detail/{eid:path}")
def detail(eid: str, request: Request):
    u = user_of(request); c = ctx(u); k = EN.kind_of(eid)
    if not RB.can_open_entity(u["role"], k): raise HTTPException(403, f"Access denied (403): the {u['role']} role cannot open {k} details.")
    if u["role"] == "Customer":
        own = k == "quote" and len(B.Q("SELECT 1 FROM freight_quotes WHERE quote_id=? AND customer_id=?", (eid, u.get("customer_id"))))
        if not own: raise HTTPException(403, "Access denied (403): you can only open your own quotes.")
        r = B.Q("SELECT final_price, status, created_at FROM freight_quotes WHERE quote_id=?", (eid,)).iloc[0]
        return JSONResponse(B.clean({"title": eid, "sub": r.status, "blocks": [B.kpis(("Price", f"${r.final_price:,.0f}"), ("Status", r.status), ("Created", r.created_at))]}))
    return JSONResponse(B.clean(V.detail(c, eid)))

# ------------------------------------------------------------------ tickets, cycles, policy
@app.post("/api/run_cycle")
def run_cycle(request: Request):
    u = user_of(request); c = ctx(u); need(c, "run"); t = time.time(); LE.run_cycle(dom, "manual", ""); audit(u, "manual_cycle", "all"); return ok(f"Agents finished in {time.time() - t:.1f}s")
@app.post("/api/run_agent")
async def run_agent(request: Request):
    u = user_of(request); c = ctx(u); need(c, "run"); a = (await request.json()).get("agent")
    if a not in dom.AGENTS: raise HTTPException(404, "unknown agent")
    await asyncio.to_thread(_run_one, a); audit(u, "manual_agent", a); return ok(f"{a} re-ran")
def _run_one(agent):
    cid = time.strftime("%m%d-%H%M%S") + "-page"; t = time.time(); s = {"cycle_id": cid, "trigger": "page", "request": "", "queue": [], "results": {}}
    if agent == "digest":
        for a in ("outlet", "inventory", "staff", "marketing", "audit"): s["results"][a] = dom.AGENTS[a](s)
    res = dom.AGENTS[agent](s); LE._log_run(cid, "page", res, int((time.time() - t) * 1000)); s["results"][agent] = res; LE.act(dom, {"actions": res.actions}, cid, {agent: res})

# ------------------------------------------------------------------ admin users

# ------------------------------------------------------------------ digital twin

# ------------------------------------------------------------------ data feeds

# ------------------------------------------------------------------ Freight actions
@app.post("/api/freight/quote")
async def fq_quote(request: Request):
    u = user_of(request); c = ctx(u); need(c, "quote"); d = await request.json()
    if d["o"] == d["d"]: return bad("Origin and destination must differ.")
    if u["role"] == "Customer": d["customer"] = u.get("customer_id")          # a Customer can only quote for their own account, whatever the request says
    rid = dom.submit_request(d["o"], d["d"], d["cargo"], float(d["weight"]), d["hs"], d["customer"], "English", d.get("docs") or []); await asyncio.to_thread(dom.run_quote, rid); audit(u, "quote_created", str(rid), f"{d['o']}->{d['d']}")
    return ok("Quote generated", blocks=B.clean(V.quote_result(rid, customer=(u["role"] == "Customer"))))
@app.post("/api/freight/storm")
async def fq_storm(request: Request):
    PRESET_STORMS = {"Red Sea": (18.0, 40.0), "Arabian Sea": (16.0, 66.0), "Bay of Bengal": (14.0, 88.0), "South China Sea": (14.0, 114.0), "Mediterranean": (35.0, 18.0)}; u = user_of(request); c = ctx(u); need(c, "storm"); d = await request.json() if request.headers.get("content-length", "0") not in ("0", "2") else {}
    la, lo = PRESET_STORMS.get(d.get("area") or "Red Sea"); dom.inject_storm(la, lo, wind=float(d.get("wind", 24))); audit(u, "inject_storm", d.get("area", "Red Sea")); return ok("Storm injected. The weather agent will propose re-routes on its next cycle.", refresh=True)
@app.post("/api/freight/reroute")
async def fq_reroute(request: Request):
    u = user_of(request); c = ctx(u); need(c, "approve"); j = await request.json(); sid = j["shipment"]; opt = j.get("opt")
    idx = int(str(opt).split(".")[0]) - 1 if opt else None; act = {"type": "reroute", "target": sid}
    if idx is not None: act["opt_label"] = str(opt).split(" · ", 1)[-1]; act["opt_idx"] = idx
    with LE._DB_LOCK, LE.conn() as cn: note = dom.apply(act, cn)
    audit(u, "manual_reroute", sid, note); return ok(note or "no change", refresh=True)
@app.post("/api/freight/attach")
async def fq_attach(request: Request):
    u = user_of(request); c = ctx(u); need(c, "resolve")
    d = await request.json(); dom.attach_doc(d["ref"], d["doc"], u["email"]); audit(u, "doc_attached", d["ref"], d["doc"]); return ok("Attached. Quotes re-run immediately; shipments clear on the next customs cycle.", refresh=True)
@app.post("/api/freight/readdoc")
async def fq_readdoc(request: Request, file: UploadFile = File(...)):
    u = user_of(request); need(ctx(u), "resolve"); data = await file.read(); txt = ""
    try:
        if file.filename.lower().endswith(".pdf"):
            import pdfplumber
            with pdfplumber.open(io.BytesIO(data)) as pdf: txt = "\n".join((pg.extract_text() or "") for pg in pdf.pages)
        else: txt = data.decode("utf-8", errors="ignore")
    except Exception as e: return bad(f"Could not read the file: {e}")
    hs = re.findall(r"HS[- ]?\d{2,8}", txt); q = re.findall(r"QTE-L\d+", txt); wt = re.findall(r"(\d+(?:\.\d+)?)\s*(?:t|tons|kg)\b", txt)
    return ok("Read", blocks=[B.table([{"field": "HS code", "value": ", ".join(dict.fromkeys(hs)) or "-"}, {"field": "Quote reference", "value": ", ".join(dict.fromkeys(q)) or "-"}, {"field": "Weights", "value": ", ".join(wt[:3]) or "-"}], ["field", "value"], title="Fields found", h=170, key="ocr"), B.md("```\n" + txt[:2500] + "\n```")])

# ------------------------------------------------------------------ language, knowledge base

# ------------------------------------------------------------------ Copilot (agents first, then the GPU LLM; NLLB for every language) + executive brief
def _sse(ev, **kw): return f"data: {json.dumps({'ev': ev, **kw}, default=str)}\n\n"
SYSTEM = "You are the operations copilot of an agentic platform. Answer ONLY from the EVIDENCE. Quote numbers exactly as given, never invent any. Be concise: a one-line answer, then up to 5 short bullets with the entity ids. If the evidence does not answer, say so. When you use a numbered source, cite it as [1], [2]."
_BRIEF = {}
def _chat_db():
    c = sqlite3.connect(DB_PATH, timeout=30); c.execute("PRAGMA journal_mode=WAL;"); c.execute("CREATE TABLE IF NOT EXISTS copilot_chats(id TEXT PRIMARY KEY, email TEXT, title TEXT, ts TEXT, data TEXT)"); c.execute("CREATE INDEX IF NOT EXISTS ix_chat_email ON copilot_chats(email, ts)"); return c
@app.get("/api/chats")
async def chats_list(request: Request):
    u = user_of(request)
    with _chat_db() as c: rows = c.execute("SELECT id, title, ts FROM copilot_chats WHERE email=? ORDER BY ts DESC LIMIT 60", (u["email"],)).fetchall()
    return {"ok": True, "chats": [{"id": r[0], "title": r[1], "ts": r[2]} for r in rows]}
@app.get("/api/chats/{cid}")
async def chats_get(cid: str, request: Request):
    u = user_of(request)
    with _chat_db() as c: r = c.execute("SELECT title, data FROM copilot_chats WHERE id=? AND email=?", (cid, u["email"])).fetchone()
    if not r: raise HTTPException(404, "chat not found")
    return {"ok": True, "title": r[0], "msgs": json.loads(r[1])}
@app.post("/api/chats/save")
async def chats_save(request: Request):
    u = user_of(request); d = await request.json(); cid = re.sub(r"[^a-zA-Z0-9_-]", "", str(d.get("id") or ""))[:40] or ("c" + str(int(time.time() * 1000))); msgs = d.get("msgs") or []
    if not isinstance(msgs, list) or not msgs: return bad("nothing to save")
    msgs = [{"role": m.get("role"), "text": str(m.get("text", ""))[:20000], "en": (str(m["en"])[:20000] if m.get("en") else None), "lang": m.get("lang"), "sql": str(m.get("sql") or "")[:1500], "rows": m.get("rows"), "table": m.get("table"), "note": str(m.get("note") or "")[:300], "sug": m.get("sug") if isinstance(m.get("sug"), list) else None, "kind": m.get("kind")} for m in msgs[-80:]]
    title = next((m["text"] for m in msgs if m["role"] == "user"), "Chat")[:70]; now = datetime.datetime.now().isoformat(timespec="seconds")
    with _chat_db() as c:
        o = c.execute("SELECT email FROM copilot_chats WHERE id=?", (cid,)).fetchone()
        if o and o[0] != u["email"]: raise HTTPException(403, "not your chat")
        c.execute("INSERT OR REPLACE INTO copilot_chats VALUES(?,?,?,?,?)", (cid, u["email"], title, now, json.dumps(msgs)))
    return {"ok": True, "id": cid}
@app.delete("/api/chats/{cid}")
async def chats_del(cid: str, request: Request):
    u = user_of(request)
    with _chat_db() as c: c.execute("DELETE FROM copilot_chats WHERE id=? AND email=?", (cid, u["email"]))
    return {"ok": True}
@app.post("/api/weather/refresh")
async def wx_refresh(request: Request):
    u = user_of(request); c = ctx(u); need(c, "run")
    if KIND != "Freight": raise HTTPException(404)
    import weather as WX; n, msg = await asyncio.to_thread(WX.refresh); audit(u, "weather_refresh", "ports", msg)
    return ok(("Real weather updated: " if n else "Could not reach Open-Meteo, simulated values stay: ") + msg, refresh=True)

# ------------------------------------------------------------------ reports, exports, profile, invitations


@app.get("/api/health")
def health(): return {"ok": True, "gpu": gpu.status()}
@app.get("/")
def index(): return FileResponse(os.path.join(STATIC, "index.html"), headers={"Cache-Control": "no-store"})
app.mount("/static", StaticFiles(directory=STATIC), name="static")


# ------------------------------------------------------------------ AI Copilot (SQL only: greetings + exact database answers)
@app.post("/api/copilot")
async def copilot(request: Request):
    u = user_of(request); c = ctx(u); need(c, "copilot"); d = await request.json(); q = str(d.get("q", "")).strip()[:500]
    if not q: return bad("Type a question first.")
    t0 = time.time(); r = await asyncio.to_thread(CP.answer, u, q)
    if r.get("kind") in ("answer", "denied", "unknown"): AUTH.log_activity(u["username"], "copilot_" + r["kind"], r.get("intent") or "", q[:120])
    # Admin and Auditor (read everything) may fall back to the older generic SQL engine when the new intents do not match
    if r.get("kind") == "unknown" and u["role"] in ("Admin", "Auditor"):
        try:
            ex = AN.answer(KIND, q, None, VC.decision(), dom) or QA.answer(KIND, q, None, None)
            if ex and ex.get("answer"):
                df = ex.get("df"); tbl = None if df is None or not len(df) else {"cols": [str(x) for x in df.columns], "rows": [[None if (isinstance(v, float) and v != v) else (v.item() if hasattr(v, "item") else v) for v in row] for row in df.head(25).itertuples(index=False, name=None)], "total": int(len(df))}
                r = {"kind": "answer", "text": re.sub(r"\*\*|`", "", ex["answer"]) if False else ex["answer"], "sql": ex.get("sql", ""), "rows": 0 if df is None else int(len(df)), "table": tbl, "note": "", "intent": "legacy"}
        except Exception as e: print("copilot legacy failed:", repr(e), file=sys.stderr)
    r["ms"] = round((time.time() - t0) * 1000); r.setdefault("table", None); r.setdefault("suggestions", [])
    return JSONResponse(B.clean(r))


# ------------------------------------------------------------------ authentication, profile, administration
async def _json(request):
    try: d = await request.json()
    except Exception: d = {}
    return d if isinstance(d, dict) else {}
@app.post("/api/auth/signup")
async def auth_signup(request: Request): return AUTH.signup(await _json(request))
@app.post("/api/auth/login")
async def auth_login(request: Request): return AUTH.login(await _json(request))
@app.post("/api/auth/question")
async def auth_question(request: Request): return {"ok": True, "question": AUTH.question_for(str((await _json(request)).get("username") or ""))}
@app.post("/api/auth/forgot")
async def auth_forgot(request: Request): return AUTH.forgot(await _json(request))
@app.post("/api/auth/reset")
async def auth_reset(request: Request): return AUTH.reset(await _json(request))
@app.post("/api/auth/logout")
async def auth_logout(request: Request):
    u, _ = AUTH.user_from_request(request); AUTH.logout(AUTH.bearer(request))
    if u: AUTH.log_activity(u["username"], "logout", u["email"])
    return {"ok": True}
@app.get("/api/profile")
async def profile_get(request: Request):
    u = user_of(request); return {"ok": True, "user": AUTH.public_user(u), "blurb": RB.ROLE_BLURB.get(u["role"], ""), "pages": [m["label"] for m in menu_for(u) if m["id"] not in ("profile",)], "caps": sorted(RB.caps(u["role"]))}
@app.post("/api/profile/username")
async def profile_username(request: Request): u = user_of(request); return AUTH.change_username(u, (await _json(request)).get("username"))
@app.post("/api/profile/password")
async def profile_password(request: Request):
    u = user_of(request); d = await _json(request); return AUTH.change_password(u, d.get("current"), d.get("password"), d.get("confirm"))
@app.post("/api/profile/avatar")
async def profile_avatar(request: Request): u = user_of(request); return AUTH.set_avatar(u, (await _json(request)).get("avatar"))
@app.post("/api/admin/users/{uid}/password")
async def admin_password(uid: int, request: Request): u = user_of(request); need(ctx(u), "users"); d = await _json(request); return AUTH.admin_set_password(u, uid, d.get("password"), d.get("kind"))
@app.get("/api/admin/overview")
async def admin_overview(request: Request): u = user_of(request); need(ctx(u), "users"); return AUTH.overview()
@app.post("/api/admin/users")
async def admin_add(request: Request): u = user_of(request); need(ctx(u), "users"); return AUTH.admin_add_user(u, await _json(request))
@app.post("/api/admin/users/{uid}/role")
async def admin_role(uid: int, request: Request): u = user_of(request); need(ctx(u), "users"); return AUTH.admin_set_role(u, uid, (await _json(request)).get("role"))
@app.post("/api/admin/users/{uid}/lock")
async def admin_lock(uid: int, request: Request): u = user_of(request); need(ctx(u), "users"); return AUTH.admin_set_lock(u, uid, bool((await _json(request)).get("locked")))
@app.post("/api/admin/users/{uid}/customer")
async def admin_customer(uid: int, request: Request): u = user_of(request); need(ctx(u), "users"); return AUTH.admin_set_customer(u, uid, (await _json(request)).get("customer_id"))
@app.delete("/api/admin/users/{uid}")
async def admin_delete(uid: int, request: Request, confirm: str = ""): u = user_of(request); need(ctx(u), "users"); return AUTH.admin_delete(u, uid, confirm == "yes")


In [ ]:
%%writefile /content/app/views_core.py
"""Shared page builders: helpers, Command Center (tickets, agents, anomalies), Knowledge Graph."""
import json, datetime
import numpy as np, pandas as pd
import live_engine as LE, entities as EN
import blocks as B
from blocks import Q, clean, L, TEAL, BLUE, AMBER, RED, VIOLET, TONE

import rbac as _RB
def can(c, cap): return cap in _RB.caps((c or {}).get("role"))      # capabilities come from rbac.ROLE_CAPS
def latest(agent):
    r = LE.conn().execute("SELECT ts,score,risk,latency_ms,summary,output,cycle_id FROM agent_runs WHERE agent=? ORDER BY run_id DESC LIMIT 1", (agent,)).fetchone()
    if not r: return None
    try: out = json.loads(r[5])
    except Exception: out = {}
    return {"ts": r[0], "score": r[1], "risk": r[2], "ms": r[3], "summary": r[4], "out": out, "cycle": r[6]}
def decision():
    d = Q("SELECT decision_id, ts, score, risk, narrative, evidence FROM decisions ORDER BY decision_id DESC LIMIT 1")
    if not len(d): return None
    try: ev = json.loads(d.evidence[0])
    except Exception: ev = {}
    return {"id": int(d.decision_id[0]), "ts": d.ts[0], "score": float(d.score[0]), "risk": d.risk[0], "narrative": d.narrative[0], "ev": ev}
def scoped(df, c, col="target"):
    return df[df[col].astype(str).str.startswith(c["scope"])] if c.get("scope") and len(df) else df

# ------------------------------------------------------------------ tickets
def tk_df(c, agent=None, types=None, target=None, statuses=("open", "proposed"), limit=300):
    dom = c["dom"]; sql = "SELECT action_id id, severity, status, type, target, detail, owner, assignee, due, note, ts FROM agent_actions WHERE status IN (%s)" % ",".join("?" * len(statuses)); p = list(statuses)
    ts = types or ([t for t, a in dom.ACTION_AGENT.items() if a == agent] if agent else None)
    if ts: sql += f" AND type IN ({','.join('?' * len(ts))})"; p += list(ts)
    if target: sql += " AND target=?"; p.append(target)
    df = Q(sql + " ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT " + str(limit), p)
    return scoped(df, c)
def ticket_block(c, agent=None, types=None, title="Open tickets", target=None, key=None, limit=60):
    """Read-only list of what the agent has raised (type, target, severity, detail). There is no approval workflow in this version."""
    df = tk_df(c, agent, types, target, limit=limit)
    if not len(df): return B.callout("Nothing raised right now.", "info", title)
    return B.table(df, ["id", "ts", B.pill_col("severity"), "type", "target", "detail"], title=title, h=300, key=key or f"tk_{agent or target or 'all'}", open={"col": "target"} if False else None)
def hero(c, agent, title, blurb): return B.hero(agent, title, blurb, latest(agent), can_run=can(c, "run"))
def cross(c, targets, own, title="Flagged by other agents for these entities"):
    t = [str(x) for x in list(dict.fromkeys(targets))][:60]
    if not t: return None
    df = Q("SELECT type, target, severity, detail FROM agent_actions WHERE status IN ('open','proposed') AND target IN (%s)" % ",".join("?" * len(t)), t)
    df["agent"] = df.type.map(lambda x: c["dom"].ACTION_AGENT.get(x, "engine")); df = df[df.agent != own].head(30)
    if not len(df): return None
    return B.table(df, ["target", "agent", "type", B.pill_col("severity"), "detail"], title=title, h=240, open={"col": "target"}, key="cross_" + own)

# ------------------------------------------------------------------ Command Center

# ------------------------------------------------------------------ Approvals inbox (human in the loop)
SOURCE = {"manual_request": "Approval request", "quote_approval": "Quote approval", "ml_review": "ML second opinion", "twin_mitigation": "Digital Twin"}





# ------------------------------------------------------------------ Knowledge graph
def entity_explorer(c):
    kind, dom = c["kind"], c["dom"]; e = (c.get("params") or {}).get("e")
    ents = Q("SELECT DISTINCT target FROM agent_actions WHERE status IN ('open','proposed') LIMIT 300").target.tolist()
    base = Q("SELECT outlet_id e FROM outlets ORDER BY outlet_id").e.tolist() if kind == "Franchise" else Q("SELECT port_name e FROM ports ORDER BY congestion_index DESC LIMIT 60").e.tolist()
    opts = list(dict.fromkeys(([c["scope"]] if c.get("scope") else ents + base))); e = e or opts[0]
    d = EN.describe(kind, dom, e); nodes = [{"id": e, "kind": "focus", "size": 34, "color": "#ffffff", "tip": "selected", "label": True}]; links = []
    for n, k, tip in d["rel"]: nodes.append({"id": n, "kind": k, "size": 16, "tip": tip, "label": True, "open": n if EN.kind_of(n) != "port" or kind == "Freight" else None}); links.append((e, n))
    for r in d["tickets"][d["tickets"].status.isin(["open", "proposed"])].itertuples():
        n = f"#{r.id} {r.type}"; ag = dom.ACTION_AGENT.get(r.type, "engine"); nodes += [{"id": n, "kind": "ticket", "size": 14, "color": TONE.get(r.severity), "tip": r.detail[:120], "label": True}, {"id": ag, "kind": "agent", "size": 26, "color": TEAL, "tip": "agent", "label": True}]; links += [(e, n), (n, ag)]
    seen = set(); nodes = [n for n in nodes if not (n["id"] in seen or seen.add(n["id"]))]
    sel = {"t": "form", "id": "kg_pick", "fields": [{"name": "e", "label": "Entity", "type": "select", "options": opts, "value": e}], "submit": "Explore", "url": "page:kg:explore", "title": None, "tone": "primary", "inline": True}
    info = B.table([{"field": k, "value": v} for k, v in d["info"].items()], ["field", "value"], title="What the platform knows", h=230, key="kg_info") if d["info"] else B.callout("No master record for this entity.", "info")
    tk = d["tickets"]
    return [sel, B.row(B.card("Facts", info, B.table(tk, ["id", B.pill_col("severity"), B.pill_col("status"), "type", "owner", "detail"], title="Why is it flagged", h=260, key="kg_tk")), B.card("Relations", B.chart(B.graph(nodes, links), h=470, open={"graph": True}, key="kg_ent")), ratio=[2, 3])]


In [ ]:
%%writefile /content/app/views_freight.py
"""FreightQuote AI pages (server-driven blocks)."""
import os, re, json, datetime
import numpy as np, pandas as pd
import live_engine as LE, entities as EN
import domain_freight as dom
import blocks as B
from blocks import Q, clean, L, TEAL, BLUE, AMBER, RED, VIOLET, GREEN, TONE
from views_core import can, latest, decision, ticket_block, tk_df, cross, hero, scoped

NAME = "FreightQuote AI"
MENU = [('copilot', 'AI Copilot', 'bot'), ('route', 'Agent 1 · Route AI', 'route'), ('quotes', 'Agent 2 · Spot Quotes', 'dollar'), ('carriers', 'Agent 3 · Carriers', 'truck'), ('weather', 'Agent 4 · Weather Risk', 'cloud'), ('margin', 'Agent 5 · Margin', 'trend'), ('customs', 'Agent 6 · Customs', 'file')]
SEVC = {2: RED, 1: AMBER, 0: BLUE, -1: TEAL}
FLEET_LEGEND = [("Vessel · no ticket", TEAL), ("Vessel · info ticket", BLUE), ("Vessel · warning ticket", AMBER), ("Vessel · critical ticket", RED), ("Port · low congestion (big dot)", "#8b8d98"), ("Port · congestion 3+", "#d98e04"), ("Port · congestion 4+", "#b4262b"), ("Storm cell (circle)", "#e5484d")]
def dport(ports, key, alt=0): return next((p for p in ports if key.lower() in p.lower()), ports[min(alt, len(ports) - 1)])

# ------------------------------------------------------------------ map layers
def fleet_layers(vessels=True, ports=True, storms=True, v=None):
    layers = []
    if ports:
        p = Q("SELECT port_name, lat, lon, congestion_index ci FROM ports"); layers.append({"type": "points", "id": "ports", "pts": [{"id": r.port_name, "lat": r.lat, "lon": r.lon, "color": "#b4262b" if r.ci >= 4 else "#d98e04" if r.ci >= 3 else "#8b8d98", "r": 3 + r.ci, "tip": f"<b>{r.port_name}</b><br>congestion {r.ci:.1f}", "open": r.port_name, "z": 1} for r in p.itertuples()]})
    if vessels:
        v = Q("SELECT v.shipment_id, v.lat, v.lon, v.route_name, v.progress, s.origin_port o, s.dest_port d FROM vessels v JOIN shipments s USING(shipment_id)") if v is None else v
        tk = Q("SELECT target, MAX(CASE severity WHEN 'Critical' THEN 2 WHEN 'Warning' THEN 1 ELSE 0 END) sv FROM agent_actions WHERE status IN ('open','proposed') GROUP BY target").set_index("target").sv
        layers.append({"type": "points", "id": "vessels", "pts": [{"id": r.shipment_id, "lat": r.lat, "lon": r.lon, "color": SEVC[int(tk.get(r.shipment_id, -1))], "r": 5, "tip": f"<b>{r.shipment_id}</b><br>{r.o} → {r.d}<br>{r.route_name} · {r.progress:.0%}", "open": r.shipment_id, "z": 2} for r in v.itertuples()]})
    if storms: layers.append({"type": "circles", "items": [{"lat": s["lat"], "lon": s["lon"], "r_m": float(s.get("r", 700)) * 1852, "color": RED, "tip": f"Storm · wind {s.get('wind', '?')} m/s"} for s in dom.storms()]})
    return layers

# ------------------------------------------------------------------ Command Center: Operations
def ops(c):
    d = decision(); oa = tk_df(c, limit=500); ex = dom.kpi_extra(); fl = Q("SELECT COUNT(*) n FROM vessels").n[0]
    mp = B.map_(fleet_layers(), h=510, view={"center": [20, 60], "zoom": 2}, key="ops_map", legend=FLEET_LEGEND)
    q = Q("SELECT quote_id, ROUND(final_price) price, margin_pct margin, status FROM freight_quotes ORDER BY rowid DESC LIMIT 7"); p = Q("SELECT port_name, ROUND(congestion_index,1) ci FROM ports ORDER BY congestion_index DESC LIMIT 7")
    h = Q("SELECT substr(ts,12,8) t, score FROM decisions ORDER BY decision_id DESC LIMIT 80").iloc[::-1]; by = oa.assign(agent=oa.type.map(lambda t: dom.ACTION_AGENT.get(t, "engine"))).groupby("agent").size().sort_values() if len(oa) else pd.Series(dtype=int)
    bl = [B.kpis(("Fleet health", d["score"] if d else "…", (d or {}).get("risk", "") + " risk", "Critical" if d and d["risk"] == "High" else "Warning" if d and d["risk"] == "Medium" else "Low"), ("Vessels tracked", int(fl)), ("Open tickets", int((oa.status == "open").sum()) if len(oa) else 0), ("Critical", int(((oa.severity == "Critical") & (oa.status == "open")).sum()) if len(oa) else 0, None, "Critical"),
                  ("Awaiting approval", int((oa.status == "proposed").sum()) if len(oa) else 0, "agent proposals", "Warning"), ("Active storms", len(dom.storms()), None, "Critical" if dom.storms() else None))]
    if d: bl.append({"t": "brief", "text": d["narrative"], "id": d["id"], "can_brief": True})
    bl.append(B.row(B.card("Live fleet · click a vessel or port", mp), B.col(B.card("Latest quotes", B.table(q, ["quote_id", B.num_col("price", "price", "money"), B.num_col("margin", "margin %", "num"), B.pill_col("status")], h=230, open={"col": "quote_id"}, key="lq")), B.card("Most congested ports", B.table(p, ["port_name", B.num_col("ci", "index", "bar", 5)], h=230, open={"col": "port_name"}, key="cp"))), ratio=[3, 2]))
    bl.append(B.row(B.card("Health trend", B.chart(B.line(h.t, h.score, ymin=0, ymax=100, colors=[TEAL]), h=220, key="trend")), B.card("Open tickets by agent", B.chart(B.bar(by.index, by.values, horizontal=True, colors=[BLUE]), h=220, key="byag"))))
    if can(c, "storm"): bl.append(B.button("storm", "Inject a demo storm (tests the re-route loop)", "/api/freight/storm", "ghost", payload={}))
    return bl

# ------------------------------------------------------------------ detail drawer
def detail(c, e):
    k = EN.kind_of(e)
    if k == "shipment":
        v = Q("SELECT v.lat, v.lon, v.route_name, v.progress, v.chain, s.origin_port o, s.dest_port d, s.carrier, s.status, s.eta, s.weight_kg, s.hs_code, s.predicted_delay_risk risk FROM vessels v RIGHT JOIN shipments s USING(shipment_id) WHERE s.shipment_id=?", (e,))
        if not len(v): return {"title": e, "blocks": [B.callout("Unknown shipment.", "warn")]}
        r = v.iloc[0]; bl = []
        if pd.notna(r.lat):
            w = dom.weather_at(float(r.lat), float(r.lon), allow_live=False); ch = json.loads(r.chain) if r.chain else []
            bl.append(B.kpis(("Progress", f"{r.progress * 100:.0f}%"), ("Wind m/s", round(w["wind"], 1)), ("Wave m", round(w["wave"], 1)), ("Weather", w["level"], w["src"], TONE.get(w["level"]))))
            bl.append(B.map_([{"type": "lines", "lines": [{"pts": ch, "color": TEAL, "w": 3, "dash": "6 6"}]}, {"type": "points", "id": "me", "pts": [{"id": e, "lat": r.lat, "lon": r.lon, "color": "#1f2430", "r": 8, "tip": e}]}, {"type": "circles", "items": [{"lat": s["lat"], "lon": s["lon"], "r_m": float(s.get("r", 700)) * 1852, "color": RED, "tip": "storm"} for s in dom.storms()]}], h=230, view={"center": [float(r.lat), float(r.lon)], "zoom": 3}, key="d_map", legend=[("This port", "#1f2430"), ("Planned voyage route", TEAL), ("Storm cell (red circle)", RED)]))
        else: bl.append(B.callout(f"Not at sea: status {r.status}.", "info"))
        bl.append(B.table([{"field": "Route", "value": f"{r.o} → {r.d}"}, {"field": "Carrier", "value": r.carrier}, {"field": "Status", "value": r.status}, {"field": "ETA", "value": r.eta}, {"field": "Cargo weight", "value": f"{(r.weight_kg or 0) / 1000:,.1f} t"}, {"field": "HS code", "value": r.hs_code}, {"field": "Predicted delay risk", "value": r.risk}], ["field", "value"], h=250, key="d_f"))
        bl.append(ticket_block(c, target=e, title=f"Tickets for {e}", key="d_tk"))
        try:
            import alts; po, pd_ = dom.port_xy(r.o), dom.port_xy(r.d); start = (float(r.lat), float(r.lon)) if pd.notna(r.lat) else (po[0], po[1]) if po else None
            if start and pd_:
                ao = alts.route_alternatives(dom, start, (pd_[0], pd_[1]), tonnes=float(r.weight_kg or 20000) / 1000, interp=dom.interp, dwell=(pd_[3] if pd_ else 0) * .5); bl += alts.route_blocks(ao, e, "Route alternatives" + (" from the vessel's position" if pd.notna(r.lat) else ""))
                if can(c, "approve") and pd.notna(r.lat) and ao: bl.append(B.form("rr_" + e, [{"name": "opt", "label": "Switch to", "type": "select", "options": [f"{i + 1}. {x['label']} · {x['name']}" for i, x in enumerate(ao)], "value": "1. " + ao[0]["label"] + " · " + ao[0]["name"]}, {"name": "shipment", "type": "hidden", "value": e}], "Re-route this voyage", "/api/freight/reroute", title="Apply a route", sub="Needs approve rights. The vessel is re-planned from its current position and the change is audited.", tone="ghost"))
        except Exception as ex: bl.append(B.callout(f"Route alternatives unavailable: {ex}", "info"))
        return {"title": e, "sub": f"{r.o} → {r.d}", "blocks": bl}
    if k == "quote":
        q = Q("SELECT * FROM freight_quotes WHERE quote_id=?", (e,))
        if not len(q): return {"title": e, "blocks": [B.callout("Unknown quote.", "warn")]}
        r = q.iloc[0]; cost = r.base_cost + r.fuel_surcharge + r.customs_fee + r.insurance
        return {"title": e, "sub": r.status, "blocks": [B.kpis(("Price", f"${r.final_price:,.0f}"), ("Margin", f"{r.margin_pct}%"), ("Status", r.status)), B.chart(B.waterfall(["Base (ML)", "Fuel", "Customs", "Insurance", "Margin"], [r.base_cost, r.fuel_surcharge, r.customs_fee, r.insurance, r.final_price - cost]), "How the price is built", h=260, key="d_wf"), ticket_block(c, target=e, title="Tickets", key="d_tk")]}
    p = Q("SELECT * FROM ports WHERE port_name=?", (e,))
    if not len(p): return {"title": e, "blocks": [B.callout("No detail view for this entity.", "info")]}
    r = p.iloc[0]; w = Q("SELECT current_severity sev, wind_speed, wave_height FROM weather_risks WHERE port_name=?", (e,)); sh = Q("SELECT shipment_id, carrier, status FROM shipments WHERE origin_port=? OR dest_port=? LIMIT 12", (e, e))
    bl = [B.kpis(("Congestion index", round(float(r.congestion_index), 2), None, "Critical" if r.congestion_index >= 4 else None), ("Avg dwell days", r.avg_dwell_days), ("Country", r.country)), B.table(sh, ["shipment_id", "carrier", B.pill_col("status")], title="Shipments via this port", h=250, open={"col": "shipment_id"}, key="d_sh")]
    try:
        import alts; bl += alts.port_alternatives(e)
    except Exception: pass
    if len(w): bl.insert(1, B.kpis(("Weather", w.sev[0], f"wind {w.wind_speed[0]:.0f} km/h · wave {w.wave_height[0]:.1f} m", TONE.get(w.sev[0]))))
    return {"title": e, "sub": f"{r.country} · port", "blocks": bl}

# ------------------------------------------------------------------ pages
def route(c, tab=None):
    v = Q("SELECT COUNT(*) n FROM vessels").n[0]; p = Q("SELECT port_name, country, region, congestion_index ci, avg_dwell_days dw FROM ports"); sh = Q("SELECT status, COUNT(*) n FROM shipments GROUP BY status").set_index("status").n
    ports = sorted(p.port_name); pr = c.get("params") or {}; o = pr.get("o") or dport(ports, "JNPT"); d = pr.get("d") or dport(ports, "Rotterdam", 1); cg = pr.get("cargo") or "Electronics"
    bl = [hero(c, "route_watch", "Route & congestion agent", "Compares Suez and Cape routings with a RandomForest congestion classifier and a live weather check along the lane. Watches every port."),
          B.kpis(("Vessels tracked", int(v)), ("Ports monitored", len(p)), ("Congested (≥ 4)", int((p.ci >= 4).sum()), None, "Warning"), ("Active storms", len(dom.storms())))]
    bl.append(B.card("Live fleet", B.map_(fleet_layers(), h=400, view={"center": [20, 60], "zoom": 2}, key="rt_map", legend=FLEET_LEGEND)))
    top = p.sort_values("ci", ascending=False).head(12)
    bl.append(B.row(B.card("Most congested ports (click)", B.chart(B.bar(top.port_name, top.ci.round(2), horizontal=True, itemcolors=[RED if x >= 4 else AMBER for x in top.ci], max_=5), h=330, open={"bar": "y"}, key="rt_c")), B.card("Shipments by status", B.chart(B.bar(sh.index, sh.values.astype(int), colors=[BLUE]), h=330, key="rt_s"))))
    bl.append(B.form("rt_plan", [{"name": "o", "label": "Origin", "type": "select", "options": ports, "value": o}, {"name": "d", "label": "Destination", "type": "select", "options": ports, "value": d}, {"name": "cargo", "label": "Cargo", "type": "select", "options": list(dom.CARGO), "value": cg}], "Plan route", "page:route", title="Route planner", sub="Same route, weather and congestion logic as the quote pipeline.", tone="ghost"))
    po, pd_ = dom.port_xy(o), dom.port_xy(d)
    if po and pd_ and o != d:
        opts = dom.route_options(po[:2], pd_[:2]); cong = dom.predict_congestion((po[2] + pd_[2]) / 2, (po[3] + pd_[3]) / 2, cg); rows, lines = [], []
        for i, op in enumerate(opts):
            ws = [dom.weather_at(*dom.interp(op["chain"], f), allow_live=False) for f in (0, .25, .5, .75, 1)]; worst = max(ws, key=lambda w: w["wind"]); days = op["nm"] / (14 * 24) + po[3] * .5 + pd_[3] * .5
            rows.append({"route": op["name"], "distance_nm": op["nm"], "transit_days": round(days, 1), "weather_risk": worst["level"], "max_wind_ms": worst["wind"]}); lines.append({"pts": op["chain"], "color": TEAL if i == 0 else AMBER, "w": 4, "dash": None if i == 0 else "8 8", "tip": op["name"]})
        bl.append(B.row(B.card("Route options", B.table(rows, ["route", B.num_col("distance_nm", "nm", "int"), B.num_col("transit_days", "days", "num"), B.pill_col("weather_risk"), B.num_col("max_wind_ms", "max wind m/s", "num")], h=150, key="rt_o"), B.callout(f"Predicted congestion level {cong}/3 (RandomForest: port congestion index, dwell time, month, weekday, cargo).", "info")),
                        B.map_([{"type": "lines", "lines": lines}, {"type": "points", "id": "ends", "pts": [{"id": o, "lat": po[0], "lon": po[1], "color": TEAL, "r": 7, "tip": o}, {"id": d, "lat": pd_[0], "lon": pd_[1], "color": BLUE, "r": 7, "tip": d}]}], h=300, key="rt_pm", view={"bounds": [[min(p[0] for ln in lines[:1] for p in ln["pts"]) - 4, min(p[1] for ln in lines[:1] for p in ln["pts"]) - 4], [max(p[0] for ln in lines[:1] for p in ln["pts"]) + 4, max(p[1] for ln in lines[:1] for p in ln["pts"]) + 4]]}, legend=[("Recommended (sea route)", TEAL), ("Alternative", AMBER)])))
    x = cross(c, p[p.ci >= 4].port_name.tolist(), "route_watch"); bl += [x] if x else []
    bl.append(ticket_block(c, "route_watch", title="Open tickets from this agent", key="tk_rt")); return B.page("Route AI", "Agent 1 · routing, congestion, lane planning", bl, live=10)

def _docs_all(): return sorted({d for s in Q("SELECT docs FROM customs_rules").docs for d in str(s).split(";") if d})
def quote_form(c):
    ports = Q("SELECT port_name FROM ports ORDER BY port_name").port_name.tolist(); cust = Q("SELECT customer_id FROM customers").customer_id.tolist()
    if c.get("role") == "Customer": cust = [c.get("customer_id")]        # a Customer quotes only for their own account
    alld = _docs_all(); ch, base, *_ = dom._required_docs("HS-8501")
    return B.form("quote", [{"name": "o", "label": "Origin", "type": "select", "options": ports, "value": dport(ports, "JNPT")}, {"name": "d", "label": "Destination", "type": "select", "options": ports, "value": dport(ports, "Rotterdam", 1)}, {"name": "cargo", "label": "Cargo", "type": "select", "options": list(dom.CARGO), "value": "Electronics"},
                                  {"name": "weight", "label": "Weight (tons)", "type": "number", "value": 120, "min": 1, "max": 5000}, {"name": "hs", "label": "HS code", "type": "text", "value": "HS-8501"}, {"name": "customer", "label": "Customer", "type": "select", "options": cust, "value": cust[0]},
                                  {"name": "docs", "label": "Documents you are submitting (tick all that apply; the customs agent checks them against the rules for the HS code)", "type": "checks", "options": alld, "value": base}], "Generate quote", "/api/freight/quote", title="New quote", sub="Runs the 7-agent pipeline: route → weather → (re-plan loop) → pricing (RandomForest) → customs → margin → documents → alerts.")
def quote_result(rid, customer=False):
    r = Q("SELECT quote_id, result FROM quote_requests WHERE req_id=?", (rid,))
    if not len(r) or not r.result[0]: return [B.callout("The pipeline did not return a result.", "warn")]
    res = json.loads(r.result[0])
    if customer:      # external shipper: price, decision and transit only (no margin, no internal agent trace)
        bl = [B.callout("Your quote is ready.", "success", ("Quote " + str(r.quote_id[0])) if r.quote_id[0] else "Your quote"), B.kpis(("Price (USD)", f"{res.get('price', 0):,.0f}"), ("Status", res.get("decision")), ("Transit", f"{res.get('days')} days"))]
        if res.get("missing"): bl.append(B.callout("Some customs documents are still missing: " + ", ".join(res["missing"]) + ". Please send them to your freight broker so the quote can be released.", "warn"))
        return bl
    bl = [B.callout(res.get("narrative", ""), "success", "Pipeline result"), B.kpis(("Price (USD)", f"{res.get('price', 0):,.0f}"), ("Decision", res.get("decision")), ("Margin", f"{res.get('margin_pct')}%"), ("Transit", f"{res.get('days')} days"))]
    if res.get("missing"): bl.append(B.callout("Missing documents: " + ", ".join(res["missing"]) + ". Attach them on the Customs page and the quote re-runs.", "warn"))
    tr = Q("SELECT agent, risk, score, latency_ms ms, summary FROM agent_runs WHERE cycle_id LIKE ? ORDER BY run_id", (f"Q{rid}-%",)); bl.append(B.table(tr, ["agent", B.pill_col("risk"), B.num_col("score", "score", "num"), B.num_col("ms", "ms", "int"), "summary"], title="Agent trace for this quote", key="qr_tr"))
    return bl
def _own_quotes_page(c):
    q = Q("SELECT quote_id, final_price price, status, created_at FROM freight_quotes WHERE customer_id=? ORDER BY created_at DESC, quote_id DESC LIMIT 200", (c.get("customer_id"),)); bl = [quote_form(c)]
    bl.append(B.kpis(("My quotes", len(q)), ("Approved / accepted", int(q.status.isin(["Approved", "Accepted"]).sum()), None, "Low"), ("Waiting", int(q.status.isin(["Pending Approval", "Submitted", "Draft", "On Hold"]).sum()), None, "Warning"), ("Total value", f"${q.price.sum():,.0f}")))
    if len(q): bl.append(B.table(q, ["quote_id", B.num_col("price", "price", "money"), B.pill_col("status"), "created_at"], title="My quotes", h=340, open={"col": "quote_id"}, key="qt_own", search=True))
    else: bl.append(B.callout("You have no quotes yet. Fill in the form above to get your first spot quote.", "info"))
    return B.page("My Quotes", "Request a spot quote and follow its status", bl, live=15)
def quotes(c, tab=None):
    if c.get("role") == "Customer": return _own_quotes_page(c)
    bl = [hero(c, "quote_watch", "Spot quotes agent", "Every quote goes through one pipeline: ML price (RandomForest) + 12% fuel + port fees + 15% operating cost, target margin ~20%. Quotes above $50,000 need admin approval.")]
    if can(c, "quote"): bl.append(quote_form(c))
    if can(c, "view"):
        q = Q("SELECT quote_id, customer_id, carrier, final_price price, margin_pct margin, status, created_at FROM freight_quotes ORDER BY created_at DESC, quote_id DESC LIMIT 400"); st = q.status.value_counts()
        bl.append(B.kpis(("Quotes", len(q)), ("Approved", int((q.status == "Approved").sum()), None, "Low"), ("Pending approval", int((q.status == "Pending Approval").sum()), None, "Warning"), ("On hold", int((q.status == "On Hold").sum()), None, "Critical")))
        bl.append(B.row(B.card("Quote decisions", B.chart(B.pie(st.to_dict(), colors={"Approved": TEAL, "Pending Approval": AMBER, "On Hold": RED, "Rejected": "#8b8d98"}), h=280, key="qt_p")), B.card("Margin % across quotes", B.chart(B.hist(q.margin, 24, BLUE), h=280, key="qt_h"))))
        bl.append(B.table(q.head(30), ["quote_id", "customer_id", "carrier", B.num_col("price", "price", "money"), B.num_col("margin", "margin %", "num"), B.pill_col("status"), "created_at"], title="Latest quotes · click a row", h=340, open={"col": "quote_id"}, key="qt_t", search=True))
        bl.append(ticket_block(c, "quote_watch", title="Open tickets from this agent", key="tk_qt"))
    return B.page("Spot Quotes", "Agent 2 · the 7-agent quote pipeline", bl, live=10)

def carriers(c, tab=None):
    k = Q("SELECT name, rating, on_time_pct, avg_cost_index, risk_level FROM carriers"); k["score"] = (k.rating / 5 * 40 + k.on_time_pct * .4 - k.avg_cost_index * 12 - (k.risk_level == "High") * 8).round(1)
    s = Q("SELECT carrier name, COUNT(*) shipments, SUM(status LIKE '%Delay%' OR status='Customs Hold') problems, AVG(predicted_delay_risk) delay_risk FROM shipments GROUP BY carrier"); k = k.merge(s, on="name", how="left").fillna({"shipments": 0, "problems": 0})
    top = k.sort_values("score", ascending=False).head(12); rc = {"Low": TEAL, "Medium": AMBER, "High": RED}
    sc = {t: [{"x": a.on_time_pct, "y": a.avg_cost_index, "label": a.name, "size": 6 + a.shipments ** .5} for a in k[k.risk_level == t].itertuples()] for t in ["Low", "Medium", "High"] if (k.risk_level == t).any()}
    bl = [hero(c, "route_watch", "Carrier intelligence", "Score = rating × 40% + on-time % × 0.4 − cost-index penalty − 8 points for High risk. Shipments and delays come from the live shipments table."),
          B.kpis(("Carriers", len(k)), ("High-risk carriers", int((k.risk_level == "High").sum()), None, "Warning"), ("Avg on-time", f"{k.on_time_pct.mean():.1f}%"), ("Best score", k.sort_values("score").name.iloc[-1])),
          B.row(B.card("Top carriers by agent score", B.chart(B.bar(top.name, top.score, horizontal=True, colors=[TEAL], sort=True), h=360, key="ca_t")), B.card("On-time vs cost index (size = shipments)", B.chart(B.scatter(sc, "on-time %", "cost index", colors=rc, sized=True), h=360, key="ca_s"))),
          B.table(k.sort_values("score", ascending=False), ["name", B.num_col("score", "score", "bar", 100), "rating", B.num_col("on_time_pct", "on-time %", "num"), B.num_col("avg_cost_index", "cost idx", "num"), B.pill_col("risk_level", "risk"), B.num_col("shipments", "shipments", "int"), B.num_col("problems", "problems", "int")], title="Carrier league table", h=340, key="ca_tb", search=True)]
    return B.page("Carriers", "Agent 3 · ranking and risk", bl, live=15)

METRICS = {"wind": ("Wind speed", [("Calm / light  < 8 m/s", TEAL), ("Strong  8–15 m/s", AMBER), ("Storm-force  > 15 m/s", RED)]),
           "waves": ("Wave height", [("Slight  < 2.5 m", TEAL), ("Rough  2.5–4 m", AMBER), ("Very rough  > 4 m", RED)]),
           "rain": ("Rain now", [("Dry", "#8b8d98"), ("Light rain  < 2.5 mm/h", BLUE), ("Heavy rain  ≥ 2.5 mm/h", VIOLET)]),
           "temp": ("Temperature", [("Cold  < 10 °C", BLUE), ("Mild  10–30 °C", TEAL), ("Hot  30–35 °C", AMBER), ("Very hot  > 35 °C", RED)])}
def _wxcolor(metric, r):
    if metric == "wind": return RED if r.wind_ms > 15 else AMBER if r.wind_ms >= 8 else TEAL
    if metric == "waves": w = r.wave_m if r.wave_m == r.wave_m and r.wave_m is not None else 0; return RED if w > 4 else AMBER if w >= 2.5 else TEAL
    if metric == "rain": p = r.precip_mm or 0; return VIOLET if p >= 2.5 else BLUE if p > 0 else "#8b8d98"
    t = r.temp_c if r.temp_c is not None else 25; return RED if t > 35 else AMBER if t >= 30 else TEAL if t >= 10 else BLUE
def wx_layers(metric, hi):
    import weather as WX
    t = WX.table(); layers = []
    if len(t):
        pts = []
        for r in t.itertuples():
            w = r.wave_m if r.wave_m is not None and r.wave_m == r.wave_m else None
            pts.append({"id": r.port_name, "lat": r.lat, "lon": r.lon, "color": _wxcolor(metric, r), "r": 4 + min(9, r.wind_ms / 2.2), "z": 1, "open": r.port_name,
                        "tip": f"<b>{r.port_name}</b> · {WX.WMO.get(r.wcode, '')}<br>wind {r.wind_ms:.1f} m/s ({r.wind_ms * 3.6:.0f} km/h), gusts {r.gust_ms or 0:.0f} m/s<br>waves {'-' if w is None else f'{w:.1f} m'} · rain {r.precip_mm or 0:.1f} mm/h · {r.temp_c:.0f} °C<br><i>real reading {r.ts[11:16]} UTC</i>"})
        layers.append({"type": "points", "id": "wx_ports", "pts": pts})
    else:
        w = Q("SELECT w.port_name, w.current_severity sev, w.wind_speed, w.wave_height, w.temperature, p.lat, p.lon FROM weather_risks w JOIN ports p USING(port_name)")
        layers.append({"type": "points", "id": "wx_ports", "pts": [{"id": r.port_name, "lat": r.lat, "lon": r.lon, "color": RED if r.sev == "High" else AMBER if r.sev == "Medium" else TEAL, "r": 4 + min(9, r.wind_speed / 8), "z": 1, "open": r.port_name, "tip": f"<b>{r.port_name}</b><br>wind {r.wind_speed:.0f} km/h · waves {r.wave_height:.1f} m<br><i>simulated, not a real reading</i>"} for r in w.itertuples()]})
    v = Q("SELECT v.shipment_id, v.lat, v.lon, v.route_name, s.origin_port o, s.dest_port d FROM vessels v JOIN shipments s USING(shipment_id)"); hs = set(hi)
    layers.append({"type": "points", "id": "wx_v", "pts": [{"id": r.shipment_id, "lat": r.lat, "lon": r.lon, "color": "#1f2430" if r.shipment_id in hs else "#6b7080", "r": 4.5 if r.shipment_id in hs else 3, "z": 2, "open": r.shipment_id, "tip": f"<b>{r.shipment_id}</b> {r.o} → {r.d}" + ("<br><b>in storm-force wind</b>" if r.shipment_id in hs else "")} for r in v.itertuples()]})
    layers.append({"type": "circles", "items": [{"lat": s["lat"], "lon": s["lon"], "r_m": float(s.get("r", 700)) * 1852, "color": RED, "tip": f"Storm cell · wind {s.get('wind', '?')} m/s"} for s in dom.storms()]})
    return layers
def weather(c, tab=None):
    import weather as WX
    p = c.get("params") or {}; metric = p.get("metric") if p.get("metric") in METRICS else "wind"; st = WX.status()
    r = (latest("weather_watch") or {}).get("out", {}); hi = r.get("vessels_high", []); w = Q("SELECT port_name, current_severity sev, wind_speed, wave_height, temperature, forecast FROM weather_risks")
    src = (f"Real · Open-Meteo, {st['ok']}/{st['total']} ports, updated {st['ts'][11:16]} UTC ({st['age_min']:.0f} min ago)" if st["live"] else "Simulated · Open-Meteo not reachable" + (f" ({st['err']})" if st["err"] else "") + ". Press Refresh to retry.")
    lg = list(METRICS[metric][1]) + [("Big dot = port (size = wind)", "#8b8d98"), ("Black dot = vessel in storm-force wind", "#1f2430"), ("Grey dot = other vessel", "#6b7080"), ("Red circle = storm cell", "#e5484d")]
    pick = {"t": "form", "id": "wx_pick", "fields": [{"name": "metric", "label": "Colour ports by", "type": "select", "options": list(METRICS), "value": metric}], "submit": "Show", "url": "page:weather", "title": None, "tone": "ghost", "inline": True}
    hot = w.sort_values("wind_speed", ascending=False)
    bl = [B.callout(src, "success" if st["live"] else "warning", "Weather data source"), hero(c, "weather_watch", "Weather risk agent", "Real wind, wave, rain and temperature from Open-Meteo for every port (refreshed every 20 min) plus injected storm cells. Vessels in High risk (wind > 15 m/s) get a re-route ticket; the system re-plans the voyage once it is approved."),
          B.kpis(("Vessels in High risk", len(hi), None, "Critical" if hi else None), ("Active storms", len(dom.storms())), ("Ports with weather data", len(w)), ("Ports in High risk", int((w.sev == "High").sum()), None, "Warning"), ("Data", "Real" if st["live"] else "Simulated", None, "Low" if st["live"] else "Warning")),
          pick, B.card(f"Port weather · coloured by {METRICS[metric][0].lower()} · click a port or vessel", B.map_(wx_layers(metric, hi), h=470, view={"center": [20, 60], "zoom": 2}, key="wx_map", legend=lg)),
          B.row(B.card("Strongest port winds (km/h)", B.chart(B.bar(hot.head(12).port_name, hot.head(12).wind_speed.round(0), horizontal=True, colors=[AMBER], sort=True), h=330, open={"bar": "y"}, key="wx_w")), B.card("Port weather risk levels", B.chart(B.pie({str(k): int(v) for k, v in w.sev.value_counts().items()}, colors={"High": RED, "Medium": AMBER, "Low": TEAL}), h=330, key="wx_d")))]
    if hi: bl.append(B.table(Q("SELECT shipment_id, lat, lon, route_name FROM vessels WHERE shipment_id IN (%s)" % ",".join("?" * len(hi[:50])), hi[:50]), ["shipment_id", B.num_col("lat", "lat", "num"), B.num_col("lon", "lon", "num"), "route_name"], title="Vessels in High-risk weather now · click to inspect", h=240, open={"col": "shipment_id"}, key="wx_hv"))
    bl.append(B.table(w.sort_values("wind_speed", ascending=False).head(60).rename(columns={"wind_speed": "wind km/h", "wave_height": "waves m", "temperature": "temp °C"}), ["port_name", B.pill_col("sev"), "wind km/h", "waves m", "temp °C", "forecast"], title="Port weather (real where the forecast says Open-Meteo live)", h=300, open={"col": "port_name"}, key="wx_t"))
    if can(c, "run"): bl.append(B.button("wx_ref", "Refresh real weather now", "/api/weather/refresh", "ghost", payload={}))
    bl.append(B.table(pd.DataFrame(dom.feed_status(), columns=["feed", "status"]), title="Data sources", h=190, key="wx_f"))
    if can(c, "storm"):
        PRESET_STORMS = {"Red Sea": (18.0, 40.0), "Arabian Sea": (16.0, 66.0), "Bay of Bengal": (14.0, 88.0), "South China Sea": (14.0, 114.0), "Mediterranean": (35.0, 18.0)}; bl.append(B.form("storm", [{"name": "area", "label": "Area", "type": "select", "options": list(PRESET_STORMS), "value": "Red Sea"}, {"name": "wind", "label": "Wind m/s", "type": "range", "min": 15, "max": 35, "value": 24}], "Inject storm", "/api/freight/storm", title="Inject a storm", sub="Adds a storm cell for testing the re-route loop; it does not change the real readings."))
    bl.append(ticket_block(c, "weather_watch", title="Open tickets from this agent", key="tk_wx")); return B.page("Weather Risk", "Agent 4 · real wind, waves, rain, storms", bl, live=10)

def margin(c, tab=None):
    q = Q("SELECT quote_id, customer_id, base_cost, fuel_surcharge fuel, customs_fee, insurance, final_price price, margin_pct margin, status FROM freight_quotes WHERE base_cost>0 ORDER BY created_at DESC, quote_id DESC LIMIT 600")
    bl = [hero(c, "quote_watch", "Margin & approvals agent", "Margin = (price − cost) ÷ price with tiered targets (Platinum 18%, Gold 20%, Silver/Bronze 22–24%). Quotes > $50,000 or < 10% margin wait for an approver."),
          B.kpis(("Average margin", f"{q.margin.mean():.1f}%"), ("Below 10% margin", int((q.margin < 10).sum()), None, "Warning"), ("Over $50,000", int((q.price > 50000).sum())), ("Revenue in quotes", f"${q.price.sum():,.0f}"))]
    sc = {s: [{"x": a.base_cost, "y": a.price, "label": a.quote_id} for a in q[q.status == s].head(250).itertuples()] for s in q.status.unique()}; cm = q.groupby("customer_id").margin.mean().round(1).sort_values().head(10)
    bl.append(B.row(B.card("Price vs base cost (click a dot)", B.chart(B.scatter(sc, "base cost", "price", colors={"Approved": TEAL, "Pending Approval": AMBER, "On Hold": RED, "Rejected": "#8b8d98"}), h=330, open={"scatter": True}, key="mg_s")), B.card("Lowest-margin customers", B.chart(B.bar(cm.index, cm.values, horizontal=True, colors=[AMBER]), h=330, key="mg_c"))))
    if len(q):
        sel = (c.get("params") or {}).get("q") or q.quote_id.iloc[0]; r = q[q.quote_id == sel]; r = r.iloc[0] if len(r) else q.iloc[0]; cost = r.base_cost + r.fuel + r.customs_fee + r.insurance
        bl.append({"t": "form", "id": "mg_pick", "fields": [{"name": "q", "label": "Cost build-up for quote", "type": "select", "options": q.quote_id.tolist()[:200], "value": r.quote_id}], "submit": "Show", "url": "page:margin", "title": None, "tone": "ghost", "inline": True})
        bl.append(B.card(f"{r.quote_id}: how the price is built", B.chart(B.waterfall(["Base (ML)", "Fuel 12%", "Customs", "Insurance", "Margin"], [r.base_cost, r.fuel, r.customs_fee, r.insurance, r.price - cost]), h=300, key="mg_w")))
    bl.append(ticket_block(c, "quote_watch", ["quote_approval", "margin_review"], title="Margin actions", key="tk_mg")); return B.page("Margin", "Agent 5 · margin and pricing tiers", bl, live=10)

def customs(c, tab="customs"):
    tabs = [{"id": "customs", "label": "Customs & tariff"}, {"id": "docs", "label": "Documents (OCR)"}]
    if tab == "docs":
        ddir = os.path.join(dom.DATA_DIR, "docs"); files = sorted(os.listdir(ddir), reverse=True) if os.path.isdir(ddir) else []
        bl = [B.callout("PDFs are generated by the Document agent for every quote (Bill of Lading, Packing List). Upload a document to read its fields and attach it to the matching quote.", "info"), B.kpis(("Generated documents", len(files)))]
        bl += [B.button("dl_" + f, "Download " + f, f"/api/doc/{f}", "ghost", payload={"download": True}) for f in files[:10]]
        bl.append(B.form("readdoc", [{"name": "file", "label": "PDF or TXT", "type": "file"}], "Read document", "/api/freight/readdoc", title="Read an uploaded document (OCR / text extraction)", tone="primary")); return B.page("Customs", "Agent 6 · documents", bl, tabs, live=0)
    h = Q("SELECT shipment_id, hs_code, origin_port, dest_port, carrier FROM shipments WHERE status='Customs Hold' LIMIT 200"); rules = Q("SELECT * FROM customs_rules ORDER BY chapter"); rows = []
    for r in h.head(30).itertuples():
        ch, docs, rs, duty, goods = dom._required_docs(r.hs_code); miss = [d for d in docs if d not in dom._attached(r.shipment_id)]; rows.append({"shipment": r.shipment_id, "hs": r.hs_code, "goods": goods, "duty_pct": duty, "missing": ", ".join(miss), "route": f"{r.origin_port} → {r.dest_port}"})
    bl = [hero(c, "customs_watch", "Customs & tariff agent", "Requirements come from the customs_rules table by HS chapter. A hold lifts as soon as every required document is attached: the quote re-runs and the shipment clears."),
          B.kpis(("Shipments on hold", len(h), None, "Warning" if len(h) else None), ("Quotes on hold", int(Q("SELECT COUNT(*) n FROM freight_quotes WHERE status='On Hold'").n[0])), ("Rules in force", len(rules))),
          B.table(rows, ["shipment", "hs", "goods", B.num_col("duty_pct", "duty %", "num"), "missing", "route"], title="Held shipments and what each is missing · click a row", h=300, open={"col": "shipment"}, key="cu_h"), B.table(rules, list(rules.columns), title="Customs rules", h=230, key="cu_r")]
    if can(c, "resolve") or can(c, "quote"):
        held = Q("SELECT shipment_id ref FROM shipments WHERE status='Customs Hold' UNION SELECT quote_id FROM freight_quotes WHERE status='On Hold' LIMIT 200").ref.tolist(); alld = _docs_all()
        if held: bl.append(B.form("attach", [{"name": "ref", "label": "Shipment or quote on hold", "type": "select", "options": held, "value": held[0]}, {"name": "doc", "label": "Document", "type": "select", "options": alld, "value": alld[0]}], "Attach document", "/api/freight/attach", title="Attach a document", sub="The Customs agent re-checks on its next cycle; quotes re-run immediately."))
    if can(c, "view"): bl.append(ticket_block(c, "customs_watch", title="Open tickets from this agent", key="tk_cu"))
    return B.page("Customs", "Agent 6 · rules, holds, documents", bl, tabs, live=12)

PAGES = {"route": route, "quotes": quotes, "carriers": carriers, "weather": weather, "margin": margin, "customs": customs}


In [ ]:
%%writefile /content/app/rbac.py
"""Role-Based Access Control for FreightQuote AI.

ROLE_MENU  : role -> set of page ids the role may open (also enforced on /api/page/<id>)
ROLE_CAPS  : role -> set of capability names (checked by need(ctx, capability) in every action route)

Capability names
  copilot   use the AI Copilot
  view      read the internal operational pages (tables, agent tickets, margins ...)
  view_own  read only the records that belong to the user's own customer account
  quote     create a spot quote
  run       run agents / refresh weather
  approve   re-route a voyage
  resolve   edit customs data (attach documents to a held shipment or quote)
  storm     inject a demo storm
  users     manage users and roles, read the activity log (Admin dashboard)
"""
ROLES = ["Admin", "Operations Manager", "Freight Broker", "Auditor", "Customer"]
ROLE_BLURB = {
    "Admin": "Platform operator",
    "Operations Manager": "Runs logistics",
    "Freight Broker": "Sells and prices",
    "Auditor": "Compliance reviewer",
    "Customer": "External shipper",
}
PUBLIC_SIGNUP_ROLE = "Customer"          # self-registration always creates a Customer

PAGES = ["copilot", "route", "quotes", "carriers", "weather", "margin", "customs", "runagents", "profile", "admin"]
PAGE_LABEL = {"runagents": "Run agents", "copilot": "AI Copilot", "route": "Route", "quotes": "Quotes", "carriers": "Carriers", "weather": "Weather",
              "margin": "Margin", "customs": "Customs", "admin": "Users (Admin)"}

ROLE_MENU = {
    "Admin":              {"copilot", "route", "quotes", "carriers", "weather", "margin", "customs", "runagents", "profile", "admin"},
    "Operations Manager": {"copilot", "route", "carriers", "weather", "customs", "runagents", "profile"},
    "Freight Broker":     {"copilot", "quotes", "carriers", "margin", "profile"},
    "Auditor":            {"copilot", "route", "quotes", "carriers", "weather", "margin", "customs", "profile"},
    "Customer":           {"copilot", "quotes", "profile"},
}

ROLE_CAPS = {
    "Admin":              {"copilot", "view", "quote", "run", "approve", "resolve", "storm", "users"},
    "Operations Manager": {"copilot", "view", "run", "approve", "resolve"},
    "Freight Broker":     {"copilot", "view", "quote"},
    "Auditor":            {"copilot", "view"},
    "Customer":           {"copilot", "quote", "view_own"},
}

# Which parts of the data each role's Copilot may read (see copilot_plus.py). A question about a domain the role may not read is refused.
ROLE_DATA = {
    "Admin":              {"ports", "carriers", "shipments", "quotes", "weather", "customs", "tickets", "customers"},
    "Operations Manager": {"ports", "carriers", "shipments", "weather", "customs", "tickets"},
    "Freight Broker":     {"ports", "carriers", "quotes", "customers"},
    "Auditor":            {"ports", "carriers", "shipments", "quotes", "weather", "customs", "tickets", "customers"},
    "Customer":           {"own_quotes", "own_shipments"},
}

# Which entity kinds (drawer panels) are reachable from which pages
KIND_PAGES = {"quote": {"quotes", "margin"}, "shipment": {"route", "weather", "customs", "carriers"}, "port": {"route", "weather", "carriers", "customs"}}


def role_of(user):
    r = (user or {}).get("role")
    return r if r in ROLE_MENU else None


def menu_ids(role):
    return ROLE_MENU.get(role, set())


def caps(role):
    return ROLE_CAPS.get(role, set())


def has_cap(role, cap):
    return cap in ROLE_CAPS.get(role, set())


def can_open_page(role, pid):
    return pid in ROLE_MENU.get(role, set())


def can_open_entity(role, kind):
    return bool(KIND_PAGES.get(kind, set()) & ROLE_MENU.get(role, set()))


def permission_rows():
    """Rows for the permission table shown in the README and on the Admin dashboard."""
    cols = [("copilot", "Copilot"), ("route", "Route"), ("quotes", "Quotes"), ("carriers", "Carriers"), ("weather", "Weather"),
            ("margin", "Margin"), ("customs", "Customs")]
    rows = []
    for r in ROLES:
        d = {"role": r, "who": ROLE_BLURB[r]}
        for k, lab in cols: d[lab] = k in ROLE_MENU[r]
        d["Run agents"] = "run" in ROLE_CAPS[r]
        d["Users"] = "users" in ROLE_CAPS[r]
        rows.append(d)
    return rows


def permission_markdown():
    rows = permission_rows(); heads = [k for k in rows[0] if k not in ("role", "who")]
    out = "| Role | " + " | ".join(heads) + " |\n|---|" + "---|" * len(heads) + "\n"
    for r in rows: out += f"| {r['role']} | " + " | ".join("✓" if r[h] else "✗" for h in heads) + " |\n"
    return out


# Sidebar: every item is listed; the ones the role may not open are shown greyed out and disabled. (id, label, icon, group)
NAV = [("copilot", "AI Copilot", "bot", "Assistant"), ("route", "Route AI", "route", "Agents"), ("quotes", "Spot Quotes", "dollar", "Agents"), ("carriers", "Carriers", "truck", "Agents"),
       ("weather", "Weather Risk", "cloud", "Agents"), ("margin", "Margin", "trend", "Agents"), ("customs", "Customs", "file", "Agents"),
       ("runagents", "Run agents", "play", "Control"), ("admin", "Users", "users", "Control")]

def nav_for(role):
    return [{"id": i, "label": l, "icon": ic, "group": g, "enabled": i in ROLE_MENU.get(role, set())} for i, l, ic, g in NAV]


In [ ]:
%%writefile /content/app/auth.py
"""Authentication + user administration for FreightQuote AI (Milestone 1 login, rebuilt as FastAPI routes).

* users live in their own SQLite file (auth_users.db) so they never mix with the freight data
* passwords and security answers are bcrypt hashes; JWTs are signed with JWT_SECRET (Colab Secret)
* every request re-reads the user row: a locked / deleted user or a changed role takes effect immediately
* the same error text is returned for an unknown account and a wrong password
"""
import os, re, time, string, secrets, sqlite3, datetime, threading, html
from contextlib import contextmanager
import bcrypt, jwt
from fastapi import Request, HTTPException
from fastapi.responses import JSONResponse
import config, rbac

AUTH_DB = os.path.join(config.DATA_DIR, "auth_users.db")
SESSION_HOURS = int(os.environ.get("SESSION_HOURS", "8"))
OTP_MINUTES = 5
MAX_ATTEMPTS = 5                 # wrong passwords before the account locks
BASE_LOCK_SECONDS, MAX_LOCK_SECONDS, STALE_SECONDS = 60, 3600, 86400
SECURITY_QUESTIONS = ["What was the name of your first school?", "What was the name of your first pet?", "What is your mother's maiden name?",
                      "In which city were you born?", "What is the name of your favourite teacher?"]
GENERIC_LOGIN = "Invalid username/email or password."
LOCKED_MSG = "This account is locked after too many wrong attempts. Reset your password or ask an administrator to unlock it."


def _jwt_secret():
    v = (os.environ.get("JWT_SECRET") or "").strip()
    if v: return v
    p = os.path.join(config.DATA_DIR, ".jwt_secret")             # no Colab Secret: a random key kept beside the data (never in git)
    try:
        if os.path.exists(p): return open(p).read().strip()
        v = secrets.token_urlsafe(48); fd = os.open(p, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600); os.write(fd, v.encode()); os.close(fd); return v
    except Exception: return secrets.token_urlsafe(48)
JWT_SECRET = _jwt_secret()
JWT_FROM_SECRET = bool((os.environ.get("JWT_SECRET") or "").strip())

_lock = threading.RLock()


@contextmanager
def _db():
    with _lock:
        c = sqlite3.connect(AUTH_DB, timeout=30); c.row_factory = sqlite3.Row; c.execute("PRAGMA journal_mode=WAL;"); c.execute("PRAGMA busy_timeout=30000;")
        try: yield c; c.commit()
        finally: c.close()


def now_s(): return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


# ------------------------------------------------------------------ hashing
def hash_text(t): return bcrypt.hashpw(t.encode("utf-8")[:72], bcrypt.gensalt()).decode("utf-8")
def check_text(t, h):
    try: return bool(h) and bcrypt.checkpw(t.encode("utf-8")[:72], h.encode("utf-8"))
    except ValueError: return False
_DUMMY = hash_text("dummy-password-for-constant-time")


# ------------------------------------------------------------------ validation (Milestone 1 rules)
USERNAME_RE = re.compile(r"^[A-Za-z0-9_]{3,30}$")
EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}$")
COMMON = {"password@123", "welcome@123", "admin@123", "qwerty@123", "password123!", "admin@12345"}

def is_valid_email(e):
    """2+ letters before @, 2+ letters between @ and the last dot, 2+ letters after it."""
    if not EMAIL_RE.match(e or ""): return False
    local, domain = e.split("@"); host, tld = domain.rsplit(".", 1); letters = lambda s: sum(ch.isalpha() for ch in s)
    return letters(local) >= 2 and letters(host) >= 2 and len(tld) >= 2

def is_strong_password(pw, *names):
    pw = pw or ""
    if len(pw) < 8: return False, "Password must be at least 8 characters long."
    if len(pw) > 64: return False, "Password must be at most 64 characters long."
    if " " in pw: return False, "Password must not contain spaces."
    if not re.search(r"[A-Z]", pw): return False, "Add at least one uppercase letter."
    if not re.search(r"[a-z]", pw): return False, "Add at least one lowercase letter."
    if not re.search(r"\d", pw): return False, "Add at least one number."
    if not re.search(r"[^A-Za-z0-9]", pw): return False, "Add at least one special character (e.g. @ # $ !)."
    for n in names:
        part = (n or "").split("@")[0].lower()
        if len(part) >= 3 and part in pw.lower(): return False, "Password must not contain your username or email name."
    if pw.lower() in COMMON: return False, "This password is too common."
    return True, "Strong"

def gen_password():
    while True:
        p = "".join(secrets.choice(string.ascii_letters + string.digits + "@#$%!") for _ in range(12))
        if is_strong_password(p)[0]: return p


# ------------------------------------------------------------------ database
def init_db():
    with _db() as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS users(id INTEGER PRIMARY KEY AUTOINCREMENT, email TEXT NOT NULL UNIQUE COLLATE NOCASE, username TEXT NOT NULL UNIQUE COLLATE NOCASE,
            password_hash TEXT NOT NULL, role TEXT NOT NULL DEFAULT 'Customer', locked INTEGER NOT NULL DEFAULT 0, lock_reason TEXT DEFAULT '', failed_attempts INTEGER NOT NULL DEFAULT 0,
            customer_id TEXT, security_question TEXT, security_answer_hash TEXT, token_version INTEGER NOT NULL DEFAULT 0, created_at TEXT NOT NULL, last_login TEXT, prev_login TEXT, avatar TEXT DEFAULT '', last_seen REAL DEFAULT 0, must_change INTEGER NOT NULL DEFAULT 0, pw_kind TEXT DEFAULT 'permanent');
        CREATE TABLE IF NOT EXISTS lockouts(key TEXT PRIMARY KEY, fails INTEGER NOT NULL, locked_until REAL NOT NULL DEFAULT 0, last_fail REAL NOT NULL);
        CREATE TABLE IF NOT EXISTS otp_codes(email TEXT PRIMARY KEY COLLATE NOCASE, otp_hash TEXT NOT NULL, expires REAL NOT NULL, tries INTEGER NOT NULL DEFAULT 0);
        CREATE TABLE IF NOT EXISTS revoked_tokens(jti TEXT PRIMARY KEY, exp REAL NOT NULL);
        CREATE TABLE IF NOT EXISTS activity_log(id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT NOT NULL, actor TEXT NOT NULL, action TEXT NOT NULL, target TEXT, detail TEXT);
        CREATE INDEX IF NOT EXISTS ix_activity_id ON activity_log(id);""")
        have = {r[1] for r in c.execute("PRAGMA table_info(users)")}          # older databases: add the new columns once
        for col, ddl in [("avatar", "TEXT DEFAULT ''"), ("last_seen", "REAL DEFAULT 0"), ("must_change", "INTEGER NOT NULL DEFAULT 0"), ("pw_kind", "TEXT DEFAULT 'permanent'")]:
            if col not in have: c.execute(f"ALTER TABLE users ADD COLUMN {col} {ddl}")


def log_activity(actor, action, target="", detail=""):
    try:
        with _db() as c: c.execute("INSERT INTO activity_log(ts,actor,action,target,detail) VALUES(?,?,?,?,?)", (now_s(), str(actor or "system"), action, str(target or ""), str(detail or "")[:400]))
    except Exception: pass


def public_user(u):
    """Never contains the password hash, the security answer or the token version."""
    if not u: return None
    return {"id": u["id"], "email": u["email"], "username": u["username"], "role": u["role"], "locked": bool(u["locked"]), "customer_id": u.get("customer_id"),
            "created_at": u["created_at"], "last_login": u.get("last_login"), "prev_login": u.get("prev_login"),
            "avatar": u.get("avatar") or "", "must_change": bool(u.get("must_change")), "pw_kind": u.get("pw_kind") or "permanent"}


def _row(r): return dict(r) if r else None
def find_user(ident):
    ident = (ident or "").strip()
    if not ident: return None
    with _db() as c: return _row(c.execute("SELECT * FROM users WHERE username=? OR email=?", (ident, ident)).fetchone())
def get_user(uid):
    with _db() as c: return _row(c.execute("SELECT * FROM users WHERE id=?", (uid,)).fetchone())
def count_admins():
    with _db() as c: return c.execute("SELECT COUNT(*) FROM users WHERE role='Admin'").fetchone()[0]


# ------------------------------------------------------------------ presence: Active (signed in and recently seen) / Inactive
ACTIVE_SECONDS = 180             # a signed-in user is "Active" while the app has been seen within the last 3 minutes (the page pings every minute)
_seen = {}
def touch(uid):
    """Record that this user is using the app. Throttled to one write per 20 s per user."""
    t = time.time()
    if t - _seen.get(uid, 0) < 20: return
    _seen[uid] = t
    try:
        with _db() as c: c.execute("UPDATE users SET last_seen=? WHERE id=?", (t, uid))
    except Exception: pass
def clear_seen(uid):
    _seen.pop(uid, None)
    with _db() as c: c.execute("UPDATE users SET last_seen=0 WHERE id=?", (uid,))
def is_active(u): return (not u.get("locked")) and (time.time() - float(u.get("last_seen") or 0)) <= ACTIVE_SECONDS
def _ts(epoch):
    return datetime.datetime.fromtimestamp(epoch, datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S") if epoch else None


# ------------------------------------------------------------------ profile picture (avatar)
AVATAR_PRESETS = 12              # preset:0 ... preset:11 are drawn by the front end; an uploaded photo is a small data URL
AVATAR_MAX_CHARS = 90000
def _image_ok(raw):
    return raw[:3] == b"\xff\xd8\xff" or raw[:8] == b"\x89PNG\r\n\x1a\n" or (raw[:4] == b"RIFF" and raw[8:12] == b"WEBP")
def set_avatar(u, value):
    import base64, binascii
    v = (value or "").strip()
    if v:
        if v.startswith("preset:"):
            if not (v[7:].isdigit() and int(v[7:]) < AVATAR_PRESETS): return _bad("Unknown avatar.")
        else:
            m = re.match(r"^data:image/(jpeg|png|webp);base64,([A-Za-z0-9+/=]+)$", v)
            if not m: return _bad("Use a JPG, PNG or WebP picture.")
            if len(v) > AVATAR_MAX_CHARS: return _bad("That picture is too large. Choose a smaller one.")
            try: raw = base64.b64decode(m.group(2), validate=True)
            except (binascii.Error, ValueError): return _bad("That picture could not be read.")
            if not _image_ok(raw): return _bad("That file is not a valid picture.")
    with _db() as c: c.execute("UPDATE users SET avatar=? WHERE id=?", (v, u["id"]))
    log_activity(u["username"], "avatar_changed", u["email"], "removed" if not v else ("preset" if v.startswith("preset:") else "photo"))
    return {"ok": True, "msg": "Profile picture updated." if v else "Profile picture removed.", "user": public_user(get_user(u["id"]))}


# ------------------------------------------------------------------ customer accounts (a Customer user is tied to a customers row)
def _freight_conn():
    c = sqlite3.connect(config.DB_PATH, timeout=30); c.execute("PRAGMA busy_timeout=30000;"); return c

def ensure_customer(customer_id=None, name="New customer"):
    """Return a valid customer_id. Creates a customers row when needed (self-signup / admin-created Customer)."""
    try:
        with _freight_conn() as c:
            if customer_id:
                if c.execute("SELECT 1 FROM customers WHERE customer_id=?", (customer_id,)).fetchone(): return customer_id
            else:
                n = max([int(m.group(1)) for (x,) in c.execute("SELECT customer_id FROM customers") if (m := re.match(r"CUST-(\d+)$", x or ""))] or [0]) + 1; customer_id = f"CUST-{n:03d}"
            c.execute("INSERT OR IGNORE INTO customers(customer_id,name,industry,priority_tier,credit_risk) VALUES(?,?,?,?,?)", (customer_id, name, "Self-registered", "Silver", 0.2))
        return customer_id
    except Exception: return customer_id


# ------------------------------------------------------------------ tokens
def make_token(u):
    n = datetime.datetime.now(datetime.timezone.utc)
    return jwt.encode({"sub": str(u["id"]), "tv": u["token_version"], "jti": secrets.token_hex(8), "type": "session", "iat": n, "exp": n + datetime.timedelta(hours=SESSION_HOURS)}, JWT_SECRET, algorithm="HS256")

def user_from_token(token):
    """-> (user, error). Re-reads the row on every call so lock / delete / role change / password change act immediately."""
    if not token: return None, "Please sign in."
    try: p = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except jwt.ExpiredSignatureError: return None, "Your session expired. Please sign in again."
    except jwt.InvalidTokenError: return None, "Invalid session. Please sign in again."
    if p.get("type") != "session": return None, "Invalid session."
    with _db() as c:
        if c.execute("SELECT 1 FROM revoked_tokens WHERE jti=?", (p.get("jti"),)).fetchone(): return None, "You have signed out."
        r = c.execute("SELECT * FROM users WHERE id=?", (p.get("sub"),)).fetchone()
    if not r: return None, "This account no longer exists."
    u = dict(r)
    if u["locked"]: return None, "Your account is locked. Contact an administrator."
    if u["token_version"] != p.get("tv"): return None, "Your password changed. Please sign in again."
    touch(u["id"]); return u, None

def bearer(request):
    h = request.headers.get("authorization") or ""
    return h[7:].strip() if h.lower().startswith("bearer ") else ""

def user_from_request(request):
    return user_from_token(bearer(request))


# ------------------------------------------------------------------ timed lockouts (OTP, security answer, unknown accounts)
def _fails(row, now): return 0 if (not row or now - row["last_fail"] > STALE_SECONDS) else row["fails"]
def failed_attempt(key):
    key, now = key.lower(), time.time()
    with _db() as c:
        row = c.execute("SELECT fails,locked_until,last_fail FROM lockouts WHERE key=?", (key,)).fetchone(); fails = _fails(row, now) + 1; lock = 0; until = 0
        if fails >= MAX_ATTEMPTS: lock = min(BASE_LOCK_SECONDS * (fails - MAX_ATTEMPTS + 1), MAX_LOCK_SECONDS); until = now + lock
        c.execute("INSERT OR REPLACE INTO lockouts(key,fails,locked_until,last_fail) VALUES(?,?,?,?)", (key, fails, until, now))
    return fails, lock
def attempts_left(key):
    with _db() as c: row = c.execute("SELECT fails,locked_until,last_fail FROM lockouts WHERE key=?", (key.lower(),)).fetchone()
    return max(0, MAX_ATTEMPTS - _fails(row, time.time()))
def lockout_remaining(key):
    with _db() as c: row = c.execute("SELECT locked_until FROM lockouts WHERE key=?", (key.lower(),)).fetchone()
    return 0 if not row else max(0, int(row["locked_until"] - time.time()) + (1 if row["locked_until"] > time.time() else 0))
def clear_attempts(key):
    with _db() as c: c.execute("DELETE FROM lockouts WHERE key=?", (key.lower(),))


# ------------------------------------------------------------------ e-mail (OTP)
def email_ready():
    import mailer; return mailer.configured()

def send_otp_email(to, otp):
    import mailer
    text = f"Your FreightQuote AI verification code is {otp}. It expires in {OTP_MINUTES} minutes. If you did not request it, ignore this e-mail."
    page = (f'<html><body style="font-family:Arial,sans-serif;background:#FBF4EC;padding:20px"><div style="max-width:480px;margin:0 auto;background:#fff;border:1.5px solid #E3D2C2;border-radius:16px;padding:30px;text-align:center">'
            f'<div style="font-size:20px;font-weight:bold;color:#26304D;margin-bottom:12px">Password reset verification</div>'
            f'<div style="color:#4A5578;font-size:15px;line-height:1.5">Use this code to reset the password for <b>{html.escape(to)}</b>:</div>'
            f'<div style="display:inline-block;margin:18px 0;padding:14px 22px;background:#E4EAF5;color:#26304D;font-size:28px;font-weight:bold;letter-spacing:6px;border:1.5px solid #54668E;border-radius:12px">{otp}</div>'
            f'<div style="color:#4A5578;font-size:14px">This code expires in <b>{OTP_MINUTES} minutes</b>.</div>'
            f'<div style="color:#667090;font-size:12px;margin-top:22px;border-top:1px solid #E3D2C2;padding-top:14px">If you did not request this, you can safely ignore this e-mail.</div></div></body></html>')
    return mailer.send(to, "Your FreightQuote AI verification code", text, html=page)


# ------------------------------------------------------------------ core operations
def _bad(msg, code=400, **kw): return JSONResponse({"ok": False, "msg": msg, **kw}, status_code=code)

def signup(d):
    username, email = (d.get("username") or "").strip(), (d.get("email") or "").strip().lower(); pw, cpw = d.get("password") or "", d.get("confirm") or ""
    q, ans = d.get("question") or "", (d.get("answer") or "").strip()
    if not (username and email and pw and cpw and ans): return _bad("All fields are mandatory.")
    if not USERNAME_RE.match(username): return _bad("Username must be 3-30 characters: letters, numbers or underscore only.")
    if not is_valid_email(email): return _bad("Enter a valid email (at least 2 letters before @, between @ and the dot, and after the dot).")
    ok, m = is_strong_password(pw, username, email)
    if not ok: return _bad(m)
    if pw != cpw: return _bad("Password and Confirm Password do not match.")
    if q not in SECURITY_QUESTIONS: return _bad("Choose one of the security questions.")
    if len(ans) < 2: return _bad("Security answer is too short.")
    cust = ensure_customer(None, username)
    try:
        with _db() as c:
            if c.execute("SELECT 1 FROM users WHERE username=?", (username,)).fetchone(): return _bad("Username already taken. Please choose another one.")
            if c.execute("SELECT 1 FROM users WHERE email=?", (email,)).fetchone(): return _bad("This email is already registered.")
            c.execute("INSERT INTO users(email,username,password_hash,role,customer_id,security_question,security_answer_hash,created_at) VALUES(?,?,?,?,?,?,?,?)",
                      (email, username, hash_text(pw), rbac.PUBLIC_SIGNUP_ROLE, cust, q, hash_text(ans.lower()), now_s()))
    except sqlite3.IntegrityError: return _bad("Username or email already exists.")
    log_activity(username, "signup", email, f"role {rbac.PUBLIC_SIGNUP_ROLE}")
    return {"ok": True, "msg": "Account created. You can sign in now."}

def login(d):
    ident, pw = (d.get("identifier") or "").strip(), d.get("password") or ""
    if not ident or not pw: return _bad("Please fill in all fields.")
    key = "login:" + ident.lower(); u = find_user(ident)
    if u is None:                                                      # unknown account: same text, same timing, same lock rule as a real one
        check_text(pw, _DUMMY)
        if lockout_remaining(key): return _bad(LOCKED_MSG, 423)
        fails, lock = failed_attempt(key)
        return _bad(LOCKED_MSG, 423) if lock else _bad(f"{GENERIC_LOGIN} Attempts left: {attempts_left(key)}.", 401)
    if u["locked"]:
        check_text(pw, _DUMMY); log_activity(u["username"], "login_blocked", u["email"], "account locked"); return _bad(LOCKED_MSG, 423)
    if not check_text(pw, u["password_hash"]):
        n = u["failed_attempts"] + 1
        with _db() as c:
            if n >= MAX_ATTEMPTS: c.execute("UPDATE users SET locked=1, lock_reason='attempts', failed_attempts=0 WHERE id=?", (u["id"],))
            else: c.execute("UPDATE users SET failed_attempts=? WHERE id=?", (n, u["id"]))
        if n >= MAX_ATTEMPTS: log_activity(u["username"], "account_locked", u["email"], f"{MAX_ATTEMPTS} wrong passwords"); return _bad(LOCKED_MSG, 423)
        log_activity(u["username"], "login_failed", u["email"], f"attempt {n}")
        return _bad(f"{GENERIC_LOGIN} Attempts left: {MAX_ATTEMPTS - n}.", 401)
    with _db() as c: c.execute("UPDATE users SET failed_attempts=0, prev_login=last_login, last_login=?, last_seen=? WHERE id=?", (now_s(), time.time(), u["id"])); _seen[u["id"]] = time.time()
    u = get_user(u["id"]); log_activity(u["username"], "login", u["email"], u["role"])
    return {"ok": True, "token": make_token(u), "user": public_user(u)}

def logout(token):
    try:
        p = jwt.decode(token, JWT_SECRET, algorithms=["HS256"]);
        with _db() as c: c.execute("INSERT OR IGNORE INTO revoked_tokens(jti,exp) VALUES(?,?)", (p.get("jti"), float(p.get("exp", 0)))); c.execute("DELETE FROM revoked_tokens WHERE exp<?", (time.time(),))
        clear_seen(int(p.get("sub")))                                   # signing out makes the user Inactive straight away
    except Exception: pass

def question_for(username):
    u = find_user(username)
    if u and u.get("security_question"): return u["security_question"]
    import zlib; return SECURITY_QUESTIONS[zlib.crc32((username or "").lower().encode()) % len(SECURITY_QUESTIONS)]   # unknown user: a stable fake question

def forgot(d):
    """Send an e-mail OTP. The answer never reveals whether the address is registered."""
    email = (d.get("email") or "").strip().lower()
    if not email: return _bad("Please enter your email.")
    if not is_valid_email(email): return _bad("Enter a valid email address.")
    if not email_ready(): return _bad("E-mail sending is not configured. Add EMAIL_ADDRESS and EMAIL_PASSWORD in Colab Secrets, or use the security question.")
    key = "otp:" + email; left = lockout_remaining(key)
    if left: return _bad(f"Too many attempts. Try again in {left}s.", 429, wait=left)
    u = find_user(email)
    if u and u["email"].lower() == email:
        otp = f"{secrets.randbelow(900000) + 100000}"
        with _db() as c: c.execute("INSERT OR REPLACE INTO otp_codes(email,otp_hash,expires,tries) VALUES(?,?,?,0)", (email, hash_text(otp), time.time() + OTP_MINUTES * 60))
        ok, m = send_otp_email(email, otp); log_activity(u["username"], "otp_sent" if ok else "otp_failed", email, "" if ok else m[:120])
    return {"ok": True, "msg": f"If this email is registered, a 6-digit code was sent. It is valid for {OTP_MINUTES} minutes."}

def _apply_new_password(u, pw):
    with _db() as c:
        c.execute("UPDATE users SET password_hash=?, must_change=0, pw_kind='permanent', token_version=token_version+1, failed_attempts=0, locked=CASE WHEN lock_reason='attempts' THEN 0 ELSE locked END, lock_reason=CASE WHEN lock_reason='attempts' THEN '' ELSE lock_reason END WHERE id=?", (hash_text(pw), u["id"]))
    clear_attempts("login:" + u["email"].lower()); clear_attempts("login:" + u["username"].lower())

def reset(d):
    method = d.get("method") or "otp"; pw, cpw = d.get("password") or "", d.get("confirm") or ""
    if not pw or not cpw: return _bad("All fields are mandatory.")
    if method == "question":
        uname = (d.get("username") or "").strip(); ans = (d.get("answer") or "").strip().lower(); key = "sq:" + uname.lower(); left = lockout_remaining(key)
        if not uname or not ans: return _bad("All fields are mandatory.")
        if left: return _bad(f"Too many wrong answers. Try again in {left}s.", 429, wait=left)
        u = find_user(uname)
        if not (u and u["username"].lower() == uname.lower() and u.get("security_answer_hash") and check_text(ans, u["security_answer_hash"])):
            if u is None: check_text(ans, _DUMMY)
            fails, lock = failed_attempt(key); return _bad(f"Incorrect security answer. Attempts left: {attempts_left(key)}." if not lock else f"Too many wrong answers. Locked for {lock // 60} min.", 400)
        ok, m = is_strong_password(pw, u["username"], u["email"])
        if not ok: return _bad(m)
        if pw != cpw: return _bad("Password and Confirm Password do not match.")
        _apply_new_password(u, pw); clear_attempts(key); log_activity(u["username"], "password_reset", u["email"], "security question")
        return {"ok": True, "msg": "Password reset successful. You can sign in now."}
    email = (d.get("email") or "").strip().lower(); code = (d.get("code") or "").strip(); key = "otp:" + email; left = lockout_remaining(key)
    if not (email and code): return _bad("All fields are mandatory.")
    if not (code.isdigit() and len(code) == 6): return _bad("Please enter the 6-digit code.")
    if left: return _bad(f"Too many wrong codes. Try again in {left}s.", 429, wait=left)
    with _db() as c: r = c.execute("SELECT * FROM otp_codes WHERE email=?", (email,)).fetchone()
    u = find_user(email)
    if not (r and u and u["email"].lower() == email) or time.time() > r["expires"] or r["tries"] >= MAX_ATTEMPTS:
        failed_attempt(key); return _bad("This code is invalid or expired. Request a new one.")
    if not check_text(code, r["otp_hash"]):
        with _db() as c: c.execute("UPDATE otp_codes SET tries=tries+1 WHERE email=?", (email,))
        failed_attempt(key); return _bad("Incorrect code.")
    ok, m = is_strong_password(pw, u["username"], email)
    if not ok: return _bad(m)
    if pw != cpw: return _bad("Password and Confirm Password do not match.")
    _apply_new_password(u, pw); clear_attempts(key)
    with _db() as c: c.execute("DELETE FROM otp_codes WHERE email=?", (email,))
    log_activity(u["username"], "password_reset", email, "e-mail OTP")
    return {"ok": True, "msg": "Password reset successful. You can sign in now."}


# ------------------------------------------------------------------ profile
def change_username(u, new):
    new = (new or "").strip()
    if not USERNAME_RE.match(new): return _bad("Username must be 3-30 characters: letters, numbers or underscore only.")
    if new.lower() == u["username"].lower() and new == u["username"]: return _bad("That is already your username.")
    with _db() as c:
        if c.execute("SELECT 1 FROM users WHERE username=? AND id<>?", (new, u["id"])).fetchone(): return _bad("Username already taken. Please choose another one.")
        c.execute("UPDATE users SET username=? WHERE id=?", (new, u["id"]))
    log_activity(new, "username_changed", u["email"], f"{u['username']} -> {new}")
    return {"ok": True, "msg": "Username updated.", "user": public_user(get_user(u["id"]))}

def change_password(u, cur, new, confirm):
    if not (cur and new and confirm): return _bad("All fields are mandatory.")
    if not check_text(cur, u["password_hash"]): return _bad("Current password is incorrect.", 400)
    ok, m = is_strong_password(new, u["username"], u["email"])
    if not ok: return _bad(m)
    if new != confirm: return _bad("Password and Confirm Password do not match.")
    if check_text(new, u["password_hash"]): return _bad("The new password must be different from the current one.")
    with _db() as c: c.execute("UPDATE users SET password_hash=?, must_change=0, pw_kind='permanent', token_version=token_version+1 WHERE id=?", (hash_text(new), u["id"]))
    log_activity(u["username"], "password_changed", u["email"], "replaced the temporary password" if u.get("must_change") else "")
    nu = get_user(u["id"])
    return {"ok": True, "msg": "Password changed. Other devices were signed out.", "token": make_token(nu), "user": public_user(nu)}


# ------------------------------------------------------------------ admin operations (callers must have passed need(ctx, "users"))
def list_users():
    with _db() as c:
        rows = [dict(r) for r in c.execute("SELECT * FROM users ORDER BY id")]
    return [{**public_user(r), "status": "Locked" if r["locked"] else "Enabled", "lock_reason": r.get("lock_reason") or "",
             "presence": "Active" if is_active(r) else "Inactive", "last_seen_at": _ts(r.get("last_seen"))} for r in rows]

def recent_activity(n=50):
    with _db() as c: return [dict(r) for r in c.execute("SELECT id,ts,actor,action,target,detail FROM activity_log ORDER BY id DESC LIMIT ?", (n,))]

def overview():
    us = list_users(); per = {r: sum(1 for u in us if u["role"] == r) for r in rbac.ROLES}
    return {"ok": True, "kpis": {"total": len(us), "per_role": per, "locked": sum(1 for u in us if u["locked"]), "active": sum(1 for u in us if u["presence"] == "Active"), "inactive": sum(1 for u in us if u["presence"] == "Inactive")},
            "users": us, "roles": rbac.ROLES, "matrix": rbac.permission_rows(), "log": recent_activity(50)}

def admin_add_user(actor, d):
    email, username, pw, role = (d.get("email") or "").strip().lower(), (d.get("username") or "").strip(), d.get("password") or "", d.get("role") or ""
    kind = "permanent" if (d.get("password_type") or "temporary") == "permanent" else "temporary"      # temporary: must be changed at first sign-in; permanent: the user keeps it (and may change it any time)
    if not (email and username and pw and role): return _bad("Email, username, password and role are all required.")
    if role not in rbac.ROLES: return _bad("Unknown role.")
    if not USERNAME_RE.match(username): return _bad("Username must be 3-30 characters: letters, numbers or underscore only.")
    if not is_valid_email(email): return _bad("Enter a valid email (at least 2 letters before @, between @ and the dot, and after the dot).")
    ok, m = is_strong_password(pw, username, email)
    if not ok: return _bad(m)
    cust = None
    if role == "Customer": cust = ensure_customer((d.get("customer_id") or "").strip().upper() or None, username)
    try:
        with _db() as c:
            if c.execute("SELECT 1 FROM users WHERE username=?", (username,)).fetchone(): return _bad("Username already taken.")
            if c.execute("SELECT 1 FROM users WHERE email=?", (email,)).fetchone(): return _bad("This email is already registered.")
            c.execute("INSERT INTO users(email,username,password_hash,role,customer_id,created_at,must_change,pw_kind) VALUES(?,?,?,?,?,?,?,?)", (email, username, hash_text(pw), role, cust, now_s(), 1 if kind == "temporary" else 0, kind))
    except sqlite3.IntegrityError: return _bad("Username or email already exists.")
    log_activity(actor["username"], "user_added", email, f"role {role}, {kind} password")
    return {"ok": True, "msg": f"User {username} created as {role} with a {kind} password." + (" They must set a new password at first sign-in." if kind == "temporary" else " They can change it themselves from their Profile page.")}

def _target(uid):
    t = get_user(uid)
    if not t: raise HTTPException(404, "User not found.")
    return t

def admin_set_role(actor, uid, role):
    t = _target(uid)
    if role not in rbac.ROLES: return _bad("Unknown role.")
    if t["id"] == actor["id"]: return _bad("You cannot change your own role. Ask another administrator.", 403)
    if t["role"] == role: return _bad("The user already has that role.")
    if t["role"] == "Admin" and count_admins() <= 1: return _bad("The last remaining Admin cannot be demoted.", 403)
    cust = t.get("customer_id")
    if role == "Customer" and not cust: cust = ensure_customer(None, t["username"])
    with _db() as c: c.execute("UPDATE users SET role=?, customer_id=? WHERE id=?", (role, cust, t["id"]))
    log_activity(actor["username"], "role_changed", t["email"], f"{t['role']} -> {role}")
    return {"ok": True, "msg": f"{t['username']} is now {role}."}

def admin_set_lock(actor, uid, lock):
    t = _target(uid)
    if t["id"] == actor["id"]: return _bad("You cannot lock or unlock your own account.", 403)
    with _db() as c:
        if lock: c.execute("UPDATE users SET locked=1, lock_reason='admin', token_version=token_version WHERE id=?", (t["id"],))
        else: c.execute("UPDATE users SET locked=0, lock_reason='', failed_attempts=0 WHERE id=?", (t["id"],))
    clear_attempts("login:" + t["email"].lower()); clear_attempts("login:" + t["username"].lower())
    if lock: clear_seen(t["id"])
    log_activity(actor["username"], "user_locked" if lock else "user_unlocked", t["email"])
    return {"ok": True, "msg": f"{t['username']} is now {'locked' if lock else 'unlocked'}."}

def admin_set_password(actor, uid, pw, kind):
    """Admin sets a new password for another user: temporary (must be changed at next sign-in) or permanent (the user may change it later)."""
    t = _target(uid)
    if t["id"] == actor["id"]: return _bad("Change your own password on the Profile page.", 403)
    kind = "permanent" if kind == "permanent" else "temporary"
    ok, m = is_strong_password(pw or "", t["username"], t["email"])
    if not ok: return _bad(m)
    with _db() as c:
        c.execute("UPDATE users SET password_hash=?, must_change=?, pw_kind=?, token_version=token_version+1, failed_attempts=0, locked=CASE WHEN lock_reason='attempts' THEN 0 ELSE locked END, lock_reason=CASE WHEN lock_reason='attempts' THEN '' ELSE lock_reason END WHERE id=?",
                  (hash_text(pw), 1 if kind == "temporary" else 0, kind, t["id"]))
    clear_attempts("login:" + t["email"].lower()); clear_attempts("login:" + t["username"].lower()); clear_seen(t["id"])
    log_activity(actor["username"], "password_set", t["email"], f"{kind} password")
    return {"ok": True, "msg": f"New {kind} password set for {t['username']}. They were signed out." + (" They must change it at next sign-in." if kind == "temporary" else "")}

def admin_set_customer(actor, uid, cid):
    t = _target(uid)
    if t["role"] != "Customer": return _bad("Only Customer accounts have a customer id.")
    cid = ensure_customer((cid or "").strip().upper() or None, t["username"])
    with _db() as c: c.execute("UPDATE users SET customer_id=? WHERE id=?", (cid, t["id"]))
    log_activity(actor["username"], "customer_linked", t["email"], cid)
    return {"ok": True, "msg": f"{t['username']} is linked to {cid}."}

def admin_delete(actor, uid, confirm):
    t = _target(uid)
    if not confirm: return _bad("Deletion must be confirmed.")
    if t["id"] == actor["id"]: return _bad("You cannot delete your own account.", 403)
    if t["role"] == "Admin" and count_admins() <= 1: return _bad("The last remaining Admin cannot be deleted.", 403)
    with _db() as c: c.execute("DELETE FROM users WHERE id=?", (t["id"],)); c.execute("DELETE FROM otp_codes WHERE email=?", (t["email"],))
    log_activity(actor["username"], "user_deleted", t["email"], t["role"])
    return {"ok": True, "msg": f"User {t['username']} deleted."}


# ------------------------------------------------------------------ first-run accounts
def seed_defaults():
    """One Admin on first start (and optional demo accounts for the other roles). Passwords come from Colab Secrets
    (ADMIN_PASSWORD / DEMO_PASSWORD) or are generated and written to initial_credentials.txt beside the data (never to git)."""
    init_db(); made = []
    def mk(username, email, role, pw, cust=None):
        with _db() as c:
            if c.execute("SELECT 1 FROM users WHERE username=? OR email=?", (username, email)).fetchone(): return
            c.execute("INSERT INTO users(email,username,password_hash,role,customer_id,created_at) VALUES(?,?,?,?,?,?)", (email, username, hash_text(pw), role, cust, now_s()))
        made.append((role, username, email, pw))
    if count_admins() == 0:
        ap = os.environ.get("ADMIN_PASSWORD") or gen_password()
        mk("admin", os.environ.get("ADMIN_EMAIL") or "admin@freightquote.ai", "Admin", ap)
        if os.environ.get("ADMIN_PASSWORD"): made[:] = [(r, u, e, "(your ADMIN_PASSWORD secret)") if r == "Admin" else (r, u, e, p) for r, u, e, p in made]
    if os.environ.get("SEED_DEMO_USERS", "1") == "1":
        dp = os.environ.get("DEMO_PASSWORD"); shown = "(your DEMO_PASSWORD secret)" if dp else None
        for username, email, role, cust in [("ops_manager", "ops.manager@freightquote.ai", "Operations Manager", None), ("freight_broker", "broker@freightquote.ai", "Freight Broker", None),
                                            ("auditor", "auditor@freightquote.ai", "Auditor", None), ("customer_demo", "customer@freightquote.ai", "Customer", "CUST-001")]:
            pw = dp or gen_password()
            if cust: cust = ensure_customer(cust, "Corporate Client 001")
            n0 = len(made); mk(username, email, role, pw, cust)
            if shown and len(made) > n0: made[-1] = (role, username, email, shown)
    if made:
        p = os.path.join(config.DATA_DIR, "initial_credentials.txt")
        try:
            with open(p, "a") as f:
                for r, u, e, pw in made: f.write(f"{r} | {u} | {e} | {pw}\n")
            os.chmod(p, 0o600)
        except Exception: pass
    return made


In [ ]:
%%writefile /content/app/views_platform.py
"""Profile and Admin pages. They are rendered by dedicated front-end blocks that talk to /api/profile and /api/admin/*."""
import blocks as B

def profile_page(c):
    return B.page("My profile", "Your account, role and password", [{"t": "profile", "key": "profile"}], live=0)

def admin_page(c):
    return B.page("Admin dashboard", "Users, active / inactive status, roles, passwords, locks and the activity log", [{"t": "admin", "key": "admin"}], live=0)


def run_agents_page(c):
    import views_core as VC
    AG = [("route_watch", "Route & congestion agent", "Compares Suez and Cape routings, classifies port congestion and checks the weather along each lane."),
          ("weather_watch", "Weather risk agent", "Reads live wind, waves and rain for every port and raises re-route tickets for vessels in high-risk weather."),
          ("customs_watch", "Customs & tariff agent", "Checks required documents per HS chapter, holds shipments that miss documents and clears them when complete."),
          ("quote_watch", "Quotes, margin & approvals agent", "Prices spot quotes, checks the margin and sends quotes above $50,000 or below 10% margin to an approver."),
          ("anomaly_watch", "Anomaly agent", "Looks for unusual prices, delays and carrier behaviour across the live tables.")]
    bl = [B.callout("Each card shows the latest run of one agent. Press Run now to run it immediately, or use Run all agents at the top. The agents also run by themselves in the background.", "info", "Run agents")]
    bl += [VC.hero(c, a, t, d) for a, t, d in AG]
    return B.page("Run agents", "Start any agent now, or all of them", bl, live=10, actions=[{"label": "Run all agents", "url": "/api/run_cycle", "tone": "primary"}])


In [ ]:
%%writefile /content/app/copilot_plus.py
"""FreightQuote AI Copilot, Milestone 2 edition: greetings + exact answers from SQL only (no LLM, no RAG).

* every answer is produced by a fixed, whitelisted, parameterised SELECT; the SQL and the row count are returned with the answer
* spelling mistakes and different wordings are handled (vocabulary correction + several phrasings per intent)
* RBAC: every intent belongs to a data domain; a role only reads the domains in rbac.ROLE_DATA (a Customer only reads own quotes / shipments)
* out-of-scope questions get a polite reply with three valid example questions for the user's role
"""
import re, math, json, difflib
import pandas as pd
from blocks import Q
import rbac

# ------------------------------------------------------------------ vocabulary used for spelling correction
VOCAB = set("""cheapest cheaper cheap costliest expensive carrier carriers port ports congested congestion busiest dwell shipment shipments shipping delayed delay delays customs hold held
quote quotes quotation margin margins weather storm storms wind waves wave temperature reliable reliability punctual rated rating risk risky highest lowest least most worst best top bottom
average total count many number status pending approved rejected draft submitted accepted transit delivered anchorage vessels vessel customers customer platinum gold silver tier credit
duty tariff documents document required restricted pharmaceuticals machinery electronics textiles chemicals tickets ticket critical warning open heaviest weight region regions asia europe africa
americas oceania mediterranean middle east suez panama malacca hormuz bosphorus cape red sea latest recent below above under over between revenue price value fastest slowest longest shortest arriving
departing destination origin route routes hapag maersk cosco evergreen yang ming""".split())
PHRASE_FIX = {"on time": "on-time", "ontime": "on-time", "kustoms": "customs", "carrer": "carrier"}
COMMON_WORDS = set("""the a an of to in on at is are was were be by for from with and or not do does did have has had which what who whom how why when where me my our your show tell give list
find get see can could would should please any all some there their them this that these those it its as into out than then also about more less much each every per vs versus i we you
shipments are ports""".split())

REGIONS = {"asia": "Asia", "europe": "Europe", "africa": "Africa", "americas": "Americas", "america": "Americas", "oceania": "Oceania", "middle east": "Middle East", "mediterranean": "Europe/Mediterranean"}
CHOKE = {"red sea": ("Red Sea", 18.0, 40.0), "suez": ("Suez Canal", 30.5, 32.3), "panama": ("Panama Canal", 9.1, -79.7), "malacca": ("Strait of Malacca", 2.5, 101.5),
         "hormuz": ("Strait of Hormuz", 26.5, 56.5), "bosphorus": ("Bosphorus", 41.1, 29.0), "cape of good hope": ("Cape of Good Hope", -34.4, 18.5),
         "bay of bengal": ("Bay of Bengal", 14.0, 88.0), "arabian sea": ("Arabian Sea", 16.0, 66.0), "south china sea": ("South China Sea", 14.0, 114.0), "mediterranean": ("Mediterranean", 35.0, 18.0)}
STATUS_WORDS = {"customs hold": "Customs Hold", "on hold": "Customs Hold", "in transit": "In Transit", "delivered": "Delivered", "anchorage": "Anchorage Pending",
                "port congestion delay": "Port Congestion Delay"}
QSTATUS = {"pending approval": "Pending Approval", "pending": "Pending Approval", "approved": "Approved", "rejected": "Rejected", "on hold": "On Hold", "draft": "Draft", "submitted": "Submitted", "accepted": "Accepted"}


def _norm(t):
    t = (t or "").lower().replace("%", " % "); t = re.sub(r"(?<=\d),(?=\d)", "", t); t = re.sub(r"[^a-z0-9%.\- ]", " ", t); t = re.sub(r"\s+", " ", t).strip()
    for a, b in PHRASE_FIX.items(): t = t.replace(a, b)
    return t


def _dynamic_vocab():
    """Port, carrier and place names are added to the vocabulary so 'rotterdm' or 'maerks' are corrected too."""
    try:
        v = set()
        for n in Q("SELECT port_name FROM ports").port_name: v.update(w for w in re.sub(r"[^a-z ]", " ", n.lower()).split() if len(w) >= 5)
        for n in Q("SELECT name FROM carriers LIMIT 40").name: v.update(w for w in re.sub(r"[^a-z ]", " ", n.lower()).split() if len(w) >= 5)
        return v
    except Exception: return set()


def correct(text):
    """-> (corrected text, [(wrong, right)]). Only words that are not ordinary English and are close to a domain word are changed."""
    vocab = VOCAB | _dynamic_vocab(); out, fixes = [], []
    for w in _norm(text).split():
        if len(w) < 4 or w in vocab or w in COMMON_WORDS or re.search(r"\d", w) or w.startswith("-") or w.rstrip("s") in vocab or w + "s" in vocab: out.append(w); continue
        m = difflib.get_close_matches(w, vocab, n=1, cutoff=0.8)
        if m and m[0] != w: fixes.append((w, m[0])); out.append(m[0])
        else: out.append(w)
    return " ".join(out), fixes


# ------------------------------------------------------------------ helpers
def _has(q, *phrases):
    return any(re.search(r"(?<![a-z0-9])" + re.escape(p) + r"(?:s|es)?(?![a-z0-9])", q) for p in phrases)

def _num(q, default=None):
    m = re.search(r"(\d+(?:\.\d+)?)\s*(k|m)?\b", q.replace(",", ""))
    if not m: return default
    v = float(m.group(1)); return v * {"k": 1e3, "m": 1e6}.get(m.group(2) or "", 1)

def _topn(q, default=5):
    m = re.search(r"\b(?:top|first|bottom|best|worst|highest|lowest|last)\s+(\d{1,2})\b", q) or re.search(r"\b(\d{1,2})\s+(?:ports|carriers|shipments|quotes|customers|vessels)\b", q)
    return max(1, min(int(m.group(1)), 25)) if m else default

def _pct_threshold(q, default):
    m = re.search(r"(?:below|under|less than|lower than|above|over|more than|greater than|at least|at most|<|>)\s*(\d+(?:\.\d+)?)\s*(?:%|percent)?", q)
    return float(m.group(1)) if m else default

def _alias(name):
    base = re.sub(r"\(.*?\)", " ", name.lower()); base = re.sub(r"[^a-z0-9 ]", " ", base).strip(); out = {base}
    core = [w for w in base.split() if w not in ("port", "terminal", "line", "lines", "shipping", "container", "international")]
    if core: out.add(" ".join(core))
    for inner in re.findall(r"\((.*?)\)", name): out.add(re.sub(r"[^a-z0-9 ]", " ", inner.lower()).strip())
    return {a for a in out if len(a) >= 3}

_CACHE = {}
def _lists():
    import time
    if _CACHE.get("t", 0) + 120 > time.time(): return _CACHE["ports"], _CACHE["carriers"]
    ports = Q("SELECT port_name FROM ports").port_name.tolist(); carriers = Q("SELECT name FROM carriers").name.tolist()
    first = {}
    for p in ports:
        core = [w for w in _alias(p) if " " not in w]
        for w in core: first.setdefault(w, []).append(p)
    pa = {}
    for p in ports:
        for a in _alias(p): pa.setdefault(a, p)
    for w, ps in first.items():
        if len(ps) == 1 and len(w) >= 5: pa.setdefault(w, ps[0])
    ca = {}
    for c in carriers:
        for a in _alias(c): ca.setdefault(a, c)
    _CACHE.update(t=time.time(), ports=pa, carriers=ca); return pa, ca

def find_ports(q):
    pa, _ = _lists(); hits = []
    for a in sorted(pa, key=len, reverse=True):
        m = re.search(r"(?<![a-z0-9])" + re.escape(a) + r"(?![a-z0-9])", q)
        if m and not any(m.start() < h[1] and h[0] < m.end() for h in hits): hits.append((m.start(), m.end(), pa[a]))
    hits.sort(); seen = []
    for h in hits:
        if h[2] not in seen: seen.append(h[2])
    return seen

def find_carrier(q):
    _, ca = _lists()
    for a in sorted(ca, key=len, reverse=True):
        if len(a) >= 3 and re.search(r"(?<![a-z0-9])" + re.escape(a) + r"(?![a-z0-9])", q): return ca[a]
    return None

def find_region(q):
    for k in sorted(REGIONS, key=len, reverse=True):
        if _has(q, k): return REGIONS[k]
    return None

def hav(la1, lo1, la2, lo2):
    r = 6371.0; p1, p2 = math.radians(la1), math.radians(la2); dp = p2 - p1; dl = math.radians(lo2 - lo1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2; return 2 * r * math.asin(math.sqrt(a))

def show_sql(sql, params):
    """The query with its bound values written in, for display only (execution always uses bound parameters)."""
    it = iter(params)
    def sub(_):
        try: v = next(it)
        except StopIteration: return "?"
        return str(v) if isinstance(v, (int, float)) else "'" + str(v).replace("'", "''") + "'"
    return re.sub(r"\?", sub, " ".join(sql.split()))

def run(sql, params=()):
    s = sql.strip()
    if not s.upper().startswith("SELECT") or ";" in s: raise ValueError("only a single SELECT is allowed")
    df = Q(s, tuple(params)); return df, show_sql(s, params)

def fmt_df(df, rename=None):
    d = df.copy()
    for c in d.columns:
        if d[c].dtype.kind == "f": d[c] = d[c].round(2)
    return d.rename(columns=rename or {})

class Ctx:
    def __init__(self, user, q, orig, fixes):
        self.user, self.q, self.orig, self.fixes = user, q, orig, fixes
        self.role = (user or {}).get("role"); self.cid = (user or {}).get("customer_id"); self.own = self.role == "Customer"
        self.ports = find_ports(q); self.carrier = find_carrier(q); self.region = find_region(q); self.n = _topn(q)
    def qscope(self, alias=""):
        a = alias + "." if alias else ""
        return (f" AND {a}customer_id = ?", [self.cid]) if self.own else ("", [])
    def sscope(self, alias=""):
        a = alias + "." if alias else ""
        return (f" AND {a}shipment_id IN (SELECT shipment_id FROM freight_quotes WHERE customer_id = ?)", [self.cid]) if self.own else ("", [])

def A(text, df=None, sql="", params=(), note=""):
    return {"text": text, "df": df, "sql": sql, "rows": 0 if df is None else int(len(df)), "note": note}


# ------------------------------------------------------------------ intents: ports
def i_port_congestion_top(c):
    least = _has(c.q, "least", "lowest", "smoothest", "quietest", "fastest", "emptiest")
    w, p = ("WHERE region = ?", [c.region]) if c.region else ("", [])
    order = "ASC" if least else "DESC"
    df, s = run(f"SELECT port_name, country, region, ROUND(congestion_index,2) AS congestion_index, avg_dwell_days FROM ports {w} ORDER BY congestion_index {order}, avg_dwell_days {order} LIMIT ?", p + [c.n])
    if not len(df): return None
    top = df.iloc[0]
    return A(f"{'Least' if least else 'Most'} congested port{' in ' + c.region if c.region else ''}: **{top.port_name}** (index {top.congestion_index}, dwell {top.avg_dwell_days} days). Top {len(df)} below.", df, s)

def i_port_congestion_lookup(c):
    if not c.ports: return None
    ph = ",".join("?" * len(c.ports[:5]))
    df, s = run(f"SELECT port_name, country, region, ROUND(congestion_index,2) AS congestion_index, avg_dwell_days FROM ports WHERE port_name IN ({ph}) ORDER BY congestion_index DESC", c.ports[:5])
    if not len(df): return None
    r = df.iloc[0]; lvl = "high" if r.congestion_index >= 4 else "moderate" if r.congestion_index >= 3 else "low"
    return A(f"**{r.port_name}** has a congestion index of **{r.congestion_index}** ({lvl}) and an average dwell time of **{r.avg_dwell_days} days**.", df, s)

def i_port_dwell(c):
    short = _has(c.q, "shortest", "lowest", "least", "fastest", "quickest")
    df, s = run(f"SELECT port_name, country, region, avg_dwell_days, ROUND(congestion_index,2) AS congestion_index FROM ports ORDER BY avg_dwell_days {'ASC' if short else 'DESC'}, congestion_index {'ASC' if short else 'DESC'} LIMIT ?", [c.n])
    r = df.iloc[0]; return A(f"{'Shortest' if short else 'Longest'} average dwell time: **{r.port_name}** with **{r.avg_dwell_days} days**.", df, s)

def i_ports_by_region(c):
    if c.region and _has(c.q, "how many", "number of", "count"):
        df, s = run("SELECT region, COUNT(*) AS ports, ROUND(AVG(congestion_index),2) AS avg_congestion FROM ports WHERE region = ? GROUP BY region", [c.region])
        return A(f"There are **{int(df.ports[0]) if len(df) else 0}** ports in {c.region}.", df, s) if len(df) else None
    df, s = run("SELECT region, COUNT(*) AS ports, ROUND(AVG(congestion_index),2) AS avg_congestion, ROUND(AVG(avg_dwell_days),1) AS avg_dwell_days FROM ports GROUP BY region ORDER BY ports DESC")
    return A(f"The platform monitors **{int(df.ports.sum())}** ports in {len(df)} regions; the busiest region by average congestion is **{df.sort_values('avg_congestion', ascending=False).region.iloc[0]}**.", df, s)

def i_congested_count(c):
    thr = _pct_threshold(c.q, 4.0)
    df, s = run("SELECT COUNT(*) AS congested_ports FROM ports WHERE congestion_index >= ?", [thr])
    return A(f"**{int(df.congested_ports[0])}** ports have a congestion index of {thr:g} or more.", df, s)

# ------------------------------------------------------------------ carriers
def i_carrier_cheapest(c):
    dear = _has(c.q, "costliest", "expensive", "dearest", "priciest", "highest cost")
    order = "DESC" if dear else "ASC"
    if c.ports:
        dest = c.ports[0]
        df, s = run(f"SELECT s.carrier, ROUND(c.avg_cost_index,2) AS cost_index, c.on_time_pct, c.risk_level, COUNT(*) AS shipments_to_port FROM shipments s JOIN carriers c ON c.name = s.carrier WHERE s.dest_port = ? GROUP BY s.carrier ORDER BY c.avg_cost_index {order} LIMIT ?", [dest, c.n])
        if len(df):
            r = df.iloc[0]; return A(f"{'Most expensive' if dear else 'Cheapest'} carrier serving **{dest}**: **{r.carrier}** (cost index {r.cost_index}, {r.on_time_pct}% on time). Cost index is relative: lower means cheaper.", df, s)
        o, so = run(f"SELECT name AS carrier, ROUND(avg_cost_index,2) AS cost_index, on_time_pct, risk_level FROM carriers ORDER BY avg_cost_index {order} LIMIT ?", [c.n])
        r = o.iloc[0]; return A(f"No shipment to **{dest}** is on record, so here is the overall ranking: **{r.carrier}** has the {'highest' if dear else 'lowest'} cost index ({r.cost_index}).", o, so)
    df, s = run(f"SELECT name AS carrier, ROUND(avg_cost_index,2) AS cost_index, on_time_pct, risk_level FROM carriers ORDER BY avg_cost_index {order}, on_time_pct DESC LIMIT ?", [c.n])
    r = df.iloc[0]; return A(f"{'Most expensive' if dear else 'Cheapest'} carrier overall: **{r.carrier}** (cost index {r.cost_index}). Lower index = cheaper.", df, s)

def i_carrier_reliable(c):
    worst = _has(c.q, "least", "worst", "unreliable", "lowest", "late", "slowest")
    df, s = run(f"SELECT name AS carrier, on_time_pct, rating, risk_level FROM carriers ORDER BY on_time_pct {'ASC' if worst else 'DESC'}, rating DESC LIMIT ?", [c.n])
    r = df.iloc[0]; return A(f"{'Least' if worst else 'Most'} reliable carrier: **{r.carrier}** with **{r.on_time_pct}%** on-time delivery.", df, s)

def i_carrier_risk(c):
    lvl = "High" if _has(c.q, "high") or not _has(c.q, "medium", "low") else ("Medium" if _has(c.q, "medium") else "Low")
    df, s = run("SELECT name AS carrier, rating, on_time_pct, risk_level FROM carriers WHERE risk_level = ? ORDER BY on_time_pct ASC LIMIT 25", [lvl])
    tot = Q("SELECT COUNT(*) n FROM carriers WHERE risk_level = ?", (lvl,)).n[0]
    return A(f"**{int(tot)}** carriers are rated **{lvl}** risk" + (f"; showing the {len(df)} with the lowest on-time rate." if len(df) < tot else "."), df, s)

def i_carrier_rating(c):
    low = _has(c.q, "lowest", "worst", "poorest")
    df, s = run(f"SELECT name AS carrier, rating, on_time_pct, risk_level FROM carriers ORDER BY rating {'ASC' if low else 'DESC'}, on_time_pct DESC LIMIT ?", [c.n])
    r = df.iloc[0]; return A(f"{'Lowest' if low else 'Highest'} rated carrier: **{r.carrier}** ({r.rating} / 5).", df, s)

def i_carrier_lookup(c):
    if not c.carrier: return None
    df, s = run("SELECT c.name AS carrier, c.rating, c.on_time_pct, c.avg_cost_index AS cost_index, c.risk_level, COUNT(sh.shipment_id) AS shipments, SUM(CASE WHEN sh.status LIKE '%Delay%' OR sh.status = 'Customs Hold' THEN 1 ELSE 0 END) AS problem_shipments FROM carriers c LEFT JOIN shipments sh ON sh.carrier = c.name WHERE c.name = ? GROUP BY c.name", [c.carrier])
    if not len(df): return None
    r = df.iloc[0]; return A(f"**{r.carrier}**: rating {r.rating}/5, {r.on_time_pct}% on time, cost index {r.cost_index}, {r.risk_level} risk. It carries **{int(r.shipments)}** shipments, **{int(r.problem_shipments or 0)}** of them delayed or on customs hold.", df, s)

# ------------------------------------------------------------------ shipments
def i_customs_hold(c):
    sc, sp = c.sscope()
    df, s = run(f"SELECT dest_port, COUNT(*) AS shipments_on_hold FROM shipments WHERE status = 'Customs Hold'{sc} GROUP BY dest_port ORDER BY shipments_on_hold DESC LIMIT 10", sp)
    tot, st = run(f"SELECT COUNT(*) AS n FROM shipments WHERE status = 'Customs Hold'{sc}", sp)
    return A(f"**{int(tot.n[0])}** shipments are on customs hold" + (f"; most at **{df.dest_port.iloc[0]}** ({int(df.shipments_on_hold.iloc[0])})." if len(df) else "."), df, st + "  -- plus the breakdown: " + s)

def i_shipments_delayed(c):
    sc, sp = c.sscope()
    df, s = run(f"SELECT status, COUNT(*) AS shipments FROM shipments WHERE (status LIKE '%Delay%' OR status = 'Anchorage Pending'){sc} GROUP BY status ORDER BY shipments DESC", sp)
    n = int(df.shipments.sum()) if len(df) else 0
    return A(f"**{n}** shipments are delayed or waiting at anchorage.", df, s)

def i_shipments_by_status(c):
    sc, sp = c.sscope(); st = next((v for k, v in STATUS_WORDS.items() if _has(c.q, k)), None)
    df, s = run(f"SELECT status, COUNT(*) AS shipments FROM shipments WHERE 1=1{sc} GROUP BY status ORDER BY shipments DESC", sp)
    if not len(df): return A("You have no shipments on record yet.", df, s)
    if st and st in set(df.status): n = int(df[df.status == st].shipments.iloc[0]); return A(f"**{n}** shipments are **{st}** (out of {int(df.shipments.sum())}).", df, s)
    return A(f"There are **{int(df.shipments.sum())}** shipments; the most common status is **{df.status.iloc[0]}** ({int(df.shipments.iloc[0])}).", df, s)

def i_shipments_risk(c):
    sc, sp = c.sscope()
    df, s = run(f"SELECT shipment_id, origin_port, dest_port, carrier, status, ROUND(predicted_delay_risk,2) AS delay_risk FROM shipments WHERE 1=1{sc} ORDER BY predicted_delay_risk DESC LIMIT ?", sp + [c.n])
    if not len(df): return None
    r = df.iloc[0]; return A(f"Highest predicted delay risk: **{r.shipment_id}** ({r.origin_port} to {r.dest_port}, risk {r.delay_risk}).", df, s)

def i_shipments_to_port(c):
    if not c.ports: return None
    dep = _has(c.q, "from", "leaving", "departing", "origin", "out of"); col = "origin_port" if dep else "dest_port"; sc, sp = c.sscope()
    df, s = run(f"SELECT shipment_id, origin_port, dest_port, carrier, status, eta FROM shipments WHERE {col} = ?{sc} ORDER BY eta LIMIT 15", [c.ports[0]] + sp)
    tot = Q(f"SELECT COUNT(*) n FROM shipments WHERE {col} = ?{sc}", tuple([c.ports[0]] + sp)).n[0]
    return A(f"**{int(tot)}** shipments {'leave' if dep else 'are heading to'} **{c.ports[0]}**" + (f"; the next {len(df)} by ETA are below." if len(df) else "."), df, s)

def i_shipments_heaviest(c):
    sc, sp = c.sscope()
    light = _has(c.q, "lightest", "smallest")
    df, s = run(f"SELECT shipment_id, origin_port, dest_port, carrier, ROUND(weight_kg/1000.0,1) AS weight_tonnes, status FROM shipments WHERE 1=1{sc} ORDER BY weight_kg {'ASC' if light else 'DESC'} LIMIT ?", sp + [c.n])
    if not len(df): return None
    r = df.iloc[0]; return A(f"{'Lightest' if light else 'Heaviest'} shipment: **{r.shipment_id}** at **{r.weight_tonnes} t** ({r.origin_port} to {r.dest_port}).", df, s)

def i_vessels_at_sea(c):
    sc, sp = c.sscope("s")
    df, s = run(f"SELECT COUNT(*) AS vessels, ROUND(AVG(v.progress)*100,0) AS avg_progress_pct, ROUND(AVG(v.speed_kn),1) AS avg_speed_kn FROM vessels v JOIN shipments s ON s.shipment_id = v.shipment_id WHERE 1=1{sc}", sp)
    return A(f"**{int(df.vessels[0])}** vessels are being tracked at sea, on average {int(df.avg_progress_pct[0] or 0)}% of the way, at {df.avg_speed_kn[0]} knots.", df, s)

def i_shipment_lookup(c):
    ids = re.findall(r"shp-?\d+", c.orig.lower())
    if not ids: return None
    sid = ids[0].upper().replace("SHP", "SHP-") if "-" not in ids[0] else ids[0].upper()
    sc, sp = c.sscope("s")
    df, s = run(f"SELECT s.shipment_id, s.origin_port, s.dest_port, s.carrier, s.status, s.eta, ROUND(s.weight_kg/1000.0,1) AS weight_tonnes, s.hs_code, ROUND(s.predicted_delay_risk,2) AS delay_risk, ROUND(v.progress*100,0) AS progress_pct FROM shipments s LEFT JOIN vessels v ON v.shipment_id = s.shipment_id WHERE s.shipment_id = ?{sc}", [sid] + sp)
    if not len(df): return A(f"I could not find shipment **{sid}**" + (" among your shipments." if c.own else "."), df, s)
    r = df.iloc[0]; return A(f"**{r.shipment_id}** ({r.origin_port} to {r.dest_port}) is **{r.status}** with {r.carrier}, ETA {r.eta}, delay risk {r.delay_risk}.", df, s)

# ------------------------------------------------------------------ quotes
def i_quotes_low_margin(c):
    thr = _pct_threshold(c.q, 10.0)
    df, s = run("SELECT quote_id, customer_id, ROUND(final_price,0) AS price, margin_pct, status FROM freight_quotes WHERE margin_pct < ? ORDER BY margin_pct ASC LIMIT 25", [thr])
    n = int(Q("SELECT COUNT(*) n FROM freight_quotes WHERE margin_pct < ?", (thr,)).n[0])
    return A(f"**{n}** quote{'s' if n != 1 else ''} {'have' if n != 1 else 'has'} a margin below **{thr:g}%**" + (f"; the lowest is **{df.quote_id.iloc[0]}** at {df.margin_pct.iloc[0]}%." if len(df) else "."), df, s)

def i_quotes_high_margin(c):
    thr = _pct_threshold(c.q, 20.0)
    df, s = run("SELECT quote_id, customer_id, ROUND(final_price,0) AS price, margin_pct, status FROM freight_quotes WHERE margin_pct > ? ORDER BY margin_pct DESC LIMIT 25", [thr])
    n = int(Q("SELECT COUNT(*) n FROM freight_quotes WHERE margin_pct > ?", (thr,)).n[0]); return A(f"**{n}** quotes have a margin above **{thr:g}%**.", df, s)

def i_quotes_high_value(c):
    m = re.search(r"(?:above|over|more than|greater than|exceed\w*|>)\s*\$?\s*(\d[\d,]*(?:\.\d+)?)\s*(k|m)?", c.q.replace(",", ""))
    thr = float(m.group(1)) * {"k": 1e3, "m": 1e6}.get(m.group(2) or "", 1) if m else 50000.0; sc, sp = c.qscope()
    df, s = run(f"SELECT quote_id, customer_id, ROUND(final_price,0) AS price, status, created_at FROM freight_quotes WHERE final_price > ?{sc} ORDER BY final_price DESC LIMIT 25", [thr] + sp)
    n = int(Q(f"SELECT COUNT(*) n FROM freight_quotes WHERE final_price > ?{sc}", tuple([thr] + sp)).n[0]); return A(f"**{n}** quotes are priced above **${thr:,.0f}**.", df, s)

def i_quotes_by_status(c):
    sc, sp = c.qscope(); st = next((v for k, v in sorted(QSTATUS.items(), key=lambda kv: -len(kv[0])) if _has(c.q, k)), None)
    df, s = run(f"SELECT status, COUNT(*) AS quotes, ROUND(SUM(final_price),0) AS total_value FROM freight_quotes WHERE 1=1{sc} GROUP BY status ORDER BY quotes DESC", sp)
    if not len(df): return A("There are no quotes yet.", df, s)
    if st: return A(f"**{int(df[df.status == st].quotes.iloc[0]) if st in set(df.status) else 0}** quotes are **{st}** (of {int(df.quotes.sum())} in total).", df, s)
    return A(f"There are **{int(df.quotes.sum())}** quotes in total; **{df.status.iloc[0]}** is the most common status ({int(df.quotes.iloc[0])}).", df, s)

def i_quotes_avg(c):
    df, s = run("SELECT COUNT(*) AS quotes, ROUND(AVG(margin_pct),2) AS avg_margin_pct, ROUND(MIN(margin_pct),1) AS min_margin_pct, ROUND(MAX(margin_pct),1) AS max_margin_pct, ROUND(AVG(final_price),0) AS avg_price FROM freight_quotes")
    r = df.iloc[0]; return A(f"Across **{int(r.quotes)}** quotes the average margin is **{r.avg_margin_pct}%** (range {r.min_margin_pct}% to {r.max_margin_pct}%) and the average price is **${r.avg_price:,.0f}**.", df, s)

def i_quotes_latest(c):
    sc, sp = c.qscope()
    df, s = run(f"SELECT quote_id, customer_id, ROUND(final_price,0) AS price, status, created_at FROM freight_quotes WHERE 1=1{sc} ORDER BY created_at DESC, quote_id DESC LIMIT ?", sp + [c.n])
    if not len(df): return A("There are no quotes yet.", df, s)
    r = df.iloc[0]; return A(f"The latest quote is **{r.quote_id}** (${r.price:,.0f}, {r.status}). Showing the {len(df)} most recent.", df, s)

def i_quote_lookup(c):
    ids = re.findall(r"qte-?\d+", c.orig.lower())
    if not ids: return None
    qid = ids[0].upper() if "-" in ids[0] else ids[0].upper().replace("QTE", "QTE-"); sc, sp = c.qscope()
    cols = "quote_id, customer_id, shipment_id, carrier, ROUND(final_price,0) AS price, status, created_at" + ("" if c.own else ", margin_pct, ROUND(base_cost,0) AS base_cost")
    df, s = run(f"SELECT {cols} FROM freight_quotes WHERE quote_id = ?{sc}", [qid] + sp)
    if not len(df): return A(f"I could not find quote **{qid}**" + (" on your account." if c.own else "."), df, s)
    r = df.iloc[0]; return A(f"**{r.quote_id}** is **{r.status}**, priced at **${r.price:,.0f}**" + ("." if c.own else f" with a margin of {r.margin_pct}%."), df, s)

def i_top_customers(c):
    df, s = run("SELECT q.customer_id, cu.name, cu.priority_tier, COUNT(*) AS quotes, ROUND(SUM(q.final_price),0) AS total_value FROM freight_quotes q LEFT JOIN customers cu ON cu.customer_id = q.customer_id GROUP BY q.customer_id ORDER BY total_value DESC LIMIT ?", [c.n])
    if not len(df): return None
    r = df.iloc[0]; return A(f"Top customer by quoted value: **{r.customer_id}** ({r['name']}) with **${r.total_value:,.0f}** across {int(r.quotes)} quotes.", df, s)

def i_customer_credit(c):
    df, s = run("SELECT customer_id, name, priority_tier, credit_risk FROM customers ORDER BY credit_risk DESC LIMIT ?", [c.n]); r = df.iloc[0]
    return A(f"Highest credit risk: **{r.customer_id}** ({r['name']}) at {r.credit_risk:.2f}.", df, s)

def i_customer_tiers(c):
    tier = next((t for t in ("platinum", "gold", "silver") if _has(c.q, t)), None)
    if tier:
        df, s = run("SELECT customer_id, name, industry, priority_tier, credit_risk FROM customers WHERE priority_tier = ? ORDER BY credit_risk DESC LIMIT 25", [tier.title()])
        return A(f"There are **{len(df)}** {tier.title()} customers.", df, s)
    if _has(c.q, "credit", "risky", "risk"):
        df, s = run("SELECT customer_id, name, priority_tier, credit_risk FROM customers ORDER BY credit_risk DESC LIMIT ?", [c.n]); r = df.iloc[0]
        return A(f"Highest credit risk: **{r.customer_id}** ({r['name']}) at {r.credit_risk:.2f}.", df, s)
    df, s = run("SELECT priority_tier, COUNT(*) AS customers, ROUND(AVG(credit_risk),2) AS avg_credit_risk FROM customers GROUP BY priority_tier ORDER BY customers DESC")
    return A(f"There are **{int(df.customers.sum())}** customers across {len(df)} priority tiers.", df, s)

# ------------------------------------------------------------------ weather
def _storm_rows():
    df, s = run("SELECT v AS storms_json FROM live_state WHERE k = 'storms'")
    try: items = json.loads(df.storms_json[0]) if len(df) else []
    except Exception: items = []
    return items, s

def i_storms(c):
    items, s = _storm_rows(); place = next((v for k, v in sorted(CHOKE.items(), key=lambda kv: -len(kv[0])) if _has(c.q, k)), None)
    rows = [{"storm": i + 1, "lat": round(x.get("lat", 0), 2), "lon": round(x.get("lon", 0), 2), "wind_ms": x.get("wind"), "radius_nm": x.get("r", 700)} for i, x in enumerate(items)]
    if place:
        name, la, lo = place
        for r in rows: r["km_from_" + name.split()[0].lower()] = round(hav(la, lo, r["lat"], r["lon"]))
        near = [r for r in rows if r["km_from_" + name.split()[0].lower()] <= 1500]; pr = Q("SELECT p.port_name, p.country, ROUND(p.congestion_index,1) AS congestion, w.current_severity AS weather, w.wind_speed AS wind_kmh, p.lat, p.lon FROM ports p LEFT JOIN weather_risks w ON w.port_name = p.port_name")
        pr["km"] = [round(hav(la, lo, a, b)) for a, b in zip(pr.lat, pr.lon)]; ports = pr[pr.km <= 1200].sort_values("km").drop(columns=["lat", "lon"]).head(10)
        txt = (f"**{len(near)}** active storm cell(s) within 1,500 km of the **{name}**" if near else f"No active storm cell is within 1,500 km of the **{name}**") + f". **{len(pr[pr.km <= 1200])}** ports lie within 1,200 km; the nearest are listed below."
        return A(txt, ports, s + "  -- ports: SELECT p.port_name, ... FROM ports p LEFT JOIN weather_risks w ON w.port_name = p.port_name (distance computed from lat/lon)")
    if not rows: return A("There are **no active storms** right now.", pd.DataFrame(columns=["storm", "lat", "lon", "wind_ms", "radius_nm"]), s)
    return A(f"**{len(rows)}** active storm cell(s). The strongest has winds of **{max(r['wind_ms'] or 0 for r in rows)} m/s**.", pd.DataFrame(rows), s)

def i_weather_worst(c):
    waves = _has(c.q, "wave", "swell")
    col = "wave_height" if waves else "wind_speed"
    df, s = run(f"SELECT port_name, current_severity AS severity, ROUND(wind_speed,1) AS wind_kmh, ROUND(wave_height,1) AS wave_m, ROUND(temperature,1) AS temp_c FROM weather_risks ORDER BY {col} DESC LIMIT ?", [c.n])
    r = df.iloc[0]; return A(f"Strongest {'waves' if waves else 'winds'}: **{r.port_name}** ({r.wave_m} m waves, {r.wind_kmh} km/h wind).", df, s)

def i_weather_port(c):
    if not c.ports: return None
    ph = ",".join("?" * len(c.ports[:5]))
    df, s = run(f"SELECT port_name, current_severity AS severity, ROUND(wind_speed,1) AS wind_kmh, ROUND(wave_height,1) AS wave_m, ROUND(temperature,1) AS temp_c, forecast FROM weather_risks WHERE port_name IN ({ph})", c.ports[:5])
    if not len(df): return A(f"I have no weather record for **{c.ports[0]}**.", df, s)
    r = df.iloc[0]; return A(f"Weather at **{r.port_name}**: wind {r.wind_kmh} km/h, waves {r.wave_m} m, {r.temp_c} °C, severity **{r.severity}**.", df, s)

def i_weather_high(c):
    df, s = run("SELECT port_name, current_severity AS severity, ROUND(wind_speed,1) AS wind_kmh, ROUND(wave_height,1) AS wave_m FROM weather_risks WHERE current_severity IN ('High', '3', '4', 3, 4) ORDER BY wind_speed DESC LIMIT 25")
    return A(f"**{len(df)}** ports are at high weather severity.", df, s)

# ------------------------------------------------------------------ customs
def i_customs_rules(c):
    goods = Q("SELECT goods FROM customs_rules").goods.tolist(); g = next((x for x in goods if _has(c.q, x.lower())), None)
    if g:
        df, s = run("SELECT chapter, goods, duty_pct, docs AS required_documents, restricted FROM customs_rules WHERE goods = ?", [g]); r = df.iloc[0]
        return A(f"**{r.goods}** (HS chapter {r.chapter}): duty **{r.duty_pct}%**, " + ("restricted goods, " if r.restricted else "") + f"required documents: {r.required_documents.replace(';', ', ')}.", df, s)
    if _has(c.q, "restricted", "prohibited", "controlled"):
        df, s = run("SELECT chapter, goods, duty_pct, docs AS required_documents FROM customs_rules WHERE restricted = 1 ORDER BY duty_pct DESC"); return A(f"**{len(df)}** goods categories are restricted.", df, s)
    df, s = run("SELECT chapter, goods, duty_pct, restricted FROM customs_rules ORDER BY duty_pct DESC LIMIT ?", [max(c.n, 5)]); r = df.iloc[0]
    return A(f"Highest import duty: **{r.goods}** at **{r.duty_pct}%**.", df, s)

def i_customs_docs_needed(c):
    df, s = run("SELECT type, severity, COUNT(*) AS open_tickets FROM agent_actions WHERE type = 'customs_docs' AND status IN ('open','proposed') GROUP BY type, severity")
    n = int(df.open_tickets.sum()) if len(df) else 0; return A(f"**{n}** shipments currently have an open ticket for missing customs documents.", df, s)

# ------------------------------------------------------------------ tickets
def i_tickets(c):
    df, s = run("SELECT type, severity, COUNT(*) AS tickets FROM agent_actions WHERE status IN ('open','proposed') GROUP BY type, severity ORDER BY tickets DESC")
    n = int(df.tickets.sum()) if len(df) else 0; crit = int(df[df.severity == "Critical"].tickets.sum()) if len(df) else 0
    return A(f"There are **{n}** open agent tickets, **{crit}** of them critical.", df, s)


# ------------------------------------------------------------------ intent registry
# must: list of phrase groups, every group needs one phrase;  boost: extra groups that add score.  domain -> rbac.ROLE_DATA.  cust: allowed for a Customer (own data only)
def I(id, domain, must, fn, examples, boost=(), cust=False):
    return {"id": id, "domain": domain, "must": must, "boost": boost, "fn": fn, "examples": examples, "cust": cust}

CHEAP = ("cheapest", "cheap", "cheaper", "costliest", "expensive", "dearest", "priciest", "lowest cost", "least expensive", "lowest price", "best price", "most economical")
INTENTS = [
    I("carrier_cheapest", "carriers", [CHEAP, ("carrier", "line", "shipping line", "liner", "who", "operator")], i_carrier_cheapest, ["Which carrier is cheapest to Rotterdam?", "Which carrier is the most expensive?"], boost=[("to", "for", "serving", "port")]),
    I("carrier_reliable", "carriers", [("reliable", "reliability", "on-time", "punctual", "on time", "late", "unreliable"), ("carrier", "line", "shipping line", "liner")], i_carrier_reliable, ["Which carrier is least reliable?", "Which carrier has the best on-time rate?"]),
    I("carrier_risk", "carriers", [("risk", "risky", "dangerous"), ("carrier", "line", "liner")], i_carrier_risk, ["Which carriers are high risk?", "How many carriers have medium risk?"]),
    I("carrier_rating", "carriers", [("rated", "rating", "stars", "top carrier", "best carrier"), ("carrier", "line", "liner")], i_carrier_rating, ["Which carrier has the highest rating?"]),
    I("carrier_lookup", "carriers", [("tell me about", "about", "details", "info", "how is", "how many", "does", "profile", "performance", "carry", "carries")], i_carrier_lookup, ["Tell me about Maersk Line", "How many shipments does MSC Cargo carry?"]),
    I("port_congestion_top", "ports", [("congested", "congestion", "busiest", "crowded", "jammed", "bottleneck")], i_port_congestion_top, ["Which ports are most congested?", "Which ports in Asia are most congested?", "Which ports are least congested?"], boost=[("ports", "port", "which", "top")]),
    I("port_congestion_lookup", "ports", [("congested", "congestion", "busy", "crowded", "dwell"), ("how", "what", "is", "level", "at", "in")], i_port_congestion_lookup, ["How congested is Singapore?", "What is the congestion at Rotterdam?"], boost=[("port",)]),
    I("port_dwell", "ports", [("dwell", "waiting time", "wait time", "turnaround")], i_port_dwell, ["Which port has the longest dwell time?"]),
    I("ports_by_region", "ports", [("region", "regions", "how many ports", "number of ports", "ports in")], i_ports_by_region, ["How many ports are in Asia?", "Show ports by region"], boost=[("how many", "number of", "count", "by region")]),
    I("congested_count", "ports", [("how many", "number of", "count"), ("congested", "congestion")], i_congested_count, ["How many ports have congestion above 4?"], boost=[("above", "over", "more than")]),
    I("customs_hold", "shipments", [("customs hold", "on hold", "held in customs", "held by customs", "customs delay", "held at customs"), ("shipments", "shipment", "how many", "cargo", "containers", "which")], i_customs_hold, ["How many shipments are on customs hold?"], cust=True),
    I("shipments_delayed", "shipments", [("delayed", "late", "delay", "behind schedule", "running late")], i_shipments_delayed, ["How many shipments are delayed?"], boost=[("shipments", "shipment", "cargo")], cust=True),
    I("shipments_by_status", "shipments", [("status", "in transit", "delivered", "anchorage", "by status", "how many shipments", "number of shipments", "shipments are")], i_shipments_by_status, ["Show shipments by status", "How many shipments are in transit?"], boost=[("shipments", "shipment")], cust=True),
    I("shipments_risk", "shipments", [("delay risk", "risk of delay", "at risk", "riskiest", "most likely to be late", "predicted delay"), ("shipments", "shipment", "cargo")], i_shipments_risk, ["Which shipments have the highest delay risk?"], cust=True),
    I("shipments_to_port", "shipments", [("shipments", "shipment", "cargo", "containers", "arriving", "heading", "bound", "departing", "leaving"), ("to", "from", "at", "for", "arriving", "heading", "bound", "departing", "leaving")], i_shipments_to_port, ["Which shipments are heading to Rotterdam?", "Shipments leaving Mundra Port"], boost=[("arriving", "heading", "bound", "departing", "leaving", "eta")], cust=True),
    I("shipments_heaviest", "shipments", [("heaviest", "lightest", "biggest", "largest", "smallest", "heavy", "weight"), ("shipments", "shipment", "cargo", "container")], i_shipments_heaviest, ["Which are the heaviest shipments?"], cust=True),
    I("vessels_at_sea", "shipments", [("vessels", "vessel", "ships at sea", "at sea", "tracked", "ais")], i_vessels_at_sea, ["How many vessels are at sea?"], cust=True),
    I("shipment_lookup", "shipments", [("shp",)], i_shipment_lookup, ["Where is SHP-0001?"], cust=True),
    I("quotes_low_margin", "quotes", [("margin", "margins", "low margin", "thin margin"), ("below", "under", "less than", "low", "lower than", "poor", "thin", "weak", "lowest")], i_quotes_low_margin, ["Which quotes are below 10% margin?", "Show quotes with a low margin"]),
    I("quotes_high_margin", "quotes", [("margin", "margins"), ("above", "over", "more than", "high", "highest", "best", "greater than")], i_quotes_high_margin, ["Show quotes above 20% margin"]),
    I("quotes_avg", "quotes", [("average", "mean", "avg", "overall"), ("margin", "margins", "quote value", "price")], i_quotes_avg, ["What is the average margin?"]),
    I("quotes_high_value", "quotes", [("quotes", "quote", "quotation"), ("above", "over", "more than", "exceed", "exceeding", "greater than", "big", "large", "expensive", "high value")], i_quotes_high_value, ["Show quotes above $50,000"], boost=[("price", "value", "worth", "$")], cust=True),
    I("quotes_by_status", "quotes", [("quotes", "quote", "quotation"), ("status", "pending", "approved", "rejected", "on hold", "draft", "submitted", "accepted", "how many", "number of", "by status")], i_quotes_by_status, ["How many quotes are pending approval?", "Show quotes by status"], cust=True),
    I("quotes_latest", "quotes", [("latest", "recent", "newest", "last", "new"), ("quotes", "quote", "quotation")], i_quotes_latest, ["Show the latest quotes"], cust=True),
    I("quote_lookup", "quotes", [("qte",)], i_quote_lookup, ["What is the status of QTE-0001?"], cust=True),
    I("customer_credit", "customers", [("customer", "customers", "client", "clients", "shipper"), ("credit", "credit risk", "risky")], i_customer_credit, ["Which customers have the highest credit risk?"], boost=[("credit",)]),
    I("top_customers", "customers", [("customer", "customers", "client", "clients", "shipper"), ("most", "top", "biggest", "largest", "best", "highest", "quoted value", "quote value", "revenue", "spend", "spends")], i_top_customers, ["Which customer has the highest quoted value?"]),
    I("customer_tiers", "customers", [("platinum", "gold", "silver", "tier", "how many customers", "number of customers", "customers")], i_customer_tiers, ["How many Platinum customers do we have?"]),
    I("storms", "weather", [("storm", "storms", "cyclone", "typhoon", "hurricane", "bad weather")], i_storms, ["Show storms near the Red Sea", "Are there any active storms?"]),
    I("weather_worst", "weather", [("wind", "winds", "windy", "waves", "wave", "swell", "rough", "weather"), ("worst", "strongest", "highest", "roughest", "most", "top")], i_weather_worst, ["Which ports have the strongest winds?", "Which ports have the highest waves?"]),
    I("weather_high", "weather", [("weather",), ("high risk", "severe", "high severity", "severity", "dangerous", "risk")], i_weather_high, ["Which ports are at high weather severity?"]),
    I("weather_port", "weather", [("weather", "wind", "waves", "temperature", "forecast", "conditions")], i_weather_port, ["What is the weather at Singapore?"], boost=[("at", "in", "for")]),
    I("customs_rules", "customs", [("duty", "tariff", "duties", "customs rules", "restricted", "documents required", "required documents", "hs code", "documents", "document", "import")], i_customs_rules, ["What is the duty on pharmaceuticals?", "Which goods are restricted?"]),
    I("customs_docs_needed", "customs", [("missing", "need", "needs", "require", "required", "lack"), ("documents", "document", "docs", "paperwork")], i_customs_docs_needed, ["How many shipments need customs documents?"]),
    I("tickets", "tickets", [("tickets", "ticket", "alerts", "alert", "issues", "incidents"), ("open", "critical", "how many", "pending", "active")], i_tickets, ["How many critical tickets are open?"]),
]
EXAMPLE_POOL = [(e, it["domain"], it["id"], it["cust"]) for it in INTENTS for e in it["examples"]]

GREETING = re.compile(r"^\W*(hi|hello|hey|hii+|good (morning|afternoon|evening)|namaste|yo|hola)\b[\w\s!.,]*$", re.I)
THANKS = re.compile(r"^\W*(thanks|thank you|thx|ty|great|cool|nice|awesome|perfect)\b[\w\s!.,]*$", re.I)
HELP = re.compile(r"\b(help|what can you do|what can i ask|examples?|capabilit\w+)\b", re.I)


def allowed(role, intent):
    data = rbac.ROLE_DATA.get(role, set())
    if intent["domain"] in data: return True
    if role == "Customer" and intent["cust"] and intent["domain"] in ("quotes", "shipments"): return True
    return False


def suggestions(role, text, k=3):
    pool = [e for e, dom, iid, cust in EXAMPLE_POOL if allowed(role, next(i for i in INTENTS if i["id"] == iid))]
    if role == "Customer": pool = [e for e in pool if "margin" not in e.lower() and "customer" not in e.lower()]
    if not pool: return []
    ranked = sorted(pool, key=lambda e: -difflib.SequenceMatcher(None, _norm(text), _norm(e)).ratio()); out = []
    for e in ranked:
        if e not in out: out.append(e)
        if len(out) == k: break
    return out


def _score(it, q):
    s = 50 if it["id"] in ("quote_lookup", "shipment_lookup") else 0
    for g in it["must"]:
        if not _has(q, *g): return 0
        s += 10
    for g in it["boost"]:
        if _has(q, *g): s += 2
    return s


def answer(user, text):
    """-> dict(kind, text, sql, rows, table, note, intent, suggestions). kind: answer | denied | unknown | greeting."""
    role = (user or {}).get("role"); raw = (text or "").strip()
    if GREETING.match(raw): return {"kind": "greeting", "text": "Hello! I answer questions about your freight data with exact database queries, and I show the SQL behind every answer. Try one of the examples below.", "sql": "", "rows": 0, "suggestions": suggestions(role, "")}
    if THANKS.match(raw): return {"kind": "greeting", "text": "You're welcome! Ask me anything else about your data.", "sql": "", "rows": 0, "suggestions": []}
    if HELP.search(raw) and len(raw.split()) <= 6: return {"kind": "greeting", "text": "I can answer questions about ports, carriers, shipments, quotes, weather, customs and open tickets (what your role may see). Pick one:", "sql": "", "rows": 0, "suggestions": suggestions(role, "", 5)}
    q, fixes = correct(raw); c = Ctx(user, q, raw, fixes)
    note = ("Interpreted as: “" + q + "”") if fixes else ""
    cands = sorted(((_score(it, q), n, it) for n, it in enumerate(INTENTS)), key=lambda x: (-x[0], x[1])); cands = [x for x in cands if x[0] > 0]
    denied = None
    for sc, _, it in cands:
        if not allowed(role, it):
            denied = it; break                # the best match is data this role may not read: refuse instead of answering something else
        try: r = it["fn"](c)
        except Exception as e: r = None
        if r:
            r.update(kind="answer", intent=it["id"], note=note)
            df = r.pop("df", None); r["table"] = None if df is None or not len(df) else {"cols": list(map(str, df.columns)), "rows": [[(None if (isinstance(v, float) and v != v) else v) for v in row] for row in fmt_df(df).head(25).itertuples(index=False, name=None)], "total": int(len(df))}
            return r
    if denied is not None:
        need = {"quotes": "quote and margin", "customers": "customer", "weather": "weather", "customs": "customs", "ports": "port", "carriers": "carrier", "shipments": "shipment", "tickets": "agent ticket"}.get(denied["domain"], denied["domain"])
        who = "your own shipments and quotes" if role == "Customer" else "the pages your role can open"
        return {"kind": "denied", "text": f"I'm sorry, I can't answer that for your role (**{role}**): it needs {need} data that this role is not allowed to read. I can only use {who}. Here are questions I can answer for you:", "sql": "", "rows": 0, "note": note, "suggestions": suggestions(role, raw)}
    return {"kind": "unknown", "text": "I'm sorry, I couldn't match that to the data I can query. I only answer from the database, so I won't guess. Here are three questions that work:", "sql": "", "rows": 0, "note": note, "suggestions": suggestions(role, raw)}


In [ ]:
%%writefile /content/app/weather.py
"""Real weather for every port: Open-Meteo forecast + marine APIs (free, no key), fetched in batches and stored in wx_live.
Falls back to the simulated values already in weather_risks when the network is closed; the page says which one it is showing."""
import time, threading, datetime, json
import live_engine as LE

try: import requests
except Exception: requests = None

FORECAST = "https://api.open-meteo.com/v1/forecast"
MARINE = "https://marine-api.open-meteo.com/v1/marine"
WMO = {0: "Clear", 1: "Mostly clear", 2: "Partly cloudy", 3: "Overcast", 45: "Fog", 48: "Rime fog", 51: "Light drizzle", 53: "Drizzle", 55: "Heavy drizzle", 56: "Freezing drizzle", 57: "Freezing drizzle", 61: "Light rain", 63: "Rain", 65: "Heavy rain", 66: "Freezing rain", 67: "Freezing rain",
       71: "Light snow", 73: "Snow", 75: "Heavy snow", 77: "Snow grains", 80: "Rain showers", 81: "Rain showers", 82: "Violent showers", 85: "Snow showers", 86: "Snow showers", 95: "Thunderstorm", 96: "Thunderstorm, hail", 99: "Thunderstorm, heavy hail"}
_get = None   # tests can replace this with a function(url, params) -> json

def ensure():
    c = LE.conn(); c.execute("CREATE TABLE IF NOT EXISTS wx_live(port_name TEXT PRIMARY KEY, ts TEXT, wind_ms REAL, gust_ms REAL, wind_dir REAL, temp_c REAL, precip_mm REAL, wcode INT, wave_m REAL, wave_dir REAL, src TEXT)"); c.commit()

def _fetch(url, params):
    if _get: return _get(url, params)
    r = requests.get(url, params=params, timeout=10); r.raise_for_status(); return r.json()

def _listify(j): return j if isinstance(j, list) else [j]

def refresh(batch=50):
    """Fetch current weather for every port. Returns (rows_updated, message)."""
    ensure(); c = LE.conn(); ports = c.execute("SELECT port_name, lat, lon FROM ports").fetchall()
    if not ports: return 0, "no ports"
    n = 0; err = ""; ts = datetime.datetime.utcnow().isoformat(timespec="seconds")
    for i in range(0, len(ports), batch):
        chunk = ports[i:i + batch]; la = ",".join(f"{p[1]:.3f}" for p in chunk); lo = ",".join(f"{p[2]:.3f}" for p in chunk)
        try:
            f = _listify(_fetch(FORECAST, {"latitude": la, "longitude": lo, "current": "temperature_2m,precipitation,weather_code,wind_speed_10m,wind_direction_10m,wind_gusts_10m", "wind_speed_unit": "ms", "timezone": "UTC"}))
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:120]}"; break
        try: m = _listify(_fetch(MARINE, {"latitude": la, "longitude": lo, "current": "wave_height,wave_direction", "timezone": "UTC"}))
        except Exception: m = [{}] * len(chunk)
        for p, a, b in zip(chunk, f, m + [{}] * (len(chunk) - len(m))):
            cur = (a or {}).get("current") or {}
            if cur.get("wind_speed_10m") is None: continue
            mc = (b or {}).get("current") or {}; wave = mc.get("wave_height")
            c.execute("INSERT OR REPLACE INTO wx_live VALUES(?,?,?,?,?,?,?,?,?,?,?)", (p[0], ts, cur.get("wind_speed_10m"), cur.get("wind_gusts_10m"), cur.get("wind_direction_10m"), cur.get("temperature_2m"), cur.get("precipitation") or 0.0, cur.get("weather_code"), wave, mc.get("wave_direction"), "Open-Meteo"))
            wind = float(cur["wind_speed_10m"]); lvl = "High" if wind > 15 else "Medium" if wind >= 8 else "Low"
            c.execute("INSERT OR REPLACE INTO weather_risks(port_name,current_severity,forecast,wind_speed,wave_height,temperature) VALUES(?,?,?,?,?,?)", (p[0], lvl, f"Open-Meteo live {ts[11:16]} UTC · {WMO.get(cur.get('weather_code'), '')}", wind * 3.6, float(wave) if wave is not None else 1.2, cur.get("temperature_2m")))
            n += 1
    c.commit(); LE.set_state("wx_status", {"ts": ts if n else (LE.get_state("wx_status", {}) or {}).get("ts"), "ok": n, "total": len(ports), "err": err, "tried": ts})
    return n, (err or f"{n} ports updated")

def status():
    s = LE.get_state("wx_status", {}) or {}; ok = int(s.get("ok") or 0); age = None
    if s.get("ts"):
        try: age = (datetime.datetime.utcnow() - datetime.datetime.fromisoformat(s["ts"])).total_seconds() / 60
        except Exception: pass
    live = bool(ok) and age is not None and age < 120
    return {"live": live, "ts": s.get("ts"), "age_min": age, "ok": ok, "total": s.get("total"), "err": s.get("err", ""), "tried": s.get("tried")}

def fresh_ports(max_age_min=90):
    """Ports whose real weather is recent: the simulator must not overwrite them."""
    try:
        cut = (datetime.datetime.utcnow() - datetime.timedelta(minutes=max_age_min)).isoformat(timespec="seconds")
        return {r[0] for r in LE.conn().execute("SELECT port_name FROM wx_live WHERE ts>=?", (cut,)).fetchall()}
    except Exception: return set()

_T = {"t": None}
def start(every_min=20):
    if _T["t"]: return
    def loop():
        time.sleep(5)
        while True:
            try: refresh()
            except Exception: pass
            time.sleep(every_min * 60 if status()["live"] else 120)
    _T["t"] = threading.Thread(target=loop, daemon=True); _T["t"].start()

def table():
    import pandas as pd
    ensure()
    return pd.read_sql("SELECT w.*, p.lat, p.lon FROM wx_live w JOIN ports p USING(port_name)", LE.conn())


In [ ]:
%%writefile /content/app/weather_context.py
import requests
from seed_data import BASE_PORTS

# Auto-built from BASE_PORTS in seed_data.py — always in sync with the port list
# (currently 50 ports). Each BASE_PORTS row is:
# (name, country, congestion_index, avg_dwell_days, lat, lon, region)
PORT_COORDS = {p[0]: (p[4], p[5]) for p in BASE_PORTS}

_DEFAULT_COORDS = (1.29, 103.85)  # Singapore Port as a safe fallback

def fetch_port_weather(port_name):
    coords = PORT_COORDS.get(port_name, _DEFAULT_COORDS)
    try:
        url = f"https://api.open-meteo.com/v1/forecast?latitude={coords[0]}&longitude={coords[1]}&current_weather=true"
        r = requests.get(url, timeout=3)
        if r.status_code == 200:
            cw = r.json().get('current_weather', {})
            windspeed = cw.get('windspeed', 15.0)
            severity = 1 if windspeed < 20 else (2 if windspeed < 40 else 3)
            return {'windspeed': windspeed, 'temperature': cw.get('temperature', 22.0), 'severity': severity}
    except Exception:
        pass
    return {'windspeed': 18.5, 'temperature': 24.0, 'severity': 2}

In [ ]:
%%writefile /content/app/worker.py
"""Background worker process: the simulator, the agent cycles, escalation and the feed threads run here so the web server (and the GPU models) never share a GIL with them."""
import os, sys, time, importlib
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config, live_engine as LE
dom = importlib.import_module("domain_franchise" if config.KIND == "Franchise" else "domain_freight")
if __name__ == "__main__":
    rt = LE.Runtime(dom, int(os.environ.get("LIVE_TICK_SEC", 5)), int(os.environ.get("LIVE_CYCLE_SEC", 60)))
    if config.KIND == "Freight":
        import weather as WX; WX.start()
    while True: time.sleep(3600)


In [ ]:
%%writefile /content/app/static/app.js
"use strict";
const $=(s,r=document)=>r.querySelector(s);
const esc=s=>String(s==null?"":s).replace(/[&<>"']/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));
function h(tag,attrs,...kids){const e=document.createElement(tag);for(const k in(attrs||{})){const v=attrs[k];if(v==null||v===false)continue;if(k==="class")e.className=v;else if(k==="html")e.innerHTML=v;else if(k.startsWith("on"))e.addEventListener(k.slice(2),v);else e.setAttribute(k,v===true?"":v)}for(const c of kids.flat())if(c!=null&&c!==false)e.append(c.nodeType?c:document.createTextNode(c));return e}
const TONE={Critical:"#cd2b31",High:"#cd2b31",Fail:"#cd2b31",rejected:"#cd2b31","On Hold":"#cd2b31",overdue:"#cd2b31",Warning:"#ab6400",Medium:"#ab6400","Pending Approval":"#ab6400",proposed:"#793aaf",Info:"#3451b2",open:"#3451b2",Low:"#0b7a6c",Pass:"#0b7a6c",Approved:"#0b7a6c",approved:"#0b7a6c",resolved:"#0b7a6c","auto-closed":"#6b7080",live:"#0b7a6c",Healthy:"#0b7a6c",Watch:"#ab6400",Rejected:"#6b7080"};
const tone=v=>TONE[v]||(/crit|high|fail|bad|reject|hold/i.test(v)?"#cd2b31":/warn|med|pend/i.test(v)?"#ab6400":/ok|pass|low|good|appr|done|clear|live/i.test(v)?"#0b7a6c":"#6b7080");
const ICON={bot:"M12 8V4H8M4 12a2 2 0 0 1 2-2h12a2 2 0 0 1 2 2v6a2 2 0 0 1-2 2H6a2 2 0 0 1-2-2zM2 14h2M20 14h2M9 15v1M15 15v1",users:"M16 21v-2a4 4 0 0 0-4-4H6a4 4 0 0 0-4 4v2M9 11a4 4 0 1 0 0-8 4 4 0 0 0 0 8M22 21v-2a4 4 0 0 0-3-3.9M16 3.1a4 4 0 0 1 0 7.8",store:"M3 9l1-5h16l1 5M3 9v11h18V9M3 9a3 3 0 0 0 6 0 3 3 0 0 0 6 0 3 3 0 0 0 6 0M9 20v-6h6v6",boxes:"M21 8l-9-5-9 5 9 5zM3 8v8l9 5 9-5V8M12 13v8",megaphone:"M3 11v2a1 1 0 0 0 1 1h3l8 5V5L7 10H4a1 1 0 0 0-1 1zM19 8a5 5 0 0 1 0 8",messages:"M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z",clipboard:"M9 4h6v3H9zM8 5H6a2 2 0 0 0-2 2v13a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V7a2 2 0 0 0-2-2h-2M9 13l2 2 4-4",bell:"M18 8a6 6 0 0 0-12 0c0 7-3 9-3 9h18s-3-2-3-9M13.7 21a2 2 0 0 1-3.4 0",network:"M12 2v6M12 16v6M5 6l5 4M19 6l-5 4M5 18l5-4M19 18l-5-4M12 8a4 4 0 1 0 0 8 4 4 0 0 0 0-8",flask:"M10 2v7L4 20a1 1 0 0 0 1 1h14a1 1 0 0 0 1-1l-6-11V2M8 2h8M7 15h10",upload:"M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4M17 8l-5-5-5 5M12 3v12",shield:"M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10",radar:"M12 12l7-7M12 2a10 10 0 1 0 10 10M12 6a6 6 0 1 0 6 6M12 10a2 2 0 1 0 2 2",route:"M6 19a3 3 0 1 0 0-6 3 3 0 0 0 0 6M18 11a3 3 0 1 0 0-6 3 3 0 0 0 0 6M9 16h6a3 3 0 0 0 0-6H9a3 3 0 0 1 0-6h3",dollar:"M12 2v20M17 5H9.5a3.5 3.5 0 0 0 0 7h5a3.5 3.5 0 0 1 0 7H6",truck:"M1 3h15v13H1zM16 8h4l3 3v5h-7M5.5 21a2.5 2.5 0 1 0 0-5 2.5 2.5 0 0 0 0 5M18.5 21a2.5 2.5 0 1 0 0-5 2.5 2.5 0 0 0 0 5",cloud:"M17.5 19a4.5 4.5 0 1 0-1.4-8.8A6 6 0 1 0 5 15a4 4 0 0 0 1 4z",trend:"M3 17l6-6 4 4 8-8M15 7h6v6",chart:"M3 3v18h18M7 15l4-4 3 3 5-6",check:"M22 11.1V12a10 10 0 1 1-5.9-9.1M22 4L12 14l-3-3",user:"M20 21v-2a4 4 0 0 0-4-4H8a4 4 0 0 0-4 4v2M12 11a4 4 0 1 0 0-8 4 4 0 0 0 0 8",file:"M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8zM14 2v6h6M9 13h6M9 17h6"};
const icon=n=>{const s=document.createElementNS("http://www.w3.org/2000/svg","svg");s.setAttribute("viewBox","0 0 24 24");const p=document.createElementNS("http://www.w3.org/2000/svg","path");p.setAttribute("d",ICON[n]||ICON.file);s.append(p);return s};
const S={me:null,authMsg:null,pub:null,es:null,page:null,tab:null,params:{},lang:localStorage.getItem("lang")||"English",pulse:null,tree:null,timer:null,seq:0,boot:null};
const LANGS_CODE={};
async function api(path,opt={}){const o={credentials:"same-origin",...opt};const hd={...(opt.headers||{})};const tk=localStorage.getItem("fq_token");if(tk)hd.Authorization="Bearer "+tk;
 if(opt.json!==undefined){o.method=o.method||"POST";hd["Content-Type"]="application/json";o.body=JSON.stringify(opt.json)}o.headers=hd;
 let r;try{r=await fetch(path,o)}catch(e){return{ok:false,status:0,msg:"Cannot reach the server. Check your connection and try again."}}
 let d=null;try{d=await r.json()}catch(e){}
 if(r.status===401&&!path.startsWith("/api/auth/")){if(S.me){const m=(d&&(d.detail||d.msg))||"Please sign in again.";S.me=null;localStorage.removeItem("fq_token");S.authMsg=m;if(S.es){S.es.abort();S.es=null}render()}return{ok:false,status:401,msg:(d&&(d.detail||d.msg))||"Please sign in."}}
 if(!d)d={msg:"Error "+r.status};if(!r.ok){d.ok=false;if(!d.msg)d.msg=(typeof d.detail==="string"?d.detail:"Error "+r.status)}d.status=r.status;return d}
function toast(msg,bad){const t=h("div",{class:bad?"bad":""},msg);$("#toast").append(t);setTimeout(()=>t.remove(),bad?7000:4500)}
const fmt={num:v=>v==null?"–":(typeof v==="number"?(Math.abs(v)>=1000?v.toLocaleString(undefined,{maximumFractionDigits:1}):+v.toFixed(2)):v),int:v=>v==null?"–":Math.round(v).toLocaleString(),money:v=>v==null?"–":"$"+Math.round(v).toLocaleString()};
function md(t){let s=esc(t||"");s=s.replace(/```([\s\S]*?)```/g,(_,c)=>"<pre>"+c.trim()+"</pre>").replace(/`([^`]+)`/g,"<code>$1</code>").replace(/\*\*([^*]+)\*\*/g,"<b>$1</b>");const L=s.split("\n");let o="",inl=false,tb=null;const row=l=>l.trim().replace(/^\||\|$/g,"").split("|").map(x=>x.trim());const endT=()=>{if(tb){o+='<div class="tbl" style="margin:8px 0"><table><thead><tr>'+row(tb[0]).map(x=>"<th>"+x+"</th>").join("")+"</tr></thead><tbody>"+tb.slice(2).map(r=>"<tr>"+row(r).map(x=>"<td>"+x+"</td>").join("")+"</tr>").join("")+"</tbody></table></div>";tb=null}};for(let i=0;i<L.length;i++){const l=L[i];if(/^\s*\|.*\|\s*$/.test(l)){if(inl){o+="</ul>";inl=false}(tb=tb||[]).push(l);continue}endT();const m=l.match(/^\s*[-*•]\s+(.*)/);if(m){if(!inl){o+="<ul>";inl=true}o+="<li>"+m[1]+"</li>"}else{if(inl){o+="</ul>";inl=false}const hd=l.match(/^(#{1,3})\s+(.*)/);o+=hd?`<h${hd[1].length}>${hd[2]}</h${hd[1].length}>`:(l.trim()?l+"<br>":"")}}endT();if(inl)o+="</ul>";return o}

// ---------------------------------------------------------------- block renderers (each returns {el, update(b)})
function mount(b,key){const f=R[b.t];if(!f){return{el:h("div",{class:"card"},"Unsupported block: "+b.t),update(){}}}const m=f(b,key);m.t=b.t;m.k=b.key||b.id||"";return m}
function mountList(parent,blocks,key){const hs=blocks.map((b,i)=>mount(b,key+"/"+i));hs.forEach(x=>parent.append(x.el));return hs}
function updList(parent,hs,blocks,key){if(hs.length!==blocks.length||hs.some((x,i)=>x.t!==blocks[i].t||x.k!==(blocks[i].key||blocks[i].id||""))){parent.replaceChildren();return mountList(parent,blocks,key)}hs.forEach((x,i)=>x.update(blocks[i]));return hs}
const R={};
R.kpis=b=>{const el=h("div",{class:"kpis"});const up=b=>{el.replaceChildren(...b.items.map(i=>h("div",{class:"kpi "+(i.tone||"")},h("div",{class:"l"},i.label),h("div",{class:"v"},typeof i.value==="number"?fmt.num(i.value):i.value),i.sub?h("div",{class:"s"},i.sub):null)))};up(b);return{el,update:up}};
R.callout=b=>{const el=h("div");const up=b=>{el.className="callout "+(b.tone||"info");el.replaceChildren(h("div",null,b.title?h("b",null,b.title+" "):null,h("span",{html:md(b.text)})))};up(b);return{el,update:up}};
R.md=b=>{const el=h("div",{class:"md"});const up=b=>el.innerHTML=md(b.text);up(b);return{el,update:up}};
R.row=(b,k)=>{const el=h("div",{class:"row"});const sty=r=>el.style.gridTemplateColumns=(r||b.cols.map(()=>1)).map(x=>x+"fr").join(" ");sty(b.ratio);let hs=[];const wrap=[];const up=nb=>{sty(nb.ratio);if(hs.length!==nb.cols.length||hs.some((x,i)=>x.t!==nb.cols[i].t)){el.replaceChildren();hs=nb.cols.map((c,i)=>{const m=mount(c,k+"/"+i);el.append(m.el);return m})}else hs.forEach((x,i)=>x.update(nb.cols[i]))};up(b);return{el,update:up}};
R.col=(b,k)=>{const el=h("div",{class:"col"});let hs=mountList(el,b.blocks,k);return{el,update:nb=>{hs=updList(el,hs,nb.blocks,k)}}};
R.card=(b,k)=>{const el=h("div",{class:"card"});const hd=h("h3"),sub=h("div",{class:"sub"}),body=h("div");el.append(hd,sub,body);let hs=[];const up=(nb,first)=>{hd.textContent=nb.title||"";hd.style.display=nb.title?"":"none";sub.textContent=nb.sub||"";sub.style.display=nb.sub?"":"none";hs=first?mountList(body,nb.blocks,k):updList(body,hs,nb.blocks,k)};up(b,true);return{el,update:nb=>up(nb)}};
R.hero=b=>{const el=h("div",{class:"hero"});const up=b=>{const c=b.risk?tone(b.risk):"#8b8d98";el.replaceChildren(h("div",{class:"sc",style:"color:"+c},b.score==null?"–":Math.round(b.score)),h("div",{style:"flex:1"},h("h2",null,b.title," ",b.risk?h("span",{class:"pill",style:`color:${c};border-color:${c}`},b.risk):null),h("p",null,b.blurb),h("p",{style:"color:var(--tx2)"},b.summary||"No run yet."),h("p",null,b.ts?`last run ${b.ts.slice(11,19)} · ${b.ms} ms`:"")),b.can_run?h("button",{class:"btn primary",onclick:async e=>{e.target.disabled=true;const r=await api("/api/run_agent",{json:{agent:b.agent}});toast(r.msg,!r.ok);e.target.disabled=false;reloadPage()}},"Run now"):null)};up(b);return{el,update:up}};
R.brief=b=>{const el=h("div",{class:"card"});const out=h("div",{class:"md"});const up=b=>{out.dataset.def=b.text;if(!out.dataset.ai)out.innerHTML=md(b.text)};el.append(h("h3",null,"Executive briefing"),out,b.can_brief?h("button",{class:"btn sm",style:"margin-top:8px",onclick:async e=>{e.target.disabled=true;out.dataset.ai="1";out.textContent="";let acc="";await sse("/api/brief",{lang:S.lang},ev=>{if(ev.ev==="tok"){acc+=ev.text;out.innerHTML=md(acc)}if(ev.ev==="final")out.innerHTML=md(ev.text)});e.target.disabled=false}},"Write with the GPU model"):null);up(b);return{el,update:up}};
R.button=b=>{const el=h("button",{class:"btn "+(b.tone||"primary"),onclick:async()=>{if(b.payload&&b.payload.download){window.open(b.url,"_blank");return}if(b.confirm&&!confirm(b.confirm))return;el.disabled=true;const r=await api(b.url,{json:b.payload||{}});toast(r.msg||"Done",!r.ok);el.disabled=false;if(r.refresh)reloadPage()}},b.label);return{el,update(){}}};
R.table=b=>{const el=h("div"),title=h("h3",{style:"margin:0 0 6px;font-size:13.5px"}),sub=h("div",{class:"sub"}),srch=h("input",{class:"srch",placeholder:"Search…",oninput:()=>{st.q=srch.value.toLowerCase();draw()}}),box=h("div",{class:"tbl"}),st={sort:null,dir:1,q:""};let cur=b;el.append(title,sub,srch,box);
 const draw=()=>{const b=cur;title.textContent=b.title||"";title.style.display=b.title?"":"none";sub.textContent=b.sub||"";sub.style.display=b.sub?"":"none";srch.style.display=b.search?"":"none";if(b.h)box.style.maxHeight=b.h+"px";
  let rows=b.rows;if(st.q)rows=rows.filter(r=>Object.values(r).some(v=>String(v).toLowerCase().includes(st.q)));if(st.sort)rows=[...rows].sort((x,y)=>{const a=x[st.sort],c=y[st.sort];return(a==null)-(c==null)||(a>c?1:a<c?-1:0)*st.dir});
  const mx={};b.cols.forEach(c=>{if(c.type==="pct"){mx[c.k]=Math.max(...b.rows.map(r=>Math.abs(r[c.k]||0)))<=1.0001?100:1}});
  const th=h("tr",null,...b.cols.map(c=>h("th",{onclick:()=>{st.dir=st.sort===c.k?-st.dir:1;st.sort=c.k;draw()}},c.label+(st.sort===c.k?(st.dir>0?" ▲":" ▼"):""))));
  const tb=rows.slice(0,400).map(r=>{const tr=h("tr",{class:b.open?"cl":""},...b.cols.map(c=>{const v=r[c.k];const t=c.type||"text";let n;
    if(t==="pill"&&v!=null&&v!=="")n=h("span",{class:"pill",style:`color:${tone(v)};border-color:${tone(v)}55;background:${tone(v)}18`},v);
    else if(t==="bar"){const m=c.max||100,p=Math.max(0,Math.min(100,(v||0)/m*100)),col=c.k.match(/risk|prob|congest|ci|index/i)?(p>66?"#e5484d":p>40?"#f5a524":"#12a594"):(p<60?"#e5484d":p<80?"#f5a524":"#12a594");n=h("div",{class:"bar"},h("span",null,v==null?"–":fmt.num(v)),h("i",null,h("b",{style:`width:${p}%;background:${col}`})))}
    else if(t==="pct")n=document.createTextNode(v==null?"–":(+(v*mx[c.k]).toFixed(1))+"%");else if(t==="money")n=document.createTextNode(fmt.money(v));else if(t==="int")n=document.createTextNode(fmt.int(v));else if(t==="num")n=document.createTextNode(fmt.num(v));else n=document.createTextNode(v==null?"":v);
    return h("td",{title:typeof v==="string"&&v.length>40?v:null},n)}));if(b.open)tr.onclick=()=>UI.openEntity(r[b.open.col]);return tr});
  const sc=box.scrollTop;box.replaceChildren(h("table",null,h("thead",null,th),h("tbody",null,...tb)));box.scrollTop=sc;if(!rows.length)box.append(h("div",{class:"sub",style:"padding:12px"},"No rows."))};
 draw();return{el,update:nb=>{cur=nb;draw()}}};
R.tickets=b=>{const el=h("div"),st={f:"all",open:new Set()};let cur=b;const body=h("div"),fl=h("div",{class:"filters"});el.append(h("h3",{style:"margin:0 0 6px;font-size:13.5px"},b.title||"Tickets"),h("div",{class:"sub"},b.sub||""),fl,body);
 const act=async(id,a,json)=>{const r=await api(`/api/ticket/${id}/${a}`,{json:json||{}});toast(r.msg,!r.ok);if(document.activeElement)document.activeElement.blur();st.sig=null;reloadPage(true)};
 const draw=()=>{const items=cur.items;st.sig=JSON.stringify(items)+st.f+[...st.open];const cnt=k=>k==="all"?items.length:k==="proposed"?items.filter(i=>i.status==="proposed").length:k==="overdue"?items.filter(i=>i.overdue).length:items.filter(i=>i.sev===k).length;
  fl.replaceChildren(...["all","proposed","Critical","Warning","Info","overdue"].map(k=>h("span",{class:"chip"+(st.f===k?" on":""),onclick:()=>{st.f=k;draw()}},(k==="proposed"?"awaiting approval":k)+" · "+cnt(k))));
  const list=items.filter(i=>st.f==="all"||(st.f==="proposed"?i.status==="proposed":st.f==="overdue"?i.overdue:i.sev===st.f));
  body.replaceChildren(...list.slice(0,80).map(i=>{const c=tone(i.sev);const t=h("div",{class:"tk "+i.sev+" "+i.status},
   h("div",{class:"h"},h("b",null,"#"+i.id+" "+i.type.replace(/_/g," ")),h("span",{class:"pill",style:`color:${c};border-color:${c}66`},i.sev),i.status==="proposed"?h("span",{class:"pill",style:"color:#8e4ec6;border-color:#8e4ec666"},"awaiting approval"):null,i.overdue?h("span",{class:"pill",style:"color:#e5484d;border-color:#e5484d66"},"overdue"):null,h("a",{href:"#",onclick:e=>{e.preventDefault();UI.openEntity(i.target)}},i.target)),
   h("div",{class:"d"},i.detail),h("div",{class:"m"},h("span",null,"agent: "+i.agent),h("span",null,"owner: "+(i.owner||"–")),h("span",null,"assignee: "+(i.assignee||"unassigned")),h("span",null,"due "+(i.due||"–")),h("span",null,i.ts)),
   i.chain&&i.chain.length?h("div",{class:"m"},h("span",null,"approval chain: "),...i.chain.map(s=>h("span",{class:"pill",style:`color:${s.status==="approved"?"#1a9e6e":s.status==="rejected"?"#e5484d":"#8e4ec6"};border-color:currentColor`},"L"+s.step+" "+s.role+(s.status==="approved"?" ✓ "+s.by:s.status==="rejected"?" ✗":" …")))):null,
   i.status==="proposed"&&i.turn&&!i.can_approve?h("div",{class:"m"},"Waiting for a "+i.turn+" to decide"):null,
   i.note?h("div",{class:"m"},"note: "+i.note):null,
   h("div",{class:"a"},i.status==="proposed"&&i.can_approve?[h("button",{class:"btn ok sm",onclick:()=>act(i.id,"approve")},"Approve & execute"),h("button",{class:"btn danger sm",onclick:()=>act(i.id,"reject")},"Reject")]:null,
    i.quote&&i.can_approve&&i.status!=="proposed"?[h("button",{class:"btn ok sm",onclick:()=>act(i.id,"approve")},"Approve quote"),h("button",{class:"btn danger sm",onclick:()=>act(i.id,"reject")},"Reject")]:null,
    i.can_resolve?[h("button",{class:"btn sm",onclick:()=>act(i.id,"resolve")},"Resolve"),(()=>{const a=h("input",{class:"in",style:"width:170px",placeholder:"assign to email",value:i.assignee||""});return h("span",{style:"display:flex;gap:4px"},a,h("button",{class:"btn sm",onclick:()=>a.value&&act(i.id,"assign",{assignee:a.value})},"Assign"))})()]:null,
    h("button",{class:"btn sm",onclick:()=>{st.open.has(i.id)?st.open.delete(i.id):st.open.add(i.id);draw()}},"Comments ("+i.comments.length+")")),
   st.open.has(i.id)?h("div",{class:"cm"},...i.comments.map(c=>h("div",null,h("b",null,c.by),` ${c.ts}: ${c.body}`)),(()=>{const t=h("input",{class:"in",placeholder:"Write a comment and press Enter",onkeydown:e=>{if(e.key==="Enter"&&t.value.trim())act(i.id,"comment",{body:t.value})}});return t})()):null);return t}));if(!list.length)body.append(h("div",{class:"sub"},"Nothing here."))};
 draw();return{el,update:nb=>{if(el.contains(document.activeElement)&&document.activeElement.tagName==="INPUT"){cur=nb;return}if(JSON.stringify(nb.items)+st.f+[...st.open]===st.sig)return;cur=nb;draw()}}};
// ---- charts
const FN={size3:v=>Math.max(8,Math.min(34,Math.sqrt(Math.abs(v[3]||10))*1.6)),scatterTip:p=>`<b>${esc(p.value[2]||"")}</b><br>${esc(p.seriesName||"")}<br>${p.value[0]}, ${p.value[1]}`,nodeTip:p=>p.dataType==="node"?`<b>${esc(p.name)}</b><br>${esc(p.data.tip||"")}`:"",nodeLabel:p=>p.data.showLabel?p.name:""};
function revive(o){if(Array.isArray(o))return o.map(revive);if(o&&typeof o==="object"){if(o.__fn)return FN[o.__fn];const r={};for(const k in o)r[k]=revive(o[k]);return r}return o}
const CH=[];function lazy(box,fn){let d=0;const t=()=>{if(d)return;if(box.isConnected&&box.clientWidth>0){d=1;fn()}else setTimeout(t,60)};t()}
R.chart=b=>{const el=h("div"),box=h("div",{style:`height:${b.h||290}px`});el.append(h("h3",{style:"margin:0 0 4px;font-size:13.5px;display:"+(b.title?"block":"none")},b.title||""),box);let ch=null,cur=b;
 const init=()=>{ch=echarts.init(box,document.documentElement.dataset.theme==="dark"?"dk":"lt",{renderer:"canvas"});CH.push(ch);ch.setOption(revive(cur.opt),true);
  ch.on("click",p=>{const o=cur.open;if(!o)return;let id=null;if(o.bar)id=p.name;else if(o.scatter)id=p.value&&p.value[2];else if(o.graph)id=p.data&&p.data.open;else if(p.data&&p.data.open)id=p.data.open;if(id)UI.openEntity(id)});
  if(cur.open)box.style.cursor="pointer"};
 lazy(box,init);
 return{el,update:nb=>{cur=nb;if(ch){const r=JSON.stringify(nb.opt);if(box._h!==r){box._h=r;ch.setOption(revive(nb.opt),{replaceMerge:["series","xAxis","yAxis"]})}}},};};
window.addEventListener("resize",()=>CH.forEach(c=>{try{c.resize()}catch(e){}}));
// ---- map
function shiftGeo(g,sh){const mv=c=>typeof c[0]==="number"?[c[0]+sh,c[1]]:c.map(mv);return{type:"FeatureCollection",features:g.features.map(f=>({type:"Feature",properties:f.properties,geometry:{type:f.geometry.type,coordinates:mv(f.geometry.coordinates)}}))}}
let WORLD=null;async function world(){if(WORLD)return WORLD;try{const t=await(await fetch("/static/vendor/world.topo.json")).json();WORLD=topojson.feature(t,t.objects.countries)}catch(e){WORLD={type:"FeatureCollection",features:[]}}return WORLD}
R.map=b=>{const el=h("div"),box=h("div",{class:"map",style:`height:${b.h||470}px`}),lg=h("div",{class:"lg"});el.append(box,lg);let M=null,grp=null,cur=b;
 const draw=()=>{grp.clearLayers();const SH=[-360,0,360];cur.layers.forEach(L=>{if(L.type==="lines")L.lines.forEach(l=>{if(l.pts.length<2)return;SH.forEach(sh=>{const p=L2.polyline(l.pts.map(q=>[q[0],q[1]+sh]),{color:l.color||"#5b5bd6",weight:l.w||2,dashArray:l.dash||null,opacity:.9,lineJoin:"round",renderer:M._r});if(l.tip)p.bindTooltip(l.tip,{sticky:true});grp.addLayer(p)})});
   else if(L.type==="circles")L.items.forEach(c=>SH.forEach(sh=>{const k=L2.circle([c.lat,c.lon+sh],{radius:c.r_m,color:c.color||"#e5484d",weight:1,fillOpacity:.18,renderer:M._r});if(c.tip)k.bindTooltip(c.tip);grp.addLayer(k)}));
   else if(L.type==="points")[...L.pts].sort((a,c)=>(a.z||0)-(c.z||0)).forEach(p=>SH.forEach(sh=>{const m=L2.circleMarker([p.lat,p.lon+sh],{radius:p.r||5,color:"#ffffff",weight:1.2,fillColor:p.color||"#5b5bd6",fillOpacity:.95,renderer:M._r});if(p.tip)m.bindTooltip(p.tip,{sticky:true});const o=p.open||(cur.open?p.id:null)||p.id;if(o)m.on("click",()=>UI.openEntity(o));grp.addLayer(m)}))});
  lg.replaceChildren(...(cur.legend||[]).map(([t,c])=>h("span",null,h("i",{style:"background:"+c}),t)))};
 const L2=L;const go=async()=>{M=L2.map(box,{zoomControl:true,worldCopyJump:true,attributionControl:false,minZoom:1,preferCanvas:true});M._r=L2.canvas({padding:.5});const w=await world();[-360,0,360].forEach(sh=>L2.geoJSON(sh?shiftGeo(w,sh):w,{renderer:M._r,style:{color:getComputedStyle(document.documentElement).getPropertyValue("--landline").trim()||"#b9b2a0",weight:.8,fillColor:getComputedStyle(document.documentElement).getPropertyValue("--land").trim()||"#f3efe6",fillOpacity:1},interactive:false}).addTo(M));grp=L2.layerGroup().addTo(M);
  const v=cur.view||{};if(v.bounds)M.fitBounds(v.bounds);else M.setView(v.center||[20,0],v.zoom||2);draw();setTimeout(()=>M.invalidateSize(),50)};
 lazy(box,go);
 return{el,update:nb=>{cur=nb;if(M&&grp)draw()}}};
// ---- forms
R.form=b=>{const el=h("div",{class:b.title||b.inline?"":""});const res=h("div",{class:"res"});const f=h("form",{class:"f"+(b.inline?" inline":"")});let ins={};
 b.fields.forEach(fl=>{const w=h("div",{class:"fld"+(fl.type==="textarea"||fl.type==="checks"||fl.type==="multi"?" wide":"")});if(fl.type!=="hidden")w.append(h("label",null,fl.label||fl.name));let i;
  if(fl.type==="select")i=h("select",{name:fl.name},...fl.options.map(o=>h("option",{value:o,selected:String(o)===String(fl.value)},o)));
  else if(fl.type==="multi"){i=h("select",{name:fl.name,multiple:true,size:5},...fl.options.map(o=>h("option",{value:o,selected:(fl.value||[]).includes(o)},o)))}
  else if(fl.type==="checks"){i=h("div",{class:"chk"},...fl.options.map(o=>h("label",null,h("input",{type:"checkbox",value:o,checked:(fl.value||[]).includes(o)}),o)));i.get=()=>[...i.querySelectorAll("input:checked")].map(x=>x.value)}
  else if(fl.type==="textarea")i=h("textarea",{name:fl.name,placeholder:fl.placeholder||""},fl.value||"");
  else if(fl.type==="range"){const v=h("span",{style:"color:#12a594"},fl.value);i=h("input",{type:"range",name:fl.name,min:fl.min,max:fl.max,step:fl.step||1,value:fl.value,oninput:e=>v.textContent=e.target.value});w.firstChild.append(" ",v)}
  else if(fl.type==="file")i=h("input",{type:"file",name:fl.name,accept:fl.accept||null});
  else if(fl.type==="hidden"){i=h("input",{type:"hidden",name:fl.name,value:fl.value});w.style.display="none"}
  else i=h("input",{type:fl.type==="number"?"number":fl.type==="date"?"date":fl.type==="password"?"password":"text",name:fl.name,value:fl.value==null?"":fl.value,min:fl.min,max:fl.max,step:fl.step||(fl.type==="number"?"any":null),placeholder:fl.placeholder||""});
  i.fl=fl;ins[fl.name]=i;w.append(i);f.append(w)});
 const btn=h("button",{class:"btn "+(b.tone||"primary"),type:"submit"},b.submit||"Submit");f.append(h("div",{class:"fld"},btn));
 f.addEventListener("submit",async e=>{e.preventDefault();const v={};let fd=null;for(const n in ins){const i=ins[n],fl=i.fl;if(fl.type==="file"){if(i.files[0]){fd=fd||new FormData();fd.append(n,i.files[0])}}else if(fl.type==="checks")v[n]=i.get();else if(fl.type==="multi")v[n]=[...i.selectedOptions].map(o=>o.value);else if(fl.type==="number"||fl.type==="range")v[n]=i.value===""?null:+i.value;else v[n]=i.value}
  if(b.url.startsWith("get:")){const q=new URLSearchParams();for(const n in v)if(v[n]!==""&&v[n]!=null)q.set(n,v[n]);window.open(b.url.slice(4)+"?"+q,"_blank");return}
  if(b.url.startsWith("page:")){const[,id,tab]=b.url.split(":");UI.go(id,tab||S.tab,v);return}
  btn.disabled=true;const t0=btn.textContent;btn.textContent="Working…";let r;
  if(fd){for(const n in v)fd.append(n,Array.isArray(v[n])?v[n].join(","):v[n]);r=await api(b.url,{method:"POST",body:fd})}else r=await api(b.url,{json:v});
  btn.disabled=false;btn.textContent=t0;toast(r.msg||(r.ok?"Done":"Failed"),!r.ok);res.replaceChildren();if(r.blocks){const hs=mountList(res,r.blocks,"res");}if(r.refresh)reloadPage(true)});
 if(b.title||b.sub){const c=h("div",{class:"card"},b.title?h("h3",null,b.title):null,b.sub?h("div",{class:"sub"},b.sub):null,f,res);el.append(c)}else el.append(f,res);return{el,update(){}}};
// ================================================================ Milestone 2 front end: theme, auth, profile, admin, copilot, shell
ICON.play="M7 4l13 8-13 8z";ICON.lock="M7 11V7a5 5 0 0 1 10 0v4M5 11h14v10H5zM12 15v3";
const sleep=ms=>new Promise(r=>setTimeout(r,ms));
const PAGE_IDS=["copilot","route","quotes","carriers","weather","margin","customs","runagents","profile","admin"];
const ROLE_ICON={"Admin":"🛡️","Operations Manager":"🧭","Freight Broker":"💼","Auditor":"🔍","Customer":"📦"};
const when=s=>{if(!s)return "–";const d=new Date(String(s).replace(" ","T")+"Z");return isNaN(d)?s:d.toLocaleString([],{dateStyle:"medium",timeStyle:"short"})};
const initials=n=>(n||"?").replace(/[^A-Za-z0-9 ]/g," ").trim().split(/\s+/).map(x=>x[0]).slice(0,2).join("").toUpperCase()||"?";
const AVATARS=[["🧑‍✈️","#54668E","#26304D"],["🚢","#2a7fb8","#16476b"],["⚓","#3e63dd","#1f2f7a"],["🧭","#12a594","#0b5e55"],["📦","#d9822b","#8a4a0b"],["🚀","#8e4ec6","#4b2373"],["🦊","#e5733a","#a13d0f"],["🐼","#6b7080","#2c303b"],["🦁","#d6a21f","#8a6208"],["🌊","#1aa3c8","#0c5670"],["🛡️","#30a46c","#176340"],["🌟","#d6409f","#7d1d5c"]];
function avEl(u,cls){const a=(u&&u.avatar)||"",e=h("div",{class:"av "+(cls||"")});
 if(a.startsWith("data:image/")){e.append(h("img",{src:a,alt:""}));return e}
 const p=a.startsWith("preset:")?AVATARS[+a.slice(7)]:null;
 if(p){e.style.background="linear-gradient(135deg,"+p[1]+","+p[2]+")";e.classList.add("emo");e.textContent=p[0];return e}
 e.textContent=initials(u&&u.username);return e}
function resizeImage(file,sz){return new Promise((res,rej)=>{const url=URL.createObjectURL(file),img=new Image();
 img.onload=()=>{const c=document.createElement("canvas");c.width=c.height=sz;const x=c.getContext("2d"),s=Math.min(img.width,img.height);x.fillStyle="#fff";x.fillRect(0,0,sz,sz);x.drawImage(img,(img.width-s)/2,(img.height-s)/2,s,s,0,0,sz,sz);URL.revokeObjectURL(url);res(c.toDataURL("image/jpeg",.85))};
 img.onerror=()=>{URL.revokeObjectURL(url);rej(new Error("bad image"))};img.src=url})}
function genPw(){const A="ABCDEFGHJKLMNPQRSTUVWXYZ",a="abcdefghijkmnpqrstuvwxyz",n="23456789",s="@#$%!",rnd=m=>crypto.getRandomValues(new Uint32Array(1))[0]%m,pick=t=>t[rnd(t.length)],all=A+a+n+s,p=[pick(A),pick(a),pick(n),pick(s)];
 while(p.length<12)p.push(pick(all));for(let i=p.length-1;i>0;i--){const k=rnd(i+1);[p[i],p[k]]=[p[k],p[i]]}return p.join("")}
const agoText=s=>{if(!s)return "never";const d=new Date(String(s).replace(" ","T")+"Z"),m=Math.round((Date.now()-d)/60000);if(isNaN(m))return s;if(m<1)return "just now";if(m<60)return m+" min ago";if(m<1440)return Math.round(m/60)+" h ago";return Math.round(m/1440)+" days ago"};
const TOKEN={get:()=>localStorage.getItem("fq_token")||"",set:t=>localStorage.setItem("fq_token",t),clear:()=>localStorage.removeItem("fq_token")};
const THEME={get:()=>localStorage.getItem("fq_theme")||(window.matchMedia&&matchMedia("(prefers-color-scheme: dark)").matches?"dark":"light"),
 apply(){document.documentElement.dataset.theme=THEME.get();echarts.registerTheme("lt",mkTheme(false));echarts.registerTheme("dk",mkTheme(true))},
 toggle(){localStorage.setItem("fq_theme",THEME.get()==="dark"?"light":"dark");THEME.apply();const tb=document.querySelector(".sbtools .btn");if(tb)tb.textContent=THEME.get()==="dark"?"☀ Light":"☾ Dark";if(S.me&&S.page){S.tree=null;loadPage(false)}}};
function mkTheme(d){const tx=d?"#C3CCE6":"#454a58",ln=d?"#2B3558":"#e6e8f0",mu=d?"#93A0C6":"#6b7080";
 return{color:["#5b5bd6","#12a594","#f5a524","#8e4ec6","#e5484d","#30a46c","#d6409f","#3e63dd","#8b8d98"],backgroundColor:"transparent",textStyle:{fontFamily:"Inter, system-ui, sans-serif",color:tx},legend:{textStyle:{color:tx}},
  categoryAxis:{axisLine:{lineStyle:{color:ln}},axisTick:{show:false},axisLabel:{color:mu},splitLine:{lineStyle:{color:ln}}},valueAxis:{axisLine:{show:false},axisTick:{show:false},axisLabel:{color:mu},splitLine:{lineStyle:{color:ln}}},
  tooltip:{backgroundColor:d?"#222B4D":"#ffffff",borderColor:ln,textStyle:{color:d?"#E8ECF8":"#1c2024"},extraCssText:"box-shadow:0 8px 24px rgba(0,0,0,.2);border-radius:10px"}}}
THEME.apply();

// ---------------------------------------------------------------- password rules (same as Milestone 1 / server)
const PW_RULES=[["8+ characters",p=>p.length>=8],["Uppercase",p=>/[A-Z]/.test(p)],["Lowercase",p=>/[a-z]/.test(p)],["Number",p=>/\d/.test(p)],["Special",p=>/[^A-Za-z0-9]/.test(p)]];
const pwScore=p=>PW_RULES.filter(r=>r[1](p||"")).length;
function pwMeter(inputEl){const bars=[0,1,2,3].map(()=>h("i")),box=h("div",{class:"meter"},bars),chips=PW_RULES.map(r=>h("span",null,r[0])),rules=h("div",{class:"rules"},chips);
 const up=()=>{const p=inputEl.value,s=pwScore(p),lvl=s<=2?1:s===3?2:s===4?3:4;bars.forEach((b,i)=>{b.className=p&&i<lvl?"on"+lvl:""});chips.forEach((c,i)=>{c.className=PW_RULES[i][1](p)?"ok":""})};inputEl.addEventListener("input",up);up();return h("div",null,box,rules)}
const validEmail=e=>{if(!/^[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}$/.test(e))return false;const[l,d]=e.split("@"),i=d.lastIndexOf("."),host=d.slice(0,i),tld=d.slice(i+1),n=s=>(s.match(/[A-Za-z]/g)||[]).length;return n(l)>=2&&n(host)>=2&&tld.length>=2};
function field(label,input,hint){const w=h("div",{class:"afld"},h("label",null,label),input.nodeType&&input.classList.contains("iw")?input:h("div",{class:"iw"},input));if(hint)w.append(h("div",{class:"hint"},hint));return w}
function passField(label,id,ph){const inp=h("input",{type:"password",id,placeholder:ph||"",autocomplete:"off"}),eye=h("button",{type:"button",class:"eye",onclick:()=>{const s=inp.type==="password";inp.type=s?"text":"password";eye.textContent=s?"Hide":"Show"}},"Show");return{inp,el:h("div",{class:"afld"},h("label",null,label),h("div",{class:"iw"},inp,eye))}}
const msgBox=()=>{const b=h("div",{style:"display:none"});b.show=(t,k)=>{b.className="amsg "+(k||"err");b.textContent=t;b.style.display=""};b.hide=()=>{b.style.display="none"};return b};
async function busy(btn,label,fn){const old=btn.textContent;btn.disabled=true;btn.textContent=label;try{return await fn()}finally{btn.disabled=false;btn.textContent=old}}

// ---------------------------------------------------------------- sign in / sign up / forgot password
function renderAuth(mode,note){const app=$("#app");mode=mode||"login";const pub=S.pub||{};let card;
 const seg=()=>h("div",{class:"seg"},h("b",{class:mode==="login"?"on":"",onclick:()=>renderAuth("login")},"Sign in"),h("b",{class:mode==="signup"?"on":"",onclick:()=>renderAuth("signup")},"Sign up"));
 const msg=msgBox();if(note)msg.show(note.t,note.k);else if(S.authMsg){msg.show(S.authMsg,"info");S.authMsg=null}
 if(mode==="login"){
  const id=h("input",{type:"text",placeholder:"Username or email",autocomplete:"username"});
  const pw=passField("Password","lp","Your password");
  const btn=h("button",{class:"abtn",type:"submit"},"Sign in");
  const go=async e=>{
   e&&e.preventDefault();msg.hide();
   if(!id.value.trim()||!pw.inp.value)return msg.show("Please fill in all fields.");
   await busy(btn,"Signing in…",async()=>{
    const r=await api("/api/auth/login",{json:{identifier:id.value.trim(),password:pw.inp.value}});
    if(r.token){TOKEN.set(r.token);S.me=null;render()}
    else{msg.show(r.msg||"Sign in failed.","err")}
   });
  };
  card=h("form",{onsubmit:go},seg(),h("h1",null,"Welcome back"),
   h("p",{class:"lead"},"Sign in with your account credentials. Your assigned access is loaded automatically."),
   msg,field("Username or email",id),pw.el,
   h("div",{class:"arow"},h("span",{class:"hint"},"Accounts lock after 5 wrong passwords."),
    h("button",{type:"button",class:"alink",onclick:()=>renderAuth("forgot")},"Forgot password?")),btn);
  if(S.prefill){id.value=S.prefill;S.prefill=null}
  setTimeout(()=>(id.value?pw.inp:id).focus(),50)}
 else if(mode==="signup"){const un=h("input",{type:"text",placeholder:"letters, numbers, underscore",autocomplete:"username",maxlength:30}),em=h("input",{type:"email",placeholder:"name@company.com",autocomplete:"email"}),pw=passField("Password","sp","Create a strong password"),cp=passField("Confirm password","sc","Repeat the password"),
   q=h("select",null,(pub.questions||[]).map(x=>h("option",{value:x},x))),an=h("input",{type:"text",placeholder:"Your answer (used to reset your password)",autocomplete:"off"}),btn=h("button",{class:"abtn",type:"submit"},"Create account");
  const go=async e=>{e.preventDefault();msg.hide();const u=un.value.trim(),m=em.value.trim().toLowerCase();
   if(!u||!m||!pw.inp.value||!cp.inp.value||!an.value.trim())return msg.show("All fields are mandatory.");if(!/^[A-Za-z0-9_]{3,30}$/.test(u))return msg.show("Username must be 3-30 characters: letters, numbers or underscore only.");
   if(!validEmail(m))return msg.show("Enter a valid email (at least 2 letters before @, between @ and the dot, and after the dot).");const bad=PW_RULES.find(r=>!r[1](pw.inp.value));if(bad)return msg.show("Password rule missing: "+bad[0]+".");if(pw.inp.value!==cp.inp.value)return msg.show("Password and Confirm Password do not match.");
   await busy(btn,"Creating account…",async()=>{const r=await api("/api/auth/signup",{json:{username:u,email:m,password:pw.inp.value,confirm:cp.inp.value,question:q.value,answer:an.value.trim()}});if(r.ok){localStorage.setItem("fq_role","Customer");S.prefill=u;renderAuth("login",{t:r.msg+" Customer is selected for you: enter your password.",k:"ok"})}else msg.show(r.msg||"Sign up failed.")})};
  card=h("form",{onsubmit:go},seg(),h("h1",null,"Create your account"),h("p",{class:"lead"},"Self-registered accounts are Customers. An administrator can give you another role later."),msg,field("Username",un),field("Email",em),pw.el,pwMeter(pw.inp),h("div",{style:"height:12px"}),cp.el,field("Security question",q),field("Security answer",an),btn)}
 else{let method="otp",step=1,email="",uname="",question="";const draw=()=>{const m2=msgBox(),box=h("div");
   const tabs=h("div",{class:"seg"},h("b",{class:method==="otp"?"on":"",onclick:()=>{method="otp";step=1;draw()}},"Email code"),h("b",{class:method==="question"?"on":"",onclick:()=>{method="question";step=1;draw()}},"Security question"));
   const back=h("div",{style:"margin-top:14px;text-align:center"},h("button",{type:"button",class:"alink",onclick:()=>renderAuth("login")},"← Back to sign in"));
   const head=[tabs,h("h1",null,"Reset your password"),h("p",{class:"lead"},step===1?(method==="otp"?"We will e-mail you a 6-digit code.":"Answer your security question to continue."):"Choose a new password."),m2];
   const np=passField("New password","np","New password"),nc=passField("Confirm new password","nc","Repeat the new password");
   if(method==="otp"&&step===1){const em=h("input",{type:"email",placeholder:"name@company.com"}),btn=h("button",{class:"abtn",type:"submit"},"Send code");if(pub.email_enabled===false)m2.show("E-mail sending is not configured on this server. Use the security question instead.","info");
    box.append(h("form",{onsubmit:async e=>{e.preventDefault();m2.hide();if(!validEmail(em.value.trim()))return m2.show("Enter a valid email address.");email=em.value.trim().toLowerCase();await busy(btn,"Sending…",async()=>{const r=await api("/api/auth/forgot",{json:{email}});if(r.ok){step=2;draw();setTimeout(()=>{const x=$(".auth-card .amsg");},0)}else m2.show(r.msg)})}},...head,field("Registered email",em),btn,back));}
   else if(method==="otp"){const code=h("input",{type:"text",inputmode:"numeric",maxlength:6,placeholder:"6-digit code",style:"letter-spacing:6px;font-weight:700;text-align:center"}),btn=h("button",{class:"abtn",type:"submit"},"Reset password");m2.show("If this email is registered, a 6-digit code was sent. It is valid for 5 minutes.","info");
    box.append(h("form",{onsubmit:async e=>{e.preventDefault();m2.hide();if(!/^\d{6}$/.test(code.value.trim()))return m2.show("Please enter the 6-digit code.");const bad=PW_RULES.find(r=>!r[1](np.inp.value));if(bad)return m2.show("Password rule missing: "+bad[0]+".");if(np.inp.value!==nc.inp.value)return m2.show("Password and Confirm Password do not match.");
     await busy(btn,"Resetting…",async()=>{const r=await api("/api/auth/reset",{json:{method:"otp",email,code:code.value.trim(),password:np.inp.value,confirm:nc.inp.value}});if(r.ok)renderAuth("login",{t:r.msg,k:"ok"});else m2.show(r.msg)})}},...head,field("Verification code",code),np.el,pwMeter(np.inp),h("div",{style:"height:12px"}),nc.el,btn,back))}
   else if(step===1){const un=h("input",{type:"text",placeholder:"Your username"}),btn=h("button",{class:"abtn",type:"submit"},"Continue");
    box.append(h("form",{onsubmit:async e=>{e.preventDefault();m2.hide();if(!un.value.trim())return m2.show("Please enter your username.");uname=un.value.trim();await busy(btn,"Checking…",async()=>{const r=await api("/api/auth/question",{json:{username:uname}});if(r.ok){question=r.question;step=2;draw()}else m2.show(r.msg)})}},...head,field("Username",un),btn,back))}
   else{const an=h("input",{type:"text",placeholder:"Your answer",autocomplete:"off"}),btn=h("button",{class:"abtn",type:"submit"},"Reset password");
    box.append(h("form",{onsubmit:async e=>{e.preventDefault();m2.hide();if(!an.value.trim())return m2.show("Please enter your answer.");const bad=PW_RULES.find(r=>!r[1](np.inp.value));if(bad)return m2.show("Password rule missing: "+bad[0]+".");if(np.inp.value!==nc.inp.value)return m2.show("Password and Confirm Password do not match.");
     await busy(btn,"Resetting…",async()=>{const r=await api("/api/auth/reset",{json:{method:"question",username:uname,answer:an.value.trim(),password:np.inp.value,confirm:nc.inp.value}});if(r.ok)renderAuth("login",{t:r.msg,k:"ok"});else m2.show(r.msg)})}},...head,h("div",{class:"amsg info"},question),field("Your answer",an),np.el,pwMeter(np.inp),h("div",{style:"height:12px"}),nc.el,btn,back))}
   card.replaceChildren(box)};card=h("div");draw()}
 const feat=(i,t)=>h("div",{class:"feat"},h("i",null,i),t);
 app.replaceChildren(h("div",{class:"auth-shell"},h("div",{class:"orb o1"}),h("div",{class:"orb o2"}),h("div",{class:"orb o3"}),
  h("div",{class:"auth-top"},h("div",{class:"brand"},h("i",null,icon("radar")),h("div",null,"FreightQuote AI",h("small",null,"Infosys Springboard Internship 7.0 · Batch 4"))),h("label",{class:"theme-sw",onclick:()=>{THEME.toggle();renderAuth(mode)}},h("span",null,THEME.get()==="dark"?"Dark":"Light"),h("i"))),
  h("div",{class:"auth-grid"},
   h("div",{class:"hero-l"},h("span",{class:"tag"},"MILESTONE 2 · MULTI-USER PLATFORM"),h("h2",null,"Freight quotes, routes and risk. One secure workspace for every role."),h("p",null,"Six AI agents price spot quotes, rank carriers, watch the weather and clear customs. Sign in to see exactly what your role is allowed to do."),
    h("div",{class:"promo"},h("img",{src:"/static/promo.webp",alt:"Infosys Springboard",onerror:e=>e.target.parentElement.style.display="none"})),
    h("div",{class:"feats"},feat("🔐","JWT sessions · bcrypt hashing"),feat("🛡️","Role-based access control"),feat("🤖","SQL-exact AI Copilot"),feat("⛈️","Live weather and customs checks"))),
   h("div",{class:"auth-card"},card)),
  h("div",{class:"auth-foot"},"Secrets live in Colab Secrets · passwords are never stored in plain text")))}

// ---------------------------------------------------------------- Copilot
R.chat=b=>{let cid=null,hist=[],work=false,viaVoice=false,rec=null;const el=h("div",{class:"card chat"}),msgs=h("div",{class:"msgs"}),hsel=h("select",{class:"in",style:"max-width:260px",onchange:()=>hsel.value&&openChat(hsel.value)}),
  inp=h("input",{placeholder:"Ask about ports, carriers, shipments, quotes, weather or customs…",maxlength:500,autocomplete:"off",onkeydown:e=>{if(e.key==="Enter"&&!e.isComposing)send(inp.value)}}),go=h("button",{class:"btn primary",onclick:()=>send(inp.value)},"Ask");
 // ---- voice: speak a question (Web Speech API), it is sent automatically and the answer can be read aloud
 const SR=window.SpeechRecognition||window.webkitSpeechRecognition,vst=h("span",{class:"vst"},SR?"Tap the mic and ask your question out loud.":"Voice input works in Chrome, Edge and Safari."),
  vlang=h("select",{title:"Speech language"},[["en-IN","English (India)"],["en-US","English (US)"],["en-GB","English (UK)"]].map(([v,t])=>h("option",{value:v},t))),
  say=h("input",{type:"checkbox"}),mic=h("button",{class:"btn mic",type:"button",title:"Ask with your voice",onclick:()=>toggleMic()},"🎤");
 say.checked=localStorage.getItem("fq_speak")!=="0";say.onchange=()=>{localStorage.setItem("fq_speak",say.checked?"1":"0");if(!say.checked&&window.speechSynthesis)speechSynthesis.cancel()};
 const vsay=(t,k)=>{vst.textContent=t;vst.className="vst "+(k||"")};
 function speak(t){if(!window.speechSynthesis||!t)return;const x=String(t).replace(/```[\s\S]*?```/g," ").replace(/\[([^\]]*)\]\([^)]*\)/g,"$1").replace(/[*_`#>|]/g," ").replace(/\s+/g," ").trim().slice(0,450);if(!x)return;speechSynthesis.cancel();const u=new SpeechSynthesisUtterance(x);u.lang=vlang.value;u.rate=1;speechSynthesis.speak(u)}
 function toggleMic(){if(rec){try{rec.stop()}catch(e){}return}
  if(!SR)return vsay("Your browser has no speech recognition. Use Chrome, Edge or Safari, or type your question.","err");
  if(work)return;if(window.speechSynthesis)speechSynthesis.cancel();
  const r=new SR();rec=r;r.lang=vlang.value;r.interimResults=true;r.continuous=false;r.maxAlternatives=1;let fin="",failed=false;
  r.onstart=()=>{mic.classList.add("on");mic.title="Stop listening";vsay("Listening… speak your question now.","on")};
  r.onresult=e=>{let tmp="";for(let i=e.resultIndex;i<e.results.length;i++){const x=e.results[i];if(x.isFinal)fin+=x[0].transcript;else tmp+=x[0].transcript}inp.value=(fin+tmp).trim()};
  r.onerror=e=>{failed=true;const M={"not-allowed":"Microphone access is blocked. Allow the microphone for this site (lock icon in the address bar) and try again.","service-not-allowed":"Microphone access is blocked for this page. Open the app in its own browser tab and allow the microphone.","no-speech":"I did not hear anything. Tap the mic and try again.","audio-capture":"No microphone was found.","network":"Speech recognition needs an internet connection.","aborted":""};vsay(e.error in M?M[e.error]:"Voice input error: "+e.error,e.error==="aborted"?"":"err")};
  r.onend=()=>{rec=null;mic.classList.remove("on");mic.title="Ask with your voice";const q=(fin||inp.value).trim();if(failed)return;if(q){vsay("Heard: “"+q+"”");viaVoice=true;send(q)}else vsay("I did not catch that. Tap the mic and try again.","err")};
  try{r.start()}catch(e){rec=null;vsay("Could not start the microphone. Try again.","err")}}
 const sug=(arr)=>arr&&arr.length?h("div",{class:"sugs"},arr.map(t=>h("span",{class:"chip",onclick:()=>send(t)},t))):null;
 const table=t=>{if(!t||!t.rows||!t.rows.length)return null;const f=v=>v==null?"–":typeof v==="number"?(Number.isInteger(v)?v.toLocaleString():+v.toFixed(2)):String(v);return h("div",{class:"mini"},h("table",null,h("thead",null,h("tr",null,t.cols.map(c=>h("th",null,c.replace(/_/g," "))))),h("tbody",null,t.rows.map(r=>h("tr",null,r.map(v=>h("td",null,f(v))))))),t.total>t.rows.length?h("div",{class:"note",style:"padding:6px 10px"},`Showing ${t.rows.length} of ${t.total} rows`):null)};
 const draw=m=>{if(m.role==="user")return h("div",{class:"m u"},m.text);const d=h("div",{class:"m a"},h("div",{class:"txt md",html:md(m.text)}));if(m.note)d.append(h("div",{class:"note"},m.note));const t=table(m.table);if(t)d.append(t);
  if(m.sql)d.append(h("details",{class:"sqlbox"},h("summary",null,"SQL behind this answer",h("span",{class:"rowsb"},(m.rows==null?0:m.rows)+" row"+(m.rows===1?"":"s")+(m.ms!=null?" · "+m.ms+" ms":""))),h("pre",null,m.sql)));const s=sug(m.sug);if(s)d.append(s);return d};
 const paint=()=>{msgs.replaceChildren(...hist.map(draw));if(!hist.length)msgs.append(h("div",{class:"m a"},h("div",{class:"txt md",html:md("Hi! I answer from your database with **exact SQL**, never guesses. Pick a question or type your own.")}),sug(b.examples)));msgs.scrollTop=msgs.scrollHeight;el.scrollIntoView&&0};
 async function save(){const r=await api("/api/chats/save",{json:{id:cid,msgs:hist.map(m=>({role:m.role,text:m.text,sql:m.sql,rows:m.rows,table:m.table,note:m.note,sug:m.sug,kind:m.kind}))}});if(r.id){const nw=!cid;cid=r.id;if(nw)loadList()}}
 async function loadList(){const r=await api("/api/chats");if(!r.chats)return;hsel.replaceChildren(h("option",{value:""},"Chat history ("+r.chats.length+")"),...r.chats.map(c=>h("option",{value:c.id},c.title+" · "+(c.ts||"").slice(5,16).replace("T"," "))));hsel.value=cid||""}
 async function openChat(id){const r=await api("/api/chats/"+id);if(r.msgs){cid=id;hist=r.msgs.map(m=>({...m}));paint()}}
 async function send(q){q=(q||"").trim();if(!q||work)return;const voice=viaVoice;viaVoice=false;work=true;inp.value="";go.disabled=true;hist.push({role:"user",text:q});paint();const tp=h("div",{class:"m a"},h("div",{class:"typing"},h("i"),h("i"),h("i")));msgs.append(tp);msgs.scrollTop=msgs.scrollHeight;
  const r=await api("/api/copilot",{json:{q}});tp.remove();
  if(r.kind)hist.push({role:"assistant",text:r.text,sql:r.sql,rows:r.rows,table:r.table,note:r.note,sug:r.suggestions,kind:r.kind,ms:r.ms});else hist.push({role:"assistant",text:"**Something went wrong.** "+(r.msg||"Please try again."),kind:"error",sug:b.examples.slice(0,3)});
  paint();work=false;go.disabled=false;inp.focus();save();if(voice&&say.checked&&r.kind&&r.text)speak(r.text)}
 el.append(h("div",{class:"chat-top"},h("button",{class:"btn sm",onclick:()=>{cid=null;hist=[];hsel.value="";paint()}},"+ New chat"),hsel,h("button",{class:"btn sm ghost",onclick:async()=>{if(!cid)return;await api("/api/chats/"+cid,{method:"DELETE"});cid=null;hist=[];paint();loadList()}},"Delete chat"),h("span",{class:"sp"}),h("span",{class:"chip"},"Every answer = one SQL query")),msgs,h("div",{class:"cin"},mic,inp,go),h("div",{class:"vbar"},vst,vlang,h("label",{title:"Read answers aloud when you ask by voice"},say,"🔊 Read answers aloud")));paint();loadList();return{el,update(){}}};

// ---------------------------------------------------------------- Profile page
R.profile=b=>{const el=h("div",{class:"prof"}),left=h("div",{class:"card id"}),right=h("div",{class:"pcols"});el.append(left,right);
 const load=async()=>{const r=await api("/api/profile");if(!r.user){el.replaceChildren(h("div",{class:"callout error"},r.msg||"Could not load your profile."));return}const u=r.user;S.me.user=u;
  left.replaceChildren(h("div",{class:"avwrap"},avEl(u,"bigav"),h("button",{class:"avedit",type:"button",title:"Change profile picture",onclick:()=>avatarModal(u,load)},"📷")),h("h2",null,u.username),h("div",{class:"em"},u.email),h("span",{class:"rolebadge "+u.role.split(" ")[0]},u.role),h("p",{class:"sub",style:"margin-top:8px"},r.blurb),
   h("div",{class:"irow"},h("span",null,"Username"),h("b",null,u.username)),h("div",{class:"irow"},h("span",null,"Email"),h("b",null,u.email)),h("div",{class:"irow"},h("span",null,"Role"),h("b",null,u.role)),h("div",{class:"irow"},h("span",null,"Joined"),h("b",null,when(u.created_at))),
   h("div",{class:"irow"},h("span",null,"Last login"),h("b",null,u.prev_login?when(u.prev_login):"This is your first sign-in")),u.customer_id?h("div",{class:"irow"},h("span",null,"Customer account"),h("b",null,u.customer_id)):null,
   h("div",{class:"irow",style:"display:block"},h("span",null,"Pages you can open"),h("div",{class:"caps"},r.pages.map(p=>h("span",{class:"chip"},p)))));
  const un=h("input",{type:"text",value:u.username,maxlength:30}),ub=h("button",{class:"btn primary",type:"submit"},"Save username"),um=msgBox();
  const cur=passField("Current password","cp1"),np=passField("New password","np1"),cf=passField("Confirm new password","cf1"),pb=h("button",{class:"btn primary",type:"submit"},"Change password"),pm=msgBox();
  right.replaceChildren(h("div",{class:"card"},h("h3",null,"Change username"),h("div",{class:"sub"},"3-30 characters: letters, numbers or underscore."),um,h("form",{onsubmit:async e=>{e.preventDefault();um.hide();await busy(ub,"Saving…",async()=>{const r=await api("/api/profile/username",{json:{username:un.value.trim()}});if(r.ok){S.me.user=r.user;toast(r.msg);renderShellUser();load()}else um.show(r.msg)})}},h("div",{class:"afld"},h("label",null,"Username"),un),ub)),
   h("div",{class:"card"},h("h3",null,"Change password"),h("div",{class:"sub"},"Enter your current password first. The new one must follow the same rules as sign-up. Other devices are signed out."),pm,
    h("form",{onsubmit:async e=>{e.preventDefault();pm.hide();if(!cur.inp.value||!np.inp.value||!cf.inp.value)return pm.show("All fields are mandatory.");const bad=PW_RULES.find(r=>!r[1](np.inp.value));if(bad)return pm.show("Password rule missing: "+bad[0]+".");if(np.inp.value!==cf.inp.value)return pm.show("Password and Confirm Password do not match.");
     await busy(pb,"Updating…",async()=>{const r=await api("/api/profile/password",{json:{current:cur.inp.value,password:np.inp.value,confirm:cf.inp.value}});if(r.ok){if(r.token)TOKEN.set(r.token);toast(r.msg);cur.inp.value=np.inp.value=cf.inp.value="";np.inp.dispatchEvent(new Event("input"))}else pm.show(r.msg)})}},cur.el,np.el,pwMeter(np.inp),h("div",{style:"height:12px"}),cf.el,pb)))};
 el.append(h("div",{class:"skel",style:"height:260px"}));load().then(()=>{el.querySelectorAll(".skel").forEach(x=>x.remove())});return{el,update(){}}};

// ---------------------------------------------------------------- Profile picture dialog
function avatarModal(u,done){let pick=u.avatar||"";const prev=h("div",{class:"avprev"}),grid=h("div",{class:"avgrid"}),msg=msgBox();
 const file=h("input",{type:"file",accept:"image/png,image/jpeg,image/webp",style:"display:none",onchange:async()=>{const f=file.files[0];file.value="";if(!f)return;if(f.size>8e6)return msg.show("Choose a picture under 8 MB.");try{pick=await resizeImage(f,160);msg.hide();paint()}catch(e){msg.show("That picture could not be read.")}}});
 const paint=()=>{prev.replaceChildren(avEl({username:u.username,avatar:pick},"bigav"));grid.replaceChildren(...AVATARS.map((p,i)=>h("button",{type:"button",class:"avopt"+(pick==="preset:"+i?" on":""),style:"background:linear-gradient(135deg,"+p[1]+","+p[2]+")",title:"Avatar "+(i+1),onclick:()=>{pick="preset:"+i;paint()}},p[0])))};
 const close=()=>ov.remove(),save=h("button",{class:"btn primary",type:"button",onclick:async e=>{await busy(e.target,"Saving…",async()=>{const r=await api("/api/profile/avatar",{json:{avatar:pick}});if(r.ok){S.me.user=r.user;renderShellUser();toast(r.msg);close();done&&done()}else msg.show(r.msg)})}},"Save");
 const ov=h("div",{class:"modal avmodal",onclick:e=>{if(e.target===ov)close()}},h("div",null,h("h3",null,"Profile picture"),h("p",{style:"margin-bottom:6px"},"Pick an avatar or upload your own photo."),msg,prev,grid,
  h("div",{class:"avtools"},h("button",{class:"btn sm",type:"button",onclick:()=>file.click()},"⬆ Upload photo"),h("button",{class:"btn sm",type:"button",onclick:()=>{pick="";paint()}},"Use my initials"),file),
  h("div",{class:"mb"},h("button",{class:"btn",type:"button",onclick:close},"Cancel"),save)));
 paint();document.body.append(ov)}

// ---------------------------------------------------------------- Admin dashboard
function confirmModal(title,text,okLabel,tone){return new Promise(res=>{const done=v=>{ov.remove();res(v)};const ov=h("div",{class:"modal",onclick:e=>{if(e.target===ov)done(false)}},h("div",null,h("h3",null,title),h("p",null,text),h("div",{class:"mb"},h("button",{class:"btn",onclick:()=>done(false)},"Cancel"),h("button",{class:"btn "+(tone||"danger"),onclick:()=>done(true)},okLabel||"Confirm"))));document.body.append(ov)})}
function promptModal(title,text,val){return new Promise(res=>{const inp=h("input",{class:"in",value:val||"",placeholder:"CUST-001"});const done=v=>{ov.remove();res(v)};const ov=h("div",{class:"modal",onclick:e=>{if(e.target===ov)done(null)}},h("div",null,h("h3",null,title),h("p",null,text),inp,h("div",{class:"mb",style:"margin-top:14px"},h("button",{class:"btn",onclick:()=>done(null)},"Cancel"),h("button",{class:"btn primary",onclick:()=>done(inp.value)},"Save"))));document.body.append(ov);inp.focus()})}
R.admin=b=>{const el=h("div",{class:"col"});let data=null,flt="all",addCard=null;
 const act=async(p,opt,okmsg)=>{const r=await api(p,opt);toast(r.msg||okmsg||"Done",!r.ok);load();return r};
 const tagFor=a=>/denied|blocked|failed|locked|deleted/.test(a)?"bad":/added|login$|unlocked|signup|created|password_set/.test(a)?"good":"";
 const PTYPES=[["temporary","Temporary: the user must set a new password at first sign-in"],["permanent","Permanent: the user keeps it, and can change it later on their Profile page"]];
 const pwHint=k=>k==="permanent"?"Permanent password: no forced change. The user can still change it themselves from My profile whenever they like.":"Temporary password: the user can sign in only once, then is asked to choose their own password.";
 function passwordModal(u){return new Promise(res=>{const pw=h("input",{class:"in",type:"text",placeholder:"new password",autocomplete:"off"}),kind=h("select",{class:"in"},PTYPES.map(([v,t])=>h("option",{value:v},t))),hint=h("div",{class:"fhint"},pwHint("temporary")),msg=msgBox();kind.onchange=()=>hint.textContent=pwHint(kind.value);
  const done=v=>{ov.remove();res(v)},ok=h("button",{class:"btn primary",type:"button",onclick:async e=>{msg.hide();const bad=PW_RULES.find(r=>!r[1](pw.value));if(bad)return msg.show("Password rule missing: "+bad[0]+".");await busy(e.target,"Saving…",async()=>{const r=await api(`/api/admin/users/${u.id}/password`,{json:{password:pw.value,kind:kind.value}});if(r.ok){toast(r.msg);done(true);load()}else msg.show(r.msg)})}},"Set password");
  const ov=h("div",{class:"modal",onclick:e=>{if(e.target===ov)done(false)}},h("div",null,h("h3",null,"Set password for "+u.username),h("p",{style:"margin-bottom:10px"},"The user is signed out everywhere. Share the password through a safe channel."),msg,h("div",{class:"pwrow",style:"margin-bottom:10px"},pw,h("button",{class:"btn sm",type:"button",onclick:()=>pw.value=genPw()},"Generate")),kind,hint,h("div",{class:"mb",style:"margin-top:14px"},h("button",{class:"btn",type:"button",onclick:()=>done(false)},"Cancel"),ok)));document.body.append(ov);pw.focus()})}
 function buildAdd(d){const f={email:h("input",{class:"in",type:"email",placeholder:"name@company.com"}),user:h("input",{class:"in",placeholder:"username",maxlength:30}),pw:h("input",{class:"in",type:"text",placeholder:"password for the user",autocomplete:"off"}),ptype:h("select",{class:"in"},PTYPES.map(([v,t])=>h("option",{value:v},v==="temporary"?"Temporary password":"Permanent password"))),role:h("select",{class:"in"},d.roles.filter(r=>r!=="Admin").concat(["Admin"]).map(r=>h("option",{value:r},r))),cust:h("input",{class:"in",placeholder:"CUST-001 (Customer only)"})};
  const addb=h("button",{class:"btn primary",type:"submit"},"Add user"),am=msgBox(),hint=h("div",{class:"fhint",style:"grid-column:1/-1;margin:0"},pwHint("temporary"));f.role.value="Customer";f.ptype.onchange=()=>hint.textContent=pwHint(f.ptype.value);
  return h("div",{class:"card"},h("h3",null,"Add user / customer"),h("div",{class:"sub"},"Choose the role (pick Customer to add a customer), set their password and decide whether it is temporary or permanent. Same validation as sign-up."),am,
   h("form",{class:"addgrid",onsubmit:async e=>{e.preventDefault();am.hide();if(!validEmail(f.email.value.trim()))return am.show("Enter a valid email (at least 2 letters before @, between @ and the dot, and after the dot).");const bad=PW_RULES.find(r=>!r[1](f.pw.value));if(bad)return am.show("Password rule missing: "+bad[0]+".");
    await busy(addb,"Adding…",async()=>{const r=await api("/api/admin/users",{json:{email:f.email.value.trim(),username:f.user.value.trim(),password:f.pw.value,password_type:f.ptype.value,role:f.role.value,customer_id:f.cust.value.trim()}});if(r.ok){toast(r.msg);f.email.value=f.user.value=f.pw.value=f.cust.value="";load()}else am.show(r.msg)})}},
    h("div",{class:"fld"},h("label",null,"Email"),f.email),h("div",{class:"fld"},h("label",null,"Username"),f.user),h("div",{class:"fld"},h("label",null,"Password"),h("div",{class:"pwrow"},f.pw,h("button",{class:"btn sm",type:"button",onclick:()=>f.pw.value=genPw()},"Generate"))),h("div",{class:"fld"},h("label",null,"Password type"),f.ptype),h("div",{class:"fld"},h("label",null,"Role"),f.role),h("div",{class:"fld"},h("label",null,"Customer id"),f.cust),addb,hint))}
 const paint=()=>{const d=data,k=d.kpis,me=S.me.user;if(!addCard)addCard=buildAdd(d);
  const kp=h("div",{class:"kpis"},h("div",{class:"kpi"},h("div",{class:"l"},"Total users"),h("div",{class:"v"},k.total),h("div",{class:"s"},k.active+" active · "+k.inactive+" inactive")),h("div",{class:"kpi Low"},h("div",{class:"l"},"Active now"),h("div",{class:"v"},k.active),h("div",{class:"s"},"signed in and in use")),h("div",{class:"kpi Info"},h("div",{class:"l"},"Inactive"),h("div",{class:"v"},k.inactive),h("div",{class:"s"},"signed out or idle")),h("div",{class:"kpi "+(k.locked?"Warning":"")},h("div",{class:"l"},"Locked users"),h("div",{class:"v"},k.locked),h("div",{class:"s"},"cannot sign in")),
   ...d.roles.map(r=>h("div",{class:"kpi Info"},h("div",{class:"l"},r),h("div",{class:"v"},k.per_role[r]||0),h("div",{class:"s"},(d.matrix.find(m=>m.role===r)||{}).who||""))));
  const FL=[["all","All",d.users.length],["active","Active",k.active],["inactive","Inactive",k.inactive],["locked","Locked",k.locked]];
  const chips=h("div",{class:"fchips"},FL.map(([id,t,n])=>h("span",{class:"chip"+(flt===id?" on":""),onclick:()=>{flt=id;paint()}},t+" · "+n)));
  const shown=d.users.filter(u=>flt==="all"||(flt==="active"&&u.presence==="Active")||(flt==="inactive"&&u.presence==="Inactive")||(flt==="locked"&&u.locked));
  const rows=shown.map(u=>{const self=u.id===me.id;const sel=h("select",{disabled:self,title:self?"You cannot change your own role":"",onchange:async()=>{if(await confirmModal("Change role",`Change ${u.username} from ${u.role} to ${sel.value}?`,"Change role","primary"))act(`/api/admin/users/${u.id}/role`,{json:{role:sel.value}});else sel.value=u.role}},d.roles.map(r=>h("option",{value:r,selected:r===u.role},r)));
   const on=u.presence==="Active";
   return h("tr",null,h("td",null,h("div",{class:"ucell"},avEl(u,"sm"),h("div",null,h("b",null,u.username),self?h("span",{class:"me-tag"},"you"):null,h("div",{class:"sub",style:"margin:0"},u.email)))),
    h("td",null,sel,u.role==="Customer"?h("div",{class:"sub",style:"margin:3px 0 0"},(u.customer_id||"no account")+" · ",h("a",{href:"#",onclick:async e=>{e.preventDefault();const v=await promptModal("Customer account","Link this user to a customer id (e.g. CUST-001). Leave empty to create a new one.",u.customer_id);if(v!==null)act(`/api/admin/users/${u.id}/customer`,{json:{customer_id:v}})}},"change")):null),
    h("td",null,h("span",{class:"pres "+(on?"on":"off")},h("i"),u.presence),h("div",{class:"sub",style:"margin:3px 0 0"},on?"online now":(u.last_seen_at?"last seen "+agoText(u.last_seen_at):"never seen"))),
    h("td",null,h("span",{class:"st-pill "+u.status},u.status)),
    h("td",null,h("span",{class:"pwk "+u.pw_kind},u.pw_kind==="temporary"?(u.must_change?"Temporary · not changed yet":"Temporary"):"Permanent")),
    h("td",null,when(u.created_at)),h("td",null,u.last_login?when(u.last_login):"never"),
    h("td",null,h("div",{class:"acts"},h("button",{class:"btn sm",disabled:self,title:self?"Change your own password on the Profile page":"Set a new temporary or permanent password",onclick:()=>passwordModal(u)},"Password"),h("button",{class:"btn sm "+(u.locked?"ok":""),disabled:self,onclick:async()=>{if(u.locked||await confirmModal("Lock user",`${u.username} will be signed out immediately and cannot sign in until unlocked.`,"Lock","danger"))act(`/api/admin/users/${u.id}/lock`,{json:{locked:!u.locked}})}},u.locked?"Unlock":"Lock"),
     h("button",{class:"btn sm danger",disabled:self,onclick:async()=>{if(await confirmModal("Delete user",`Permanently delete ${u.username} (${u.email})? This cannot be undone.`,"Delete user","danger"))act(`/api/admin/users/${u.id}?confirm=yes`,{method:"DELETE"})}},"Delete"))))});
  if(!rows.length)rows.push(h("tr",null,h("td",{colspan:8,style:"text-align:center;color:var(--mut);padding:18px"},"No users in this view.")));
  const mxcols=["Copilot","Route","Quotes","Carriers","Weather","Margin","Customs","Run agents","Users"];
  const mx=h("div",{class:"card"},h("h3",null,"Role permissions"),h("div",{class:"sub"},"Enforced on the server for every page and action."),h("div",{style:"overflow:auto"},h("table",{class:"mx"},h("thead",null,h("tr",null,h("th",null,"Role"),mxcols.map(c=>h("th",null,c)))),h("tbody",null,d.matrix.map(r=>h("tr",null,h("td",null,h("b",null,r.role),h("div",{class:"sub",style:"margin:0"},r.who)),mxcols.map(c=>h("td",{class:r[c]?"y":"n"},r[c]?"✓":"✗"))))))));
  const lg=h("div",{class:"card"},h("h3",null,"Activity log"),h("div",{class:"sub"},"Latest 50 events: who, what, when."),h("div",{class:"logl"},h("table",null,h("thead",null,h("tr",null,["When","Who","Action","Target","Detail"].map(x=>h("th",null,x)))),h("tbody",null,d.log.map(l=>h("tr",null,h("td",null,when(l.ts)),h("td",null,l.actor),h("td",null,h("span",{class:"act-tag "+tagFor(l.action)},l.action.replace(/_/g," "))),h("td",null,l.target),h("td",{class:"dt"},l.detail)))))));
  el.replaceChildren(kp,h("div",{class:"card"},h("h3",null,"Users"),h("div",{class:"sub"},"Active = signed in and using the app in the last 3 minutes. Inactive = signed out or idle. The list refreshes by itself. You cannot lock, delete or change yourself, and the last Admin is protected."),chips,h("div",{style:"overflow:auto"},h("table",{class:"utbl"},h("thead",null,h("tr",null,["User","Role","Presence","Account","Password","Created","Last login","Actions"].map(x=>h("th",null,x)))),h("tbody",null,rows)))),addCard,h("div",{class:"two"},mx,lg))};
 async function load(){const r=await api("/api/admin/overview");if(!r.users){el.replaceChildren(h("div",{class:"callout error"},r.msg||"Could not load the dashboard."));return}data=r;paint()}
 el.append(h("div",{class:"kpis"},[1,2,3,4].map(()=>h("div",{class:"skel",style:"height:84px"}))),h("div",{class:"skel",style:"height:300px"}));
 load().then(()=>{const t=setInterval(()=>{if(!el.isConnected){clearInterval(t);return}if(document.hidden)return;const f=document.activeElement;if(f&&el.contains(f)&&/SELECT|INPUT/.test(f.tagName))return;load()},20000)});return{el,update(){}}};

// ---------------------------------------------------------------- shell
const UI={};
UI.openEntity=async id=>{if(!id)return;$("#dt").textContent=id;$("#db").replaceChildren(h("div",{class:"boot"},h("div",{class:"spin"}),"Loading…"));$("#drawer").classList.add("on");const d=await api("/api/detail/"+encodeURIComponent(id));if(d.title===undefined){$("#db").replaceChildren(h("div",{class:"errbox "+(d.status===403?"e403":"")},h("div",{class:"big"},d.status||"!"),h("h3",null,d.status===403?"Access denied":"Not available"),h("p",null,d.msg||"Not available")));return}$("#dt").textContent=d.title||id;$("#db").replaceChildren();mountList($("#db"),d.blocks||[],"d")};
UI.closeDrawer=()=>$("#drawer").classList.remove("on");
UI.go=(id,tab,params)=>{S.page=id;S.tab=tab||null;S.params=params||{};history.replaceState(null,"","#"+id+(S.tab?":"+S.tab:""));loadPage(false);renderNav()};
function pageError(d){const c=d.status===403;$("#ttl").textContent=c?"Access denied":"Could not load this page";$("#sub").textContent="";$("#tabs").style.display="none";$("#acts").replaceChildren();
 $("#blocks").replaceChildren(h("div",{class:"errbox "+(c?"e403":"")},h("div",{class:"big"},c?"403":(d.status||"!")),h("h3",null,c?"You do not have permission":"Something went wrong"),h("p",null,d.msg||"Please try again."),h("button",{class:"btn primary",onclick:()=>c?UI.go(S.me.menu[0].id):loadPage(false)},c?"Back to my workspace":"Try again")))}
async function loadPage(silent){const seq=++S.seq;clearTimeout(S.timer);const q=new URLSearchParams();if(S.tab)q.set("tab",S.tab);if(Object.keys(S.params).length)q.set("p",JSON.stringify(S.params));
 if(!silent){S.tree=null;$("#blocks").replaceChildren(h("div",{class:"kpis"},[1,2,3,4].map(()=>h("div",{class:"skel",style:"height:84px"}))),h("div",{class:"skel",style:"height:260px;margin-top:16px"}))}
 const t=performance.now();const d=await api(`/api/page/${S.page}?`+q);if(seq!==S.seq)return;
 if(d.blocks===undefined){if(!silent||!S.tree)pageError(d);return}
 S.ms=Math.round(performance.now()-t);$("#ttl").textContent=d.title;$("#sub").textContent=d.sub||"";document.title=d.title+" · "+S.me.name;
 const tabs=$("#tabs");tabs.replaceChildren(...(d.tabs||[]).map(x=>h("b",{class:(S.tab||d.tabs[0].id)===x.id?"on":"",onclick:()=>{S.params={};UI.go(S.page,x.id)}},x.label)));tabs.style.display=d.tabs?"":"none";
 $("#acts").replaceChildren(...(d.actions||[]).map(a=>h("button",{class:"btn "+(a.tone||"primary"),onclick:async e=>{e.target.disabled=true;const r=await api(a.url,{json:{}});toast(r.msg,!r.ok);e.target.disabled=false;reloadPage(true)}},a.label)));
 const box=$("#blocks");if(!S.tree||silent===false||!S.tree.ok){box.replaceChildren();S.tree={hs:mountList(box,d.blocks,"p"),ok:1,id:S.page,tab:S.tab}}else S.tree.hs=updList(box,S.tree.hs,d.blocks,"p");
 S.live=d.live||0;sched()}
function sched(){clearTimeout(S.timer);if(S.live)S.timer=setTimeout(()=>{if(!document.hidden)loadPage(true);else sched()},S.live*1000)}
function reloadPage(soft){loadPage(!!soft)}
const GROUPS=[["Assistant"],["Agents"],["Control"]];
function renderNav(){const nav=$("#nav");if(!nav)return;const items=[],role=S.me.user.role;
 const mk=m=>m.enabled?h("div",{class:"nav"+(m.id===S.page?" on":""),"data-id":m.id,onclick:()=>UI.go(m.id)},icon(m.icon),h("span",null,m.label))
   :h("div",{class:"nav off","data-id":m.id,title:"Not available for the "+role+" role","aria-disabled":"true",onclick:()=>toast("The "+role+" role cannot open "+m.label+".",true)},icon(m.icon),h("span",null,m.label),(()=>{const l=icon("lock");l.classList.add("lk");return l})());
 GROUPS.forEach(([g])=>{const ms=S.me.nav.filter(m=>m.group===g&&m.enabled);if(!ms.length)return;items.push(h("div",{class:"ngrp"},g.toUpperCase()));ms.forEach(m=>items.push(mk(m)))});nav.replaceChildren(...items);updBadges()}
function renderShellUser(){const u=S.me.user,box=$("#sbuser");if(box)box.replaceChildren(avEl(u),h("div",{style:"min-width:0"},h("b",null,u.username),h("small",null,"My profile")));const bs=$(".brand small");if(bs)bs.textContent=u.role+" · "+u.username}
UI.palette=()=>{if($(".kpal"))return;let sel=0,rows=[];const inp=h("input",{placeholder:"Jump to a page, open a record (QTE-0012, SHP-0001) or ask the Copilot…"}),lst=h("div",{class:"lst"}),ov=h("div",{class:"kpal",onclick:e=>{if(e.target===ov)close()}},h("div",null,inp,lst));
 const close=()=>{ov.remove();document.removeEventListener("keydown",kd,true)};
 const build=()=>{const q=inp.value.trim(),ql=q.toLowerCase();rows=[];S.me.menu.filter(m=>!ql||m.label.toLowerCase().includes(ql)||m.id.includes(ql)).forEach(m=>rows.push({g:"Pages",t:m.label,s:"go to",f:()=>UI.go(m.id)}));
  if(/^[A-Za-z]{2,5}-?[A-Za-z]?\d+$/.test(q))rows.unshift({g:"Open",t:q.toUpperCase(),s:"record",f:()=>UI.openEntity(q.toUpperCase())});
  if(q.length>2)rows.push({g:"Ask",t:"Ask the Copilot: "+q,s:"SQL answer",f:()=>{UI.go("copilot");setTimeout(()=>{const i=document.querySelector(".cin input");if(i){i.value=q;i.dispatchEvent(new KeyboardEvent("keydown",{key:"Enter"}))}},700)}});
  if(sel>=rows.length)sel=rows.length-1;let last="";lst.replaceChildren(...rows.flatMap((r,i)=>{const e=[];if(r.g!==last){last=r.g;e.push(h("div",{class:"gl"},r.g))}e.push(h("div",{class:"it"+(i===sel?" on":""),onclick:()=>{close();r.f()}},r.t,h("small",null,r.s)));return e}))};
 const kd=e=>{if(e.key==="Escape"){close()}else if(e.key==="ArrowDown"){sel=Math.min(rows.length-1,sel+1);build();e.preventDefault()}else if(e.key==="ArrowUp"){sel=Math.max(0,sel-1);build();e.preventDefault()}else if(e.key==="Enter"&&document.activeElement===inp){const r=rows[sel];if(r){close();r.f()}}};
 document.addEventListener("keydown",kd,true);inp.addEventListener("input",()=>{sel=0;build()});document.body.append(ov);build();inp.focus()};
document.addEventListener("keydown",e=>{if((e.ctrlKey||e.metaKey)&&e.key.toLowerCase()==="k"&&S.me){e.preventDefault();UI.palette()}});
function updBadges(){const p=S.pulse,hl=$("#hl");if(!p||!hl)return;if(p.health==null&&p.open==null){hl.innerHTML=`<span class="dot g live"></span>live`;return}
 hl.innerHTML=`<span class="dot g live"></span>live · health ${p.health==null?"–":Math.round(p.health)} · open ${p.open} · crit ${p.critical} · cycle ${p.cycles}`;if(p.worker_alive===false)hl.innerHTML+=' · <b style="color:var(--red)">worker down</b>'}
function stopStream(){if(S.es){S.es.abort();S.es=null}}
function forceSignOut(msg){stopStream();clearTimeout(S.timer);TOKEN.clear();S.me=null;S.pulse=null;S.authMsg=msg;render()}
function openStream(){stopStream();const ctl=new AbortController();S.es=ctl;
 (async()=>{while(!ctl.signal.aborted&&S.me){try{const r=await fetch("/api/stream",{headers:{Authorization:"Bearer "+TOKEN.get()},signal:ctl.signal});if(r.status===401){forceSignOut("Your session ended. Please sign in again.");return}if(!r.ok||!r.body)throw 0;
   const rd=r.body.getReader(),dec=new TextDecoder();let buf="";for(;;){const{done,value}=await rd.read();if(done)break;buf+=dec.decode(value,{stream:true});let i;while((i=buf.indexOf("\n\n"))>=0){const m=buf.slice(0,i);buf=buf.slice(i+2);if(!m.startsWith("data: "))continue;let p;try{p=JSON.parse(m.slice(6))}catch(e){continue}
     if(p.error==="signed_out"){forceSignOut("Your session ended: the account was locked, deleted or signed out.");return}if(p.error)continue;const old=S.pulse;S.pulse=p;updBadges();if(old&&p.dec!==old.dec&&S.live)loadPage(true)}}}catch(e){if(ctl.signal.aborted)return}await sleep(3000)}})()}
async function signOut(){stopStream();clearTimeout(S.timer);clearInterval(S.hb);await api("/api/auth/logout",{json:{}});TOKEN.clear();S.me=null;S.pulse=null;history.replaceState(null,"",location.pathname);render()}
function renderForceChange(){const app=$("#app"),u=S.me.user,msg=msgBox(),cur=passField("Temporary password","fc0","The password your administrator gave you"),np=passField("New password","fc1","Create a strong password"),cf=passField("Confirm new password","fc2","Repeat the new password"),btn=h("button",{class:"abtn",type:"submit"},"Save and continue");
 const go=async e=>{e.preventDefault();msg.hide();if(!cur.inp.value||!np.inp.value||!cf.inp.value)return msg.show("All fields are mandatory.");const bad=PW_RULES.find(r=>!r[1](np.inp.value));if(bad)return msg.show("Password rule missing: "+bad[0]+".");if(np.inp.value!==cf.inp.value)return msg.show("Password and Confirm Password do not match.");
  await busy(btn,"Saving…",async()=>{const r=await api("/api/profile/password",{json:{current:cur.inp.value,password:np.inp.value,confirm:cf.inp.value}});if(r.ok){if(r.token)TOKEN.set(r.token);S.me=null;toast("Password updated. Welcome!");render()}else msg.show(r.msg)})};
 app.replaceChildren(h("div",{class:"auth-shell"},h("div",{class:"auth-grid"},h("div",{class:"hero-l"},h("span",{class:"tag"},"ONE QUICK STEP"),h("h2",null,"Choose your own password"),h("p",null,"Your administrator gave you a temporary password. Set a new one that only you know. You will then reach your workspace."),h("div",{class:"feats"},feat("🔐","Stored as a bcrypt hash"),feat("🛡️","Same rules as sign-up"))),
  h("div",{class:"auth-card"},h("form",{onsubmit:go},h("h1",null,"Welcome, "+u.username),h("p",{class:"lead"},"Set a new password to continue."),msg,cur.el,np.el,pwMeter(np.inp),h("div",{style:"height:12px"}),cf.el,btn,h("div",{style:"margin-top:14px;text-align:center"},h("button",{type:"button",class:"alink",onclick:signOut},"Sign out")))))))}
async function render(){const app=$("#app");stopStream();THEME.apply();clearInterval(S.hb);
 let pub=await api("/api/public");if(pub.status&&pub.status>=500||pub.ok===false&&!pub.questions){app.replaceChildren(h("div",{class:"auth-shell"},h("div",{class:"auth-card",style:"margin-top:15vh;max-width:460px;text-align:center"},h("div",{class:"spin"}),h("h3",null,"The server is starting"),h("p",{class:"sub"},"This takes a few seconds. Retrying automatically…"))));await sleep(2500);return render()}
 S.pub=pub;if(TOKEN.get()&&!S.me){const m=await api("/api/me");if(m.user)S.me=m;else{TOKEN.clear();if(m.msg&&m.status===401&&!S.authMsg)S.authMsg=m.msg}}
 if(!S.me){renderAuth("login");return}
 if(S.me.user.must_change){renderForceChange();return}
 S.hb=setInterval(()=>{if(!document.hidden&&TOKEN.get()&&S.me)api("/api/ping")},60000);          // keeps the user "Active" on the admin dashboard
 const u=S.me.user;
 app.replaceChildren(h("div",{class:"side"},h("div",{class:"brand"},h("i",null,icon("radar")),h("div",null,S.me.name,h("small",null,u.role+" · "+u.username))),
   h("div",{class:"ksearch",onclick:()=>UI.palette()},"Search or ask…",h("kbd",null,"Ctrl K")),h("div",{id:"nav"}),h("div",{class:"sp"}),
   h("div",{class:"sbfoot"},h("div",{class:"sbuser",id:"sbuser",onclick:()=>UI.go("profile")}),h("div",{class:"sbtools"},h("button",{class:"btn sm",onclick:()=>THEME.toggle()},THEME.get()==="dark"?"☀ Light":"☾ Dark"),h("button",{class:"btn sm",onclick:signOut},"Sign out")))),
  h("div",{class:"main"},h("div",{class:"top"},h("div",null,h("h1",{id:"ttl"}),h("div",{class:"sub",id:"sub"})),h("div",{class:"sp"}),h("div",{id:"acts",style:"display:flex;gap:8px"}),h("span",{class:"chip",id:"hl"},"connecting…"),
    h("span",{class:"chip hide-m"},h("span",{class:"rolebadge "+u.role.split(" ")[0],style:"padding:0 8px"},u.role)),h("button",{class:"btn sm hamb",onclick:()=>THEME.toggle()},"◐"),h("button",{class:"btn sm hamb",onclick:signOut},"Sign out")),
   h("div",{class:"content"},h("div",{class:"tabs",id:"tabs"}),h("div",{id:"pg"},h("div",{class:"blocks",id:"blocks"})))));
 renderShellUser();renderNav();const hs=(location.hash||"").slice(1).split(":");const inMenu=S.me.menu.find(m=>m.id===hs[0]);
 S.page=inMenu?hs[0]:(PAGE_IDS.includes(hs[0])?hs[0]:S.me.menu[0].id);S.tab=hs[1]||null;S.params={};   // a known page outside the role is requested anyway: the server answers 403
 loadPage(false);openStream();api("/api/pulse").then(p=>{if(!p.error&&p.stage!==undefined||p.health!==undefined){S.pulse=p;updBadges()}})}
window.UI=UI;window.addEventListener("hashchange",()=>{if(S.me){const id=(location.hash||"").slice(1).split(":")[0];if(id&&id!==S.page&&PAGE_IDS.includes(id)){S.page=id;S.tab=null;S.params={};loadPage(false);renderNav()}}});render();


In [ ]:
%%writefile /content/app/static/index.html
<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><meta name="theme-color" content="#54668E"><title>FreightQuote AI</title>
<link rel="icon" href="data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 32 32'%3E%3Crect width='32' height='32' rx='8' fill='%2354668E'/%3E%3Cpath d='M8 20l5-6 4 4 7-9' stroke='%23fff' stroke-width='3' fill='none' stroke-linecap='round'/%3E%3C/svg%3E">
<link rel="preconnect" href="https://fonts.googleapis.com"><link rel="preconnect" href="https://fonts.gstatic.com" crossorigin><link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Poppins:wght@500;600;700;800&display=swap"><link rel="stylesheet" href="/static/vendor/leaflet.css"><link rel="stylesheet" href="/static/style.css?v=3"></head>
<body><div id="app"></div><div id="toast"></div><div id="drawer"><div class="dh"><b id="dt"></b><button class="x" onclick="UI.closeDrawer()">✕</button></div><div id="db"></div></div>
<script src="/static/vendor/echarts.min.js"></script><script src="/static/vendor/leaflet.js"></script><script src="/static/vendor/topojson-client.min.js"></script><script src="/static/app.js?v=3"></script></body></html>


In [ ]:
%%writefile /content/app/static/style.css
@font-face{font-family:"Inter";font-style:normal;font-weight:100 900;font-display:swap;src:url(/static/vendor/fonts/inter-latin-wght-normal.woff2) format("woff2");unicode-range:U+0000-00FF,U+0131,U+0152-0153,U+02BB-02BC,U+02C6,U+02DA,U+02DC,U+2000-206F,U+20AC,U+2122,U+2191,U+2193,U+2212,U+2215,U+FEFF,U+FFFD}
@font-face{font-family:"Inter";font-style:normal;font-weight:100 900;font-display:swap;src:url(/static/vendor/fonts/inter-latin-ext-wght-normal.woff2) format("woff2");unicode-range:U+0100-02BA,U+02BD-02C5,U+02C7-02CC,U+02CE-02D7,U+02DD-02FF,U+0304,U+0308,U+0329,U+1D00-1DBF,U+1E00-1E9F,U+1EF2-1EFF,U+2020,U+20A0-20AB,U+20AD-20C0,U+2113,U+2C60-2C7F,U+A720-A7FF}
:root{--bg:#f6f7fb;--card:#fff;--line:#e6e8f0;--line2:#d9dce8;--tx:#1c2024;--tx2:#454a58;--mut:#6b7080;--pri:#5b5bd6;--pri2:#4a4ac4;--pri-bg:#eef0ff;--soft:#f1f2f8;--soft2:#fafbfe;--glass:rgba(255,255,255,.85);--sea:#cfe5f7;--land:#f3efe6;--landline:#b9b2a0;--herobg:linear-gradient(120deg,#eef0ff,#e8f6f4);--toast:#1c2024;--teal:#12a594;--teal-bg:#e6f7f4;--blue:#3e63dd;--blue-bg:#edf2fe;--amber:#f5a524;--amber-bg:#fff6e0;--red:#e5484d;--red-bg:#feebec;--vio:#8e4ec6;--vio-bg:#f5eefb;--green:#30a46c;--r:14px;--sh:0 1px 2px rgba(20,24,40,.04),0 4px 14px rgba(20,24,40,.05);--head:"Inter",system-ui,sans-serif}
html[data-theme=dark]{--bg:#11162B;--bg2:#0D1122;--card:#1A2140;--line:#2B3558;--line2:#3A4670;--tx:#E8ECF8;--tx2:#C3CCE6;--mut:#93A0C6;--pri:#8b8bf0;--pri2:#b3b3ff;--pri-bg:#26264f;--soft:#222B4D;--soft2:#1E2646;--glass:rgba(26,33,64,.88);--sea:#15233f;--land:#27345a;--landline:#3a4a78;--herobg:linear-gradient(120deg,#222c52,#1d2a45);--toast:#33406b;--teal:#35c3b0;--teal-bg:#14352f;--blue:#8aa0ff;--blue-bg:#222d5c;--amber:#f0b24a;--amber-bg:#3b2f14;--red:#ff7a7f;--red-bg:#40202a;--vio:#c394ee;--vio-bg:#31244a;--green:#52cc93;--sh:0 1px 2px rgba(0,0,0,.3),0 8px 22px rgba(0,0,0,.28);color-scheme:dark}
*{box-sizing:border-box}html,body{margin:0;height:100%;background:var(--bg);color:var(--tx);font:14px/1.5 "Inter",ui-sans-serif,system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;font-feature-settings:"cv11","ss01";-webkit-font-smoothing:antialiased}
button,input,select,textarea{font:inherit;color:inherit}a{color:var(--pri);text-decoration:none}a:hover{text-decoration:underline}
#app{display:flex;height:100vh}
.side{width:248px;flex:none;background:var(--card);border-right:1px solid var(--line);display:flex;flex-direction:column;padding:14px 12px;overflow:auto}
.brand{display:flex;gap:11px;align-items:center;padding:4px 6px 14px;font-weight:700;font-size:15px;letter-spacing:-.01em}.brand i{width:34px;height:34px;border-radius:11px;background:linear-gradient(135deg,var(--pri),#8b8bf0);display:grid;place-items:center;color:#fff;box-shadow:0 4px 12px rgba(91,91,214,.35)}.brand i svg{width:18px;height:18px;stroke:#fff;fill:none;stroke-width:2}
.brand small{display:block;color:var(--mut);font-weight:500;font-size:11px;max-width:150px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.ksearch{display:flex;align-items:center;gap:8px;margin:0 2px 10px;padding:8px 10px;border:1px solid var(--line);border-radius:10px;color:var(--mut);cursor:pointer;font-size:12.5px;background:var(--bg)}.ksearch:hover{border-color:var(--line2)}.ksearch kbd{margin-left:auto;font:600 10.5px "Inter";border:1px solid var(--line2);border-radius:5px;padding:1px 5px;background:var(--card);color:var(--mut)}
.ngrp{font-size:10.5px;font-weight:700;letter-spacing:.07em;text-transform:uppercase;color:var(--mut);padding:12px 10px 4px}
.nav{display:flex;align-items:center;gap:10px;padding:8px 10px;border-radius:10px;color:var(--tx2);cursor:pointer;border:1px solid transparent;margin-bottom:1px;user-select:none;font-weight:500;transition:background .12s}
.nav:hover{background:var(--soft)}.nav.on{background:var(--pri-bg);color:var(--pri2);font-weight:600}
.nav svg{width:17px;height:17px;flex:none;stroke:currentColor;fill:none;stroke-width:1.8;stroke-linecap:round;stroke-linejoin:round;opacity:.85}.nav .bd{margin-left:auto;background:var(--red);color:#fff;border-radius:99px;font-size:10.5px;padding:1px 7px;font-weight:700}.nav .bd.a{background:var(--amber);color:#2b1c00}
.nav.cmd{margin-top:10px;background:linear-gradient(135deg,#5b5bd6,#7a6be0);color:#fff;font-weight:600;box-shadow:0 6px 16px rgba(91,91,214,.28)}.nav.cmd:hover{filter:brightness(1.05)}.nav.cmd.on{background:linear-gradient(135deg,#4a4ac4,#6a5bd8);color:#fff}
.main{flex:1;min-width:0;display:flex;flex-direction:column}
.top{display:flex;align-items:center;gap:12px;padding:12px 24px;border-bottom:1px solid var(--line);background:var(--glass);backdrop-filter:blur(10px);flex-wrap:wrap}
.top h1{font-size:19px;margin:0;letter-spacing:-.015em}.top .sub{color:var(--mut);font-size:12.5px;margin:0}.sp{flex:1}
.chip{display:inline-flex;gap:6px;align-items:center;padding:4px 11px;border-radius:99px;border:1px solid var(--line);background:var(--card);font-size:12px;white-space:nowrap;color:var(--tx2)}
.dot{width:8px;height:8px;border-radius:50%;background:#b4b8c8}.dot.g{background:var(--green);box-shadow:0 0 0 3px rgba(48,164,108,.18)}.dot.r{background:var(--red);box-shadow:0 0 0 3px rgba(229,72,77,.16)}.dot.a{background:var(--amber)}
.live{animation:pl 1.6s infinite}@keyframes pl{50%{opacity:.4}}
.content{flex:1;overflow:auto;padding:20px 24px 70px}
.tabs{display:flex;gap:2px;border-bottom:1px solid var(--line);margin-bottom:16px;flex-wrap:wrap}.tabs b{padding:9px 14px;cursor:pointer;color:var(--mut);border-bottom:2px solid transparent;font-weight:600;margin-bottom:-1px}.tabs b:hover{color:var(--tx)}.tabs b.on{color:var(--pri2);border-color:var(--pri)}
.blocks>*{margin-bottom:16px}
.card{background:var(--card);border:1px solid var(--line);border-radius:var(--r);padding:16px;box-shadow:var(--sh);min-width:0}.card>h3{margin:0 0 2px;font-size:14px;letter-spacing:-.005em}.card>.sub,.sub{color:var(--mut);font-size:12.5px;margin-bottom:8px}
.row{display:grid;gap:16px}.row>*{min-width:0}.col>*{margin-bottom:16px}.col>*:last-child{margin-bottom:0}
@media(max-width:1000px){.row{grid-template-columns:1fr!important}.side{width:64px}.nav span,.brand div,.side .foot,.ngrp,.ksearch{display:none}}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:14px}.kpi{background:var(--card);border:1px solid var(--line);border-radius:var(--r);padding:14px 16px;position:relative;overflow:hidden;box-shadow:var(--sh)}
.kpi:before{content:"";position:absolute;left:0;top:14px;bottom:14px;width:3px;border-radius:0 3px 3px 0;background:var(--teal)}.kpi.Critical:before,.kpi.High:before{background:var(--red)}.kpi.Warning:before,.kpi.Medium:before{background:var(--amber)}.kpi.Info:before{background:var(--blue)}
.kpi .l{color:var(--mut);font-size:11.5px;font-weight:600;text-transform:uppercase;letter-spacing:.05em}.kpi .v{font-size:26px;font-weight:700;margin-top:3px;letter-spacing:-.02em}.kpi .s{color:var(--mut);font-size:12px}
.pill{display:inline-block;padding:1px 9px;border-radius:99px;font-size:11.5px;font-weight:600;border:1px solid}
.tbl{overflow:auto;border:1px solid var(--line);border-radius:12px;background:var(--card)}.tbl table{width:100%;border-collapse:collapse;font-size:12.8px}.tbl th{position:sticky;top:0;background:var(--soft2);text-align:left;padding:9px 12px;color:var(--mut);font-weight:600;cursor:pointer;white-space:nowrap;text-transform:capitalize;z-index:1;border-bottom:1px solid var(--line)}
.tbl td{padding:8px 12px;border-top:1px solid var(--line);max-width:380px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}.tbl tr.cl{cursor:pointer}.tbl tr.cl:hover td{background:var(--pri-bg)}
.bar{display:flex;align-items:center;gap:8px;min-width:110px}.bar i{display:block;height:6px;border-radius:4px;flex:1;background:var(--soft);overflow:hidden}.bar i b{display:block;height:100%;border-radius:4px}
.srch{margin:0 0 8px;padding:7px 11px;border-radius:9px;border:1px solid var(--line2);background:var(--card);width:240px}
.callout{padding:11px 14px;border-radius:12px;border:1px solid;display:flex;gap:10px;color:var(--tx2)}.callout.info{border-color:var(--line2);background:var(--blue-bg)}.callout.warn,.callout.warning{border-color:var(--amber);background:var(--amber-bg)}.callout.error,.callout.critical,.callout.bad{border-color:var(--red);background:var(--red-bg)}.callout.good,.callout.ok,.callout.success{border-color:var(--teal);background:var(--teal-bg)}
.md{line-height:1.65}.md code,.md pre{background:var(--soft);border-radius:6px;padding:1px 5px;font-family:ui-monospace,"JetBrains Mono",monospace;font-size:12px}.md pre{padding:10px;overflow:auto;white-space:pre-wrap}.md h1,.md h2,.md h3{margin:.7em 0 .3em;letter-spacing:-.01em}.md ul{margin:.3em 0;padding-left:20px}.md table{border-collapse:collapse;margin:.5em 0}.md td,.md th{border:1px solid var(--line);padding:5px 10px}
.btn{border:1px solid var(--line2);background:var(--card);padding:7px 14px;border-radius:10px;cursor:pointer;font-weight:600;color:var(--tx2);transition:all .12s}.btn:hover{background:var(--soft);border-color:#c9cde0}
.btn.primary{background:var(--pri);color:#fff;border-color:var(--pri);box-shadow:0 2px 8px rgba(91,91,214,.3)}.btn.primary:hover{background:var(--pri2)}.btn.danger{background:var(--red-bg);border-color:var(--red);color:var(--red)}.btn.ok{background:var(--teal-bg);border-color:var(--teal);color:var(--teal)}.btn.ghost{background:transparent}.btn.sm{padding:3px 10px;font-size:12px}.btn:disabled{opacity:.5;cursor:wait}
form.f{display:grid;grid-template-columns:repeat(auto-fit,minmax(210px,1fr));gap:12px;align-items:end}form.f.inline{grid-template-columns:repeat(auto-fit,minmax(160px,max-content))}
.fld label{display:block;color:var(--mut);font-size:12px;font-weight:500;margin-bottom:4px}.fld input,.fld select,.fld textarea,.in{width:100%;padding:8px 11px;border-radius:10px;border:1px solid var(--line2);background:var(--card);transition:border .12s,box-shadow .12s}.fld input:focus,.fld select:focus,.fld textarea:focus,.in:focus,.srch:focus{outline:0;border-color:var(--pri);box-shadow:0 0 0 3px rgba(91,91,214,.15)}.fld textarea{min-height:80px;grid-column:1/-1}.fld.wide{grid-column:1/-1}.fld input[type=range]{padding:0;accent-color:var(--pri)}
.chk{display:flex;flex-wrap:wrap;gap:6px}.chk label{display:flex;gap:6px;align-items:center;margin:0;color:var(--tx2);padding:3px 10px;border:1px solid var(--line2);border-radius:99px;font-size:12px;background:var(--card)}
.res{margin-top:12px}
.tk{border:1px solid var(--line);border-left:4px solid var(--blue);border-radius:12px;padding:12px 14px;margin-bottom:10px;background:var(--card);box-shadow:var(--sh)}.tk.Critical{border-left-color:var(--red)}.tk.Warning{border-left-color:var(--amber)}.tk.proposed{background:var(--vio-bg)}
.tk .h{display:flex;gap:8px;align-items:center;flex-wrap:wrap}.tk .d{margin:6px 0;color:var(--tx2)}.tk .m{color:var(--mut);font-size:11.8px;display:flex;gap:10px;flex-wrap:wrap}.tk .a{display:flex;gap:6px;margin-top:8px;flex-wrap:wrap;align-items:center}.tk .cm{margin-top:8px;border-top:1px dashed var(--line2);padding-top:6px;font-size:12.5px}
.filters{display:flex;gap:6px;margin-bottom:12px;flex-wrap:wrap;align-items:center}.filters .chip{cursor:pointer}.filters .chip:hover{border-color:var(--pri)}.filters .chip.on{border-color:var(--pri);background:var(--pri-bg);color:var(--pri2);font-weight:600}
.hero{display:flex;gap:18px;align-items:center;background:var(--herobg);border:1px solid var(--line);border-radius:16px;padding:16px 20px}.hero .sc{font-size:36px;font-weight:800;min-width:84px;text-align:center;letter-spacing:-.03em}.hero h2{margin:0;font-size:16px}.hero p{margin:2px 0;color:var(--mut);font-size:12.5px}
.map{border-radius:12px;overflow:hidden;border:1px solid var(--line);background:var(--sea)}.leaflet-container{background:var(--sea);font:inherit}.lg{display:flex;gap:12px;font-size:12px;color:var(--mut);margin-top:8px;flex-wrap:wrap}.lg i{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:4px}
.leaflet-tooltip{background:var(--card);color:var(--tx);border:1px solid var(--line);border-radius:8px;box-shadow:var(--sh)}
#toast{position:fixed;right:18px;bottom:18px;display:flex;flex-direction:column;gap:8px;z-index:99}#toast div{background:var(--toast);color:#fff;padding:10px 16px;border-radius:12px;max-width:380px;box-shadow:0 10px 30px rgba(20,24,40,.25);animation:up .25s}#toast div.bad{background:#c62a2f}@keyframes up{from{transform:translateY(10px);opacity:0}}
#drawer{z-index:2000!important;position:fixed;right:0;top:0;bottom:0;width:min(720px,96vw);background:var(--card);border-left:1px solid var(--line);transform:translateX(105%);transition:.25s;z-index:50;display:flex;flex-direction:column;box-shadow:-20px 0 60px rgba(20,24,40,.12)}#drawer.on{transform:none}
#drawer .dh{display:flex;justify-content:space-between;padding:14px 18px;border-bottom:1px solid var(--line);font-size:16px}#db{padding:16px;overflow:auto;flex:1;background:var(--bg)}.x{background:none;border:0;color:var(--mut);cursor:pointer;font-size:16px}
.boot{text-align:center;padding:60px;color:var(--mut)}.spin{width:28px;height:28px;border:3px solid var(--line);border-top-color:var(--pri);border-radius:50%;animation:sp .8s linear infinite;margin:0 auto 12px}@keyframes sp{to{transform:rotate(360deg)}}
.chat{display:flex;flex-direction:column;gap:12px}.msgs{display:flex;flex-direction:column;gap:12px;min-height:280px}.m{max-width:88%;padding:11px 15px;border-radius:16px;border:1px solid var(--line)}.m.u{align-self:flex-end;background:var(--pri);color:#fff;border-color:var(--pri);border-bottom-right-radius:5px}.m.a{background:var(--card);box-shadow:var(--sh);border-bottom-left-radius:5px}.st{color:var(--mut);font-size:12px}.st:before{content:"› "}.cites{margin:4px 0;font-size:12px;color:var(--mut)}.cites summary{cursor:pointer;color:var(--pri2);font-weight:600}
.cin{display:flex;gap:8px}.cin input{flex:1;padding:11px 15px;border-radius:12px;border:1px solid var(--line2);background:var(--card)}.cin input:focus{outline:0;border-color:var(--pri);box-shadow:0 0 0 3px rgba(91,91,214,.15)}
.kpal{position:fixed;inset:0;background:rgba(20,24,40,.35);backdrop-filter:blur(2px);z-index:90;display:flex;justify-content:center;padding-top:12vh}.kpal>div{width:min(620px,94vw);background:var(--card);border-radius:16px;box-shadow:0 30px 80px rgba(20,24,40,.3);overflow:hidden;align-self:flex-start}.kpal input{width:100%;border:0;border-bottom:1px solid var(--line);padding:16px 18px;font-size:15px;outline:0}.kpal .it{display:flex;gap:10px;padding:10px 18px;cursor:pointer;align-items:center;color:var(--tx2)}.kpal .it.on{background:var(--pri-bg);color:var(--pri2)}.kpal .it small{margin-left:auto;color:var(--mut)}.kpal .lst{max-height:50vh;overflow:auto;padding:6px 0}.kpal .gl{font-size:10.5px;font-weight:700;letter-spacing:.07em;text-transform:uppercase;color:var(--mut);padding:8px 18px 2px}
::-webkit-scrollbar{width:9px;height:9px}::-webkit-scrollbar-thumb{background:var(--line2);border-radius:9px}
@media(max-width:760px){#app{flex-direction:column-reverse}.side{width:100%;flex-direction:row;padding:4px;overflow-x:auto;border-right:0;border-top:1px solid var(--line);position:sticky;bottom:0;z-index:20;gap:2px}.side .brand,.side .foot,.side .sp,.ngrp,.ksearch{display:none}#nav{display:flex;gap:2px}.nav{flex-direction:column;gap:2px;padding:6px 10px;font-size:10px;min-width:62px;text-align:center;margin:0}.nav.cmd{margin-top:0}.nav span{display:block!important;white-space:nowrap;max-width:70px;overflow:hidden;text-overflow:ellipsis}.nav .bd{position:absolute;margin:0}.nav{position:relative}.content{padding:12px 10px 20px}.top{padding:8px 10px}.top h1{font-size:15px}.kpis{grid-template-columns:repeat(2,1fr)}.hero{flex-direction:column;align-items:flex-start}#drawer{width:100vw}.tbl td{max-width:200px}.srch{width:100%}.chip{font-size:11px}}
.top>div:first-child{min-width:0;flex:1 1 280px}.top .sub{white-space:nowrap;overflow:hidden;text-overflow:ellipsis;max-width:100%}

.leaflet-container{isolation:isolate;overflow:hidden!important}.mapw,.map,.leafmap{isolation:isolate;position:relative;z-index:0}#drawer.on{z-index:2000}.kpal{z-index:3000!important}#toast{z-index:3500!important}

/* ===================================================== Milestone 2 additions ===================================================== */
html,body{background:var(--bg)}h1,h2,h3,.brand b{font-family:var(--head)}
.btn.primary{background:linear-gradient(135deg,var(--pri2),var(--pri));border-color:var(--pri2)}
.btn.primary:hover{filter:brightness(1.08)}
.nav.on{background:var(--pri-bg);color:var(--pri2)}
.side{background:var(--card)}
.tbl th{background:var(--soft2)}.tbl td{border-top:1px solid var(--line)}
.kpi .v{font-family:var(--head)}
.m.a{background:var(--card)}.cin input,.fld input,.fld select,.fld textarea,.in,.srch{background:var(--card);color:var(--tx)}
.kpal>div,#drawer{background:var(--card)}
.leaflet-tooltip{background:var(--card)}
a{color:var(--pri2)}
.map,.leaflet-container{background:var(--sea)!important}
.skel{border-radius:12px;background:linear-gradient(90deg,var(--soft) 25%,var(--soft2) 37%,var(--soft) 63%);background-size:400% 100%;animation:sk 1.4s ease infinite}@keyframes sk{0%{background-position:100% 50%}100%{background-position:0 50%}}
.errbox{text-align:center;padding:48px 20px;border:1px dashed var(--line2);border-radius:var(--r);background:var(--card)}.errbox .big{font:800 54px var(--head);color:var(--red);letter-spacing:-.03em}.errbox h3{margin:4px 0}.errbox p{color:var(--mut);max-width:460px;margin:6px auto 14px}
.errbox.e403 .big{color:var(--amber)}
.sbuser{margin:6px 2px 10px;padding:10px;border:1px solid var(--line);border-radius:12px;background:var(--soft2);display:flex;gap:10px;align-items:center}
.av{width:36px;height:36px;border-radius:50%;background:linear-gradient(135deg,var(--pri2),var(--pri));color:#fff;display:grid;place-items:center;font:700 14px var(--head);flex:none}
.sbuser b{display:block;font-size:13px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;max-width:130px}.sbuser small{display:block;color:var(--mut);font-size:11px}
.sbtools{display:flex;gap:6px;margin-top:8px}.sbtools .btn{flex:1;padding:7px 6px;font-size:12px}
.rolebadge{display:inline-block;padding:2px 10px;border-radius:99px;font-size:11px;font-weight:700;background:var(--pri-bg);color:var(--pri2);border:1px solid var(--line2)}
.rolebadge.Admin{background:#EADBF6;color:#5B2E86;border-color:#d4b8ea}.rolebadge.Auditor{background:var(--amber-bg);color:#8a5a00}.rolebadge.Customer{background:var(--teal-bg);color:#0a6b5e}.rolebadge.Freight{background:var(--blue-bg)}
html[data-theme=dark] .rolebadge.Admin{background:#3a2a57;color:#d7b8ff;border-color:#56407c}html[data-theme=dark] .rolebadge.Auditor{color:#f0c26b}html[data-theme=dark] .rolebadge.Customer{color:#74e0cf}
.theme-sw{display:inline-flex;align-items:center;gap:8px;cursor:pointer;color:var(--tx2);font-size:12px;user-select:none}.theme-sw i{width:34px;height:19px;border-radius:99px;background:var(--line2);position:relative;transition:.2s}.theme-sw i:after{content:"";position:absolute;top:2px;left:2px;width:15px;height:15px;border-radius:50%;background:#fff;transition:.2s;box-shadow:0 1px 3px rgba(0,0,0,.3)}html[data-theme=dark] .theme-sw i{background:var(--pri)}html[data-theme=dark] .theme-sw i:after{left:17px}
.hamb{display:none}

/* ---------- login (same theme as Milestone 1) ---------- */
.auth-shell{min-height:100vh;width:100%;position:relative;overflow:hidden;background:radial-gradient(1000px 520px at 85% -10%,rgba(84,102,142,.22),transparent 60%),radial-gradient(900px 520px at -10% 110%,rgba(198,166,134,.28),transparent 60%),linear-gradient(180deg,var(--bg),var(--bg2));display:flex;flex-direction:column;align-items:center;padding:28px 18px 40px}
.auth-shell:before{content:"";position:absolute;inset:0;background-image:linear-gradient(30deg,rgba(84,102,142,.05) 12%,transparent 12.5%,transparent 87%,rgba(84,102,142,.05) 87.5%),linear-gradient(150deg,rgba(84,102,142,.05) 12%,transparent 12.5%,transparent 87%,rgba(84,102,142,.05) 87.5%);background-size:80px 140px;pointer-events:none;opacity:.7}
.orb{position:absolute;border-radius:50%;filter:blur(2px);opacity:.55;animation:fl 9s ease-in-out infinite;pointer-events:none}.orb.o1{width:220px;height:220px;left:-60px;top:18%;background:radial-gradient(circle at 30% 30%,#9fb1d8,#54668E)}.orb.o2{width:150px;height:150px;right:6%;bottom:12%;background:radial-gradient(circle at 30% 30%,#f0d9bf,#c9a47c);animation-delay:-3s}.orb.o3{width:90px;height:90px;right:30%;top:8%;background:radial-gradient(circle at 30% 30%,#c7d3ee,#7f93bf);animation-delay:-5s}@keyframes fl{50%{transform:translateY(-22px) translateX(10px)}}
.auth-top{width:min(1120px,100%);display:flex;justify-content:space-between;align-items:center;position:relative;z-index:1;margin-bottom:18px}
.auth-top .brand{padding:0;font-size:17px}
.auth-grid{width:min(1120px,100%);display:grid;grid-template-columns:1.08fr 1fr;gap:28px;align-items:stretch;position:relative;z-index:1}
.hero-l{border-radius:24px;overflow:hidden;position:relative;background:linear-gradient(135deg,#34436A 0%,#54668E 55%,#7F93BF 100%);color:#fff;padding:34px;display:flex;flex-direction:column;gap:16px;box-shadow:0 20px 50px rgba(52,67,106,.35);min-height:560px}
.hero-l:after{content:"";position:absolute;right:-80px;top:-80px;width:280px;height:280px;border-radius:50%;background:rgba(255,255,255,.08)}
.hero-l .tag{align-self:flex-start;background:rgba(255,255,255,.16);border:1px solid rgba(255,255,255,.3);border-radius:99px;padding:5px 14px;font-size:12px;font-weight:600;letter-spacing:.04em}
.hero-l h2{font:800 34px/1.15 var(--head);margin:0;letter-spacing:-.02em}.hero-l p{margin:0;color:rgba(255,255,255,.88);max-width:470px;font-size:14.5px}
.hero-l .promo{flex:1;min-height:230px;position:relative;display:flex;justify-content:center;align-items:flex-end}.hero-l .promo:before{content:"";position:absolute;inset:8% 14% 0;border-radius:50% 50% 18px 18px;background:rgba(255,255,255,.13);border:1px solid rgba(255,255,255,.22)}.hero-l .promo img{position:relative;max-width:100%;max-height:330px;object-fit:contain;display:block;filter:drop-shadow(0 14px 22px rgba(0,0,0,.28))}
.feats{display:grid;grid-template-columns:1fr 1fr;gap:10px}.feat{background:rgba(255,255,255,.12);border:1px solid rgba(255,255,255,.2);border-radius:14px;padding:11px 13px;font-size:12.5px;display:flex;gap:10px;align-items:center}.feat i{width:30px;height:30px;border-radius:9px;background:rgba(255,255,255,.2);display:grid;place-items:center;flex:none;font-style:normal}
.auth-card{background:var(--card);border:1.5px solid var(--line);border-radius:24px;padding:30px;box-shadow:var(--sh),0 18px 46px rgba(84,102,142,.14);align-self:center;width:100%}
.auth-card h1{font:700 26px var(--head);margin:0 0 4px;letter-spacing:-.02em}.auth-card .lead{color:var(--mut);margin:0 0 18px}
.seg{display:grid;grid-template-columns:1fr 1fr;background:var(--soft);border:1px solid var(--line);border-radius:14px;padding:4px;margin-bottom:18px}.seg b{text-align:center;padding:9px;border-radius:10px;cursor:pointer;font-weight:600;color:var(--mut);transition:.15s}.seg b.on{background:var(--card);color:var(--pri2);box-shadow:0 2px 8px rgba(38,48,77,.12)}
.afld{margin-bottom:13px}.afld label{display:block;font-size:12.5px;font-weight:600;color:var(--tx2);margin-bottom:5px}.afld .iw{position:relative}.afld input,.afld select{width:100%;padding:11px 14px;border-radius:12px;border:1.5px solid var(--line);background:var(--bg);color:var(--tx);transition:.15s}.afld input:focus,.afld select:focus{outline:0;border-color:var(--pri);box-shadow:0 0 0 4px rgba(84,102,142,.16);background:var(--card)}.afld .eye{position:absolute;right:10px;top:50%;transform:translateY(-50%);border:0;background:none;color:var(--mut);cursor:pointer;font-size:12px;font-weight:600}
.afld .hint{font-size:11.5px;color:var(--mut);margin-top:4px}.afld.bad input{border-color:var(--red)}
.meter{display:flex;gap:4px;margin-top:6px}.meter i{flex:1;height:5px;border-radius:4px;background:var(--line)}.meter i.on1{background:var(--red)}.meter i.on2{background:var(--amber)}.meter i.on3{background:#b8c44c}.meter i.on4{background:var(--green)}
.rules{display:flex;flex-wrap:wrap;gap:6px;margin-top:7px}.rules span{font-size:11px;padding:2px 8px;border-radius:99px;border:1px solid var(--line);color:var(--mut)}.rules span.ok{background:var(--teal-bg);color:var(--teal);border-color:var(--teal)}
.abtn{width:100%;padding:12px;border:0;border-radius:13px;background:linear-gradient(135deg,#3F5078,#54668E 60%,#6b7fa8);color:#fff;font:700 15px var(--head);cursor:pointer;box-shadow:0 10px 24px rgba(63,80,120,.32);transition:.15s;margin-top:4px}.abtn:hover{transform:translateY(-1px);filter:brightness(1.06)}.abtn:disabled{opacity:.6;cursor:wait;transform:none}
.alink{background:none;border:0;color:var(--pri2);font-weight:600;cursor:pointer;padding:0}.alink:hover{text-decoration:underline}
.amsg{padding:10px 13px;border-radius:12px;font-size:13px;margin-bottom:12px;border:1px solid}.amsg.err{background:var(--red-bg);color:var(--red);border-color:var(--red)}.amsg.ok{background:var(--teal-bg);color:var(--teal);border-color:var(--teal)}.amsg.info{background:var(--blue-bg);color:var(--blue);border-color:var(--line2)}
.arow{display:flex;justify-content:space-between;align-items:center;margin:2px 0 14px;font-size:13px}
.demo-box{margin-top:16px;padding:12px;border:1px dashed var(--line2);border-radius:14px;background:var(--soft2);font-size:12px;color:var(--mut)}
.auth-foot{position:relative;z-index:1;margin-top:20px;color:var(--mut);font-size:12px;text-align:center}

/* ---------- profile ---------- */
.prof{display:grid;grid-template-columns:minmax(260px,340px) 1fr;gap:16px}.prof .card.id{text-align:center}
.prof .bigav{width:84px;height:84px;font-size:30px;margin:6px auto 10px}.prof h2{margin:0;font-size:20px}.prof .em{color:var(--mut);margin:2px 0 10px;word-break:break-all}
.irow{display:flex;justify-content:space-between;gap:12px;padding:9px 0;border-top:1px solid var(--line);font-size:13px;text-align:left}.irow span{color:var(--mut)}.irow b{font-weight:600;text-align:right}
.caps{display:flex;gap:6px;flex-wrap:wrap;margin-top:6px}.caps .chip{font-size:11.5px}
.pcols{display:grid;gap:16px}
@media(max-width:900px){.prof{grid-template-columns:1fr}}

/* ---------- admin ---------- */
.utbl{width:100%;border-collapse:collapse;font-size:12.8px}.utbl th{text-align:left;padding:9px 10px;color:var(--mut);font-weight:600;background:var(--soft2);border-bottom:1px solid var(--line);white-space:nowrap}.utbl td{padding:9px 10px;border-top:1px solid var(--line);vertical-align:middle}.utbl tr:hover td{background:var(--soft2)}
.utbl select{padding:5px 8px;border-radius:8px;border:1px solid var(--line2);background:var(--card);color:var(--tx)}
.st-pill{display:inline-flex;align-items:center;gap:6px;padding:2px 10px;border-radius:99px;font-size:11.5px;font-weight:700}.st-pill.Active{background:var(--teal-bg);color:var(--teal)}.st-pill.Locked{background:var(--red-bg);color:var(--red)}.st-pill:before{content:"";width:7px;height:7px;border-radius:50%;background:currentColor}
.acts{display:flex;gap:6px;flex-wrap:wrap}.me-tag{font-size:10.5px;color:var(--mut);border:1px solid var(--line2);border-radius:6px;padding:0 5px;margin-left:6px}
.modal{position:fixed;inset:0;background:rgba(17,22,43,.5);backdrop-filter:blur(3px);z-index:4000;display:grid;place-items:center;padding:16px}.modal>div{background:var(--card);border:1px solid var(--line);border-radius:18px;width:min(440px,100%);padding:22px;box-shadow:0 30px 80px rgba(0,0,0,.35)}.modal h3{margin:0 0 6px;font-size:18px}.modal p{color:var(--tx2);margin:6px 0 16px}.modal .mb{display:flex;justify-content:flex-end;gap:8px}
.logl{max-height:420px;overflow:auto;border:1px solid var(--line);border-radius:12px}.logl table{width:100%;border-collapse:collapse;font-size:12.3px}.logl td,.logl th{padding:7px 10px;border-top:1px solid var(--line);text-align:left;white-space:nowrap}.logl th{position:sticky;top:0;background:var(--soft2);color:var(--mut);border-top:0}.logl td.dt{white-space:normal;color:var(--tx2)}
.act-tag{font-size:11px;font-weight:700;padding:1px 8px;border-radius:6px;background:var(--pri-bg);color:var(--pri2)}.act-tag.bad{background:var(--red-bg);color:var(--red)}.act-tag.good{background:var(--teal-bg);color:var(--teal)}
.mx{width:100%;border-collapse:collapse;font-size:12.5px}.mx th,.mx td{padding:8px 10px;border-top:1px solid var(--line);text-align:center}.mx th:first-child,.mx td:first-child{text-align:left}.mx th{background:var(--soft2);color:var(--mut);border-top:0;font-weight:600}.mx .y{color:var(--teal);font-weight:800}.mx .n{color:var(--red);opacity:.7}
.two{display:grid;grid-template-columns:1fr 1fr;gap:16px}@media(max-width:1000px){.two{grid-template-columns:1fr}}
.addgrid{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:12px;align-items:end}

/* ---------- copilot v2 ---------- */
.chat{max-width:980px}.msgs{min-height:340px}.m.a .txt{line-height:1.6}.m.a .txt b{color:var(--pri2)}
.m.u{background:linear-gradient(135deg,var(--pri2),var(--pri));border-color:var(--pri2)}
.sqlbox{margin-top:10px;border:1px solid var(--line);border-radius:12px;background:var(--soft2);overflow:hidden}.sqlbox summary{cursor:pointer;padding:7px 12px;font-size:12px;font-weight:600;color:var(--pri2);display:flex;gap:8px;align-items:center}.sqlbox pre{margin:0;padding:10px 12px;font:12px/1.5 ui-monospace,Menlo,Consolas,monospace;white-space:pre-wrap;word-break:break-word;color:var(--tx2);border-top:1px solid var(--line);background:var(--card)}
.rowsb{margin-left:auto;font-weight:700;color:var(--mut);font-size:11.5px}
.mini{margin-top:10px;overflow:auto;border:1px solid var(--line);border-radius:12px;max-height:300px}.mini table{border-collapse:collapse;width:100%;font-size:12.3px}.mini th{position:sticky;top:0;background:var(--soft2);color:var(--mut);text-align:left;padding:7px 10px;font-weight:600;white-space:nowrap}.mini td{padding:6px 10px;border-top:1px solid var(--line);white-space:nowrap}
.note{font-size:11.8px;color:var(--mut);margin-top:6px;font-style:italic}
.sugs{display:flex;flex-wrap:wrap;gap:8px;margin-top:10px}.sugs .chip{cursor:pointer;background:var(--card);border-color:var(--line2);color:var(--pri2);font-weight:600}.sugs .chip:hover{background:var(--pri-bg);border-color:var(--pri)}
.typing{display:inline-flex;gap:4px;padding:4px 0}.typing i{width:7px;height:7px;border-radius:50%;background:var(--pri);animation:ty 1s infinite}.typing i:nth-child(2){animation-delay:.15s}.typing i:nth-child(3){animation-delay:.3s}@keyframes ty{50%{transform:translateY(-5px);opacity:.4}}
.chat-top{display:flex;gap:8px;flex-wrap:wrap;align-items:center;margin-bottom:6px}

/* ---------- responsive ---------- */
@media(max-width:1000px){.auth-grid{grid-template-columns:1fr}.hero-l{min-height:0}.hero-l .promo{min-height:140px;max-height:200px}}
@media(max-width:760px){.auth-card{padding:22px 18px}.hero-l{padding:22px}.hero-l h2{font-size:26px}.feats{grid-template-columns:1fr}.sbuser,.sbtools{display:none}.top .chip.hide-m{display:none}.hamb{display:inline-flex}.utbl{font-size:12px}}
.auth-top .brand small{max-width:none}

/* fix: tall auth pages (sign-up, reset) must scroll */
.auth-shell{height:100vh;min-height:0;overflow-x:hidden;overflow-y:auto;-webkit-overflow-scrolling:touch;padding-bottom:60px}
.auth-shell:before{position:fixed}
html:has(.auth-shell),body:has(.auth-shell){overflow:hidden}
@media(max-width:760px){.auth-shell{height:100dvh}}


/* the login / sign-up screens keep the Milestone 1 slate-blue + cream theme */
.auth-shell{--bg:#FBF4EC;--bg2:#F5E6D8;--card:#FFFDFA;--line:#E3D2C2;--line2:#D3BFAE;--tx:#26304D;--tx2:#4A5578;--mut:#667090;--pri:#54668E;--pri2:#3F5078;--pri-bg:#E4EAF5;--soft:#F3EADF;--soft2:#F8F0E6;--teal:#0f9d8c;--teal-bg:#E3F4F0;--blue:#4A63C9;--blue-bg:#E8EDFB;--amber:#E39A1B;--amber-bg:#FFF4DC;--red:#D9484D;--red-bg:#FDECEC;--green:#2f9e69;--head:"Poppins","Inter",system-ui,sans-serif;color:var(--tx)}
html[data-theme=dark] .auth-shell{--bg:#11162B;--bg2:#0D1122;--card:#1A2140;--line:#2B3558;--line2:#3A4670;--tx:#E8ECF8;--tx2:#C3CCE6;--mut:#93A0C6;--pri:#8FA3D6;--pri2:#B5C3EC;--pri-bg:#252F57;--soft:#222B4D;--soft2:#1E2646;--teal:#35c3b0;--teal-bg:#14352f;--blue:#8aa0ff;--blue-bg:#222d5c;--amber:#f0b24a;--amber-bg:#3b2f14;--red:#ff7a7f;--red-bg:#40202a;--green:#52cc93}
.auth-shell h1,.auth-shell h2,.auth-shell h3,.auth-shell .brand b{font-family:var(--head)}
.auth-shell .brand i{background:linear-gradient(135deg,#3F5078,#62759D)}
.auth-shell .btn.primary{background:linear-gradient(135deg,#3F5078,#54668E)}

/* ---------- role picker on the sign-in form ---------- */
.rolepick{display:grid;grid-template-columns:repeat(auto-fit,minmax(138px,1fr));gap:8px;margin-bottom:14px}
.rp{position:relative;border:1.5px solid var(--line);border-radius:14px;background:var(--bg);padding:10px 10px 9px 12px;cursor:pointer;transition:.15s;text-align:left;display:flex;gap:9px;align-items:flex-start}
.rp:hover{border-color:var(--pri);transform:translateY(-1px)}
.rp .ri{width:30px;height:30px;border-radius:9px;background:var(--pri-bg);display:grid;place-items:center;flex:none;font-size:15px}
.rp b{display:block;font:600 12.5px var(--head);color:var(--tx);line-height:1.2}.rp small{display:block;font-size:10.8px;color:var(--mut);margin-top:2px;line-height:1.25}
.rp.on{border-color:var(--pri2);background:var(--pri-bg);box-shadow:0 0 0 3px rgba(84,102,142,.18)}
.rp.on:after{content:"✓";position:absolute;top:6px;right:8px;width:16px;height:16px;border-radius:50%;background:var(--pri2);color:#fff;font-size:10px;display:grid;place-items:center}
.rplab{display:flex;justify-content:space-between;align-items:baseline;margin:2px 0 7px;font-size:12.5px;font-weight:600;color:var(--tx2)}.rplab small{font-weight:500;color:var(--mut)}
.pill-role{display:inline-flex;gap:6px;align-items:center;background:var(--pri-bg);color:var(--pri2);border-radius:99px;padding:3px 11px;font-weight:700;font-size:12px}

/* ---------- menu: items the role may not open stay visible but disabled ---------- */
.nav.off{opacity:.42;cursor:not-allowed;position:relative}.nav.off:hover{background:transparent}
.nav.off .lk{margin-left:auto;width:14px;height:14px;opacity:.9}
.nav.off span{text-decoration:none}
.sbfoot{margin-top:auto;padding-top:10px;border-top:1px solid var(--line)}
.sbuser{cursor:pointer;margin:0 0 8px}.sbuser:hover{border-color:var(--pri)}
.ngrp.ctl{margin-top:6px}

/* ---- Milestone 2 additions: avatars, voice input, presence, passwords ---- */
.av{overflow:hidden}.av img{width:100%;height:100%;object-fit:cover;display:block}.av.emo{font-size:18px;line-height:1}.av.sm{width:30px;height:30px;font-size:12px}.av.sm.emo{font-size:15px}.prof .bigav.emo{font-size:42px}
.avwrap{position:relative;width:fit-content;margin:0 auto}.avwrap .bigav{margin:6px auto 10px}.avedit{position:absolute;right:-4px;bottom:6px;width:30px;height:30px;border-radius:50%;border:2px solid var(--card);background:var(--pri);color:#fff;cursor:pointer;font-size:14px;display:grid;place-items:center;padding:0}.avedit:hover{background:var(--pri2)}
.avmodal>div{width:min(460px,100%)}.avprev{display:grid;place-items:center;margin:4px 0 14px}.avprev .av{margin:0}
.avgrid{display:grid;grid-template-columns:repeat(6,1fr);gap:10px;margin:6px 0 14px}.avopt{aspect-ratio:1;border-radius:50%;border:2px solid transparent;font-size:22px;cursor:pointer;display:grid;place-items:center;padding:0;color:#fff}.avopt:hover{transform:scale(1.06)}.avopt.on{border-color:var(--pri);box-shadow:0 0 0 3px var(--pri-bg)}
.avtools{display:flex;gap:8px;flex-wrap:wrap;margin-bottom:6px}
.cin .mic{width:46px;flex:none;border-radius:12px;font-size:18px;padding:0;display:grid;place-items:center}.cin .mic.on{background:#e5484d;border-color:#e5484d;color:#fff;animation:micpulse 1.2s infinite}
@keyframes micpulse{0%{box-shadow:0 0 0 0 rgba(229,72,77,.55)}70%{box-shadow:0 0 0 12px rgba(229,72,77,0)}100%{box-shadow:0 0 0 0 rgba(229,72,77,0)}}
.vbar{display:flex;gap:10px;align-items:center;flex-wrap:wrap;margin-top:8px;font-size:12.5px;color:var(--mut)}.vbar .vst{flex:1;min-width:160px}.vbar .vst.on{color:#e5484d;font-weight:600}.vbar .vst.err{color:var(--red)}.vbar select{padding:4px 8px;border-radius:8px;border:1px solid var(--line2);background:var(--card);color:var(--tx);font-size:12px}.vbar label{display:inline-flex;gap:5px;align-items:center;cursor:pointer}
.ucell{display:flex;gap:10px;align-items:center}.pres{display:inline-flex;align-items:center;gap:6px;padding:2px 10px;border-radius:99px;font-size:11.5px;font-weight:700}.pres i{width:8px;height:8px;border-radius:50%;background:currentColor;display:inline-block}.pres.on{background:rgba(48,164,108,.14);color:#30a46c}.pres.on i{animation:micpulse 1.6s infinite}.pres.off{background:var(--soft);color:var(--mut)}
.st-pill.Enabled{background:var(--teal-bg);color:var(--teal)}
.pwk{display:inline-block;padding:2px 9px;border-radius:99px;font-size:11.5px;font-weight:600;border:1px solid var(--line2);color:var(--tx2);white-space:nowrap}.pwk.temporary{background:rgba(245,165,36,.14);color:#ab6400;border-color:transparent}.pwk.permanent{background:var(--pri-bg);color:var(--pri2);border-color:transparent}
.fchips{display:flex;gap:6px;flex-wrap:wrap;margin:0 0 10px}.fchips .chip{cursor:pointer}.fchips .chip.on{border-color:var(--pri);background:var(--pri-bg);color:var(--pri2);font-weight:600}
.pwrow{display:flex;gap:6px}.pwrow .in{flex:1;min-width:0}.fhint{font-size:12px;color:var(--mut);margin:8px 0 0}
.modal .in,.modal select.in{width:100%;margin-bottom:10px}


## Step 3 · Front-end libraries (offline bundle)

In [ ]:
import base64, io, tarfile
B64 = (
'...'
)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(B64)), mode='r:gz').extractall('/content/app/static')
print('vendor files ready')

In [ ]:
import base64
_PROMO = (
    '...'
    'SuF0odVyIZZWsoCJGn3Uc/KQSoASFcy1xQAA'
)
open('/content/app/static/promo.webp', 'wb').write(base64.b64decode(_PROMO))
print('promo image written')

## Step 4 · Start the server and open the app
The first start seeds the database, creates the Admin and the demo accounts and runs the first agent cycle (about a minute).

In [ ]:
import os, subprocess, sys, time, urllib.request
os.chdir('/content/app')
REQUIRED = ['alts.py', 'analyst.py', 'auth.py', 'blocks.py', 'config.py', 'copilot_plus.py', 'db.py', 'domain_freight.py', 'entities.py', 'gpu.py', 'hardening.py', 'intent_router.py', 'live_engine.py', 'mailer.py', 'qa.py', 'rbac.py', 'scale_data.py', 'seanav.py', 'seed_data.py', 'server.py', 'static/app.js', 'static/index.html', 'static/style.css', 'views_core.py', 'views_freight.py', 'views_platform.py', 'weather.py', 'weather_context.py', 'worker.py']
missing = [f for f in REQUIRED + ['static/app.js', 'static/index.html', 'static/style.css', 'static/vendor/echarts.min.js'] if not os.path.exists('/content/app/' + f)]
if missing:
    raise SystemExit('These project files are missing: ' + ', '.join(missing) + '. The session was reset or the Project files cells did not run: use Runtime -> Run all from the first cell, then start the server again.')
os.environ['PYTHONPATH'] = '/content/app'
os.environ['APP_KIND'] = 'Freight'
os.environ.pop('RATE_LIMIT_OFF', None)          # login routes are rate limited
subprocess.run('fuser -k 8000/tcp', shell=True, capture_output=True)
log = open('/content/app/server.log', 'w')
proc = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'server:app', '--app-dir', '/content/app', '--host', '0.0.0.0', '--port', '8000'], stdout=log, stderr=log, cwd='/content/app', env=os.environ.copy())
for _ in range(120):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/api/health', timeout=2); print('FreightQuote AI is running'); break
    except Exception: time.sleep(2)
else:
    print(open('/content/app/server.log').read()[-3000:])

In [ ]:
# First-run accounts. Passwords you did not set as Colab Secrets were generated: note them, then CLEAR THIS OUTPUT before you submit.
import os
p = '/content/app_data/initial_credentials.txt'
if os.path.exists(p):
    print(f"{'Role':<20}{'Username':<16}{'Email':<32}Password")
    for line in open(p):
        r, u, e, pw = [x.strip() for x in line.split('|')]
        print(f"{r:<20}{u:<16}{e:<32}{pw}")
else:
    print('No generated credentials (accounts already existed, or you set ADMIN_PASSWORD / DEMO_PASSWORD).')

In [ ]:
from IPython.display import HTML, display
try:
    from google.colab import output
    url = output.eval_js("google.colab.kernel.proxyPort(8000, {'cache': false})")
    display(HTML(f'<a href="{url}" target="_blank" style="font-size:18px">Open the app</a><br><small>{url}</small>'))
except Exception as e:
    print('Colab proxy unavailable:', e)
    try:
        from google.colab import userdata
        from pyngrok import ngrok
        ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
        url = ngrok.connect(8000).public_url
        display(HTML(f'<a href="{url}" target="_blank" style="font-size:18px">Open the app</a><br><small>{url}</small>'))
    except Exception as e2:
        print('ngrok failed:', e2)